# Alpha Auditor — Executable Calculation & Verification Notebook

**Experiment Title:** PEP / AAPL Statistical Arbitrage Spread Convergence  
**Original Hypothesis:** `I think that pepsi and apple stock move together`  
**Audit Run ID:** `audit-7b3a5691f5` (Schema `2.0`)  
**Market Data As-Of Date:** `2025-12-31` | **Chronological Split Date:** `2021-12-30`  
**Spec SHA-256:** `65f4ba17b56baaca3b7a1c794197a462dfdb931cb0a5679c05e4b2f8393bdd3b`  
**Aligned Data Slice SHA-256:** `142c11239b793fbaa502c2cc80511826b6bbc73711d89ec2a7de646044438a23`

---
This notebook is self-contained and executes end-to-end offline without external network access or API keys. Every signal, trade fill, fee deduction, high-water mark drawdown, econometric test, adversarial stress test, and scorecard verdict is recomputed from the aligned daily OHLCV dataset and verified against the AuditReport.

In [1]:
# Cell 1: Environment Setup, Library Versions & Deterministic Seeds
import hashlib
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels
import statsmodels.api as sm

REPO_ROOT = Path('/Users/simbashakib/Desktop/Alpha Audit')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from engine.schemas import StrategySpec
from engine.backtester import (
    compute_causal_signals,
    compute_pair_leg_weights,
    compute_parameter_sensitivity_surface,
    compute_standard_bracket_profiles,
    simulate_strategy_core,
    _compute_period_metrics,
)
from engine.econometrics import (
    TRADING_DAYS_PER_YEAR,
    compute_pairs_econometrics,
    deflated_sharpe_ratio_bailey_lopez,
    exact_sign_test_p_value,
    lo_mertens_sharpe_se_and_psr,
    newey_west_factor_regression,
    stationary_bootstrap_cis,
)
from engine.stress_tests import (
    build_hypothesis_scorecard_and_ledger,
    run_adversarial_stress_suite,
)

BOOTSTRAP_SEED = 20261107
MONTE_CARLO_SEED = 42
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | pandas: {pd.__version__} | SciPy: {scipy.__version__} | statsmodels: {statsmodels.__version__}')
print(f'Audit Run ID: audit-7b3a5691f5 | Spec SHA-256: 65f4ba17b56baaca...')

Python: 3.14.6 | NumPy: 2.4.6 | pandas: 3.0.3 | SciPy: 1.18.0 | statsmodels: 0.14.6
Audit Run ID: audit-7b3a5691f5 | Spec SHA-256: 65f4ba17b56baaca...


## Cell 2 — Confirmed `StrategySpec` & Explicit vs. Software-Default Assumptions
Separates parameters explicitly stated in the user's natural-language prompt from literature defaults filled by the compiler.

In [2]:
# Cell 2: Reconstruct Confirmed StrategySpec & Inspect Assumptions
SPEC_JSON = '{\n  "hypothesis_text": "I think that pepsi and apple stock move together",\n  "title": "PEP / AAPL Statistical Arbitrage Spread Convergence",\n  "strategy_family": "pairs_trading",\n  "template_id": "pairs_cointegration_reversion",\n  "signal_variant": "ols_log",\n  "asset_a": "PEP",\n  "asset_b": "AAPL",\n  "benchmark_ticker": "SPY",\n  "start_date": "2018-01-01",\n  "end_date": "2025-12-31",\n  "threshold_mode": "auto_standard_bracket",\n  "parameters": {\n    "lookback_window": 20,\n    "secondary_window": 60,\n    "streak_length": 3,\n    "entry_threshold": 2.0,\n    "exit_threshold": 0.25,\n    "stop_z_threshold": 4.0,\n    "holding_period": 20,\n    "exit_mode": "threshold_reversion",\n    "direction": "both"\n  },\n  "execution": {\n    "initial_capital": 100000.0,\n    "execution_timing": "next_open",\n    "commission_bps": 5.0,\n    "slippage_bps": 5.0,\n    "train_split_ratio": 0.5\n  },\n  "risk_controls": {\n    "max_drawdown_circuit_breaker_pct": 25.0,\n    "per_trade_stop_loss_pct": 12.0,\n    "max_holding_days": 40\n  },\n  "selected_diagnostics": [\n    "stationary_bootstrap",\n    "newey_west_alpha",\n    "subperiod_regimes",\n    "cost_sensitivity",\n    "parameter_sensitivity_grid"\n  ],\n  "human_confirmed": true,\n  "user_specified_fields": [],\n  "software_inferred_defaults": {},\n  "evaluation_stage": "confirmatory",\n  "prior_holdout_touches": 0\n}'
EXPECTED_SNAPSHOT = json.loads('{\n  "run_id": "audit-7b3a5691f5",\n  "schema_version": "2.0",\n  "spec_sha256": "65f4ba17b56baaca3b7a1c794197a462dfdb931cb0a5679c05e4b2f8393bdd3b",\n  "split_date": "2021-12-30",\n  "market_as_of_date": "2025-12-31",\n  "aligned_slice_sha256": "142c11239b793fbaa502c2cc80511826b6bbc73711d89ec2a7de646044438a23",\n  "trade_count": 12,\n  "full_sample": {\n    "period_label": "Full Sample",\n    "start_date": "2018-01-02",\n    "end_date": "2025-12-31",\n    "trading_days": 2011,\n    "trade_count": 12,\n    "exposure_pct": 8.65,\n    "total_gross_return_pct": -22.5,\n    "total_net_return_pct": -24.38,\n    "cagr_net_pct": -3.44,\n    "benchmark_return_pct": 187.48,\n    "benchmark_cagr_pct": 14.15,\n    "excess_return_pct": -211.86,\n    "annualized_volatility_pct": 4.36,\n    "benchmark_volatility_pct": 19.45,\n    "max_drawdown_pct": 25.36,\n    "benchmark_max_drawdown_pct": 33.72,\n    "sharpe_ratio": -0.782,\n    "benchmark_sharpe_ratio": 0.778,\n    "sharpe_se_lo2002": 0.334,\n    "probabilistic_sharpe_psr_pct": 0.96,\n    "sortino_ratio": -0.953,\n    "calmar_ratio": -0.136,\n    "win_rate_pct": 25.0,\n    "profit_factor": 0.185,\n    "mean_trade_net_pct": -2.255,\n    "median_trade_net_pct": -3.726,\n    "std_trade_net_pct": 3.168,\n    "best_trade_net_pct": 2.872,\n    "worst_trade_net_pct": -7.799,\n    "skewness_daily": -2.822,\n    "kurtosis_daily": 45.949,\n    "t_stat_mean_return": -2.208,\n    "p_value_two_sided": 0.0273,\n    "sign_test_p_value": 0.146,\n    "circuit_breaker_triggered": true,\n    "circuit_breaker_date": "2019-04-15",\n    "is_zero_exposure": false,\n    "halted_prior_to_period": false,\n    "status_note": null\n  },\n  "in_sample": {\n    "period_label": "In-Sample Formation",\n    "start_date": "2018-01-02",\n    "end_date": "2021-12-29",\n    "trading_days": 1006,\n    "trade_count": 12,\n    "exposure_pct": 17.3,\n    "total_gross_return_pct": -22.5,\n    "total_net_return_pct": -24.38,\n    "cagr_net_pct": -6.76,\n    "benchmark_return_pct": 90.41,\n    "benchmark_cagr_pct": 17.51,\n    "excess_return_pct": -114.79,\n    "annualized_volatility_pct": 6.15,\n    "benchmark_volatility_pct": 20.82,\n    "max_drawdown_pct": 25.36,\n    "benchmark_max_drawdown_pct": 33.72,\n    "sharpe_ratio": -1.106,\n    "benchmark_sharpe_ratio": 0.88,\n    "sharpe_se_lo2002": 0.473,\n    "probabilistic_sharpe_psr_pct": 0.97,\n    "sortino_ratio": -1.348,\n    "calmar_ratio": -0.267,\n    "win_rate_pct": 25.0,\n    "profit_factor": 0.185,\n    "mean_trade_net_pct": -2.255,\n    "median_trade_net_pct": -3.726,\n    "std_trade_net_pct": 3.168,\n    "best_trade_net_pct": 2.872,\n    "worst_trade_net_pct": -7.799,\n    "skewness_daily": -1.9,\n    "kurtosis_daily": 22.875,\n    "t_stat_mean_return": -2.211,\n    "p_value_two_sided": 0.0273,\n    "sign_test_p_value": 0.146,\n    "circuit_breaker_triggered": true,\n    "circuit_breaker_date": "2019-04-15",\n    "is_zero_exposure": false,\n    "halted_prior_to_period": false,\n    "status_note": null\n  },\n  "out_of_sample": {\n    "period_label": "Out-of-Sample (Held-Out)",\n    "start_date": "2021-12-30",\n    "end_date": "2025-12-31",\n    "trading_days": 1005,\n    "trade_count": 0,\n    "exposure_pct": 0.0,\n    "total_gross_return_pct": 0.0,\n    "total_net_return_pct": 0.0,\n    "cagr_net_pct": 0.0,\n    "benchmark_return_pct": 50.98,\n    "benchmark_cagr_pct": 10.88,\n    "excess_return_pct": -50.98,\n    "annualized_volatility_pct": 0.0,\n    "benchmark_volatility_pct": 17.99,\n    "max_drawdown_pct": 0.0,\n    "benchmark_max_drawdown_pct": 24.5,\n    "sharpe_ratio": 0.0,\n    "benchmark_sharpe_ratio": 0.664,\n    "sharpe_se_lo2002": 0.0,\n    "probabilistic_sharpe_psr_pct": 0.0,\n    "sortino_ratio": 0.0,\n    "calmar_ratio": 0.0,\n    "win_rate_pct": 0.0,\n    "profit_factor": null,\n    "mean_trade_net_pct": 0.0,\n    "median_trade_net_pct": 0.0,\n    "std_trade_net_pct": 0.0,\n    "best_trade_net_pct": 0.0,\n    "worst_trade_net_pct": 0.0,\n    "skewness_daily": 0.0,\n    "kurtosis_daily": 0.0,\n    "t_stat_mean_return": 0.0,\n    "p_value_two_sided": 1.0,\n    "sign_test_p_value": null,\n    "circuit_breaker_triggered": false,\n    "circuit_breaker_date": "2019-04-15",\n    "is_zero_exposure": true,\n    "halted_prior_to_period": true,\n    "status_note": "Zero exposure in Out-of-Sample (Held-Out): portfolio drawdown circuit breaker already halted trading on 2019-04-15 during the In-Sample Formation split. Return-to-volatility ratios (Sharpe, Sortino, PSR) are undefined."\n  },\n  "factor_regression": {\n    "annualized_alpha_pct": -3.59,\n    "alpha_t_stat_hac": -2.344,\n    "alpha_p_value_hac": 0.0191,\n    "beta": 0.012,\n    "beta_t_stat_hac": 1.516,\n    "r_squared": 0.0029,\n    "information_ratio": -0.941,\n    "newey_west_lags": 7\n  },\n  "bootstrap_cis": [\n    {\n      "metric_name": "Annualized Return (Mean)",\n      "point_estimate": -3.41,\n      "ci_lower_95": -6.82,\n      "ci_upper_95": -0.44,\n      "bootstrap_std_error": 1.62,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "%"\n    },\n    {\n      "metric_name": "Annualized Sharpe Ratio",\n      "point_estimate": -0.782,\n      "ci_lower_95": -1.366,\n      "ci_upper_95": -0.112,\n      "bootstrap_std_error": 0.312,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "ratio"\n    },\n    {\n      "metric_name": "Annualized Excess Return vs. Benchmark",\n      "point_estimate": -18.54,\n      "ci_lower_95": -30.09,\n      "ci_upper_95": -6.96,\n      "bootstrap_std_error": 6.01,\n      "block_length": 13,\n      "replications": 1000,\n      "unit": "%"\n    },\n    {\n      "metric_name": "Mean Net Return per Trade",\n      "point_estimate": -2.255,\n      "ci_lower_95": -3.862,\n      "ci_upper_95": -0.567,\n      "bootstrap_std_error": 0.875,\n      "block_length": 1,\n      "replications": 1000,\n      "unit": "%"\n    }\n  ],\n  "stress_test_outcomes": [\n    {\n      "test_id": "outlier_removal",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "-24.38% (N=12)",\n      "stressed_value": "-28.84% (N=9)"\n    },\n    {\n      "test_id": "execution_latency_shock",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "-24.38% (SR -0.78)",\n      "stressed_value": "-25.37% (SR -0.88)"\n    },\n    {\n      "test_id": "regime_vulnerability",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "0 / 5 Regimes",\n      "stressed_value": "-0.39 vs -1.38"\n    },\n    {\n      "test_id": "monte_carlo_random_entry",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "-24.38% (6th pct)",\n      "stressed_value": "-5.78% (p=0.944\\u00b10.010)"\n    },\n    {\n      "test_id": "pairs_structural_break",\n      "outcome_badge": "HIGH_VULNERABILITY",\n      "baseline_value": "\\u03b2=0.290 (p=0.295)",\n      "stressed_value": "\\u0394\\u03b2=138.5% (p=0.721)"\n    }\n  ],\n  "pairs_diagnostics": {\n    "formation_start": "2018-01-02",\n    "formation_end": "2021-12-29",\n    "ols_alpha": 3.4155,\n    "ols_hedge_ratio_beta": 0.2899,\n    "ols_r_squared": 0.7838,\n    "return_correlation": 0.5079,\n    "level_correlation": 0.8924,\n    "adf_statistic": -1.981,\n    "adf_p_value": 0.2948,\n    "adf_critical_values": {\n      "1%": -3.437,\n      "5%": -2.864,\n      "10%": -2.568\n    },\n    "engle_granger_stat": -1.984,\n    "engle_granger_p_value": 0.5368,\n    "ou_lambda": -0.01364,\n    "half_life_days": 50.5,\n    "hurst_exponent": 0.492,\n    "variance_ratio_q5": 0.7329,\n    "variance_ratio_p_value": 0.1879,\n    "cointegration_status_note": "Formation spread does NOT reject the unit-root null at standard significance levels (Engle-Granger p = 0.5368, return correlation = 0.51). High return correlation does not establish cointegration; out-of-sample divergence risk is elevated."\n  },\n  "verdict_label": "CONTRADICTED BY HISTORICAL DATA",\n  "supported_checks_count": 0,\n  "evaluated_checks_count": 4,\n  "dimension_statuses": {\n    "post_signal_direction": "CONTRADICTED",\n    "statistical_significance": "CONTRADICTED",\n    "out_of_sample_persistence": "NO_EVIDENCE",\n    "transaction_cost_survival": "CONTRADICTED",\n    "stress_test_resilience": "CONTRADICTED"\n  }\n}')
spec = StrategySpec.model_validate_json(SPEC_JSON)
computed_spec_sha256 = hashlib.sha256(spec.model_dump_json().encode('utf-8')).hexdigest()
assert computed_spec_sha256 == EXPECTED_SNAPSHOT['spec_sha256'], 'Spec SHA-256 mismatch!'

assumptions_rows = [
    ('Strategy Family / Template', f'{spec.strategy_family} / {spec.template_id} ({spec.signal_variant})', 'Core Mapping'),
    ('Target Asset(s)', f'{spec.asset_a}' + (f' / {spec.asset_b}' if spec.asset_b else ''), 'User Stated' if 'asset_a' in spec.user_specified_fields else 'Default'),
    ('Benchmark Ticker', spec.benchmark_ticker, 'Default'),
    ('Primary Lookback Window', f'{spec.parameters.lookback_window} trading days', 'User Stated' if 'parameters.lookback_window' in spec.user_specified_fields else 'Software Default'),
    ('Entry Threshold', f'{spec.parameters.entry_threshold}', 'User Stated' if 'parameters.entry_threshold' in spec.user_specified_fields else 'Software Default'),
    ('Exit Threshold / Mode', f'{spec.parameters.exit_threshold} ({spec.parameters.exit_mode})', 'Software Default'),
    ('Holding Period / Max Hold', f'{spec.parameters.holding_period}d (Max {spec.risk_controls.max_holding_days}d)', 'User Stated' if 'parameters.holding_period' in spec.user_specified_fields else 'Software Default'),
    ('Execution Timing & Costs', f'{spec.execution.execution_timing}, {spec.execution.commission_bps} bps comm + {spec.execution.slippage_bps} bps slip', 'Software Default'),
    ('Risk Controls', f'Circuit Breaker: {spec.risk_controls.max_drawdown_circuit_breaker_pct}%, Stop-Loss: {spec.risk_controls.per_trade_stop_loss_pct}%', 'Software Default'),
]
assumptions_df = pd.DataFrame(assumptions_rows, columns=['Parameter', 'Configured Value', 'Provenance'])
display(assumptions_df)

,Parameter,Configured Value,Provenance
0,Strategy Family / Template,pairs_trading / pairs_cointegration_reversion ...,Core Mapping
1,Target Asset(s),PEP / AAPL,Default
2,Benchmark Ticker,SPY,Default
3,Primary Lookback Window,20 trading days,Software Default
4,Entry Threshold,2.0,Software Default
5,Exit Threshold / Mode,0.25 (threshold_reversion),Software Default
6,Holding Period / Max Hold,20d (Max 40d),Software Default
7,Execution Timing & Costs,"next_open, 5.0 bps comm + 5.0 bps slip",Software Default
8,Risk Controls,"Circuit Breaker: 25.0%, Stop-Loss: 12.0%",Software Default


## Cell 3 — Immutable Aligned Market Data Snapshot & Integrity Verification
Loads the exact inner-joined daily OHLCV market slice embedded at audit time and verifies its cryptographic SHA-256 digest, date monotonicity, and chronological In-Sample / Out-of-Sample split boundary.

In [3]:
# Cell 3: Load Embedded Aligned Market Data & Verify Integrity
DF_RECORDS = json.loads('[{"date":"2018-01-02","open_a":91.111547,"high_a":91.301474,"low_a":89.592135,"close_a":89.690895,"volume_a":4165800,"open_b":39.741914,"high_b":40.241724,"low_b":39.531711,"close_b":40.23238,"volume_b":102223600,"open_bench":234.55539,"close_bench":235.369812},{"date":"2018-01-03","open_a":89.576947,"high_a":90.00238,"low_a":89.371823,"close_a":89.455391,"volume_a":4224000,"open_b":40.29544,"high_b":40.767224,"low_b":40.162315,"close_b":40.225372,"volume_b":118071600,"open_bench":235.536252,"close_bench":236.858612},{"date":"2018-01-04","open_a":89.493391,"high_a":90.488604,"low_a":89.341446,"close_a":89.896034,"volume_a":3147000,"open_b":40.297769,"high_b":40.514978,"low_b":40.190335,"close_b":40.412212,"volume_b":89738400,"open_bench":237.497832,"close_bench":237.856857},{"date":"2018-01-05","open_a":89.956831,"high_a":90.473426,"low_a":89.835275,"close_a":90.15435,"volume_a":2714200,"open_b":40.507983,"high_b":40.958744,"low_b":40.416896,"close_b":40.87233,"volume_b":94640000,"open_bench":238.645069,"close_bench":239.441986},{"date":"2018-01-08","open_a":89.80487,"high_a":90.207513,"low_a":89.394628,"close_a":89.637733,"volume_a":5730400,"open_b":40.720509,"high_b":41.014788,"low_b":40.622412,"close_b":40.720509,"volume_b":82271200,"open_bench":239.345582,"close_bench":239.879791},{"date":"2018-01-09","open_a":89.842863,"high_a":90.389853,"low_a":89.60735,"close_a":89.706116,"volume_a":4573600,"open_b":40.767233,"high_b":40.886346,"low_b":40.50098,"close_b":40.715851,"volume_b":86336000,"open_bench":240.300206,"close_bench":240.422821},{"date":"2018-01-10","open_a":89.744094,"high_a":89.918829,"low_a":88.657714,"close_a":89.25029,"volume_a":3363900,"open_b":40.44257,"high_b":40.708824,"low_b":40.4052,"close_b":40.706486,"volume_b":95839600,"open_bench":239.669639,"close_bench":240.054962},{"date":"2018-01-11","open_a":89.371842,"high_a":89.53138,"low_a":88.414611,"close_a":88.794464,"volume_a":3685200,"open_b":40.776563,"high_b":40.986766,"low_b":40.75321,"close_b":40.937717,"volume_b":74670800,"open_bench":240.606667,"close_bench":241.806412},{"date":"2018-01-12","open_a":88.847635,"high_a":89.652927,"low_a":88.771665,"close_a":89.174309,"volume_a":4335500,"open_b":41.147918,"high_b":41.423516,"low_b":41.024133,"close_b":41.360455,"volume_b":101672400,"open_bench":242.069233,"close_bench":243.382828},{"date":"2018-01-16","open_a":89.03757,"high_a":89.911233,"low_a":88.908421,"close_a":89.538979,"volume_a":4936500,"open_b":41.549621,"high_b":41.897621,"low_b":41.138563,"close_b":41.150242,"volume_b":118263600,"open_bench":244.635068,"close_bench":242.550827},{"date":"2018-01-17","open_a":89.865656,"high_a":91.293901,"low_a":89.804878,"close_a":90.541794,"volume_a":4554600,"open_b":41.140909,"high_b":41.864935,"low_b":40.888672,"close_b":41.829903,"volume_b":137547200,"open_bench":243.479076,"close_bench":244.862717},{"date":"2018-01-18","open_a":90.473422,"high_a":90.876071,"low_a":90.139155,"close_a":90.443039,"volume_a":6221800,"open_b":41.892951,"high_b":42.06345,"low_b":41.63137,"close_b":41.86726,"volume_b":124773600,"open_bench":244.748901,"close_bench":244.451157},{"date":"2018-01-19","open_a":90.769699,"high_a":91.681344,"low_a":90.131542,"close_a":91.293892,"volume_a":6002600,"open_b":41.715463,"high_b":41.942013,"low_b":41.435197,"close_b":41.680431,"volume_b":129700400,"open_bench":245.029115,"close_bench":245.563324},{"date":"2018-01-22","open_a":91.240716,"high_a":92.570202,"low_a":91.164747,"close_a":92.311905,"volume_a":4308600,"open_b":41.409503,"high_b":41.521609,"low_b":41.246015,"close_b":41.339436,"volume_b":108434400,"open_bench":245.353093,"close_bench":247.559921},{"date":"2018-01-23","open_a":91.977624,"high_a":93.071603,"low_a":91.810488,"close_a":92.152359,"volume_a":3536900,"open_b":41.409497,"high_b":41.909307,"low_b":41.297391,"close_b":41.34877,"volume_b":130756400,"open_bench":247.603797,"close_bench":248.085464},{"date":"2018-01-24","open_a":92.106762,"high_a":92.289095,"low_a":91.787687,"close_a":92.030792,"volume_a":3445200,"open_b":41.397814,"high_b":41.409493,"low_b":40.451912,"close_b":40.69014,"volume_b":204420400,"open_bench":248.72473,"close_bench":247.98912},{"date":"2018-01-25","open_a":91.62818,"high_a":92.26633,"low_a":91.438253,"close_a":91.848495,"volume_a":2712900,"open_b":40.757873,"high_b":40.860638,"low_b":39.82832,"close_b":39.963783,"volume_b":166116000,"open_bench":248.84738,"close_bench":248.094238},{"date":"2018-01-26","open_a":91.628183,"high_a":92.562618,"low_a":91.628183,"close_a":92.501846,"volume_a":3993900,"open_b":40.171664,"high_b":40.171664,"low_b":39.718565,"close_b":40.05722,"volume_b":156572000,"open_bench":248.92613,"close_bench":250.966568},{"date":"2018-01-29","open_a":92.213129,"high_a":92.805704,"low_a":90.89884,"close_a":90.914032,"volume_a":3854700,"open_b":39.741903,"high_b":39.741903,"low_b":39.020216,"close_b":39.228081,"volume_b":202561600,"open_bench":250.397351,"close_bench":249.302689},{"date":"2018-01-30","open_a":90.921647,"high_a":92.076398,"low_a":90.784899,"close_a":91.081184,"volume_a":4945200,"open_b":38.660539,"high_b":39.090282,"low_b":38.466687,"close_b":38.996861,"volume_b":184192800,"open_bench":247.481215,"close_bench":246.745605},{"date":"2018-01-31","open_a":91.043192,"high_a":91.734528,"low_a":90.792493,"close_a":91.392662,"volume_a":4018100,"open_b":38.973518,"high_b":39.340203,"low_b":38.887104,"close_b":39.104309,"volume_b":129915600,"open_bench":247.595035,"close_bench":246.868164},{"date":"2018-02-01","open_a":91.10397,"high_a":91.932053,"low_a":90.617758,"close_a":91.392662,"volume_a":3485600,"open_b":39.043577,"high_b":39.382233,"low_b":38.947818,"close_b":39.186047,"volume_b":188923200,"open_bench":246.141256,"close_bench":246.58786},{"date":"2018-02-02","open_a":91.09639,"high_a":91.377478,"low_a":89.949232,"close_a":90.15435,"volume_a":3348900,"open_b":38.770326,"high_b":38.957172,"low_b":37.392346,"close_b":37.485767,"volume_b":346375200,"open_bench":245.274363,"close_bench":241.219757},{"date":"2018-02-05","open_a":90.078372,"high_a":90.906454,"low_a":86.917997,"close_a":86.955978,"volume_a":5347600,"open_b":37.158784,"high_b":38.275182,"low_b":36.434758,"close_b":36.549202,"volume_b":290954000,"open_bench":239.468296,"close_bench":231.131332},{"date":"2018-02-06","open_a":85.42138,"high_a":86.568532,"low_a":83.681652,"close_a":86.272247,"volume_a":6642600,"open_b":36.161497,"high_b":38.237811,"low_b":35.967645,"close_b":38.076656,"volume_b":272975200,"open_bench":227.637162,"close_bench":235.68512},{"date":"2018-02-07","open_a":86.583709,"high_a":87.951177,"low_a":85.656867,"close_a":86.226646,"volume_a":4931400,"open_b":38.090668,"high_b":38.16307,"low_b":37.151775,"close_b":37.261543,"volume_b":206434400,"open_bench":235.13343,"close_bench":234.406586},{"date":"2018-02-08","open_a":86.378594,"high_a":86.393792,"low_a":83.636054,"close_a":83.681641,"volume_a":4907900,"open_b":37.436711,"high_b":37.602538,"low_b":36.208207,"close_b":36.236233,"volume_b":217562000,"open_bench":234.704257,"close_bench":225.614182},{"date":"2018-02-09","open_a":84.183048,"high_a":85.079502,"low_a":82.716815,"close_a":84.464142,"volume_a":5301700,"open_b":36.834235,"high_b":37.02653,"low_b":35.232542,"close_b":36.679459,"volume_b":282690400,"open_bench":228.390274,"close_bench":229.003296},{"date":"2018-02-12","open_a":85.01113,"high_a":85.656879,"low_a":84.372974,"close_a":85.03392,"volume_a":4759200,"open_b":37.169575,"high_b":38.433575,"low_b":36.93741,"close_b":38.156857,"volume_b":243278000,"open_bench":231.043698,"close_bench":232.366058},{"date":"2018-02-13","open_a":85.087085,"high_a":85.596087,"low_a":84.084273,"close_a":85.193443,"volume_a":5274700,"open_b":37.978633,"high_b":38.635258,"low_b":37.90828,"close_b":38.539108,"volume_b":130196800,"open_bench":231.166301,"close_bench":232.944031},{"date":"2018-02-14","open_a":84.836394,"high_a":85.011129,"low_a":82.625652,"close_a":82.891548,"volume_a":7702700,"open_b":38.23425,"high_b":39.289538,"low_b":38.196731,"close_b":39.249672,"volume_b":162579600,"open_bench":231.46413,"close_bench":236.087982},{"date":"2018-02-15","open_a":83.210617,"high_a":84.357775,"low_a":82.595257,"close_a":84.304596,"volume_a":5125200,"open_b":39.817174,"high_b":40.591053,"low_b":39.631914,"close_b":40.567604,"volume_b":204588800,"open_bench":237.821845,"close_bench":239.100403},{"date":"2018-02-16","open_a":84.372963,"high_a":84.859174,"low_a":83.385348,"close_a":84.372963,"volume_a":5578500,"open_b":40.419866,"high_b":40.996758,"low_b":40.281506,"close_b":40.436279,"volume_b":160704400,"open_bench":238.478664,"close_bench":239.170471},{"date":"2018-02-20","open_a":84.297,"high_a":84.312197,"low_a":83.066274,"close_a":83.172638,"volume_a":3835500,"open_b":40.347167,"high_b":40.865429,"low_b":40.199426,"close_b":40.300266,"volume_b":135722000,"open_bench":238.224687,"close_bench":237.672974},{"date":"2018-02-21","open_a":83.073897,"high_a":83.446151,"low_a":82.002714,"close_a":82.048294,"volume_a":4501900,"open_b":40.530081,"high_b":40.832595,"low_b":40.103274,"close_b":40.117348,"volume_b":149886400,"open_bench":238.11095,"close_bench":236.490845},{"date":"2018-02-22","open_a":82.298989,"high_a":83.544907,"low_a":82.063476,"close_a":83.005516,"volume_a":5394700,"open_b":40.288546,"high_b":40.792738,"low_b":40.267441,"close_b":40.452702,"volume_b":123967600,"open_bench":237.410308,"close_bench":236.797287},{"date":"2018-02-23","open_a":83.081502,"high_a":83.605701,"low_a":82.481333,"close_a":83.324608,"volume_a":4760400,"open_b":40.727073,"high_b":41.191398,"low_b":40.696585,"close_b":41.156223,"volume_b":135249600,"open_bench":238.01454,"close_bench":240.571655},{"date":"2018-02-26","open_a":83.658848,"high_a":84.456536,"low_a":83.590472,"close_a":84.281807,"volume_a":5355400,"open_b":41.355549,"high_b":42.068452,"low_b":41.322718,"close_b":41.969959,"volume_b":152648800,"open_bench":241.640061,"close_bench":243.36525},{"date":"2018-02-27","open_a":84.471719,"high_a":85.147857,"low_a":84.107058,"close_a":84.243805,"volume_a":6385800,"open_b":42.000449,"high_b":42.324068,"low_b":41.78001,"close_b":41.833946,"volume_b":155712400,"open_bench":243.549187,"close_bench":240.326508},{"date":"2018-02-28","open_a":84.327388,"high_a":84.821199,"low_a":83.324576,"close_a":83.362564,"volume_a":4638900,"open_b":42.037969,"high_b":42.356901,"low_b":41.754216,"close_b":41.77063,"volume_b":151128400,"open_bench":241.421126,"close_bench":237.891937},{"date":"2018-03-01","open_a":83.580672,"high_a":84.093435,"low_a":82.072989,"close_a":82.746475,"volume_a":6896300,"open_b":41.869121,"high_b":42.159913,"low_b":40.490215,"close_b":41.038963,"volume_b":195208000,"open_bench":237.681749,"close_bench":234.4328},{"date":"2018-03-02","open_a":82.585751,"high_a":83.66485,"low_a":82.578096,"close_a":83.450562,"volume_a":4903600,"open_b":40.523045,"high_b":41.343824,"low_b":40.440966,"close_b":41.32272,"volume_b":153816000,"open_bench":232.76892,"close_bench":235.641312},{"date":"2018-03-05","open_a":83.006699,"high_a":83.879165,"low_a":82.669956,"close_a":83.749062,"volume_a":3781700,"open_b":41.08822,"high_b":41.681526,"low_b":40.926408,"close_b":41.465778,"volume_b":113605600,"open_bench":234.459061,"close_bench":238.364807},{"date":"2018-03-06","open_a":83.695474,"high_a":83.894459,"low_a":82.991381,"close_a":83.863846,"volume_a":3380800,"open_b":41.721383,"high_b":41.801115,"low_b":41.303959,"close_b":41.430592,"volume_b":95154000,"open_bench":239.336847,"close_bench":238.969055},{"date":"2018-03-07","open_a":83.335768,"high_a":84.231192,"low_a":83.121479,"close_a":84.06282,"volume_a":4612600,"open_b":41.024897,"high_b":41.2383,"low_b":40.867776,"close_b":41.046001,"volume_b":126814000,"open_bench":236.814776,"close_bench":238.881485},{"date":"2018-03-08","open_a":84.208266,"high_a":85.815438,"low_a":83.863873,"close_a":85.777176,"volume_a":5437700,"open_b":41.151524,"high_b":41.536118,"low_b":41.055378,"close_b":41.493908,"volume_b":95096400,"open_bench":239.555778,"close_bench":240.037445},{"date":"2018-03-09","open_a":85.922579,"high_a":86.152178,"low_a":85.440429,"close_a":86.129219,"volume_a":3680600,"open_b":41.733128,"high_b":42.211524,"low_b":41.599457,"close_b":42.206833,"volume_b":128740800,"open_bench":241.438613,"close_bench":244.214661},{"date":"2018-03-12","open_a":86.098572,"high_a":86.550108,"low_a":85.777138,"close_a":86.289902,"volume_a":4509600,"open_b":42.279512,"high_b":42.771981,"low_b":42.260754,"close_b":42.614861,"volume_b":128828400,"open_bench":244.503643,"close_bench":243.908127},{"date":"2018-03-13","open_a":86.68788,"high_a":87.170036,"low_a":86.266957,"close_a":86.940437,"volume_a":4590000,"open_b":42.818886,"high_b":43.03229,"low_b":42.033285,"close_b":42.204475,"volume_b":126774000,"open_bench":245.064166,"close_bench":242.331894},{"date":"2018-03-14","open_a":86.917477,"high_a":87.070539,"low_a":85.89195,"close_a":86.18277,"volume_a":4055700,"open_b":42.286553,"high_b":42.333454,"low_b":41.697935,"close_b":41.845676,"volume_b":117473600,"open_bench":243.286407,"close_bench":241.088318},{"date":"2018-03-15","open_a":86.022067,"high_a":86.672588,"low_a":85.455731,"close_a":85.708282,"volume_a":3906800,"open_b":41.859753,"high_b":42.267799,"low_b":41.758916,"close_b":41.894928,"volume_b":90975200,"open_bench":241.596299,"close_bench":240.825653},{"date":"2018-03-16","open_a":85.968472,"high_a":86.634298,"low_a":85.394482,"close_a":85.440399,"volume_a":7730800,"open_b":41.894919,"high_b":42.005138,"low_b":41.653375,"close_b":41.747181,"volume_b":157618800,"open_bench":241.350537,"close_bench":241.086777},{"date":"2018-03-19","open_a":85.448057,"high_a":85.708263,"low_a":84.223545,"close_a":84.460793,"volume_a":4216700,"open_b":41.583018,"high_b":41.618193,"low_b":40.724717,"close_b":41.10931,"volume_b":133787200,"open_bench":240.33943,"close_bench":237.824799},{"date":"2018-03-20","open_a":84.759286,"high_a":85.180215,"low_a":84.055192,"close_a":84.108765,"volume_a":3319000,"open_b":41.095249,"high_b":41.461082,"low_b":41.024896,"close_b":41.095249,"volume_b":78597600,"open_bench":238.220493,"close_bench":238.229294},{"date":"2018-03-21","open_a":84.101106,"high_a":84.246518,"low_a":83.060269,"close_a":83.220985,"volume_a":3867600,"open_b":41.048328,"high_b":41.060054,"low_b":40.161887,"close_b":40.164234,"volume_b":148219600,"open_bench":238.185383,"close_bench":237.772141},{"date":"2018-03-22","open_a":82.999051,"high_a":83.932744,"low_a":82.501597,"close_a":82.608742,"volume_a":4325500,"open_b":39.866428,"high_b":40.494909,"low_b":39.538117,"close_b":39.596745,"volume_b":165963200,"open_bench":235.556371,"close_bench":231.828415},{"date":"2018-03-23","open_a":82.723495,"high_a":83.220955,"low_a":81.078061,"close_a":81.238777,"volume_a":4873300,"open_b":39.48887,"high_b":39.847668,"low_b":38.679817,"close_b":38.679817,"volume_b":164115200,"open_bench":232.267975,"close_bench":226.887024},{"date":"2018-03-26","open_a":81.644407,"high_a":82.249011,"low_a":81.192865,"close_a":81.743896,"volume_a":4119400,"open_b":39.413822,"high_b":40.593398,"low_b":39.031572,"close_b":40.51601,"volume_b":150164800,"open_bench":230.474328,"close_bench":233.094437},{"date":"2018-03-27","open_a":82.065364,"high_a":83.067933,"low_a":81.514332,"close_a":82.379143,"volume_a":5036300,"open_b":40.729404,"high_b":41.074132,"low_b":39.144129,"close_b":39.477131,"volume_b":163690400,"open_bench":234.026491,"close_bench":229.129135},{"date":"2018-03-28","open_a":82.761805,"high_a":84.269488,"low_a":82.65466,"close_a":83.557739,"volume_a":6413000,"open_b":39.221523,"high_b":39.871112,"low_b":38.738437,"close_b":39.040951,"volume_b":166674000,"open_bench":229.260997,"close_bench":228.452087},{"date":"2018-03-29","open_a":84.277129,"high_a":84.690403,"low_a":83.397009,"close_a":83.534767,"volume_a":3829500,"open_b":39.352853,"high_b":40.276816,"low_b":39.139449,"close_b":39.345818,"volume_b":153594000,"open_bench":229.58629,"close_bench":231.37114},{"date":"2018-04-02","open_a":83.129151,"high_a":83.588342,"low_a":81.460751,"close_a":81.927597,"volume_a":4325500,"open_b":39.07847,"high_b":39.61784,"low_b":38.569587,"close_b":39.087849,"volume_b":150347200,"open_bench":230.843617,"close_bench":226.377106},{"date":"2018-04-03","open_a":82.187775,"high_a":83.06024,"low_a":81.575516,"close_a":82.800034,"volume_a":4699000,"open_b":39.312987,"high_b":39.573292,"low_b":38.665746,"close_b":39.488869,"volume_b":121112000,"open_bench":227.60802,"close_bench":229.278564},{"date":"2018-04-04","open_a":82.486273,"high_a":84.743973,"low_a":82.072999,"close_a":84.323044,"volume_a":4694200,"open_b":38.665751,"high_b":40.337794,"low_b":38.639955,"close_b":40.243992,"volume_b":138422000,"open_bench":225.744103,"close_bench":231.731705},{"date":"2018-04-05","open_a":84.514377,"high_a":85.088367,"low_a":83.993958,"close_a":84.621521,"volume_a":3275700,"open_b":40.471437,"high_b":40.858374,"low_b":40.354183,"close_b":40.523029,"volume_b":107732800,"open_bench":233.481331,"close_bench":233.560486},{"date":"2018-04-06","open_a":84.231222,"high_a":84.935321,"low_a":83.243957,"close_a":83.649582,"volume_a":3579100,"open_b":40.093895,"high_b":40.448001,"low_b":39.444305,"close_b":39.486519,"volume_b":140021200,"open_bench":231.608595,"close_bench":228.355408},{"date":"2018-04-09","open_a":83.833254,"high_a":84.529693,"low_a":83.251609,"close_a":83.7108,"volume_a":2620000,"open_b":39.838274,"high_b":40.591043,"low_b":39.831239,"close_b":39.878139,"volume_b":116070800,"open_bench":229.806164,"close_bench":229.48085},{"date":"2018-04-10","open_a":83.986303,"high_a":84.33835,"low_a":83.358741,"close_a":83.726097,"volume_a":2817000,"open_b":40.569951,"high_b":40.804459,"low_b":40.225223,"close_b":40.628578,"volume_b":113634400,"open_bench":232.355936,"close_bench":233.129669},{"date":"2018-04-11","open_a":83.090901,"high_a":83.779691,"low_a":82.976102,"close_a":83.175087,"volume_a":4000200,"open_b":40.389385,"high_b":40.785704,"low_b":40.265095,"close_b":40.438633,"volume_b":89726400,"open_bench":231.65253,"close_bench":231.907516},{"date":"2018-04-12","open_a":83.389367,"high_a":83.810295,"low_a":82.861293,"close_a":82.891907,"volume_a":2691700,"open_b":40.66609,"high_b":41.038958,"low_b":40.57932,"close_b":40.83728,"volume_b":91557200,"open_bench":233.226447,"close_bench":233.815521},{"date":"2018-04-13","open_a":83.305173,"high_a":83.772019,"low_a":83.052616,"close_a":83.618958,"volume_a":2828300,"open_b":40.987381,"high_b":41.235959,"low_b":40.76929,"close_b":40.975655,"volume_b":100497200,"open_bench":235.116753,"close_bench":233.129669},{"date":"2018-04-16","open_a":83.993981,"high_a":84.675117,"low_a":83.817955,"close_a":84.124084,"volume_a":2769800,"open_b":41.046006,"high_b":41.318037,"low_b":40.999105,"close_b":41.23127,"volume_b":86313600,"open_bench":234.756235,"close_bench":235.046371},{"date":"2018-04-17","open_a":83.182684,"high_a":83.481159,"low_a":82.639307,"close_a":83.320442,"volume_a":5022000,"open_b":41.388378,"high_b":41.962922,"low_b":41.369616,"close_b":41.798767,"volume_b":106421600,"open_bench":236.804817,"close_bench":237.560974},{"date":"2018-04-18","open_a":83.396995,"high_a":83.695469,"low_a":81.476038,"close_a":81.874008,"volume_a":6525500,"open_b":41.697933,"high_b":41.934788,"low_b":41.479841,"close_b":41.704967,"volume_b":83018000,"open_bench":238.000668,"close_bench":237.736908},{"date":"2018-04-19","open_a":81.667372,"high_a":81.843398,"low_a":80.419901,"close_a":80.794907,"volume_a":6320200,"open_b":40.748155,"high_b":41.130405,"low_b":40.490198,"close_b":40.523029,"volume_b":139235200,"open_bench":237.086293,"close_bench":236.418091},{"date":"2018-04-20","open_a":80.703067,"high_a":80.817866,"low_a":78.269346,"close_a":78.430069,"volume_a":8650000,"open_b":40.007138,"high_b":40.152532,"low_b":38.794726,"close_b":38.862736,"volume_b":261964400,"open_bench":236.347616,"close_bench":234.413284},{"date":"2018-04-23","open_a":78.483633,"high_a":78.674963,"low_a":77.94025,"close_a":78.506592,"volume_a":6670600,"open_b":39.123024,"high_b":39.144129,"low_b":38.48047,"close_b":38.750156,"volume_b":146062000,"open_bench":234.98488,"close_bench":234.378204},{"date":"2018-04-24","open_a":78.537219,"high_a":78.713239,"low_a":77.274438,"close_a":77.63414,"volume_a":5695500,"open_b":38.850984,"high_b":39.00576,"low_b":37.807423,"close_b":38.210777,"volume_b":134768000,"open_bench":235.398116,"close_bench":231.221741},{"date":"2018-04-25","open_a":77.458095,"high_a":78.338215,"low_a":77.18258,"close_a":77.412178,"volume_a":4828300,"open_b":38.135746,"high_b":38.79237,"low_b":38.086501,"close_b":38.377289,"volume_b":113528400,"open_bench":231.160146,"close_bench":231.793198},{"date":"2018-04-26","open_a":78.254048,"high_a":79.654586,"low_a":77.412196,"close_a":79.027023,"volume_a":5978800,"open_b":38.487511,"high_b":38.86507,"low_b":38.31163,"close_b":38.510963,"volume_b":111852000,"open_bench":232.813106,"close_bench":234.149536},{"date":"2018-04-27","open_a":78.23109,"high_a":78.820384,"low_a":77.794857,"close_a":77.840775,"volume_a":4765900,"open_b":38.459379,"high_b":38.536767,"low_b":37.669087,"close_b":38.065407,"volume_b":142623200,"open_bench":234.756159,"close_bench":234.369293},{"date":"2018-04-30","open_a":77.947904,"high_a":78.361177,"low_a":76.975949,"close_a":77.251465,"volume_a":5479900,"open_b":38.020832,"high_b":39.223857,"low_b":37.952823,"close_b":38.754841,"volume_b":169709600,"open_bench":234.984919,"close_bench":232.567017},{"date":"2018-05-01","open_a":76.807589,"high_a":77.243821,"low_a":75.437664,"close_a":75.866241,"volume_a":4543900,"open_b":39.024526,"high_b":39.678802,"low_b":38.757186,"close_b":39.655354,"volume_b":214277600,"open_bench":232.004212,"close_bench":232.980179},{"date":"2018-05-02","open_a":75.889209,"high_a":76.042271,"low_a":74.259072,"close_a":74.41214,"volume_a":5927700,"open_b":41.092903,"high_b":41.683865,"low_b":40.757557,"close_b":41.407146,"volume_b":266157600,"open_bench":232.786799,"close_bench":231.415192},{"date":"2018-05-03","open_a":74.603478,"high_a":75.070318,"low_a":74.006523,"close_a":74.695312,"volume_a":6668200,"open_b":41.245329,"high_b":41.625232,"low_b":40.907637,"close_b":41.482182,"volume_b":136272800,"open_bench":230.588654,"close_bench":230.905167},{"date":"2018-05-04","open_a":74.657059,"high_a":76.356067,"low_a":74.618791,"close_a":75.759117,"volume_a":5937000,"open_b":41.801101,"high_b":43.20815,"low_b":41.78234,"close_b":43.109657,"volume_b":224805200,"open_bench":229.938026,"close_bench":233.894592},{"date":"2018-05-07","open_a":75.751481,"high_a":76.004033,"low_a":75.055036,"close_a":75.17749,"volume_a":3475400,"open_b":43.426246,"high_b":44.010173,"low_b":43.325409,"close_b":43.421558,"volume_b":169805600,"open_bench":234.659522,"close_bench":234.685898},{"date":"2018-05-08","open_a":74.93257,"high_a":74.993792,"low_a":73.799893,"close_a":74.075409,"volume_a":5333700,"open_b":43.381705,"high_b":43.670149,"low_b":43.072152,"close_b":43.630283,"volume_b":113611200,"open_bench":234.316607,"close_bench":234.685898},{"date":"2018-05-09","open_a":74.381526,"high_a":74.634077,"low_a":73.424875,"close_a":73.593246,"volume_a":6201500,"open_b":43.747515,"high_b":43.946844,"low_b":43.435618,"close_b":43.937466,"volume_b":92844800,"open_bench":235.354098,"close_bench":236.954315},{"date":"2018-05-10","open_a":73.799867,"high_a":74.802435,"low_a":73.776909,"close_a":74.442734,"volume_a":3100600,"open_b":44.026597,"high_b":44.643352,"low_b":44.005489,"close_b":44.565964,"volume_b":111957200,"open_bench":237.692841,"close_bench":239.169952},{"date":"2018-05-11","open_a":74.680008,"high_a":75.200426,"low_a":74.14428,"close_a":74.565208,"volume_a":3507300,"open_b":44.60835,"high_b":44.742534,"low_b":44.128106,"close_b":44.396477,"volume_b":104848800,"open_bench":239.293089,"close_bench":239.899765},{"date":"2018-05-14","open_a":74.764188,"high_a":74.794801,"low_a":74.029481,"close_a":74.350914,"volume_a":4307900,"open_b":44.495336,"high_b":44.617752,"low_b":44.224613,"close_b":44.292881,"volume_b":83115200,"open_bench":240.330566,"close_bench":240.014053},{"date":"2018-05-15","open_a":73.937641,"high_a":74.419791,"low_a":73.746311,"close_a":74.083054,"volume_a":6555700,"open_b":43.970375,"high_b":44.038646,"low_b":43.574883,"close_b":43.890335,"volume_b":94780800,"open_bench":238.791978,"close_bench":238.36116},{"date":"2018-05-16","open_a":73.960594,"high_a":74.657039,"low_a":73.891718,"close_a":74.381523,"volume_a":5544700,"open_b":43.80324,"high_b":44.365876,"low_b":43.786759,"close_b":44.299957,"volume_b":76732400,"open_bench":238.396371,"close_bench":239.36351},{"date":"2018-05-17","open_a":74.657028,"high_a":74.993765,"low_a":74.350899,"close_a":74.940193,"volume_a":6531100,"open_b":44.257575,"high_b":44.471802,"low_b":43.871499,"close_b":44.01981,"volume_b":69176000,"open_bench":239.099671,"close_bench":239.161224},{"date":"2018-05-18","open_a":75.238672,"high_a":75.345817,"low_a":74.481006,"close_a":74.626419,"volume_a":6894000,"open_b":44.066894,"high_b":44.212849,"low_b":43.817357,"close_b":43.85973,"volume_b":73190800,"open_bench":238.818371,"close_bench":238.563385},{"date":"2018-05-21","open_a":75.169825,"high_a":76.516791,"low_a":75.04737,"close_a":76.157089,"volume_a":6678100,"open_b":44.257584,"high_b":44.556559,"low_b":44.000985,"close_b":44.170483,"volume_b":73603200,"open_bench":240.040482,"close_bench":240.356995},{"date":"2018-05-22","open_a":76.363724,"high_a":76.884142,"low_a":76.095863,"close_a":76.478523,"volume_a":6807600,"open_b":44.347037,"high_b":44.464744,"low_b":43.970376,"close_b":44.059834,"volume_b":60962800,"open_bench":240.875834,"close_bench":239.688858},{"date":"2018-05-23","open_a":76.478501,"high_a":76.991264,"low_a":76.233598,"close_a":76.876465,"volume_a":6249900,"open_b":43.869145,"high_b":44.375281,"low_b":43.730249,"close_b":44.342323,"volume_b":80233600,"open_bench":238.422656,"close_bench":240.34816},{"date":"2018-05-24","open_a":76.769335,"high_a":76.975975,"low_a":76.348412,"close_a":76.669846,"volume_a":4696700,"open_b":44.438839,"high_b":44.455316,"low_b":43.836184,"close_b":44.292881,"volume_b":92936000,"open_bench":239.952565,"close_bench":239.855835},{"date":"2018-05-25","open_a":76.669828,"high_a":77.381577,"low_a":76.455539,"close_a":76.769318,"volume_a":5324300,"open_b":44.311731,"high_b":44.646017,"low_b":44.175191,"close_b":44.394127,"volume_b":69844000,"open_bench":239.284271,"close_bench":239.284271},{"date":"2018-05-29","open_a":76.53208,"high_a":77.519345,"low_a":76.53208,"close_a":77.373932,"volume_a":8302600,"open_b":44.163411,"high_b":44.434134,"low_b":43.991557,"close_b":44.234032,"volume_b":90056400,"open_bench":237.666557,"close_bench":236.532333},{"date":"2018-05-30","open_a":77.802529,"high_a":78.246411,"low_a":77.366291,"close_a":77.932632,"volume_a":6256600,"open_b":44.191653,"high_b":44.257568,"low_b":43.970365,"close_b":44.139862,"volume_b":74762000,"open_bench":237.83368,"close_bench":239.688858},{"date":"2018-05-31","open_a":78.124171,"high_a":78.240027,"low_a":76.973357,"close_a":77.429047,"volume_a":8270100,"open_b":44.073962,"high_b":44.311727,"low_b":43.819715,"close_b":43.991566,"volume_b":109931200,"open_bench":239.284282,"close_bench":238.220413},{"date":"2018-06-01","open_a":77.653032,"high_a":77.954251,"low_a":76.965632,"close_a":77.429047,"volume_a":4414200,"open_b":44.25523,"high_b":44.789614,"low_b":44.198729,"close_b":44.784908,"volume_b":93770000,"open_bench":239.512925,"close_bench":240.559219},{"date":"2018-06-04","open_a":77.830642,"high_a":78.077797,"low_a":77.259098,"close_a":77.328613,"volume_a":3932000,"open_b":45.114481,"high_b":45.533515,"low_b":45.046213,"close_b":45.15921,"volume_b":105064800,"open_bench":241.376896,"close_bench":241.702209},{"date":"2018-06-05","open_a":77.591237,"high_a":77.977417,"low_a":77.143268,"close_a":77.398148,"volume_a":4851900,"open_b":45.451121,"high_b":45.655929,"low_b":45.283977,"close_b":45.507618,"volume_b":86264000,"open_bench":241.83412,"close_bench":241.878098},{"date":"2018-06-06","open_a":77.544877,"high_a":77.900161,"low_a":77.096907,"close_a":77.876991,"volume_a":3946700,"open_b":45.582948,"high_b":45.688883,"low_b":45.180391,"close_b":45.66534,"volume_b":83734400,"open_bench":242.484741,"close_bench":243.900299},{"date":"2018-06-07","open_a":77.629854,"high_a":78.402214,"low_a":77.49083,"close_a":78.131889,"volume_a":5052900,"open_b":45.703021,"high_b":45.717145,"low_b":45.279277,"close_b":45.542942,"volume_b":85388800,"open_bench":244.383874,"close_bench":243.873901},{"date":"2018-06-08","open_a":78.147337,"high_a":79.251812,"low_a":78.147337,"close_a":79.159126,"volume_a":4795500,"open_b":45.00384,"high_b":45.199233,"low_b":44.674264,"close_b":45.128609,"volume_b":106627200,"open_bench":243.416735,"close_bench":244.59491},{"date":"2018-06-11","open_a":79.251818,"high_a":79.869708,"low_a":78.989213,"close_a":79.715233,"volume_a":3984500,"open_b":45.046208,"high_b":45.192162,"low_b":44.777837,"close_b":45.017956,"volume_b":73234000,"open_bench":244.814738,"close_bench":244.920242},{"date":"2018-06-12","open_a":79.730679,"high_a":80.750194,"low_a":79.414009,"close_a":80.696129,"volume_a":5075900,"open_b":45.055631,"high_b":45.342835,"low_b":44.999131,"close_b":45.265148,"volume_b":67644400,"open_bench":245.333413,"close_bench":245.23671},{"date":"2018-06-13","open_a":80.448946,"high_a":81.561139,"low_a":80.448946,"close_a":81.306259,"volume_a":7553500,"open_b":45.298095,"high_b":45.406386,"low_b":44.831979,"close_b":44.893185,"volume_b":86553600,"open_bench":245.474126,"close_bench":244.454208},{"date":"2018-06-14","open_a":81.298569,"high_a":81.375804,"low_a":80.657509,"close_a":81.198158,"volume_a":4812600,"open_b":45.093304,"high_b":45.098013,"low_b":44.780204,"close_b":44.916744,"volume_b":86440400,"open_bench":245.315796,"close_bench":245.069611},{"date":"2018-06-15","open_a":81.174993,"high_a":83.221747,"low_a":80.958733,"close_a":83.113617,"volume_a":10573000,"open_b":44.735464,"high_b":44.766069,"low_b":44.318783,"close_b":44.455322,"volume_b":246876800,"open_bench":244.288932,"close_bench":244.757019},{"date":"2018-06-18","open_a":82.650193,"high_a":82.804662,"low_a":80.966449,"close_a":81.391243,"volume_a":5964400,"open_b":44.229334,"high_b":44.544786,"low_b":44.069252,"close_b":44.431789,"volume_b":73939600,"open_bench":243.308602,"close_bench":244.253616},{"date":"2018-06-19","open_a":81.476208,"high_a":82.395318,"low_a":81.267674,"close_a":81.839218,"volume_a":5504400,"open_b":43.584309,"high_b":43.864451,"low_b":43.186461,"close_b":43.713787,"volume_b":134314000,"open_bench":241.992666,"close_bench":243.317444},{"date":"2018-06-20","open_a":81.661566,"high_a":81.955061,"low_a":81.375791,"close_a":81.723351,"volume_a":4601400,"open_b":43.869147,"high_b":44.069245,"low_b":43.723188,"close_b":43.904457,"volume_b":82514800,"open_bench":243.99755,"close_bench":243.732605},{"date":"2018-06-21","open_a":82.310333,"high_a":82.974562,"low_a":82.101798,"close_a":82.827812,"volume_a":6550700,"open_b":44.081013,"high_b":44.339968,"low_b":43.53721,"close_b":43.659626,"volume_b":102847600,"open_bench":243.723776,"close_bench":242.204697},{"date":"2018-06-22","open_a":82.827799,"high_a":83.932273,"low_a":82.750565,"close_a":83.700569,"volume_a":5709400,"open_b":43.815011,"high_b":43.822073,"low_b":43.480725,"close_b":43.532516,"volume_b":108801600,"open_bench":243.458782,"close_bench":242.64624},{"date":"2018-06-25","open_a":83.190824,"high_a":84.642858,"low_a":83.190824,"close_a":84.179443,"volume_a":5183000,"open_b":43.174677,"high_b":43.532505,"low_b":42.546126,"close_b":42.88512,"volume_b":126652400,"open_bench":241.498129,"close_bench":239.343155},{"date":"2018-06-26","open_a":84.225805,"high_a":84.318484,"low_a":83.005476,"close_a":83.453445,"volume_a":5034100,"open_b":43.078157,"high_b":43.911517,"low_b":42.972219,"close_b":43.417149,"volume_b":98276800,"open_bench":239.908382,"close_bench":239.873047},{"date":"2018-06-27","open_a":83.700585,"high_a":84.156274,"low_a":83.121316,"close_a":83.623344,"volume_a":6229200,"open_b":43.605493,"high_b":44.08809,"low_b":43.322999,"close_b":43.353603,"volume_b":101141200,"open_bench":240.455951,"close_bench":237.88588},{"date":"2018-06-28","open_a":83.739218,"high_a":84.418898,"low_a":83.646538,"close_a":83.940033,"volume_a":4297400,"open_b":43.339468,"high_b":43.836189,"low_b":43.268844,"close_b":43.669044,"volume_b":69460800,"open_bench":237.832916,"close_bench":239.246017},{"date":"2018-06-29","open_a":83.970884,"high_a":84.573323,"low_a":83.260315,"close_a":84.086739,"volume_a":4721400,"open_b":43.855018,"high_b":44.066891,"low_b":43.059325,"close_b":43.577232,"volume_b":90950800,"open_bench":240.332326,"close_bench":239.590454},{"date":"2018-07-02","open_a":83.95546,"high_a":84.959529,"low_a":83.65424,"close_a":84.596519,"volume_a":4711900,"open_b":43.273558,"high_b":44.092793,"low_b":43.179391,"close_b":44.064541,"volume_b":70925200,"open_bench":238.027168,"close_bench":240.102631},{"date":"2018-07-03","open_a":83.862787,"high_a":84.627421,"low_a":83.461157,"close_a":84.032707,"volume_a":3769100,"open_b":44.208146,"high_b":44.245813,"low_b":43.207642,"close_b":43.2971,"volume_b":55819200,"open_bench":240.994682,"close_bench":239.254807},{"date":"2018-07-05","open_a":84.11769,"high_a":84.666064,"low_a":83.41484,"close_a":84.612,"volume_a":4870400,"open_b":43.612543,"high_b":43.883269,"low_b":43.381839,"close_b":43.6455,"volume_b":66416800,"open_bench":240.376496,"close_bench":241.206665},{"date":"2018-07-06","open_a":84.565648,"high_a":84.974998,"low_a":84.164024,"close_a":84.619713,"volume_a":4387300,"open_b":43.650212,"high_b":44.358803,"low_b":43.598421,"close_b":44.250515,"volume_b":69940800,"open_bench":241.233165,"close_bench":243.246826},{"date":"2018-07-09","open_a":84.287598,"high_a":84.418904,"low_a":83.044099,"close_a":83.229469,"volume_a":7866400,"open_b":44.610702,"high_b":44.888487,"low_b":44.56362,"close_b":44.864948,"volume_b":79026400,"open_bench":244.244874,"close_bench":245.43718},{"date":"2018-07-10","open_a":85.438397,"high_a":87.27661,"low_a":84.712377,"close_a":87.19165,"volume_a":16107200,"open_b":44.895551,"high_b":45.029735,"low_b":44.770779,"close_b":44.810802,"volume_b":63756400,"open_bench":245.88756,"close_bench":246.320312},{"date":"2018-07-11","open_a":86.998576,"high_a":87.230286,"low_a":86.581501,"close_a":86.921341,"volume_a":5521400,"open_b":44.375279,"high_b":44.676607,"low_b":44.165762,"close_b":44.229324,"volume_b":75326000,"open_bench":244.774655,"close_bench":244.518524},{"date":"2018-07-12","open_a":86.921328,"high_a":87.052627,"low_a":85.708724,"close_a":86.141243,"volume_a":5500100,"open_b":44.61776,"high_b":45.060337,"low_b":44.565969,"close_b":44.970879,"volume_b":72164400,"open_bench":245.772683,"close_bench":246.735352},{"date":"2018-07-13","open_a":86.257101,"high_a":87.11442,"low_a":86.210761,"close_a":87.037186,"volume_a":4417100,"open_b":44.982655,"high_b":45.161567,"low_b":44.940278,"close_b":45.041508,"volume_b":50055600,"open_bench":246.55875,"close_bench":246.929672},{"date":"2018-07-16","open_a":86.936775,"high_a":87.3693,"low_a":86.311166,"close_a":87.24572,"volume_a":5413600,"open_b":45.08624,"high_b":45.352254,"low_b":44.827285,"close_b":44.942638,"volume_b":60172400,"open_bench":246.973864,"close_bench":246.708893},{"date":"2018-07-17","open_a":87.639629,"high_a":88.844508,"low_a":87.639629,"close_a":88.728653,"volume_a":6783800,"open_b":44.669547,"high_b":45.168621,"low_b":44.540069,"close_b":45.069748,"volume_b":62138000,"open_bench":245.940494,"close_bench":247.706863},{"date":"2018-07-18","open_a":88.759535,"high_a":88.821325,"low_a":87.964006,"close_a":88.690025,"volume_a":5379000,"open_b":45.147438,"high_b":45.152147,"low_b":44.711923,"close_b":44.822567,"volume_b":65573600,"open_bench":247.786359,"close_bench":248.227951},{"date":"2018-07-19","open_a":88.643694,"high_a":90.049388,"low_a":88.62825,"close_a":89.416054,"volume_a":9241300,"open_b":44.655428,"high_b":45.328708,"low_b":44.655428,"close_b":45.170982,"volume_b":81147200,"open_bench":247.565594,"close_bench":247.291809},{"date":"2018-07-20","open_a":89.400602,"high_a":90.203857,"low_a":88.991253,"close_a":89.601418,"volume_a":27559700,"open_b":45.14743,"high_b":45.300447,"low_b":44.768415,"close_b":45.06739,"volume_b":82704800,"open_bench":247.088623,"close_bench":247.00914},{"date":"2018-07-23","open_a":89.222953,"high_a":89.578244,"low_a":87.72458,"close_a":88.350189,"volume_a":7769300,"open_b":44.888479,"high_b":45.18981,"low_b":44.624818,"close_b":45.107414,"volume_b":63957600,"open_bench":246.806071,"close_bench":247.46846},{"date":"2018-07-24","open_a":88.157099,"high_a":88.759539,"low_a":87.933115,"close_a":88.620514,"volume_a":5095100,"open_b":45.305169,"high_b":45.59002,"low_b":45.211005,"close_b":45.434647,"volume_b":74791600,"open_bench":248.872678,"close_bench":248.713684},{"date":"2018-07-25","open_a":88.774994,"high_a":89.338813,"low_a":88.435154,"close_a":88.983528,"volume_a":4762300,"open_b":45.448762,"high_b":45.870153,"low_b":45.300451,"close_b":45.863091,"volume_b":66839600,"open_bench":248.466403,"close_bench":250.833359},{"date":"2018-07-26","open_a":89.701824,"high_a":90.288819,"low_a":88.72865,"close_a":88.821335,"volume_a":5096300,"open_b":45.813654,"high_b":46.131462,"low_b":45.578241,"close_b":45.71949,"volume_b":76304000,"open_bench":250.117991,"close_bench":250.241623},{"date":"2018-07-27","open_a":88.643675,"high_a":89.091645,"low_a":88.072131,"close_a":88.265221,"volume_a":4563700,"open_b":45.903133,"high_b":45.950215,"low_b":44.751965,"close_b":44.959126,"volume_b":96096000,"open_bench":250.568372,"close_bench":248.545898},{"date":"2018-07-30","open_a":87.879043,"high_a":88.504653,"low_a":87.794083,"close_a":88.187988,"volume_a":3621800,"open_b":45.175683,"high_b":45.246308,"low_b":44.509469,"close_b":44.707214,"volume_b":84118000,"open_bench":248.625478,"close_bench":247.247711},{"date":"2018-07-31","open_a":88.558735,"high_a":89.199789,"low_a":88.133935,"close_a":88.821335,"volume_a":4892100,"open_b":44.799037,"high_b":45.232195,"low_b":44.573039,"close_b":44.79668,"volume_b":157492000,"open_bench":248.007215,"close_bench":248.466461},{"date":"2018-08-01","open_a":88.373359,"high_a":88.458319,"low_a":87.454255,"close_a":87.747749,"volume_a":4169000,"open_b":46.877719,"high_b":47.496852,"low_b":46.449267,"close_b":47.435646,"volume_b":271742800,"open_bench":248.669556,"close_bench":248.051315},{"date":"2018-08-02","open_a":87.740019,"high_a":88.767259,"low_a":87.346114,"close_a":88.581894,"volume_a":4167200,"open_b":47.219061,"high_b":49.05528,"low_b":47.164917,"close_b":48.82222,"volume_b":249616000,"open_bench":246.753079,"close_bench":249.402634},{"date":"2018-08-03","open_a":88.643686,"high_a":90.13434,"low_a":88.334747,"close_a":89.825401,"volume_a":3934500,"open_b":48.73748,"high_b":49.140038,"low_b":48.37259,"close_b":48.963478,"volume_b":133789600,"open_bench":249.526238,"close_bench":250.471252},{"date":"2018-08-06","open_a":90.559134,"high_a":91.478243,"low_a":90.288809,"close_a":90.659538,"volume_a":6704800,"open_b":48.965846,"high_b":49.260112,"low_b":48.746914,"close_b":49.217739,"volume_b":101701600,"open_bench":250.506586,"close_bench":251.389771},{"date":"2018-08-07","open_a":89.941253,"high_a":89.941253,"low_a":87.685965,"close_a":88.218895,"volume_a":6919600,"open_b":49.276585,"high_b":49.318957,"low_b":48.673925,"close_b":48.756321,"volume_b":102349600,"open_bench":252.05219,"close_bench":252.219971},{"date":"2018-08-08","open_a":88.45062,"high_a":88.7055,"low_a":87.639646,"close_a":87.65509,"volume_a":5172900,"open_b":48.506774,"high_b":48.921099,"low_b":48.146593,"close_b":48.789268,"volume_b":90102000,"open_bench":252.052165,"close_bench":252.113968},{"date":"2018-08-09","open_a":88.141656,"high_a":88.141656,"low_a":87.570106,"close_a":87.817261,"volume_a":4115400,"open_b":49.326017,"high_b":49.38487,"low_b":48.777505,"close_b":49.173,"volume_b":93970400,"open_bench":252.175819,"close_bench":251.769562},{"date":"2018-08-10","open_a":87.732325,"high_a":88.025826,"low_a":86.944521,"close_a":87.176231,"volume_a":3711900,"open_b":48.986364,"high_b":49.397419,"low_b":48.823359,"close_b":49.026524,"volume_b":98444800,"open_bench":250.33888,"close_bench":250.082748},{"date":"2018-08-13","open_a":87.129858,"high_a":87.384738,"low_a":86.511975,"close_a":87.292053,"volume_a":3170100,"open_b":49.447032,"high_b":49.834462,"low_b":49.066688,"close_b":49.343086,"volume_b":103563600,"open_bench":250.35652,"close_bench":249.146561},{"date":"2018-08-14","open_a":87.044914,"high_a":87.655079,"low_a":86.589225,"close_a":87.353859,"volume_a":3836700,"open_b":49.647831,"high_b":49.742324,"low_b":49.198976,"close_b":49.550972,"volume_b":82992000,"open_bench":249.870748,"close_bench":250.736252},{"date":"2018-08-15","open_a":87.091237,"high_a":87.848146,"low_a":86.697332,"close_a":87.624161,"volume_a":6513300,"open_b":49.425765,"high_b":49.784848,"low_b":49.215513,"close_b":49.666729,"volume_b":115230400,"open_bench":249.393777,"close_bench":248.863861},{"date":"2018-08-16","open_a":87.848176,"high_a":88.612816,"low_a":87.562406,"close_a":88.242081,"volume_a":4504600,"open_b":50.023449,"high_b":50.510099,"low_b":49.957302,"close_b":50.394344,"volume_b":114001600,"open_bench":250.294612,"close_bench":250.877518},{"date":"2018-08-17","open_a":88.048987,"high_a":88.890856,"low_a":88.048987,"close_a":88.790451,"volume_a":3391600,"open_b":50.422704,"high_b":51.488137,"low_b":50.356557,"close_b":51.40073,"volume_b":141708000,"open_bench":250.674445,"close_bench":251.760773},{"date":"2018-08-20","open_a":88.813601,"high_a":89.477831,"low_a":88.512382,"close_a":88.697746,"volume_a":3676000,"open_b":51.523567,"high_b":51.778702,"low_b":50.817214,"close_b":50.899899,"volume_b":121150800,"open_bench":252.211145,"close_bench":252.299469},{"date":"2018-08-21","open_a":89.014435,"high_a":89.014435,"low_a":87.778661,"close_a":87.832726,"volume_a":4667700,"open_b":51.21646,"high_b":51.308593,"low_b":50.562079,"close_b":50.800678,"volume_b":104639200,"open_bench":252.811706,"close_bench":252.89119},{"date":"2018-08-22","open_a":87.817288,"high_a":87.909973,"low_a":86.473384,"close_a":86.689644,"volume_a":3860900,"open_b":50.578622,"high_b":51.112519,"low_b":50.517197,"close_b":50.803047,"volume_b":76072400,"open_bench":252.484897,"close_bench":252.741028},{"date":"2018-08-23","open_a":86.63556,"high_a":86.681906,"low_a":86.125806,"close_a":86.450195,"volume_a":4508800,"open_b":50.708547,"high_b":51.275521,"low_b":50.696738,"close_b":50.90699,"volume_b":75532800,"open_bench":252.564454,"close_bench":252.405487},{"date":"2018-08-24","open_a":86.403857,"high_a":86.913612,"low_a":86.349793,"close_a":86.620117,"volume_a":3677700,"open_b":51.169203,"high_b":51.240072,"low_b":50.817207,"close_b":51.065258,"volume_b":73905600,"open_bench":252.9795,"close_bench":253.924515},{"date":"2018-08-27","open_a":86.95223,"high_a":87.098974,"low_a":86.372961,"close_a":86.82093,"volume_a":3155700,"open_b":51.299145,"high_b":51.674766,"low_b":51.105431,"close_b":51.485775,"volume_b":82100400,"open_bench":255.11684,"close_bench":255.929382},{"date":"2018-08-28","open_a":86.674177,"high_a":86.851816,"low_a":85.646938,"close_a":85.863197,"volume_a":5229400,"open_b":51.738541,"high_b":52.099986,"low_b":51.717281,"close_b":51.901546,"volume_b":91107200,"open_bench":256.388597,"close_bench":256.053009},{"date":"2018-08-29","open_a":85.963608,"high_a":86.596942,"low_a":85.901817,"close_a":86.542877,"volume_a":2403200,"open_b":52.007852,"high_b":52.796891,"low_b":51.833038,"close_b":52.676407,"volume_b":109019200,"open_bench":256.264945,"close_bench":257.430756},{"date":"2018-08-30","open_a":86.403864,"high_a":86.681914,"low_a":86.156709,"close_a":86.473373,"volume_a":2119100,"open_b":52.740198,"high_b":53.92375,"low_b":52.539394,"close_b":53.160702,"volume_b":195175200,"open_bench":256.953893,"close_bench":256.388641},{"date":"2018-08-31","open_a":86.326615,"high_a":86.95223,"low_a":85.932716,"close_a":86.511986,"volume_a":4248800,"open_b":53.510336,"high_b":54.067858,"low_b":53.389856,"close_b":53.774925,"volume_b":173360400,"open_bench":255.982363,"close_bench":256.397461},{"date":"2018-09-04","open_a":86.365245,"high_a":86.57378,"low_a":85.307111,"close_a":85.731911,"volume_a":4576300,"open_b":53.959189,"high_b":54.14109,"low_b":53.538685,"close_b":53.947376,"volume_b":109560400,"open_bench":255.982366,"close_bench":255.955872},{"date":"2018-09-05","open_a":85.546547,"high_a":87.407931,"low_a":85.500201,"close_a":87.369316,"volume_a":4392100,"open_b":54.096195,"high_b":54.256835,"low_b":53.177228,"close_b":53.595367,"volume_b":133332000,"open_bench":255.602582,"close_bench":255.266968},{"date":"2018-09-06","open_a":87.219779,"high_a":88.411263,"low_a":86.783682,"close_a":88.387901,"volume_a":4808300,"open_b":53.444181,"high_b":53.70877,"low_b":52.279528,"close_b":52.704758,"volume_b":137160000,"open_bench":255.37291,"close_bench":254.498566},{"date":"2018-09-07","open_a":87.881721,"high_a":88.387909,"low_a":86.838202,"close_a":87.796059,"volume_a":3833800,"open_b":52.40945,"high_b":53.241006,"low_b":52.140138,"close_b":52.279518,"volume_b":150479200,"open_bench":253.456504,"close_bench":254.004074},{"date":"2018-09-10","open_a":87.936256,"high_a":88.769517,"low_a":87.764931,"close_a":88.442444,"volume_a":3289700,"open_b":52.196839,"high_b":52.409456,"low_b":51.138493,"close_b":51.577896,"volume_b":158066000,"open_bench":255.010903,"close_bench":254.445679},{"date":"2018-09-11","open_a":87.102966,"high_a":88.582589,"low_a":87.102966,"close_a":88.255516,"volume_a":2807300,"open_b":51.502312,"high_b":52.988253,"low_b":51.159767,"close_b":52.881947,"volume_b":142996000,"open_bench":253.800847,"close_bench":255.284592},{"date":"2018-09-12","open_a":88.239929,"high_a":88.956382,"low_a":88.02967,"close_a":88.660454,"volume_a":3651400,"open_b":53.139442,"high_b":53.153615,"low_b":51.934625,"close_b":52.225201,"volume_b":197114800,"open_bench":255.293401,"close_bench":255.34639},{"date":"2018-09-13","open_a":89.158882,"high_a":89.563834,"low_a":88.535887,"close_a":89.548256,"volume_a":3808100,"open_b":52.803985,"high_b":53.945016,"low_b":52.579559,"close_b":53.486713,"volume_b":166825600,"open_bench":256.406222,"close_bench":256.856628},{"date":"2018-09-14","open_a":89.470352,"high_a":89.470352,"low_a":88.559219,"close_a":89.221153,"volume_a":2698300,"open_b":53.330797,"high_b":53.588296,"low_b":52.567749,"close_b":52.879581,"volume_b":127997200,"open_bench":257.059815,"close_bench":256.900848},{"date":"2018-09-17","open_a":89.244538,"high_a":89.906472,"low_a":88.948615,"close_a":89.867538,"volume_a":3092600,"open_b":52.480331,"high_b":52.669322,"low_b":51.32749,"close_b":51.471596,"volume_b":148780400,"open_bench":256.84788,"close_bench":255.540756},{"date":"2018-09-18","open_a":89.929833,"high_a":89.929833,"low_a":88.535873,"close_a":88.761711,"volume_a":4081200,"open_b":51.450323,"high_b":52.409453,"low_b":51.292044,"close_b":51.556633,"volume_b":126286800,"open_bench":255.752717,"close_bench":256.927368},{"date":"2018-09-19","open_a":88.738358,"high_a":88.964195,"low_a":88.356773,"close_a":88.598183,"volume_a":2844900,"open_b":51.618065,"high_b":51.882651,"low_b":50.862103,"close_b":51.587353,"volume_b":108495200,"open_bench":256.980376,"close_bench":257.201172},{"date":"2018-09-20","open_a":88.956397,"high_a":89.953194,"low_a":88.862945,"close_a":89.727356,"volume_a":3375900,"open_b":52.029111,"high_b":52.511036,"low_b":51.771609,"close_b":51.9795,"volume_b":106435200,"open_bench":258.455292,"close_bench":259.285461},{"date":"2018-09-21","open_a":89.758507,"high_a":89.960978,"low_a":89.431429,"close_a":89.485947,"volume_a":6989500,"open_b":52.156694,"high_b":52.293713,"low_b":51.332221,"close_b":51.419632,"volume_b":384986800,"open_bench":260.02443,"close_bench":259.048523},{"date":"2018-09-24","open_a":88.855149,"high_a":89.337975,"low_a":87.375526,"close_a":87.430038,"volume_a":3387400,"open_b":51.221191,"high_b":52.270086,"low_b":51.176305,"close_b":52.159054,"volume_b":110773600,"open_bench":258.471802,"close_bench":258.187897},{"date":"2018-09-25","open_a":87.718194,"high_a":87.803857,"low_a":87.032898,"close_a":87.196434,"volume_a":3538700,"open_b":51.913351,"high_b":52.638604,"low_b":51.901538,"close_b":52.489773,"volume_b":98217600,"open_bench":258.640549,"close_bench":257.948547},{"date":"2018-09-26","open_a":87.430055,"high_a":87.889519,"low_a":86.682457,"close_a":86.830421,"volume_a":4053500,"open_b":52.208662,"high_b":52.858318,"low_b":51.915726,"close_b":52.071644,"volume_b":95938800,"open_bench":258.090433,"close_bench":257.176636},{"date":"2018-09-27","open_a":86.77591,"high_a":87.531297,"low_a":86.370964,"close_a":86.479988,"volume_a":3841500,"open_b":52.874846,"high_b":53.493789,"low_b":52.808696,"close_b":53.141792,"volume_b":120724800,"open_bench":257.646853,"close_bench":257.895264},{"date":"2018-09-28","open_a":86.861565,"high_a":87.258728,"low_a":86.370956,"close_a":87.064041,"volume_a":5945300,"open_b":53.104001,"high_b":53.352052,"low_b":52.9221,"close_b":53.32843,"volume_b":91717600,"open_bench":257.274191,"close_bench":257.921844},{"date":"2018-10-01","open_a":87.336571,"high_a":87.593559,"low_a":86.184021,"close_a":86.215172,"volume_a":5064700,"open_b":53.850516,"high_b":54.197787,"low_b":53.472537,"close_b":53.687511,"volume_b":94403200,"open_bench":259.154917,"close_bench":258.81781},{"date":"2018-10-02","open_a":86.012704,"high_a":86.526681,"low_a":83.871148,"close_a":84.665474,"volume_a":6702200,"open_b":53.685144,"high_b":54.334799,"low_b":53.538677,"close_b":54.164707,"volume_b":99152800,"open_bench":258.667023,"close_bench":258.667023},{"date":"2018-10-03","open_a":85.047074,"high_a":85.436448,"low_a":83.458427,"close_a":83.730988,"volume_a":6107500,"open_b":54.34662,"high_b":55.154554,"low_b":54.282834,"close_b":54.823822,"volume_b":114619200,"open_bench":259.713877,"close_bench":258.80896},{"date":"2018-10-04","open_a":83.209221,"high_a":83.637535,"low_a":82.39154,"close_a":83.022324,"volume_a":4930300,"open_b":54.519054,"high_b":54.88995,"low_b":53.562289,"close_b":53.859951,"volume_b":128168000,"open_bench":258.329945,"close_bench":256.786255},{"date":"2018-10-05","open_a":82.741989,"high_a":83.333839,"low_a":82.531729,"close_a":82.928886,"volume_a":3438100,"open_b":53.852868,"high_b":53.959174,"low_b":52.109429,"close_b":52.98587,"volume_b":134322000,"open_bench":257.008057,"close_bench":255.34903},{"date":"2018-10-08","open_a":82.827651,"high_a":84.369569,"low_a":82.788712,"close_a":84.252762,"volume_a":3994100,"open_b":52.494514,"high_b":53.10637,"low_b":52.019672,"close_b":52.863045,"volume_b":118655600,"open_bench":254.665881,"close_bench":255.34903},{"date":"2018-10-09","open_a":84.634339,"high_a":84.836815,"low_a":83.730989,"close_a":84.198242,"volume_a":4105600,"open_b":52.832319,"high_b":53.689865,"low_b":52.503948,"close_b":53.595367,"volume_b":107564000,"open_bench":254.967541,"close_bench":254.976395},{"date":"2018-10-10","open_a":83.7855,"high_a":85.156098,"low_a":83.481788,"close_a":83.590813,"volume_a":5796200,"open_b":53.262276,"high_b":53.472528,"low_b":51.039274,"close_b":51.112507,"volume_b":167962400,"open_bench":254.470732,"close_bench":246.903061},{"date":"2018-10-11","open_a":84.268335,"high_a":84.4786,"low_a":81.69068,"close_a":81.815277,"volume_a":5755100,"open_b":50.677831,"high_b":51.854297,"low_b":50.158108,"close_b":50.661293,"volume_b":212497600,"open_bench":245.82066,"close_bench":241.464615},{"date":"2018-10-12","open_a":82.002181,"high_a":82.360405,"low_a":81.402542,"close_a":81.986603,"volume_a":4543000,"open_b":52.071632,"high_b":52.652779,"low_b":51.225898,"close_b":52.470875,"volume_b":161351600,"open_bench":245.545692,"close_bench":244.818222},{"date":"2018-10-15","open_a":81.924309,"high_a":83.053497,"low_a":81.900947,"close_a":82.259171,"volume_a":3805700,"open_b":52.246461,"high_b":52.40474,"low_b":51.327494,"close_b":51.348755,"volume_b":123164000,"open_bench":244.463263,"close_bench":243.443008},{"date":"2018-10-16","open_a":82.788719,"high_a":83.863395,"low_a":82.422706,"close_a":83.629768,"volume_a":4061600,"open_b":51.719644,"high_b":52.678774,"low_b":51.207007,"close_b":52.480331,"volume_b":116736000,"open_bench":245.394783,"close_bench":248.766068},{"date":"2018-10-17","open_a":83.4896,"high_a":84.712235,"low_a":83.474028,"close_a":84.346222,"volume_a":4495900,"open_b":52.515768,"high_b":52.596089,"low_b":51.816502,"close_b":52.253544,"volume_b":91541600,"open_bench":248.801575,"close_bench":248.810455},{"date":"2018-10-18","open_a":84.478584,"high_a":85.156091,"low_a":83.754348,"close_a":84.05027,"volume_a":4093700,"open_b":51.466866,"high_b":51.910995,"low_b":50.318748,"close_b":51.032188,"volume_b":130325200,"open_bench":247.878895,"close_bench":245.217346},{"date":"2018-10-19","open_a":84.564247,"high_a":86.129533,"low_a":84.486374,"close_a":85.888123,"volume_a":5109000,"open_b":51.51412,"high_b":52.270082,"low_b":51.365289,"close_b":51.809418,"volume_b":132314800,"open_bench":245.865122,"close_bench":245.084396},{"date":"2018-10-22","open_a":85.732393,"high_a":86.355394,"low_a":85.070458,"close_a":85.195061,"volume_a":5477700,"open_b":51.922808,"high_b":52.76618,"low_b":51.722007,"close_b":52.125973,"volume_b":115168400,"open_bench":245.749723,"close_bench":243.984238},{"date":"2018-10-23","open_a":84.829032,"high_a":86.075027,"low_a":84.353995,"close_a":85.763527,"volume_a":4484100,"open_b":50.987301,"high_b":52.740189,"low_b":50.720351,"close_b":52.617344,"volume_b":155071200,"open_bench":240.382228,"close_bench":242.742111},{"date":"2018-10-24","open_a":85.818016,"high_a":88.901862,"low_a":85.623329,"close_a":87.967361,"volume_a":8222900,"open_b":52.58664,"high_b":52.971706,"low_b":50.682557,"close_b":50.812489,"volume_b":163702000,"open_bench":242.493692,"close_bench":235.387375},{"date":"2018-10-25","open_a":87.967369,"high_a":88.208779,"low_a":86.768096,"close_a":87.881706,"volume_a":6712900,"open_b":51.431438,"high_b":52.298432,"low_b":51.204647,"close_b":51.925175,"volume_b":119423200,"open_bench":237.215004,"close_bench":239.610382},{"date":"2018-10-26","open_a":86.923873,"high_a":87.266518,"low_a":85.420889,"close_a":86.012733,"volume_a":5872800,"open_b":51.003838,"high_b":52.017302,"low_b":50.240789,"close_b":51.098335,"volume_b":189033600,"open_bench":235.919702,"close_bench":235.39624},{"date":"2018-10-29","open_a":86.666884,"high_a":88.239953,"low_a":86.550071,"close_a":87.91288,"volume_a":5706000,"open_b":51.781069,"high_b":51.899188,"low_b":48.686346,"close_b":50.139214,"volume_b":183742000,"open_bench":238.474804,"close_bench":234.092117},{"date":"2018-10-30","open_a":88.58261,"high_a":88.956412,"low_a":87.687049,"close_a":88.598183,"volume_a":4900400,"open_b":49.881704,"high_b":50.833743,"low_b":49.437578,"close_b":50.389618,"volume_b":146640000,"open_bench":233.923561,"close_bench":237.560989},{"date":"2018-10-31","open_a":88.380112,"high_a":88.380112,"low_a":86.588995,"close_a":87.515701,"volume_a":6449500,"open_b":51.235351,"high_b":52.07872,"low_b":51.173927,"close_b":51.703102,"volume_b":153435600,"open_bench":240.116062,"close_bench":240.098328},{"date":"2018-11-01","open_a":87.219796,"high_a":88.107568,"low_a":86.425475,"close_a":86.838211,"volume_a":5736700,"open_b":51.747993,"high_b":52.529941,"low_b":51.218818,"close_b":52.496868,"volume_b":233292800,"open_bench":240.958904,"close_bench":242.653427},{"date":"2018-11-02","open_a":86.822617,"high_a":87.297654,"low_a":85.374145,"close_a":86.573418,"volume_a":6705700,"open_b":49.503728,"high_b":50.472303,"low_b":48.530424,"close_b":49.014713,"volume_b":365314800,"open_bench":243.753563,"close_bench":241.216232},{"date":"2018-11-05","open_a":86.978369,"high_a":89.119926,"low_a":86.659079,"close_a":89.080986,"volume_a":5270400,"open_b":48.263468,"high_b":48.284728,"low_b":46.815326,"close_b":47.62326,"volume_b":264654800,"open_bench":241.704132,"close_bench":242.546967},{"date":"2018-11-06","open_a":88.800643,"high_a":89.828591,"low_a":88.668258,"close_a":89.719566,"volume_a":3913400,"open_b":47.70124,"high_b":48.362708,"low_b":47.646907,"close_b":48.138283,"volume_b":127531600,"open_bench":242.484899,"close_bench":244.081818},{"date":"2018-11-07","open_a":89.906464,"high_a":90.155663,"low_a":89.119933,"close_a":90.031067,"volume_a":4219600,"open_b":48.658005,"high_b":49.624219,"low_b":48.223327,"close_b":49.598232,"volume_b":133697600,"open_bench":246.246595,"close_bench":249.307388},{"date":"2018-11-08","open_a":89.851936,"high_a":90.724136,"low_a":89.750701,"close_a":90.381485,"volume_a":2925100,"open_b":49.778403,"high_b":49.811592,"low_b":49.012692,"close_b":49.425182,"volume_b":101450400,"open_bench":248.508768,"close_bench":248.854782},{"date":"2018-11-09","open_a":90.397087,"high_a":91.666444,"low_a":90.397087,"close_a":91.487335,"volume_a":5026300,"open_b":48.728207,"high_b":48.837254,"low_b":47.9459,"close_b":48.472179,"volume_b":137463200,"open_bench":247.55077,"close_bench":246.424057},{"date":"2018-11-12","open_a":91.113518,"high_a":92.141465,"low_a":90.88768,"close_a":91.090157,"volume_a":4781900,"open_b":47.175446,"high_b":47.376951,"low_b":45.940349,"close_b":46.030434,"volume_b":204542000,"open_bench":245.918239,"close_bench":241.819458},{"date":"2018-11-13","open_a":91.697594,"high_a":91.697594,"low_a":90.498321,"close_a":91.222557,"volume_a":3609900,"open_b":45.428295,"high_b":46.743989,"low_b":45.385621,"close_b":45.57053,"volume_b":187531600,"open_bench":242.280786,"close_bench":241.366989},{"date":"2018-11-14","open_a":91.456185,"high_a":91.487335,"low_a":90.545045,"close_a":90.840973,"volume_a":4423800,"open_b":45.966425,"high_b":46.103922,"low_b":44.077036,"close_b":44.283283,"volume_b":243204000,"open_bench":243.230141,"close_bench":239.716904},{"date":"2018-11-15","open_a":90.66964,"high_a":91.105737,"low_a":90.023278,"close_a":90.957779,"volume_a":4045700,"open_b":44.660222,"high_b":45.508907,"low_b":44.306997,"close_b":45.376152,"volume_b":185915200,"open_bench":238.456952,"close_bench":242.218597},{"date":"2018-11-16","open_a":91.152479,"high_a":92.266089,"low_a":90.988937,"close_a":92.164848,"volume_a":4484300,"open_b":45.16042,"high_b":46.22009,"low_b":44.913877,"close_b":45.878719,"volume_b":147713200,"open_bench":241.12745,"close_bench":242.848587},{"date":"2018-11-19","open_a":91.954551,"high_a":93.075944,"low_a":91.868889,"close_a":92.678787,"volume_a":4673900,"open_b":45.041879,"high_b":45.207822,"low_b":43.854197,"close_b":44.06044,"volume_b":167701200,"open_bench":242.245339,"close_bench":238.740982},{"date":"2018-11-20","open_a":93.294023,"high_a":93.667819,"low_a":90.163452,"close_a":90.334778,"volume_a":5087800,"open_b":42.284831,"high_b":43.019726,"low_b":41.606833,"close_b":41.955315,"volume_b":271300800,"open_bench":235.422913,"close_bench":234.322815},{"date":"2018-11-21","open_a":90.249115,"high_a":91.027863,"low_a":89.493729,"close_a":89.774078,"volume_a":3245000,"open_b":42.607255,"high_b":42.735271,"low_b":41.853397,"close_b":41.907921,"volume_b":124496800,"open_bench":235.8664,"close_bench":235.12117},{"date":"2018-11-23","open_a":89.384705,"high_a":90.778665,"low_a":89.384705,"close_a":89.87532,"volume_a":1444700,"open_b":41.47172,"high_b":41.865245,"low_b":40.798463,"close_b":40.843502,"volume_b":94496000,"open_bench":233.488825,"close_bench":233.550934},{"date":"2018-11-26","open_a":90.241334,"high_a":90.669648,"low_a":89.548248,"close_a":90.225761,"volume_a":2972500,"open_b":41.305777,"high_b":41.47409,"low_b":40.362266,"close_b":41.395859,"volume_b":179994000,"open_bench":235.795486,"close_bench":237.321442},{"date":"2018-11-27","open_a":89.844184,"high_a":90.747529,"low_a":89.478171,"close_a":90.677444,"volume_a":2884700,"open_b":40.658595,"high_b":41.431421,"low_b":40.509248,"close_b":41.305779,"volume_b":165549600,"open_bench":236.292294,"close_bench":238.119888},{"date":"2018-11-28","open_a":90.319219,"high_a":92.297234,"low_a":89.99993,"close_a":92.281662,"volume_a":3989400,"open_b":41.896069,"high_b":42.977074,"low_b":41.469356,"close_b":42.894104,"volume_b":184250000,"open_bench":239.184568,"close_bench":243.602722},{"date":"2018-11-29","open_a":92.20378,"high_a":92.569793,"low_a":91.456183,"close_a":92.102539,"volume_a":2878700,"open_b":43.301848,"high_b":43.335036,"low_b":42.126016,"close_b":42.564583,"volume_b":167080000,"open_bench":242.830832,"close_bench":243.070389},{"date":"2018-11-30","open_a":91.993529,"high_a":95.007287,"low_a":91.993529,"close_a":94.960564,"volume_a":8051100,"open_b":42.740004,"high_b":42.749489,"low_b":41.967181,"close_b":42.334629,"volume_b":158126000,"open_bench":242.919588,"close_bench":244.552002},{"date":"2018-12-03","open_a":93.823586,"high_a":94.158448,"low_a":91.510703,"close_a":92.655464,"volume_a":6112600,"open_b":43.728569,"high_b":43.842358,"low_b":42.958115,"close_b":43.813911,"volume_b":163210000,"open_bench":248.659625,"close_bench":247.790176},{"date":"2018-12-04","open_a":92.959153,"high_a":93.924799,"low_a":91.463952,"close_a":91.736519,"volume_a":7018200,"open_b":42.896463,"high_b":43.237833,"low_b":41.787012,"close_b":41.886578,"volume_b":165377200,"open_bench":246.965153,"close_bench":239.76123},{"date":"2018-12-06","open_a":91.295383,"high_a":92.135259,"low_a":89.560685,"close_a":91.711395,"volume_a":5191700,"open_b":40.717853,"high_b":41.433783,"low_b":40.40019,"close_b":41.419559,"volume_b":172393600,"open_bench":235.919719,"close_bench":239.397461},{"date":"2018-12-07","open_a":91.342481,"high_a":91.82129,"low_a":90.565395,"close_a":90.910767,"volume_a":4615500,"open_b":41.127974,"high_b":41.365036,"low_b":39.897619,"close_b":39.942661,"volume_b":169126400,"open_bench":239.060348,"close_bench":233.834854},{"date":"2018-12-10","open_a":91.366022,"high_a":91.538708,"low_a":89.843253,"close_a":91.201187,"volume_a":4150600,"open_b":39.115318,"high_b":40.321965,"low_b":38.719424,"close_b":40.205807,"volume_b":248104000,"open_bench":233.657358,"close_bench":234.278397},{"date":"2018-12-11","open_a":91.758498,"high_a":92.771061,"low_a":91.585812,"close_a":92.064621,"volume_a":5545100,"open_b":40.694148,"high_b":40.724964,"low_b":39.589436,"close_b":39.975849,"volume_b":189126800,"open_bench":237.463389,"close_bench":234.331635},{"date":"2018-12-12","open_a":92.888803,"high_a":93.085036,"low_a":91.373891,"close_a":91.836998,"volume_a":4889700,"open_b":40.395459,"high_b":40.755795,"low_b":40.068315,"close_b":40.08728,"volume_b":142510800,"open_bench":237.294797,"close_bench":235.511551},{"date":"2018-12-13","open_a":91.946886,"high_a":93.320516,"low_a":91.923339,"close_a":92.896652,"volume_a":6181200,"open_b":40.416802,"high_b":40.909892,"low_b":40.193962,"close_b":40.525848,"volume_b":127594400,"open_bench":236.452033,"close_bench":235.431778},{"date":"2018-12-14","open_a":92.088169,"high_a":92.268706,"low_a":89.223168,"close_a":89.442947,"volume_a":7404200,"open_b":40.063572,"high_b":40.082537,"low_b":39.181699,"close_b":39.229111,"volume_b":162814800,"open_bench":233.29361,"close_bench":231.084534},{"date":"2018-12-17","open_a":89.772651,"high_a":90.322101,"low_a":88.069344,"close_a":88.595253,"volume_a":6015300,"open_b":39.221988,"high_b":39.909472,"low_b":38.577178,"close_b":38.864025,"volume_b":177151600,"open_bench":230.135269,"close_bench":226.551056},{"date":"2018-12-18","open_a":88.869937,"high_a":89.113264,"low_a":86.931156,"close_a":87.786728,"volume_a":5846200,"open_b":39.205418,"high_b":39.715101,"low_b":38.970725,"close_b":39.368992,"volume_b":135366000,"open_bench":228.183465,"close_bench":226.302628},{"date":"2018-12-19","open_a":88.406849,"high_a":88.846414,"low_a":86.052054,"close_a":86.373878,"volume_a":6915400,"open_b":39.352381,"high_b":39.696121,"low_b":37.714278,"close_b":38.140991,"volume_b":196189200,"open_bench":226.382509,"close_bench":222.91362},{"date":"2018-12-20","open_a":86.499443,"high_a":87.857377,"low_a":85.714511,"close_a":86.507294,"volume_a":8468300,"open_b":38.024828,"high_b":38.430206,"low_b":36.815811,"close_b":37.178516,"volume_b":259092000,"open_bench":221.671559,"close_bench":219.285034},{"date":"2018-12-21","open_a":87.096026,"high_a":89.607811,"low_a":85.408421,"close_a":85.88723,"volume_a":13263600,"open_b":37.185612,"high_b":37.493794,"low_b":35.471652,"close_b":35.732418,"volume_b":382978400,"open_bench":220.181862,"close_bench":214.791977},{"date":"2018-12-24","open_a":86.444514,"high_a":87.198047,"low_a":83.17135,"close_a":83.226295,"volume_a":3354000,"open_b":35.120824,"high_b":35.926839,"low_b":34.751007,"close_b":34.807903,"volume_b":148676800,"open_bench":213.310623,"close_bench":209.116516},{"date":"2018-12-26","open_a":83.375426,"high_a":85.117974,"low_a":82.441355,"close_a":85.031631,"volume_a":4537000,"open_b":35.15637,"high_b":37.273336,"low_b":34.781811,"close_b":37.259113,"volume_b":234330000,"open_bench":210.571102,"close_bench":219.682129},{"date":"2018-12-27","open_a":84.592091,"high_a":85.926479,"low_a":83.226312,"close_a":85.88723,"volume_a":5175900,"open_b":36.943822,"high_b":37.164292,"low_b":35.575974,"close_b":37.017311,"volume_b":212468400,"open_bench":216.460697,"close_bench":221.368698},{"date":"2018-12-28","open_a":86.413103,"high_a":87.205886,"low_a":85.730217,"close_a":86.625038,"volume_a":5410300,"open_b":37.337337,"high_b":37.579141,"low_b":36.638003,"close_b":37.036266,"volume_b":169165600,"open_bench":222.716173,"close_bench":221.083145},{"date":"2018-12-31","open_a":86.797738,"high_a":87.198049,"low_a":85.808722,"close_a":86.719246,"volume_a":5019100,"open_b":37.581527,"high_b":37.778289,"low_b":37.095548,"close_b":37.394249,"volume_b":140014000,"open_bench":222.698341,"close_bench":223.019592},{"date":"2019-01-02","open_a":85.675297,"high_a":86.358184,"low_a":84.953161,"close_a":85.777336,"volume_a":4835400,"open_b":36.718613,"high_b":37.657383,"low_b":36.562151,"close_b":37.436913,"volume_b":148158800,"open_bench":219.503661,"close_bench":223.251587},{"date":"2019-01-03","open_a":85.486891,"high_a":86.460205,"low_a":84.81185,"close_a":84.976685,"volume_a":5975900,"open_b":34.13226,"high_b":34.54475,"low_b":33.662878,"close_b":33.70792,"volume_b":365248800,"open_bench":221.511472,"close_bench":217.924179},{"date":"2019-01-04","open_a":85.981408,"high_a":86.813434,"low_a":84.960995,"close_a":86.719246,"volume_a":6293900,"open_b":34.262647,"high_b":35.215639,"low_b":34.089592,"close_b":35.146889,"volume_b":234428400,"open_bench":220.94036,"close_bench":225.223709},{"date":"2019-01-07","open_a":86.350311,"high_a":86.640732,"low_a":85.282804,"close_a":85.973541,"volume_a":5220300,"open_b":35.251192,"high_b":35.282012,"low_b":34.587417,"close_b":35.068653,"volume_b":219111200,"open_bench":225.491384,"close_bench":226.999481},{"date":"2019-01-08","open_a":86.452365,"high_a":86.970423,"low_a":85.887214,"close_a":86.797737,"volume_a":6407700,"open_b":35.455078,"high_b":35.990841,"low_b":35.208534,"close_b":35.737183,"volume_b":164101200,"open_bench":229.176928,"close_bench":229.132294},{"date":"2019-01-09","open_a":86.601508,"high_a":87.025372,"low_a":84.372299,"close_a":84.372299,"volume_a":8521300,"open_b":35.865192,"high_b":36.633276,"low_b":35.471671,"close_b":36.344059,"volume_b":180396400,"open_bench":229.837222,"close_bench":230.203094},{"date":"2019-01-10","open_a":84.309503,"high_a":85.039483,"low_a":84.270254,"close_a":84.89035,"volume_a":7905000,"open_b":36.152031,"high_b":36.500513,"low_b":35.763248,"close_b":36.460213,"volume_b":143122800,"open_bench":228.677239,"close_bench":231.015228},{"date":"2019-01-11","open_a":84.662719,"high_a":85.165075,"low_a":84.490033,"close_a":84.898201,"volume_a":5939000,"open_b":36.242111,"high_b":36.4365,"low_b":35.917333,"close_b":36.102242,"volume_b":108092800,"open_bench":229.944342,"close_bench":231.104431},{"date":"2019-01-14","open_a":84.654889,"high_a":84.741232,"low_a":83.9563,"close_a":84.238876,"volume_a":5804600,"open_b":35.760892,"high_b":35.860458,"low_b":35.374479,"close_b":35.559387,"volume_b":129756800,"open_bench":229.212559,"close_bench":229.694443},{"date":"2019-01-15","open_a":84.348738,"high_a":85.612479,"low_a":83.775742,"close_a":85.549683,"volume_a":4508000,"open_b":35.623391,"high_b":36.363025,"low_b":35.571237,"close_b":36.287167,"volume_b":114843600,"open_bench":230.069195,"close_bench":232.326874},{"date":"2019-01-16","open_a":85.549689,"high_a":85.596789,"low_a":84.537126,"close_a":85.039482,"volume_a":4953500,"open_b":36.28954,"high_b":36.953316,"low_b":36.270574,"close_b":36.730476,"volume_b":122278800,"open_bench":232.755223,"close_bench":232.889099},{"date":"2019-01-17","open_a":84.88249,"high_a":85.92645,"low_a":84.843241,"close_a":85.730217,"volume_a":4612200,"open_b":36.555049,"high_b":37.375287,"low_b":36.33221,"close_b":36.948574,"volume_b":119284800,"open_bench":232.023549,"close_bench":234.656006},{"date":"2019-01-18","open_a":86.318931,"high_a":86.57796,"low_a":85.690983,"close_a":86.397423,"volume_a":5142700,"open_b":37.337348,"high_b":37.427433,"low_b":36.977012,"close_b":37.176147,"volume_b":135004000,"open_bench":236.458555,"close_bench":237.779236},{"date":"2019-01-22","open_a":86.232563,"high_a":86.52299,"low_a":85.384836,"close_a":86.271812,"volume_a":5298200,"open_b":37.078959,"high_b":37.154817,"low_b":36.180489,"close_b":36.341694,"volume_b":121576000,"open_bench":236.315779,"close_bench":234.566727},{"date":"2019-01-23","open_a":86.091324,"high_a":86.962598,"low_a":85.957887,"close_a":86.915504,"volume_a":5197400,"open_b":36.543182,"high_b":36.777875,"low_b":35.96238,"close_b":36.488659,"volume_b":92522400,"open_bench":235.593026,"close_bench":235.057602},{"date":"2019-01-24","open_a":86.585797,"high_a":86.585797,"low_a":85.094432,"close_a":85.950005,"volume_a":4478000,"open_b":36.533707,"high_b":36.621419,"low_b":35.97187,"close_b":36.199448,"volume_b":101766000,"open_bench":234.879095,"close_bench":235.182495},{"date":"2019-01-25","open_a":86.460223,"high_a":86.805589,"low_a":85.588948,"close_a":85.832275,"volume_a":5074800,"open_b":36.858484,"high_b":37.486702,"low_b":36.583495,"close_b":37.398987,"volume_b":134142000,"open_bench":237.020756,"close_bench":237.17247},{"date":"2019-01-28","open_a":85.526157,"high_a":86.028513,"low_a":84.772624,"close_a":85.416267,"volume_a":4055900,"open_b":36.931975,"high_b":37.059991,"low_b":36.427034,"close_b":37.052879,"volume_b":104768400,"open_bench":235.039733,"close_bench":235.369904},{"date":"2019-01-29","open_a":85.502604,"high_a":86.044208,"low_a":85.172934,"close_a":85.557549,"volume_a":4301500,"open_b":37.041021,"high_b":37.4867,"low_b":36.533707,"close_b":36.668831,"volume_b":166348800,"open_bench":235.512716,"close_bench":235.057602},{"date":"2019-01-30","open_a":85.510453,"high_a":87.315792,"low_a":85.196479,"close_a":86.986122,"volume_a":5023800,"open_b":38.700458,"high_b":39.387938,"low_b":37.984529,"close_b":39.174583,"volume_b":244439200,"open_bench":236.565596,"close_bench":238.778641},{"date":"2019-01-31","open_a":87.103859,"high_a":88.618777,"low_a":86.507309,"close_a":88.43824,"volume_a":6183400,"open_b":39.378457,"high_b":40.063567,"low_b":39.011009,"close_b":39.456688,"volume_b":162958400,"open_bench":238.716258,"close_bench":240.875763},{"date":"2019-02-01","open_a":88.658044,"high_a":88.917073,"low_a":87.480647,"close_a":88.061501,"volume_a":5010500,"open_b":39.579955,"high_b":40.058819,"low_b":39.335777,"close_b":39.475647,"volume_b":130672400,"open_bench":241.072054,"close_bench":240.991745},{"date":"2019-02-04","open_a":88.077176,"high_a":89.011247,"low_a":87.7946,"close_a":88.767914,"volume_a":3665000,"open_b":39.686643,"high_b":40.694158,"low_b":39.655823,"close_b":40.596962,"volume_b":125982000,"open_bench":241.036393,"close_bench":242.687271},{"date":"2019-02-05","open_a":88.720806,"high_a":89.121122,"low_a":88.296942,"close_a":88.618767,"volume_a":4593400,"open_b":40.978637,"high_b":41.504916,"low_b":40.857736,"close_b":41.291557,"volume_b":144406400,"open_bench":243.115566,"close_bench":243.704529},{"date":"2019-02-06","open_a":89.019102,"high_a":89.152539,"low_a":88.147828,"close_a":88.736526,"volume_a":4209400,"open_b":41.402971,"high_b":41.621072,"low_b":40.976262,"close_b":41.305779,"volume_b":112958400,"open_bench":243.427951,"close_bench":243.383316},{"date":"2019-02-07","open_a":88.555985,"high_a":88.822865,"low_a":88.037933,"close_a":88.767914,"volume_a":4227700,"open_b":40.869593,"high_b":41.234671,"low_b":40.381244,"close_b":40.523483,"volume_b":126966800,"open_bench":241.777082,"close_bench":241.063202},{"date":"2019-02-08","open_a":88.642311,"high_a":88.728654,"low_a":87.857379,"close_a":88.712952,"volume_a":3868300,"open_b":40.233019,"high_b":40.630611,"low_b":40.097312,"close_b":40.571091,"volume_b":95280000,"open_bench":239.822769,"close_bench":241.357635},{"date":"2019-02-11","open_a":88.822876,"high_a":89.152545,"low_a":88.414708,"close_a":88.673737,"volume_a":4851100,"open_b":40.723452,"high_b":40.761545,"low_b":40.294908,"close_b":40.337761,"volume_b":83973600,"open_bench":242.009148,"close_bench":241.491562},{"date":"2019-02-12","open_a":88.877793,"high_a":89.874653,"low_a":88.08501,"close_a":89.317352,"volume_a":6765300,"open_b":40.497294,"high_b":40.711564,"low_b":40.40206,"close_b":40.685375,"volume_b":89134000,"open_bench":243.097714,"close_bench":244.596878},{"date":"2019-02-13","open_a":89.568535,"high_a":90.055194,"low_a":89.191765,"close_a":89.576385,"volume_a":5528000,"open_b":40.804401,"high_b":41.063906,"low_b":40.454424,"close_b":40.516323,"volume_b":89960800,"open_bench":245.426892,"close_bench":245.39119},{"date":"2019-02-14","open_a":88.697267,"high_a":88.697267,"low_a":87.818142,"close_a":88.375443,"volume_a":6359200,"open_b":40.404435,"high_b":40.773455,"low_b":40.325868,"close_b":40.66394,"volume_b":87342800,"open_bench":244.311408,"close_bench":244.846832},{"date":"2019-02-15","open_a":90.934307,"high_a":91.271828,"low_a":89.568528,"close_a":90.981407,"volume_a":8889500,"open_b":40.771085,"high_b":40.87822,"low_b":40.413966,"close_b":40.573479,"volume_b":98507200,"open_bench":246.613627,"close_bench":247.514923},{"date":"2019-02-19","open_a":91.680016,"high_a":91.813453,"low_a":90.424121,"close_a":90.997124,"volume_a":5596100,"open_b":40.404439,"high_b":40.816316,"low_b":40.352062,"close_b":40.694893,"volume_b":75891200,"open_bench":246.720825,"close_bench":247.943359},{"date":"2019-02-20","open_a":90.855837,"high_a":91.554425,"low_a":90.588963,"close_a":90.918633,"volume_a":6038900,"open_b":40.756786,"high_b":41.263896,"low_b":40.709171,"close_b":40.956772,"volume_b":104457600,"open_bench":247.907569,"close_bench":248.442993},{"date":"2019-02-21","open_a":90.659619,"high_a":91.782072,"low_a":90.636072,"close_a":91.130577,"volume_a":4825900,"open_b":40.902018,"high_b":41.037721,"low_b":40.544899,"close_b":40.725838,"volume_b":68998800,"open_bench":247.809324,"close_bench":247.559464},{"date":"2019-02-22","open_a":91.138421,"high_a":91.656473,"low_a":90.730254,"close_a":91.648628,"volume_a":4865400,"open_b":40.849627,"high_b":41.187699,"low_b":40.802012,"close_b":41.180557,"volume_b":75652800,"open_bench":248.175291,"close_bench":249.094452},{"date":"2019-02-25","open_a":91.970426,"high_a":92.072471,"low_a":90.824427,"close_a":91.099152,"volume_a":3755900,"open_b":41.463885,"high_b":41.870999,"low_b":41.413887,"close_b":41.480549,"volume_b":87493600,"open_bench":250.513313,"close_bench":249.433533},{"date":"2019-02-26","open_a":91.342493,"high_a":91.444531,"low_a":90.604655,"close_a":91.028519,"volume_a":4074600,"open_b":41.356752,"high_b":41.735297,"low_b":41.228187,"close_b":41.50436,"volume_b":68280800,"open_bench":249.085515,"close_bench":249.255066},{"date":"2019-02-27","open_a":90.691008,"high_a":90.816594,"low_a":89.804038,"close_a":90.557571,"volume_a":4175100,"open_b":41.237703,"high_b":41.663863,"low_b":41.123422,"close_b":41.632912,"volume_b":111341600,"open_bench":248.541121,"close_bench":249.147949},{"date":"2019-02-28","open_a":90.793363,"high_a":91.964485,"low_a":90.373977,"close_a":91.505531,"volume_a":5586300,"open_b":41.501979,"high_b":41.642445,"low_b":41.168666,"close_b":41.223423,"volume_b":112861600,"open_bench":248.933836,"close_bench":248.683975},{"date":"2019-03-01","open_a":91.679631,"high_a":92.083194,"low_a":91.181117,"close_a":91.932846,"volume_a":3629100,"open_b":41.492459,"high_b":41.699587,"low_b":41.161529,"close_b":41.656734,"volume_b":103544800,"open_bench":250.254517,"close_bench":250.236679},{"date":"2019-03-04","open_a":92.225623,"high_a":92.526314,"low_a":91.394759,"close_a":91.924927,"volume_a":4383100,"open_b":41.828136,"high_b":42.318579,"low_b":41.41864,"close_b":41.86623,"volume_b":109744800,"open_bench":251.289627,"close_bench":249.326416},{"date":"2019-03-05","open_a":91.774587,"high_a":92.439273,"low_a":91.521372,"close_a":91.814148,"volume_a":3785400,"open_b":41.887675,"high_b":41.90196,"low_b":41.554362,"close_b":41.790062,"volume_b":78949600,"open_bench":249.451351,"close_bench":248.987305},{"date":"2019-03-06","open_a":91.6163,"high_a":92.447164,"low_a":91.458043,"close_a":92.312645,"volume_a":4770700,"open_b":41.585301,"high_b":41.780528,"low_b":41.411505,"close_b":41.549591,"volume_b":83241600,"open_bench":249.103295,"close_bench":247.479187},{"date":"2019-03-07","open_a":91.956588,"high_a":92.842843,"low_a":91.553025,"close_a":91.869545,"volume_a":4858300,"open_b":41.394847,"high_b":41.530554,"low_b":40.954403,"close_b":41.06868,"volume_b":99185600,"open_bench":247.033093,"close_bench":245.409012},{"date":"2019-03-08","open_a":91.655868,"high_a":91.996126,"low_a":90.413533,"close_a":91.181091,"volume_a":5003300,"open_b":40.549662,"high_b":41.20438,"low_b":40.354435,"close_b":41.166286,"volume_b":95997600,"open_bench":243.561799,"close_bench":244.918182},{"date":"2019-03-11","open_a":91.236496,"high_a":92.352221,"low_a":90.983281,"close_a":92.257263,"volume_a":4821600,"open_b":41.780522,"high_b":42.644747,"low_b":41.747191,"close_b":42.592369,"volume_b":128044000,"open_bench":245.632032,"close_bench":248.469742},{"date":"2019-03-12","open_a":92.383866,"high_a":92.747861,"low_a":91.798308,"close_a":92.241432,"volume_a":4356400,"open_b":42.854262,"high_b":43.489933,"low_b":42.704271,"close_b":43.070915,"volume_b":129870400,"open_bench":249.023028,"close_bench":249.406738},{"date":"2019-03-13","open_a":92.486729,"high_a":93.024811,"low_a":92.154386,"close_a":92.249344,"volume_a":5968600,"open_b":43.389956,"high_b":43.63994,"low_b":43.07331,"close_b":43.261395,"volume_b":124130000,"open_bench":250.290243,"close_bench":251.057663},{"date":"2019-03-14","open_a":92.470916,"high_a":92.565874,"low_a":91.355185,"close_a":91.394753,"volume_a":8986600,"open_b":43.782779,"high_b":43.830398,"low_b":43.463753,"close_b":43.742306,"volume_b":94318000,"open_bench":251.084422,"close_bench":250.897034},{"date":"2019-03-15","open_a":91.537193,"high_a":92.209794,"low_a":91.291893,"close_a":91.52137,"volume_a":18194200,"open_b":44.008952,"high_b":44.599387,"low_b":43.744684,"close_b":44.31131,"volume_b":156171600,"open_bench":251.446399,"close_bench":252.136536},{"date":"2019-03-18","open_a":91.924917,"high_a":92.890299,"low_a":91.829965,"close_a":92.795341,"volume_a":9476000,"open_b":44.235132,"high_b":44.851756,"low_b":44.232749,"close_b":44.763668,"volume_b":104879200,"open_bench":252.351718,"close_bench":253.050827},{"date":"2019-03-19","open_a":92.866586,"high_a":93.966488,"low_a":92.779543,"close_a":93.444229,"volume_a":6751700,"open_b":44.842222,"high_b":44.994593,"low_b":44.263688,"close_b":44.408916,"volume_b":126585600,"open_bench":254.108472,"close_bench":253.113571},{"date":"2019-03-20","open_a":93.428389,"high_a":94.54412,"low_a":92.858654,"close_a":93.626213,"volume_a":5733500,"open_b":44.337498,"high_b":45.113639,"low_b":43.98038,"close_b":44.796993,"volume_b":124140800,"open_bench":252.898438,"close_bench":252.351685},{"date":"2019-03-21","open_a":93.380897,"high_a":94.694446,"low_a":93.380897,"close_a":94.512451,"volume_a":4305600,"open_b":45.239828,"high_b":46.742108,"low_b":45.18983,"close_b":46.446888,"volume_b":204136800,"open_bench":251.536094,"close_bench":255.201935},{"date":"2019-03-22","open_a":94.559939,"high_a":95.976361,"low_a":94.520372,"close_a":95.525322,"volume_a":6668100,"open_b":46.506405,"high_b":47.065893,"low_b":45.420764,"close_b":45.485046,"volume_b":169630800,"open_bench":253.848481,"close_bench":250.290192},{"date":"2019-03-25","open_a":95.161319,"high_a":95.976354,"low_a":95.018886,"close_a":95.628181,"volume_a":4275400,"open_b":45.594551,"high_b":45.706448,"low_b":44.425584,"close_b":44.935074,"volume_b":175381200,"open_bench":249.949638,"close_bench":250.10202},{"date":"2019-03-26","open_a":96.1109,"high_a":96.664811,"low_a":95.952643,"close_a":96.554024,"volume_a":4583000,"open_b":45.630266,"high_b":45.920723,"low_b":43.944665,"close_b":44.470818,"volume_b":199202000,"open_bench":251.849771,"close_bench":251.966293},{"date":"2019-03-27","open_a":96.902188,"high_a":97.479837,"low_a":95.897244,"close_a":96.451149,"volume_a":5318500,"open_b":44.937454,"high_b":45.177913,"low_b":44.413681,"close_b":44.870792,"volume_b":119393600,"open_bench":251.957325,"close_bench":250.648743},{"date":"2019-03-28","open_a":96.45115,"high_a":96.910104,"low_a":96.213759,"close_a":96.411583,"volume_a":3694700,"open_b":44.985086,"high_b":45.130314,"low_b":44.647013,"close_b":44.930328,"volume_b":83121600,"open_bench":251.276148,"close_bench":251.598801},{"date":"2019-03-29","open_a":96.767658,"high_a":97.139568,"low_a":95.960532,"close_a":96.973396,"volume_a":5758800,"open_b":45.194595,"high_b":45.254115,"low_b":44.887471,"close_b":45.223164,"volume_b":94256000,"open_bench":253.104578,"close_bench":253.185242},{"date":"2019-04-01","open_a":97.00504,"high_a":97.242431,"low_a":96.213743,"close_a":96.538177,"volume_a":5025700,"open_b":45.625507,"high_b":45.635029,"low_b":44.84937,"close_b":45.530277,"volume_b":111448000,"open_bench":255.175016,"close_bench":256.187805},{"date":"2019-04-02","open_a":96.459075,"high_a":96.743942,"low_a":96.190031,"close_a":96.284988,"volume_a":3242600,"open_b":45.494568,"high_b":46.296898,"low_b":45.485047,"close_b":46.192142,"volume_b":91062800,"open_bench":256.376041,"close_bench":256.313293},{"date":"2019-04-03","open_a":96.284953,"high_a":96.720168,"low_a":95.541132,"close_a":96.189995,"volume_a":4534400,"open_b":46.008817,"high_b":46.782575,"low_b":45.985008,"close_b":46.508785,"volume_b":93087200,"open_bench":257.523273,"close_bench":256.716614},{"date":"2019-04-04","open_a":96.38782,"high_a":96.641035,"low_a":95.881391,"close_a":96.118782,"volume_a":3022500,"open_b":46.375452,"high_b":46.751618,"low_b":45.982623,"close_b":46.589725,"volume_b":76457200,"open_bench":257.039284,"close_bench":257.397797},{"date":"2019-04-05","open_a":96.023865,"high_a":96.27708,"low_a":95.604478,"close_a":96.237518,"volume_a":3971500,"open_b":46.770678,"high_b":46.925432,"low_b":46.646876,"close_b":46.901623,"volume_b":74106400,"open_bench":258.06106,"close_bench":258.643646},{"date":"2019-04-08","open_a":96.316615,"high_a":96.56983,"low_a":95.913053,"close_a":96.538177,"volume_a":3140100,"open_b":46.763523,"high_b":47.670604,"low_b":46.744476,"close_b":47.639656,"volume_b":103526800,"open_bench":258.222466,"close_bench":258.840912},{"date":"2019-04-09","open_a":96.332449,"high_a":96.696445,"low_a":95.628195,"close_a":96.672707,"volume_a":3864400,"open_b":47.692038,"high_b":48.294378,"low_b":47.432528,"close_b":47.496811,"volume_b":143072800,"open_bench":257.881766,"close_bench":257.514282},{"date":"2019-04-10","open_a":96.815132,"high_a":97.092084,"low_a":96.269141,"close_a":96.569832,"volume_a":2681100,"open_b":47.30158,"high_b":47.792026,"low_b":47.182541,"close_b":47.763454,"volume_b":86781200,"open_bench":257.926731,"close_bench":258.392822},{"date":"2019-04-11","open_a":96.823043,"high_a":97.131648,"low_a":95.881398,"close_a":96.277046,"volume_a":3399700,"open_b":47.818224,"high_b":47.853935,"low_b":47.244452,"close_b":47.365871,"volume_b":83603200,"open_bench":258.876713,"close_bench":258.321014},{"date":"2019-04-12","open_a":96.166258,"high_a":96.799298,"low_a":95.865567,"close_a":96.720169,"volume_a":3501100,"open_b":47.425395,"high_b":47.64919,"low_b":46.71354,"close_b":47.346828,"volume_b":111042800,"open_bench":259.925285,"close_bench":260.068695},{"date":"2019-04-15","open_a":96.886348,"high_a":98.120768,"low_a":96.823042,"close_a":97.85173,"volume_a":4419800,"open_b":47.277779,"high_b":47.580141,"low_b":47.142072,"close_b":47.432529,"volume_b":70146400,"open_bench":260.140458,"close_bench":259.898468},{"date":"2019-04-16","open_a":97.503543,"high_a":97.978319,"low_a":96.451117,"close_a":96.862595,"volume_a":5263700,"open_b":47.487285,"high_b":47.942013,"low_b":47.273011,"close_b":47.437286,"volume_b":102785600,"open_bench":260.776775,"close_bench":260.068695},{"date":"2019-04-17","open_a":98.912071,"high_a":100.581707,"low_a":97.994164,"close_a":100.502579,"volume_a":7318200,"open_b":47.506338,"high_b":48.420566,"low_b":47.284926,"close_b":48.361046,"volume_b":115627200,"open_bench":261.1801,"close_bench":259.432343},{"date":"2019-04-18","open_a":101.467957,"high_a":101.491695,"low_a":100.281019,"close_a":100.56588,"volume_a":6331200,"open_b":48.35866,"high_b":48.603881,"low_b":48.215814,"close_b":48.53484,"volume_b":96783200,"open_bench":260.015048,"close_bench":259.943329},{"date":"2019-04-22","open_a":100.312652,"high_a":100.684562,"low_a":99.608399,"close_a":100.051529,"volume_a":3707000,"open_b":48.289606,"high_b":48.791953,"low_b":48.172946,"close_b":48.69434,"volume_b":77758000,"open_bench":259.181487,"close_bench":260.167389},{"date":"2019-04-23","open_a":100.241451,"high_a":100.921967,"low_a":99.798327,"close_a":100.56588,"volume_a":4464100,"open_b":48.67054,"high_b":49.460965,"low_b":48.544359,"close_b":49.396683,"volume_b":93292000,"open_bench":260.534855,"close_bench":262.506714},{"date":"2019-04-24","open_a":100.819098,"high_a":101.270137,"low_a":100.233534,"close_a":100.431358,"volume_a":3517200,"open_b":49.368118,"high_b":49.634766,"low_b":49.294314,"close_b":49.320503,"volume_b":70162400,"open_bench":262.426087,"close_bench":261.924164},{"date":"2019-04-25","open_a":99.426425,"high_a":100.019897,"low_a":99.054514,"close_a":99.806244,"volume_a":4433700,"open_b":49.241929,"high_b":49.463341,"low_b":48.834812,"close_b":48.872906,"volume_b":74172800,"open_bench":261.825504,"close_bench":261.762756},{"date":"2019-04-26","open_a":100.114872,"high_a":100.692521,"low_a":99.798358,"close_a":100.01992,"volume_a":3033800,"open_b":48.782444,"high_b":48.806254,"low_b":48.120584,"close_b":48.639599,"volume_b":74596400,"open_bench":261.807637,"close_bench":262.981781},{"date":"2019-04-29","open_a":99.814144,"high_a":100.463007,"low_a":99.719186,"close_a":100.249359,"volume_a":2793400,"open_b":48.663407,"high_b":49.037194,"low_b":48.534846,"close_b":48.713406,"volume_b":88818800,"open_bench":263.071299,"close_bench":263.393951},{"date":"2019-04-30","open_a":100.53425,"high_a":101.594584,"low_a":99.972424,"close_a":101.325546,"volume_a":4963600,"open_b":48.344377,"high_b":48.425323,"low_b":47.403965,"close_b":47.775368,"volume_b":186139600,"open_bench":263.053468,"close_bench":263.528503},{"date":"2019-05-01","open_a":101.167277,"high_a":101.285973,"low_a":100.502591,"close_a":100.684586,"volume_a":3810800,"open_b":49.968067,"high_b":51.260835,"low_b":49.813313,"close_b":50.120438,"volume_b":259309200,"open_bench":264.155765,"close_bench":261.547546},{"date":"2019-05-02","open_a":100.684566,"high_a":100.929872,"low_a":99.837879,"close_a":100.360138,"volume_a":2875900,"open_b":49.958549,"high_b":50.627551,"low_b":49.551435,"close_b":49.794273,"volume_b":127985200,"open_bench":261.431149,"close_bench":260.983002},{"date":"2019-05-03","open_a":100.573772,"high_a":101.428368,"low_a":100.431338,"close_a":101.024811,"volume_a":2854900,"open_b":50.208547,"high_b":50.434722,"low_b":50.051414,"close_b":50.413296,"volume_b":83569600,"open_bench":262.452967,"close_bench":263.537476},{"date":"2019-05-06","open_a":100.732042,"high_a":100.993165,"low_a":100.304741,"close_a":100.708298,"volume_a":2883900,"open_b":48.637202,"high_b":49.720463,"low_b":48.449121,"close_b":49.634754,"volume_b":129772400,"open_bench":259.253164,"close_bench":262.452942},{"date":"2019-05-07","open_a":100.209806,"high_a":100.455106,"low_a":99.054514,"close_a":99.687553,"volume_a":3558900,"open_b":49.015765,"high_b":49.382405,"low_b":47.813464,"close_b":48.296764,"volume_b":155054800,"open_bench":260.059843,"close_bench":258.070068},{"date":"2019-05-08","open_a":99.545112,"high_a":100.288927,"low_a":98.872505,"close_a":99.719193,"volume_a":3844700,"open_b":48.068199,"high_b":48.887192,"low_b":48.032489,"close_b":48.306278,"volume_b":105358000,"open_bench":257.711609,"close_bench":257.711609},{"date":"2019-05-09","open_a":99.521359,"high_a":100.130661,"low_a":98.793368,"close_a":99.742928,"volume_a":3127200,"open_b":47.71109,"high_b":48.015831,"low_b":46.820676,"close_b":47.787277,"volume_b":139634400,"open_bench":255.650099,"close_bench":256.931793},{"date":"2019-05-10","open_a":99.814141,"high_a":101.507515,"low_a":99.402669,"close_a":101.293861,"volume_a":3720400,"open_b":47.182612,"high_b":47.524378,"low_b":46.071282,"close_b":47.125252,"volume_b":164834800,"open_bench":255.999592,"close_bench":258.222412},{"date":"2019-05-13","open_a":100.51049,"high_a":101.238487,"low_a":100.201885,"close_a":100.621277,"volume_a":4297200,"open_b":44.86196,"high_b":45.28498,"low_b":43.700439,"close_b":44.386356,"volume_b":229722400,"open_bench":253.131414,"close_bench":251.73317},{"date":"2019-05-14","open_a":100.937781,"high_a":101.689511,"low_a":100.724134,"close_a":100.819092,"volume_a":5150300,"open_b":44.551263,"high_b":45.337559,"low_b":44.312267,"close_b":45.089005,"volume_b":146118800,"open_bench":252.746079,"close_bench":254.009857},{"date":"2019-05-15","open_a":100.573794,"high_a":102.765683,"low_a":100.573794,"close_a":102.354218,"volume_a":3751900,"open_b":44.517808,"high_b":45.827506,"low_b":44.458059,"close_b":45.629139,"volume_b":106178800,"open_bench":252.387508,"close_bench":255.49765},{"date":"2019-05-16","open_a":102.370048,"high_a":103.462041,"low_a":101.887356,"close_a":103.082222,"volume_a":4812900,"open_b":45.387754,"high_b":45.999583,"low_b":45.132026,"close_b":45.428383,"volume_b":132125600,"open_bench":256.1968,"close_bench":257.863922},{"date":"2019-05-17","open_a":102.805248,"high_a":103.556977,"low_a":102.591594,"close_a":103.27211,"volume_a":4011700,"open_b":44.675538,"high_b":45.624352,"low_b":44.634909,"close_b":45.170261,"volume_b":131516400,"open_bench":255.569327,"close_bench":256.196716},{"date":"2019-05-20","open_a":102.900197,"high_a":103.224631,"low_a":102.512469,"close_a":102.860641,"volume_a":3965600,"open_b":43.860564,"high_b":44.058931,"low_b":43.086215,"close_b":43.757793,"volume_b":154449200,"open_bench":254.601372,"close_bench":254.502792},{"date":"2019-05-21","open_a":103.327504,"high_a":103.327504,"low_a":101.958564,"close_a":102.116821,"volume_a":4080800,"open_b":44.266856,"high_b":44.931265,"low_b":44.142577,"close_b":44.596672,"volume_b":113459200,"open_bench":256.187832,"close_bench":256.797333},{"date":"2019-05-22","open_a":102.156384,"high_a":103.374975,"low_a":102.037682,"close_a":102.923943,"volume_a":3588000,"open_b":44.133014,"high_b":44.383961,"low_b":43.628732,"close_b":43.683701,"volume_b":118994400,"open_bench":255.84728,"close_bench":256.008606},{"date":"2019-05-23","open_a":102.552027,"high_a":103.580703,"low_a":102.480801,"close_a":102.844803,"volume_a":3901400,"open_b":42.971493,"high_b":43.148348,"low_b":42.49589,"close_b":42.938034,"volume_b":146118800,"open_bench":253.794734,"close_bench":252.880524},{"date":"2019-05-24","open_a":103.129701,"high_a":103.248391,"low_a":102.045628,"close_a":102.472923,"volume_a":2702700,"open_b":43.067093,"high_b":43.530746,"low_b":42.689478,"close_b":42.773129,"volume_b":94858800,"open_bench":254.314582,"close_bench":253.454147},{"date":"2019-05-28","open_a":102.472906,"high_a":102.78942,"low_a":100.779532,"close_a":100.993179,"volume_a":8806000,"open_b":42.761179,"high_b":43.160302,"low_b":42.519795,"close_b":42.596272,"volume_b":111792800,"open_bench":253.73197,"close_bench":251.096863},{"date":"2019-05-29","open_a":100.953602,"high_a":101.531245,"low_a":100.368038,"close_a":101.420464,"volume_a":5502700,"open_b":42.163697,"high_b":42.863958,"low_b":42.063319,"close_b":42.393135,"volume_b":113924800,"open_bench":249.985537,"close_bench":249.411896},{"date":"2019-05-30","open_a":101.404645,"high_a":102.006038,"low_a":101.111869,"close_a":101.768646,"volume_a":2710300,"open_b":42.529364,"high_b":42.835279,"low_b":42.223449,"close_b":42.613014,"volume_b":84873600,"open_bench":250.164755,"close_bench":250.093063},{"date":"2019-05-31","open_a":101.285942,"high_a":101.642023,"low_a":100.486731,"close_a":101.285942,"volume_a":4191200,"open_b":42.118274,"high_b":42.53891,"low_b":41.821921,"close_b":41.841042,"volume_b":108174400,"open_bench":247.556536,"close_bench":246.722961},{"date":"2019-06-03","open_a":101.175168,"high_a":102.203856,"low_a":100.882387,"close_a":102.061417,"volume_a":3588700,"open_b":41.967714,"high_b":42.522183,"low_b":40.693864,"close_b":41.418022,"volume_b":161584400,"open_bench":246.758784,"close_bench":246.095535},{"date":"2019-06-04","open_a":102.67072,"high_a":102.718196,"low_a":101.301781,"close_a":102.132645,"volume_a":4242400,"open_b":41.929471,"high_b":42.978664,"low_b":41.709596,"close_b":42.933254,"volume_b":123872000,"open_bench":248.381117,"close_bench":251.437485},{"date":"2019-06-05","open_a":102.25134,"high_a":104.13463,"low_a":101.966473,"close_a":103.968452,"volume_a":5084200,"open_b":44.042201,"high_b":44.21189,"low_b":43.291754,"close_b":43.626347,"volume_b":119093600,"open_bench":253.050763,"close_bench":253.615433},{"date":"2019-06-06","open_a":104.482583,"high_a":105.431122,"low_a":104.068092,"close_a":105.199959,"volume_a":5528000,"open_b":43.755402,"high_b":44.326602,"low_b":43.533134,"close_b":44.266853,"volume_b":90105200,"open_bench":253.911217,"close_bench":255.264603},{"date":"2019-06-07","open_a":105.885433,"high_a":107.376,"low_a":105.686161,"close_a":106.387604,"volume_a":6332700,"open_b":44.575172,"high_b":45.868142,"low_b":44.398317,"close_b":45.445118,"volume_b":122737600,"open_bench":256.277495,"close_bench":257.819122},{"date":"2019-06-10","open_a":106.156451,"high_a":106.554996,"low_a":105.502844,"close_a":106.483261,"volume_a":2957900,"open_b":45.841839,"high_b":46.692665,"low_b":45.79643,"close_b":46.025867,"volume_b":104883600,"open_bench":259.36074,"close_bench":259.002228},{"date":"2019-06-11","open_a":106.857915,"high_a":107.081099,"low_a":105.765905,"close_a":106.172417,"volume_a":3553600,"open_b":46.570785,"high_b":46.84324,"low_b":46.269651,"close_b":46.558834,"volume_b":107731600,"open_bench":260.812767,"close_bench":258.939514},{"date":"2019-06-12","open_a":106.818045,"high_a":107.1608,"low_a":105.741981,"close_a":106.12458,"volume_a":3302800,"open_b":46.353296,"high_b":46.836069,"low_b":46.219458,"close_b":46.410656,"volume_b":73012800,"open_bench":258.706465,"close_bench":258.482391},{"date":"2019-06-13","open_a":106.371683,"high_a":106.642691,"low_a":105.096346,"close_a":105.965172,"volume_a":2851400,"open_b":46.532533,"high_b":47.032034,"low_b":46.26964,"close_b":46.401085,"volume_b":86698400,"open_bench":259.387563,"close_bench":259.548889},{"date":"2019-06-14","open_a":106.020944,"high_a":106.443401,"low_a":105.662256,"close_a":105.79776,"volume_a":3291800,"open_b":45.7797,"high_b":46.26725,"low_b":45.480955,"close_b":46.064106,"volume_b":75046000,"open_bench":259.262207,"close_bench":259.262207},{"date":"2019-06-17","open_a":105.78981,"high_a":106.2043,"low_a":105.128223,"close_a":105.630394,"volume_a":2725800,"open_b":46.102347,"high_b":46.594683,"low_b":45.927881,"close_b":46.338955,"volume_b":58676400,"open_bench":259.495126,"close_bench":259.360687},{"date":"2019-06-18","open_a":107.144866,"high_a":107.144866,"low_a":105.104317,"close_a":105.263733,"volume_a":4378200,"open_b":46.855179,"high_b":47.86852,"low_b":46.654423,"close_b":47.428768,"volume_b":106204000,"open_bench":261.171299,"close_bench":262.076538},{"date":"2019-06-19","open_a":105.263725,"high_a":106.244142,"low_a":104.681852,"close_a":105.893433,"volume_a":4349900,"open_b":47.722728,"high_b":47.77053,"low_b":47.156309,"close_b":47.290146,"volume_b":84496800,"open_bench":262.210972,"close_bench":262.668091},{"date":"2019-06-20","open_a":106.706444,"high_a":107.168758,"low_a":106.283987,"close_a":106.937607,"volume_a":3847500,"open_b":47.887638,"high_b":47.944999,"low_b":47.328388,"close_b":47.670155,"volume_b":86056000,"open_bench":265.338966,"close_bench":265.177612},{"date":"2019-06-21","open_a":106.937637,"high_a":107.256468,"low_a":106.51518,"close_a":106.778221,"volume_a":7696300,"open_b":47.512418,"high_b":48.002361,"low_b":47.357068,"close_b":47.507637,"volume_b":191202400,"open_bench":264.909326,"close_bench":264.792236},{"date":"2019-06-24","open_a":107.328181,"high_a":107.798473,"low_a":106.762253,"close_a":106.921669,"volume_a":3685800,"open_b":47.450273,"high_b":47.837449,"low_b":47.361846,"close_b":47.459835,"volume_b":72881600,"open_bench":264.999431,"close_bench":264.468048},{"date":"2019-06-25","open_a":106.794134,"high_a":107.192679,"low_a":106.443413,"close_a":106.65863,"volume_a":4176300,"open_b":47.423992,"high_b":47.62236,"low_b":46.673545,"close_b":46.740467,"volume_b":84281200,"open_bench":264.521978,"close_bench":261.874054},{"date":"2019-06-26","open_a":106.060809,"high_a":106.626748,"low_a":105.455025,"close_a":105.47097,"volume_a":6024100,"open_b":47.266257,"high_b":48.035825,"low_b":47.165879,"close_b":47.751419,"volume_b":104270000,"open_bench":262.765659,"close_bench":261.612823},{"date":"2019-06-27","open_a":105.415134,"high_a":105.566583,"low_a":104.721669,"close_a":104.968765,"volume_a":4750900,"open_b":47.868528,"high_b":48.174447,"low_b":47.696455,"close_b":47.737083,"volume_b":83598800,"open_bench":262.369431,"close_bench":262.540558},{"date":"2019-06-28","open_a":105.43911,"high_a":105.542736,"low_a":104.227529,"close_a":104.522461,"volume_a":6318500,"open_b":47.483741,"high_b":47.679719,"low_b":47.09418,"close_b":47.302105,"volume_b":124442400,"open_bench":263.513223,"close_bench":263.89151},{"date":"2019-07-01","open_a":105.207933,"high_a":105.447051,"low_a":104.474612,"close_a":105.112274,"volume_a":3544800,"open_b":48.556839,"high_b":48.872315,"low_b":47.954568,"close_b":48.169666,"volume_b":109012000,"open_bench":267.20598,"close_bench":266.287323},{"date":"2019-07-02","open_a":105.287632,"high_a":105.734,"low_a":104.697793,"close_a":105.55864,"volume_a":3233300,"open_b":48.136202,"high_b":48.547276,"low_b":48.124252,"close_b":48.451675,"volume_b":67740800,"open_bench":266.242201,"close_bench":266.980743},{"date":"2019-07-03","open_a":105.997055,"high_a":106.881826,"low_a":105.829673,"close_a":106.802124,"volume_a":2859200,"open_b":48.583117,"high_b":48.860353,"low_b":48.44211,"close_b":48.853184,"volume_b":45448000,"open_bench":267.656332,"close_bench":269.115387},{"date":"2019-07-05","open_a":105.925305,"high_a":106.403564,"low_a":104.857207,"close_a":106.028931,"volume_a":3356000,"open_b":48.599859,"high_b":49.013321,"low_b":48.492308,"close_b":48.810173,"volume_b":69062000,"open_bench":267.890425,"close_bench":268.809082},{"date":"2019-07-08","open_a":106.092705,"high_a":106.323856,"low_a":105.247779,"close_a":105.66227,"volume_a":5854500,"open_b":47.992802,"high_b":48.133809,"low_b":47.419213,"close_b":47.803997,"volume_b":101354400,"open_bench":267.503158,"close_bench":267.332031},{"date":"2019-07-09","open_a":106.012981,"high_a":106.331812,"low_a":104.195616,"close_a":105.008652,"volume_a":8233200,"open_b":47.608027,"high_b":48.160107,"low_b":47.514819,"close_b":48.095581,"volume_b":82312000,"open_bench":266.179271,"close_bench":267.665344},{"date":"2019-07-10","open_a":105.534743,"high_a":107.176759,"low_a":105.112286,"close_a":107.136902,"volume_a":6141500,"open_b":48.24137,"high_b":48.69068,"low_b":48.172059,"close_b":48.571182,"volume_b":71588400,"open_bench":268.727974,"close_bench":268.944122},{"date":"2019-07-11","open_a":106.977473,"high_a":107.39993,"low_a":106.291975,"close_a":107.089066,"volume_a":3944200,"open_b":48.590294,"high_b":48.84841,"low_b":48.207902,"close_b":48.217461,"volume_b":80767200,"open_bench":269.583722,"close_bench":269.574707},{"date":"2019-07-12","open_a":106.810089,"high_a":107.152832,"low_a":105.702134,"close_a":106.236183,"volume_a":4481200,"open_b":48.384754,"high_b":48.755199,"low_b":48.325005,"close_b":48.587902,"volume_b":70380800,"open_bench":270.060982,"close_bench":270.781494},{"date":"2019-07-15","open_a":106.491221,"high_a":106.93759,"low_a":105.654275,"close_a":106.929611,"volume_a":3077900,"open_b":48.776717,"high_b":49.20213,"low_b":48.755209,"close_b":49.044395,"volume_b":67789600,"open_bench":271.213896,"close_bench":270.871643},{"date":"2019-07-16","open_a":106.873857,"high_a":107.798485,"low_a":106.052855,"close_a":106.650673,"volume_a":4019700,"open_b":48.896207,"high_b":49.259482,"low_b":48.635703,"close_b":48.874699,"volume_b":67467200,"open_bench":270.781458,"close_bench":269.997894},{"date":"2019-07-17","open_a":106.746314,"high_a":107.033266,"low_a":105.590535,"close_a":105.837631,"volume_a":4105100,"open_b":48.76715,"high_b":49.015704,"low_b":48.580733,"close_b":48.599854,"volume_b":56430000,"open_bench":269.970937,"close_bench":268.160614},{"date":"2019-07-18","open_a":105.702125,"high_a":106.172406,"low_a":104.8333,"close_a":105.917343,"volume_a":3698600,"open_b":48.755201,"high_b":49.204515,"low_b":48.683502,"close_b":49.151936,"volume_b":74162400,"open_bench":267.665246,"close_bench":269.142303},{"date":"2019-07-19","open_a":106.148521,"high_a":106.164466,"low_a":103.629713,"close_a":103.693481,"volume_a":5022000,"open_b":49.183003,"high_b":49.352692,"low_b":48.363249,"close_b":48.418217,"volume_b":83717200,"open_bench":270.23215,"close_bench":267.647278},{"date":"2019-07-22","open_a":104.012273,"high_a":104.267336,"low_a":103.326776,"close_a":103.844879,"volume_a":4411700,"open_b":48.671566,"high_b":49.527173,"low_b":48.662008,"close_b":49.524784,"volume_b":89111600,"open_bench":268.043582,"close_bench":268.304779},{"date":"2019-07-23","open_a":104.522412,"high_a":105.407183,"low_a":103.948507,"close_a":104.211548,"volume_a":4145800,"open_b":49.821137,"high_b":49.928684,"low_b":49.541508,"close_b":49.911953,"volume_b":73420800,"open_bench":269.421607,"close_bench":270.223175},{"date":"2019-07-24","open_a":104.578221,"high_a":104.578221,"low_a":102.521727,"close_a":102.99202,"volume_a":4669700,"open_b":49.632326,"high_b":49.98604,"low_b":49.512828,"close_b":49.871323,"volume_b":59966400,"open_bench":269.466602,"close_bench":271.493073},{"date":"2019-07-25","open_a":102.52972,"high_a":103.247097,"low_a":102.338414,"close_a":102.816673,"volume_a":3335200,"open_b":49.923904,"high_b":50.007554,"low_b":49.407671,"close_b":49.476982,"volume_b":55638400,"open_bench":271.042723,"close_bench":270.196106},{"date":"2019-07-26","open_a":103.17535,"high_a":104.753585,"low_a":102.593478,"close_a":104.59417,"volume_a":4580800,"open_b":49.586907,"high_b":50.124648,"low_b":49.505649,"close_b":49.649048,"volume_b":70475600,"open_bench":270.880561,"close_bench":272.006378},{"date":"2019-07-29","open_a":104.817351,"high_a":105.26372,"low_a":104.562276,"close_a":104.841263,"volume_a":4533400,"open_b":49.821133,"high_b":50.342143,"low_b":49.816352,"close_b":50.112705,"volume_b":86693600,"open_bench":271.889395,"close_bench":271.511108},{"date":"2019-07-30","open_a":104.689839,"high_a":105.27969,"low_a":103.805069,"close_a":104.004341,"volume_a":3367900,"open_b":49.892825,"high_b":50.227422,"low_b":49.546281,"close_b":49.897606,"volume_b":135742800,"open_bench":270.114986,"close_bench":270.844513},{"date":"2019-07-31","open_a":104.004312,"high_a":104.004312,"low_a":100.624652,"close_a":101.876083,"volume_a":5757600,"open_b":51.723534,"high_b":52.906564,"low_b":50.499875,"close_b":50.915726,"volume_b":277125600,"open_bench":271.087733,"close_bench":267.881409},{"date":"2019-08-01","open_a":102.792759,"high_a":103.143468,"low_a":101.047123,"close_a":101.342049,"volume_a":6199400,"open_b":51.121266,"high_b":52.108321,"low_b":49.410057,"close_b":49.813957,"volume_b":216071600,"open_bench":268.03458,"close_bench":265.548767},{"date":"2019-08-02","open_a":101.310147,"high_a":102.681149,"low_a":100.831894,"close_a":101.96376,"volume_a":5573300,"open_b":49.120866,"high_b":49.335961,"low_b":48.188783,"close_b":48.759983,"volume_b":163448400,"open_bench":264.657069,"close_bench":263.549255},{"date":"2019-08-05","open_a":102.306522,"high_a":102.944184,"low_a":98.743522,"close_a":99.158012,"volume_a":6192800,"open_b":47.318842,"high_b":47.476577,"low_b":46.025872,"close_b":46.207508,"volume_b":209572000,"open_bench":259.469378,"close_bench":255.623596},{"date":"2019-08-06","open_a":99.460907,"high_a":101.230448,"low_a":99.142069,"close_a":100.792046,"volume_a":6258600,"open_b":46.917333,"high_b":47.337969,"low_b":46.374811,"close_b":47.082241,"volume_b":143299200,"open_bench":257.50603,"close_bench":259.208252},{"date":"2019-08-07","open_a":100.752176,"high_a":102.298526,"low_a":99.341323,"close_a":101.852158,"volume_a":6149000,"open_b":46.702233,"high_b":47.694065,"low_b":46.32223,"close_b":47.569786,"volume_b":133457600,"open_bench":256.145927,"close_bench":259.361267},{"date":"2019-08-08","open_a":102.107247,"high_a":103.422441,"low_a":101.230443,"close_a":103.199257,"volume_a":3702100,"open_b":47.84702,"high_b":48.642878,"low_b":47.653434,"close_b":48.618977,"volume_b":108038000,"open_bench":260.847336,"close_bench":264.449951},{"date":"2019-08-09","open_a":103.390561,"high_a":103.414473,"low_a":101.48551,"close_a":102.505791,"volume_a":2929500,"open_b":48.292708,"high_b":48.642966,"low_b":47.810498,"close_b":48.218338,"volume_b":98478800,"open_bench":263.51324,"close_bench":262.648621},{"date":"2019-08-12","open_a":102.386243,"high_a":102.944204,"low_a":101.923935,"close_a":102.306541,"volume_a":2920800,"open_b":47.889668,"high_b":48.472637,"low_b":47.776912,"close_b":48.095985,"volume_b":89927600,"open_bench":261.153515,"close_bench":259.451294},{"date":"2019-08-13","open_a":102.250724,"high_a":103.924618,"low_a":101.979716,"close_a":103.565918,"volume_a":4088200,"open_b":48.225526,"high_b":50.893259,"low_b":48.095976,"close_b":50.132763,"volume_b":188874000,"open_bench":259.154154,"close_bench":263.486298},{"date":"2019-08-14","open_a":103.239131,"high_a":104.044188,"low_a":102.665226,"close_a":102.920288,"volume_a":6232700,"open_b":48.738922,"high_b":49.525807,"low_b":48.602175,"close_b":48.64056,"volume_b":146189600,"open_bench":259.451325,"close_bench":255.695587},{"date":"2019-08-15","open_a":103.350704,"high_a":104.594163,"low_a":103.143453,"close_a":104.195618,"volume_a":3701800,"open_b":48.810901,"high_b":49.213939,"low_b":47.901662,"close_b":48.398266,"volume_b":108909600,"open_bench":256.578254,"close_bench":256.371094},{"date":"2019-08-16","open_a":104.753567,"high_a":105.470955,"low_a":104.131837,"close_a":105.024574,"volume_a":3405000,"open_b":49.007612,"high_b":49.698537,"low_b":48.902053,"close_b":49.540199,"volume_b":110481600,"open_bench":258.019234,"close_bench":260.153778},{"date":"2019-08-19","open_a":105.27169,"high_a":105.893419,"low_a":105.168064,"close_a":105.670235,"volume_a":3101200,"open_b":50.528604,"high_b":51.034802,"low_b":50.387062,"close_b":50.463833,"volume_b":97654400,"open_bench":263.162009,"close_bench":263.288086},{"date":"2019-08-20","open_a":105.853584,"high_a":106.092714,"low_a":105.160107,"close_a":105.263733,"volume_a":3160300,"open_b":50.59098,"high_b":51.183543,"low_b":50.456634,"close_b":50.466228,"volume_b":107537200,"open_bench":262.783691,"close_bench":261.270599},{"date":"2019-08-21","open_a":105.941248,"high_a":106.52312,"low_a":104.912994,"close_a":105.4869,"volume_a":2936900,"open_b":51.09719,"high_b":51.255524,"low_b":50.763723,"close_b":51.013222,"volume_b":86141600,"open_bench":263.423228,"close_bench":263.39621},{"date":"2019-08-22","open_a":105.016604,"high_a":105.957164,"low_a":104.673861,"close_a":105.741959,"volume_a":2484200,"open_b":51.145168,"high_b":51.445049,"low_b":50.559802,"close_b":50.970039,"volume_b":89014800,"open_bench":264.098686,"close_bench":263.315094},{"date":"2019-08-23","open_a":105.574603,"high_a":106.291992,"low_a":103.255068,"close_a":103.836952,"volume_a":4239500,"open_b":50.243129,"high_b":50.871681,"low_b":48.220739,"close_b":48.614182,"volume_b":187272000,"open_bench":262.018152,"close_bench":256.551178},{"date":"2019-08-26","open_a":104.275327,"high_a":106.610796,"low_a":104.091988,"close_a":106.602829,"volume_a":3954100,"open_b":49.386678,"high_b":49.705751,"low_b":49.194754,"close_b":49.537819,"volume_b":104174400,"open_bench":258.730818,"close_bench":259.388306},{"date":"2019-08-27","open_a":106.969502,"high_a":107.89413,"low_a":106.555012,"close_a":107.415871,"volume_a":7187800,"open_b":49.866481,"high_b":50.032015,"low_b":48.827695,"close_b":48.978836,"volume_b":103493200,"open_bench":260.775333,"close_bench":258.370575},{"date":"2019-08-28","open_a":107.208627,"high_a":107.87818,"low_a":107.033266,"close_a":107.758621,"volume_a":3972600,"open_b":48.964445,"high_b":49.353089,"low_b":48.77732,"close_b":49.307507,"volume_b":63755200,"open_bench":257.713113,"close_bench":260.189911},{"date":"2019-08-29","open_a":108.093387,"high_a":108.754973,"low_a":107.224561,"close_a":108.420197,"volume_a":3045200,"open_b":50.020015,"high_b":50.216738,"low_b":49.578592,"close_b":50.142365,"volume_b":83962000,"open_bench":262.738787,"close_bench":263.513336},{"date":"2019-08-30","open_a":108.491961,"high_a":109.289051,"low_a":108.212974,"close_a":108.986153,"volume_a":3742800,"open_b":50.418266,"high_b":50.487837,"low_b":49.708148,"close_b":50.077602,"volume_b":84573600,"open_bench":264.990357,"close_bench":263.39621},{"date":"2019-09-03","open_a":108.68323,"high_a":109.735383,"low_a":108.308597,"close_a":109.568001,"volume_a":2989800,"open_b":49.523407,"high_b":49.655355,"low_b":48.993221,"close_b":49.348278,"volume_b":80092000,"open_bench":261.703014,"close_bench":261.85611},{"date":"2019-09-04","open_a":109.846987,"high_a":110.938997,"low_a":109.400618,"close_a":110.915085,"volume_a":3201700,"open_b":49.993627,"high_b":50.255121,"low_b":49.736931,"close_b":50.185551,"volume_b":76752400,"open_bench":264.017573,"close_bench":264.828156},{"date":"2019-09-05","open_a":111.40067,"high_a":111.40067,"low_a":108.792225,"close_a":109.137337,"volume_a":4994600,"open_b":50.859675,"high_b":51.332286,"low_b":50.742121,"close_b":51.166752,"volume_b":95654800,"open_bench":267.305024,"close_bench":268.232697},{"date":"2019-09-06","open_a":109.899784,"high_a":110.293062,"low_a":109.169429,"close_a":110.25293,"volume_a":3051600,"open_b":51.351487,"high_b":51.44025,"low_b":50.982032,"close_b":51.161961,"volume_b":77449200,"open_bench":268.54786,"close_bench":268.439758},{"date":"2019-09-09","open_a":109.474403,"high_a":109.69913,"low_a":108.695882,"close_a":109.474403,"volume_a":4062100,"open_b":51.540996,"high_b":51.924844,"low_b":50.63656,"close_b":51.38026,"volume_b":109237600,"open_bench":269.421568,"close_bench":268.574951},{"date":"2019-09-10","open_a":108.944723,"high_a":109.562715,"low_a":108.093958,"close_a":109.041031,"volume_a":4175000,"open_b":51.305898,"high_b":52.006417,"low_b":50.790105,"close_b":51.987225,"volume_b":127111600,"open_bench":267.81832,"close_bench":268.511841},{"date":"2019-09-11","open_a":108.567508,"high_a":109.787449,"low_a":108.254495,"close_a":109.442337,"volume_a":5005100,"open_b":52.315906,"high_b":53.668966,"low_b":52.234336,"close_b":53.640175,"volume_b":177158400,"open_bench":268.818163,"close_bench":270.421326},{"date":"2019-09-12","open_a":109.747324,"high_a":110.991342,"low_a":109.747324,"close_a":110.188744,"volume_a":4146900,"open_b":53.930456,"high_b":54.319099,"low_b":53.465041,"close_b":53.520218,"volume_b":128906800,"open_bench":271.321998,"close_bench":271.358032},{"date":"2019-09-13","open_a":109.21761,"high_a":109.915867,"low_a":108.896563,"close_a":109.506546,"volume_a":4008200,"open_b":52.778918,"high_b":52.968441,"low_b":52.064005,"close_b":52.479038,"volume_b":159053200,"open_bench":271.799279,"close_bench":271.177826},{"date":"2019-09-16","open_a":109.466398,"high_a":109.554684,"low_a":107.853179,"close_a":108.254478,"volume_a":2953600,"open_b":52.234337,"high_b":52.810109,"low_b":52.193554,"close_b":52.754929,"volume_b":84632400,"open_bench":270.051964,"close_bench":270.340179},{"date":"2019-09-17","open_a":108.318721,"high_a":109.402221,"low_a":107.957541,"close_a":108.912636,"volume_a":4013300,"open_b":52.769315,"high_b":52.975633,"low_b":52.567793,"close_b":52.946842,"volume_b":73274800,"open_bench":270.142099,"close_bench":271.02475},{"date":"2019-09-18","open_a":109.000894,"high_a":109.15339,"low_a":107.628458,"close_a":108.75209,"volume_a":2647800,"open_b":53.033217,"high_b":53.462648,"low_b":52.644573,"close_b":53.443455,"volume_b":101360000,"open_bench":270.637385,"close_bench":271.186798},{"date":"2019-09-19","open_a":108.67984,"high_a":109.024951,"low_a":108.414981,"close_a":108.736015,"volume_a":2804500,"open_b":53.261117,"high_b":53.680949,"low_b":52.867674,"close_b":53.00922,"volume_b":88242400,"open_bench":271.574098,"close_bench":271.168793},{"date":"2019-09-20","open_a":108.350775,"high_a":108.960745,"low_a":107.93342,"close_a":108.118027,"volume_a":6671100,"open_b":53.109993,"high_b":53.393078,"low_b":52.171967,"close_b":52.234341,"volume_b":221652400,"open_bench":271.769603,"close_bench":269.887604},{"date":"2019-09-23","open_a":108.318705,"high_a":108.952753,"low_a":108.318705,"close_a":108.703949,"volume_a":3346500,"open_b":52.527005,"high_b":52.74052,"low_b":52.215129,"close_b":52.471828,"volume_b":76662000,"open_bench":269.227069,"close_bench":269.824249},{"date":"2019-09-24","open_a":109.354045,"high_a":109.514562,"low_a":108.013707,"close_a":108.639732,"volume_a":4798500,"open_b":53.026012,"high_b":53.376274,"low_b":52.104781,"close_b":52.222332,"volume_b":124763200,"open_bench":270.910077,"close_bench":267.707031},{"date":"2019-09-25","open_a":108.519355,"high_a":108.880522,"low_a":107.957526,"close_a":108.423035,"volume_a":3579500,"open_b":52.431051,"high_b":53.138768,"low_b":52.092785,"close_b":53.026012,"volume_b":87613600,"open_bench":267.788443,"close_bench":269.290436},{"date":"2019-09-26","open_a":109.049027,"high_a":109.418228,"low_a":108.358792,"close_a":108.928642,"volume_a":4309600,"open_b":52.778915,"high_b":53.004425,"low_b":52.498228,"close_b":52.752525,"volume_b":75334000,"open_bench":269.299468,"close_bench":268.729431},{"date":"2019-09-27","open_a":109.34603,"high_a":109.34603,"low_a":108.085944,"close_a":108.832367,"volume_a":3559400,"open_b":52.908462,"high_b":53.009225,"low_b":52.126376,"close_b":52.495831,"volume_b":101408000,"open_bench":269.480396,"close_bench":267.281708},{"date":"2019-09-30","open_a":109.209564,"high_a":110.541868,"low_a":109.049035,"close_a":110.03624,"volume_a":3705200,"open_b":52.994835,"high_b":53.877684,"low_b":52.968445,"close_b":53.731342,"volume_b":103909600,"open_bench":267.797492,"close_bench":268.521332},{"date":"2019-10-01","open_a":109.899784,"high_a":110.710417,"low_a":109.225605,"close_a":110.25293,"volume_a":3638900,"open_b":53.99523,"high_b":54.750927,"low_b":53.786511,"close_b":53.880074,"volume_b":139223200,"open_bench":269.399021,"close_bench":265.327362},{"date":"2019-10-02","open_a":109.867701,"high_a":110.260979,"low_a":107.138871,"close_a":107.500038,"volume_a":5728600,"open_b":53.513012,"high_b":53.637763,"low_b":52.282303,"close_b":52.529408,"volume_b":138449200,"open_bench":263.752908,"close_bench":260.64035},{"date":"2019-10-03","open_a":111.111733,"high_a":112.163135,"low_a":109.55469,"close_a":110.7024,"volume_a":8829700,"open_b":52.402257,"high_b":53.009218,"low_b":51.610577,"close_b":52.975632,"volume_b":114426000,"open_bench":260.414181,"close_bench":262.775757},{"date":"2019-10-04","open_a":110.630159,"high_a":112.724941,"low_a":110.453586,"close_a":112.588501,"volume_a":5278700,"open_b":54.13198,"high_b":54.575804,"low_b":53.712148,"close_b":54.460648,"volume_b":138478800,"open_bench":263.427193,"close_bench":266.331635},{"date":"2019-10-07","open_a":111.79393,"high_a":112.050767,"low_a":110.734506,"close_a":111.031464,"volume_a":4273300,"open_b":54.283115,"high_b":55.161161,"low_b":54.179954,"close_b":54.472637,"volume_b":122306000,"open_bench":265.535416,"close_bench":265.182526},{"date":"2019-10-08","open_a":110.477677,"high_a":111.119758,"low_a":110.020202,"close_a":110.148621,"volume_a":4074900,"open_b":54.175159,"high_b":54.712542,"low_b":53.8177,"close_b":53.834492,"volume_b":111820000,"open_bench":263.336763,"close_bench":261.065674},{"date":"2019-10-09","open_a":110.212823,"high_a":110.895024,"low_a":109.819545,"close_a":110.638199,"volume_a":3065300,"open_b":54.465435,"high_b":54.647761,"low_b":54.131968,"close_b":54.465435,"volume_b":74770400,"open_bench":263.074399,"close_bench":263.544891},{"date":"2019-10-10","open_a":110.04427,"high_a":110.959232,"low_a":109.667048,"close_a":110.670296,"volume_a":2191000,"open_b":54.681364,"high_b":55.283526,"low_b":54.530227,"close_b":55.199558,"volume_b":113013600,"open_bench":263.463449,"close_bench":265.327362},{"date":"2019-10-11","open_a":110.822773,"high_a":111.32038,"low_a":110.028196,"close_a":110.469627,"volume_a":2761400,"open_b":55.885673,"high_b":57.010824,"low_b":55.732134,"close_b":56.667763,"volume_b":166795600,"open_bench":268.069007,"close_bench":268.078064},{"date":"2019-10-14","open_a":110.630174,"high_a":110.999375,"low_a":110.148623,"close_a":110.317162,"volume_a":2371500,"open_b":56.35349,"high_b":57.128383,"low_b":56.298313,"close_b":56.586197,"volume_b":96427600,"open_bench":267.761336,"close_bench":267.779449},{"date":"2019-10-15","open_a":110.341234,"high_a":110.718457,"low_a":109.281811,"close_a":109.418251,"volume_a":3369300,"open_b":56.710946,"high_b":57.013224,"low_b":56.348692,"close_b":56.45425,"volume_b":87360000,"open_bench":268.820006,"close_bench":270.430573},{"date":"2019-10-16","open_a":109.265752,"high_a":109.546666,"low_a":108.816299,"close_a":109.490479,"volume_a":2812000,"open_b":55.986426,"high_b":56.435049,"low_b":55.945642,"close_b":56.22633,"volume_b":73903200,"open_bench":269.969042,"close_bench":269.996185},{"date":"2019-10-17","open_a":110.012191,"high_a":110.45361,"low_a":109.634968,"close_a":109.923904,"volume_a":2322500,"open_b":56.399066,"high_b":56.653364,"low_b":56.022419,"close_b":56.444649,"volume_b":67585200,"open_bench":271.154278,"close_bench":270.792358},{"date":"2019-10-18","open_a":110.172696,"high_a":110.252948,"low_a":109.337987,"close_a":109.354042,"volume_a":3630400,"open_b":56.279109,"high_b":56.996424,"low_b":56.207137,"close_b":56.715736,"volume_b":97433600,"open_bench":270.258522,"close_bench":269.607056},{"date":"2019-10-21","open_a":109.42626,"high_a":109.747307,"low_a":108.880499,"close_a":109.249687,"volume_a":2829400,"open_b":56.982034,"high_b":57.814501,"low_b":56.934053,"close_b":57.699345,"volume_b":87247200,"open_bench":270.919123,"close_bench":271.434845},{"date":"2019-10-22","open_a":109.675079,"high_a":110.236908,"low_a":109.402199,"close_a":109.683113,"volume_a":3248100,"open_b":57.855294,"high_b":58.104793,"low_b":57.485839,"close_b":57.56741,"volume_b":82293600,"open_bench":271.968662,"close_bench":270.548126},{"date":"2019-10-23","open_a":109.642982,"high_a":110.357294,"low_a":109.642982,"close_a":110.317162,"volume_a":2323600,"open_b":58.080785,"high_b":58.354275,"low_b":57.869668,"close_b":58.339878,"volume_b":75828800,"open_bench":270.294828,"close_bench":271.335358},{"date":"2019-10-24","open_a":110.5178,"high_a":110.846868,"low_a":109.787433,"close_a":110.670296,"volume_a":2292500,"open_b":58.658965,"high_b":58.728539,"low_b":58.011224,"close_b":58.435856,"volume_b":69275200,"open_bench":272.267263,"close_bench":271.778656},{"date":"2019-10-25","open_a":110.581992,"high_a":110.694355,"low_a":109.337962,"close_a":109.66703,"volume_a":2520000,"open_b":58.335084,"high_b":59.19154,"low_b":58.267911,"close_b":59.155556,"volume_b":73477200,"open_bench":271.208665,"close_bench":272.891632},{"date":"2019-10-28","open_a":109.594823,"high_a":110.397421,"low_a":109.273776,"close_a":109.610878,"volume_a":3212600,"open_b":59.35709,"high_b":59.796115,"low_b":59.189157,"close_b":59.748135,"volume_b":96572800,"open_bench":274.10409,"close_bench":274.42981},{"date":"2019-10-29","open_a":109.819562,"high_a":110.05231,"low_a":109.10525,"close_a":109.217613,"volume_a":2550300,"open_b":59.728941,"high_b":59.916066,"low_b":58.193556,"close_b":58.366283,"volume_b":142839600,"open_bench":274.158294,"close_bench":274.348297},{"date":"2019-10-30","open_a":109.041028,"high_a":110.188738,"low_a":108.679861,"close_a":109.875725,"volume_a":2653700,"open_b":58.718937,"high_b":58.848488,"low_b":57.867281,"close_b":58.359081,"volume_b":124522000,"open_bench":274.547382,"close_bench":275.189819},{"date":"2019-10-31","open_a":109.795465,"high_a":110.381358,"low_a":109.233649,"close_a":110.092422,"volume_a":4492300,"open_b":59.313903,"high_b":59.776916,"low_b":56.919657,"close_b":59.678555,"volume_b":139162000,"open_bench":275.180715,"close_bench":274.456848},{"date":"2019-11-01","open_a":110.39743,"high_a":111.031478,"low_a":109.715217,"close_a":109.899811,"volume_a":3054900,"open_b":59.865679,"high_b":61.398666,"low_b":59.774518,"close_b":61.37228,"volume_b":151125200,"open_bench":275.895578,"close_bench":276.999451},{"date":"2019-11-04","open_a":109.795479,"high_a":110.036261,"low_a":108.134094,"close_a":108.479218,"volume_a":3202400,"open_b":61.734546,"high_b":61.859301,"low_b":61.266738,"close_b":61.775333,"volume_b":103272000,"open_bench":278.546594,"close_bench":278.112274},{"date":"2019-11-05","open_a":108.214335,"high_a":108.43104,"low_a":106.424544,"close_a":107.556198,"volume_a":4300800,"open_b":61.667365,"high_b":61.940859,"low_b":61.49224,"close_b":61.686562,"volume_b":79897600,"open_bench":278.311503,"close_bench":277.80481},{"date":"2019-11-06","open_a":107.949498,"high_a":108.25449,"low_a":107.187032,"close_a":107.901344,"volume_a":3645600,"open_b":61.600181,"high_b":61.772912,"low_b":61.264317,"close_b":61.712936,"volume_b":75864400,"open_bench":277.804698,"close_bench":277.868042},{"date":"2019-11-07","open_a":107.877248,"high_a":107.949479,"low_a":106.841889,"close_a":107.002419,"volume_a":3338700,"open_b":62.259162,"high_b":62.646571,"low_b":62.107568,"close_b":62.425194,"volume_b":94940400,"open_bench":279.198105,"close_bench":278.845215},{"date":"2019-11-08","open_a":106.978352,"high_a":107.652544,"low_a":106.400492,"close_a":106.849945,"volume_a":2604000,"open_b":62.247138,"high_b":62.668231,"low_b":61.80439,"close_b":62.596046,"volume_b":69986400,"open_bench":278.501368,"close_bench":279.532867},{"date":"2019-11-11","open_a":106.769667,"high_a":106.769667,"low_a":105.918914,"close_a":106.256004,"volume_a":2227200,"open_b":62.153283,"high_b":63.15669,"low_b":62.148473,"close_b":63.091724,"volume_b":81821200,"open_bench":278.157585,"close_bench":278.999054},{"date":"2019-11-12","open_a":106.488757,"high_a":106.488757,"low_a":105.613928,"close_a":105.926941,"volume_a":3246500,"open_b":62.935324,"high_b":63.233704,"low_b":62.783737,"close_b":63.033981,"volume_b":87388800,"open_bench":279.361016,"close_bench":279.587219},{"date":"2019-11-13","open_a":106.448642,"high_a":107.203087,"low_a":105.951023,"close_a":106.978348,"volume_a":3166600,"open_b":62.834259,"high_b":63.712536,"low_b":62.819822,"close_b":63.637943,"volume_b":102734400,"open_bench":278.600883,"close_bench":279.677612},{"date":"2019-11-14","open_a":107.227165,"high_a":107.475969,"low_a":106.528896,"close_a":107.01046,"volume_a":2141900,"open_b":63.464672,"high_b":63.736578,"low_b":63.067643,"close_b":63.197582,"volume_b":89182800,"open_bench":279.3972,"close_bench":280.084839},{"date":"2019-11-15","open_a":107.275291,"high_a":107.483962,"low_a":106.585056,"close_a":107.395676,"volume_a":3648700,"open_b":63.447842,"high_b":63.953155,"low_b":63.286628,"close_b":63.948345,"volume_b":100206400,"open_bench":281.414971,"close_bench":282.111694},{"date":"2019-11-18","open_a":107.54819,"high_a":108.904583,"low_a":107.475959,"close_a":107.596344,"volume_a":3619700,"open_b":63.957966,"high_b":64.350185,"low_b":63.580192,"close_b":64.270782,"volume_b":86703200,"open_bench":281.876444,"close_bench":282.319794},{"date":"2019-11-19","open_a":108.069881,"high_a":108.398949,"low_a":107.604385,"close_a":108.005684,"volume_a":3019100,"open_b":64.463268,"high_b":64.487332,"low_b":63.859306,"close_b":64.075867,"volume_b":76167200,"open_bench":282.916952,"close_bench":282.238342},{"date":"2019-11-20","open_a":107.796998,"high_a":108.51131,"low_a":107.066631,"close_a":107.339523,"volume_a":3047700,"open_b":63.895417,"high_b":64.025349,"low_b":62.658604,"close_b":63.329948,"volume_b":106234400,"open_bench":281.650154,"close_bench":281.18869},{"date":"2019-11-21","open_a":107.090696,"high_a":107.716734,"low_a":106.817816,"close_a":107.475952,"volume_a":3391200,"open_b":63.450253,"high_b":63.527254,"low_b":62.846283,"close_b":63.046005,"volume_b":121395200,"open_bench":281.297305,"close_bench":280.736298},{"date":"2019-11-22","open_a":107.484003,"high_a":107.973589,"low_a":106.76167,"close_a":107.604401,"volume_a":3014900,"open_b":63.185563,"high_b":63.327531,"low_b":62.76447,"close_b":62.990658,"volume_b":65325200,"open_bench":281.478195,"close_bench":281.360565},{"date":"2019-11-25","open_a":108.077901,"high_a":108.174209,"low_a":106.914136,"close_a":107.411743,"volume_a":3612400,"open_b":63.214437,"high_b":64.111969,"low_b":63.168718,"close_b":64.095123,"volume_b":84020400,"open_bench":282.283614,"close_bench":283.54129},{"date":"2019-11-26","open_a":107.780969,"high_a":109.049076,"low_a":107.443879,"close_a":108.912636,"volume_a":4947400,"open_b":64.232277,"high_b":64.285214,"low_b":63.163904,"close_b":63.594624,"volume_b":105207600,"open_bench":283.577385,"close_bench":284.183594},{"date":"2019-11-27","open_a":108.912618,"high_a":109.570755,"low_a":108.615661,"close_a":109.081169,"volume_a":2476100,"open_b":63.905039,"high_b":64.482543,"low_b":63.840073,"close_b":64.448853,"volume_b":65235600,"open_bench":284.663168,"close_bench":285.450378},{"date":"2019-11-29","open_a":109.201562,"high_a":109.490498,"low_a":108.559481,"close_a":109.016968,"volume_a":2978400,"open_b":64.15048,"high_b":64.487353,"low_b":63.98204,"close_b":64.306885,"volume_b":46617600,"open_bench":284.88939,"close_bench":284.391754},{"date":"2019-12-02","open_a":109.675083,"high_a":109.675083,"low_a":107.901347,"close_a":109.057091,"volume_a":3866300,"open_b":64.311678,"high_b":64.547492,"low_b":63.392498,"close_b":63.563339,"volume_b":94487200,"open_bench":284.645165,"close_bench":281.975983},{"date":"2019-12-03","open_a":109.683094,"high_a":109.683094,"low_a":107.805017,"close_a":108.719978,"volume_a":5000000,"open_b":62.1557,"high_b":62.449262,"low_b":61.669641,"close_b":62.430016,"volume_b":114430400,"open_bench":279.270513,"close_bench":280.084839},{"date":"2019-12-04","open_a":108.310636,"high_a":110.252922,"low_a":107.981568,"close_a":110.196747,"volume_a":4228200,"open_b":62.819827,"high_b":63.358823,"low_b":62.72598,"close_b":62.981041,"volume_b":67181600,"open_bench":281.12541,"close_bench":281.813049},{"date":"2019-12-05","open_a":110.07149,"high_a":110.35436,"low_a":109.327929,"close_a":110.192719,"volume_a":3351800,"open_b":63.474317,"high_b":63.97963,"low_b":63.219256,"close_b":63.905029,"volume_b":74424400,"open_bench":282.509824,"close_bench":282.319794},{"date":"2019-12-06","open_a":110.459424,"high_a":111.283812,"low_a":110.19271,"close_a":111.017097,"volume_a":3216900,"open_b":64.362238,"high_b":65.209233,"low_b":64.31892,"close_b":65.13945,"volume_b":106075600,"open_bench":284.219889,"close_bench":284.898499},{"date":"2019-12-09","open_a":110.847356,"high_a":111.566671,"low_a":110.726128,"close_a":110.855446,"volume_a":3882900,"open_b":64.968567,"high_b":65.161064,"low_b":63.74379,"close_b":64.227448,"volume_b":128042400,"open_bench":284.50944,"close_bench":284.002747},{"date":"2019-12-10","open_a":111.219182,"high_a":111.219182,"low_a":110.055342,"close_a":110.136169,"volume_a":2940200,"open_b":64.631703,"high_b":64.985421,"low_b":63.972387,"close_b":64.602829,"volume_b":90420400,"open_bench":283.948439,"close_bench":283.686035},{"date":"2019-12-11","open_a":110.766548,"high_a":111.162567,"low_a":110.17655,"close_a":110.806961,"volume_a":3339100,"open_b":64.68224,"high_b":65.233272,"low_b":64.607647,"close_b":65.153862,"volume_b":78756800,"open_bench":284.138443,"close_bench":284.491333},{"date":"2019-12-12","open_a":110.564528,"high_a":111.453563,"low_a":110.524114,"close_a":110.871643,"volume_a":3502100,"open_b":64.434412,"high_b":65.584596,"low_b":64.323727,"close_b":65.319908,"volume_b":137310400,"open_bench":284.500262,"close_bench":286.943268},{"date":"2019-12-13","open_a":110.669575,"high_a":111.736432,"low_a":110.0715,"close_a":111.485886,"volume_a":3557900,"open_b":65.319897,"high_b":66.243894,"low_b":65.192367,"close_b":66.207802,"volume_b":133587600,"open_bench":286.708149,"close_bench":287.115326},{"date":"2019-12-16","open_a":110.685746,"high_a":111.146437,"low_a":110.047257,"close_a":110.774651,"volume_a":4701000,"open_b":66.652966,"high_b":67.564935,"low_b":66.648156,"close_b":67.341148,"volume_b":128186000,"open_bench":288.834483,"close_bench":289.08783},{"date":"2019-12-17","open_a":110.766566,"high_a":111.081771,"low_a":109.667386,"close_a":110.055328,"volume_a":5491800,"open_b":67.271364,"high_b":67.800734,"low_b":67.086079,"close_b":67.473488,"volume_b":114158400,"open_bench":289.467813,"close_bench":289.151123},{"date":"2019-12-18","open_a":110.766571,"high_a":110.968626,"low_a":109.6189,"close_a":109.893692,"volume_a":5370500,"open_b":67.3267,"high_b":67.832013,"low_b":67.163078,"close_b":67.312263,"volume_b":116028400,"open_bench":289.540197,"close_bench":289.16922},{"date":"2019-12-19","open_a":110.095751,"high_a":110.516029,"low_a":109.877528,"close_a":110.297806,"volume_a":3641700,"open_b":67.254509,"high_b":67.658756,"low_b":67.122169,"close_b":67.379631,"volume_b":98369200,"open_bench":289.359192,"close_bench":290.354492},{"date":"2019-12-20","open_a":111.057539,"high_a":111.639458,"low_a":109.239043,"close_a":111.469727,"volume_a":10311300,"open_b":67.911434,"high_b":68.012492,"low_b":67.028339,"close_b":67.240089,"volume_b":275978000,"open_bench":291.381954,"close_bench":291.627472},{"date":"2019-12-23","open_a":111.283811,"high_a":111.63943,"low_a":110.596819,"close_a":110.758461,"volume_a":2992900,"open_b":67.502359,"high_b":68.397482,"low_b":67.463858,"close_b":68.337326,"volume_b":98572000,"open_bench":292.409329,"close_bench":292.072906},{"date":"2019-12-24","open_a":110.79891,"high_a":111.0333,"low_a":110.136175,"close_a":110.637268,"volume_a":1220600,"open_b":68.503356,"high_b":68.551484,"low_b":68.077453,"close_b":68.40229,"volume_b":48478800,"open_bench":292.300306,"close_bench":292.082092},{"date":"2019-12-26","open_a":110.612989,"high_a":111.000932,"low_a":109.998747,"close_a":110.475594,"volume_a":2934700,"open_b":68.53463,"high_b":69.776253,"low_b":68.505756,"close_b":69.759407,"volume_b":93121200,"open_bench":292.463886,"close_bench":293.636841},{"date":"2019-12-27","open_a":110.629174,"high_a":111.194926,"low_a":110.427119,"close_a":111.16259,"volume_a":2554000,"open_b":70.050574,"high_b":70.736355,"low_b":69.328701,"close_b":69.732948,"volume_b":146266000,"open_bench":294.364333,"close_bench":293.564178},{"date":"2019-12-30","open_a":110.483674,"high_a":111.510117,"low_a":110.305864,"close_a":110.572578,"volume_a":3024700,"open_b":69.651134,"high_b":70.428354,"low_b":68.630889,"close_b":70.14682,"volume_b":144114400,"open_bench":293.646021,"close_bench":291.945679},{"date":"2019-12-31","open_a":110.475595,"high_a":110.936274,"low_a":109.942167,"close_a":110.459427,"volume_a":3229400,"open_b":69.76424,"high_b":70.666581,"low_b":69.665583,"close_b":70.659363,"volume_b":100805600,"open_bench":291.445571,"close_bench":292.654877},{"date":"2020-01-02","open_a":110.621059,"high_a":110.912019,"low_a":109.222842,"close_a":109.772438,"volume_a":3784100,"open_b":71.282575,"high_b":72.331702,"low_b":71.029923,"close_b":72.271545,"volume_b":135480400,"open_bench":294.182418,"close_bench":295.391724},{"date":"2020-01-03","open_a":109.481516,"high_a":110.556438,"low_a":109.214802,"close_a":109.618912,"volume_a":4000100,"open_b":71.501534,"high_b":72.326874,"low_b":71.34513,"close_b":71.568909,"volume_b":146322800,"open_bench":292.018422,"close_bench":293.154999},{"date":"2020-01-06","open_a":109.35219,"high_a":110.176578,"low_a":109.271363,"close_a":110.039169,"volume_a":4085100,"open_b":70.693036,"high_b":72.177684,"low_b":70.442784,"close_b":72.139183,"volume_b":118387200,"open_bench":291.409241,"close_bench":294.273438},{"date":"2020-01-07","open_a":109.91794,"high_a":109.974522,"low_a":108.261087,"close_a":108.309578,"volume_a":5718100,"open_b":72.148835,"high_b":72.403897,"low_b":71.580965,"close_b":71.799934,"volume_b":108872000,"open_bench":293.70968,"close_bench":293.446014},{"date":"2020-01-08","open_a":108.673284,"high_a":109.400677,"low_a":108.358078,"close_a":108.867249,"volume_a":3681400,"open_b":71.503931,"high_b":73.255675,"low_b":71.503931,"close_b":72.954895,"volume_b":132079200,"open_bench":293.636924,"close_bench":295.009918},{"date":"2020-01-09","open_a":109.344074,"high_a":109.416823,"low_a":108.729832,"close_a":108.939964,"volume_a":3954700,"open_b":73.929451,"high_b":74.697044,"low_b":73.679207,"close_b":74.504547,"volume_b":170108400,"open_bench":296.564847,"close_bench":297.010376},{"date":"2020-01-10","open_a":109.247103,"high_a":109.449158,"low_a":108.616693,"close_a":108.729843,"volume_a":3099500,"open_b":74.737947,"high_b":75.236041,"low_b":74.172478,"close_b":74.672974,"volume_b":140644800,"open_bench":297.592197,"close_bench":296.155548},{"date":"2020-01-13","open_a":108.826825,"high_a":110.394774,"low_a":108.81067,"close_a":110.346283,"volume_a":5127200,"open_b":74.988202,"high_b":76.294791,"low_b":74.870291,"close_b":76.268318,"volume_b":121532000,"open_bench":296.773937,"close_bench":298.192383},{"date":"2020-01-14","open_a":110.063409,"high_a":110.677664,"low_a":109.675467,"close_a":110.604927,"volume_a":4159700,"open_b":76.205749,"high_b":76.415091,"low_b":75.11572,"close_b":75.238434,"volume_b":161954400,"open_bench":297.755846,"close_bench":297.737671},{"date":"2020-01-15","open_a":110.758476,"high_a":112.544648,"low_a":110.564511,"close_a":112.504234,"volume_a":5002600,"open_b":75.038714,"high_b":75.916991,"low_b":74.485273,"close_b":74.915993,"volume_b":121923600,"open_bench":297.646777,"close_bench":298.410553},{"date":"2020-01-16","open_a":112.625461,"high_a":113.280118,"low_a":112.019296,"close_a":112.835594,"volume_a":4977900,"open_b":75.45741,"high_b":75.965131,"low_b":75.096473,"close_b":75.854439,"volume_b":108829200,"open_bench":299.783522,"close_bench":300.892822},{"date":"2020-01-17","open_a":113.231656,"high_a":114.217685,"low_a":113.070002,"close_a":114.169182,"volume_a":6908700,"open_b":76.10228,"high_b":76.696623,"low_b":75.79669,"close_b":76.694221,"volume_b":137816400,"open_bench":301.602061,"close_bench":301.829376},{"date":"2020-01-21","open_a":113.643839,"high_a":114.864247,"low_a":113.425616,"close_a":114.654114,"volume_a":4440000,"open_b":76.323663,"high_b":76.764002,"low_b":76.037319,"close_b":76.174477,"volume_b":110843200,"open_bench":300.874582,"close_bench":301.238281},{"date":"2020-01-22","open_a":114.61368,"high_a":116.036155,"low_a":114.540943,"close_a":115.882591,"volume_a":5366100,"open_b":76.658124,"high_b":76.997405,"low_b":76.352534,"close_b":76.446381,"volume_b":101832400,"open_bench":302.093049,"close_bench":301.274719},{"date":"2020-01-23","open_a":115.648176,"high_a":116.262431,"low_a":115.066256,"close_a":116.100777,"volume_a":3915700,"open_b":76.49932,"high_b":76.89394,"low_b":75.953098,"close_b":76.814537,"volume_b":104472000,"open_bench":300.629148,"close_bench":301.620239},{"date":"2020-01-24","open_a":116.246309,"high_a":116.504933,"low_a":115.130962,"close_a":115.510826,"volume_a":2791500,"open_b":77.059975,"high_b":77.801095,"low_b":76.403067,"close_b":76.593163,"volume_b":146537600,"open_bench":302.274806,"close_bench":298.937805},{"date":"2020-01-27","open_a":114.484377,"high_a":115.866439,"low_a":113.999452,"close_a":114.880409,"volume_a":3895300,"open_b":74.608009,"high_b":75.019475,"low_b":73.361576,"close_b":74.340919,"volume_b":161940000,"open_bench":293.718734,"close_bench":294.146088},{"date":"2020-01-28","open_a":114.87234,"high_a":115.510829,"low_a":114.573291,"close_a":115.122887,"volume_a":2908300,"open_b":75.219185,"high_b":76.614803,"low_b":75.120528,"close_b":76.443962,"volume_b":162234000,"open_bench":295.564491,"close_bench":297.228455},{"date":"2020-01-29","open_a":114.767234,"high_a":115.130931,"low_a":114.09641,"close_a":114.549011,"volume_a":3263200,"open_b":78.070593,"high_b":78.888715,"low_b":77.331875,"close_b":78.044121,"volume_b":216229200,"open_bench":298.58328,"close_bench":296.982971},{"date":"2020-01-30","open_a":115.025877,"high_a":116.351357,"low_a":114.379298,"close_a":116.246284,"volume_a":4058500,"open_b":77.129732,"high_b":77.983945,"low_b":76.699012,"close_b":77.931007,"volume_b":126743200,"open_bench":294.928053,"close_bench":297.946808},{"date":"2020-01-31","open_a":116.230141,"high_a":116.909043,"low_a":114.419735,"close_a":114.783432,"volume_a":4753100,"open_b":77.223589,"high_b":77.644682,"low_b":74.1821,"close_b":74.475662,"volume_b":199588400,"open_bench":297.328511,"close_bench":292.536713},{"date":"2020-02-03","open_a":115.389596,"high_a":115.632065,"low_a":114.637945,"close_a":115.179451,"volume_a":3977800,"open_b":73.222,"high_b":75.433338,"low_b":72.721504,"close_b":74.271126,"volume_b":173788400,"open_bench":294.009717,"close_bench":294.709839},{"date":"2020-02-04","open_a":115.73712,"high_a":116.43219,"low_a":114.848086,"close_a":115.753288,"volume_a":3494300,"open_b":75.871301,"high_b":76.913209,"low_b":75.467054,"close_b":76.723114,"volume_b":136616400,"open_bench":298.30147,"close_bench":299.20163},{"date":"2020-02-05","open_a":116.12504,"high_a":116.860523,"low_a":115.922985,"close_a":116.626144,"volume_a":3504900,"open_b":77.846781,"high_b":78.14516,"low_b":76.747133,"close_b":77.348694,"volume_b":118826800,"open_bench":302.120338,"close_bench":302.656799},{"date":"2020-02-06","open_a":116.900922,"high_a":117.43435,"low_a":116.440243,"close_a":116.650375,"volume_a":3571400,"open_b":77.618242,"high_b":78.255895,"low_b":77.0624,"close_b":78.253487,"volume_b":105425600,"open_bench":303.611485,"close_bench":303.67514},{"date":"2020-02-07","open_a":116.593834,"high_a":117.773841,"low_a":116.246306,"close_a":117.490959,"volume_a":3138500,"open_b":77.754178,"high_b":78.002609,"low_b":76.700155,"close_b":77.189781,"volume_b":117684000,"open_bench":302.620439,"close_bench":302.056702},{"date":"2020-02-10","open_a":117.943552,"high_a":118.339585,"low_a":117.296986,"close_a":117.725342,"volume_a":3775600,"open_b":75.778802,"high_b":77.556412,"low_b":75.699211,"close_b":77.556412,"volume_b":109348800,"open_bench":301.17471,"close_bench":304.311646},{"date":"2020-02-11","open_a":118.169846,"high_a":118.291074,"low_a":117.086822,"close_a":118.064774,"volume_a":3870500,"open_b":78.050853,"high_b":78.123209,"low_b":76.871404,"close_b":77.088478,"volume_b":94323200,"open_bench":305.657319,"close_bench":304.838989},{"date":"2020-02-12","open_a":118.549698,"high_a":118.598202,"low_a":117.474776,"close_a":118.064774,"volume_a":4217400,"open_b":77.537106,"high_b":78.92398,"low_b":77.537106,"close_b":78.919159,"volume_b":113730400,"open_bench":306.26652,"close_bench":306.803009},{"date":"2020-02-13","open_a":117.39396,"high_a":118.913418,"low_a":115.995743,"close_a":118.37999,"volume_a":5041400,"open_b":78.193167,"high_b":78.682793,"low_b":77.990563,"close_b":78.357178,"volume_b":94747600,"open_bench":305.384461,"close_bench":306.475586},{"date":"2020-02-14","open_a":118.460796,"high_a":118.929553,"low_a":117.951614,"close_a":118.800247,"volume_a":3555600,"open_b":78.325816,"high_b":78.624904,"low_b":77.869961,"close_b":78.376472,"volume_b":80113600,"open_bench":306.884814,"close_bench":306.966644},{"date":"2020-02-18","open_a":118.897273,"high_a":118.97001,"low_a":117.515211,"close_a":117.676865,"volume_a":4938700,"open_b":76.063405,"high_b":77.122257,"low_b":75.882509,"close_b":76.94136,"volume_b":152531200,"open_bench":305.975621,"close_bench":306.175659},{"date":"2020-02-19","open_a":117.692991,"high_a":118.388061,"low_a":117.587931,"close_a":117.620255,"volume_a":2978800,"open_b":77.182545,"high_b":78.28481,"low_b":77.182545,"close_b":78.055672,"volume_b":93984000,"open_bench":307.139383,"close_bench":307.639465},{"date":"2020-02-20","open_a":117.814235,"high_a":117.838481,"low_a":116.343269,"close_a":117.32122,"volume_a":2805900,"open_b":77.816903,"high_b":78.304115,"low_b":76.750816,"close_b":77.254913,"volume_b":100566000,"open_bench":307.093999,"close_bench":306.375702},{"date":"2020-02-21","open_a":117.240405,"high_a":117.935475,"low_a":117.03835,"close_a":117.878906,"volume_a":4271800,"open_b":76.849709,"high_b":77.291101,"low_b":74.891203,"close_b":75.506248,"volume_b":129554000,"open_bench":305.029941,"close_bench":303.22052},{"date":"2020-02-24","open_a":117.886977,"high_a":118.929576,"low_a":115.219849,"close_a":115.284508,"volume_a":5027700,"open_b":71.697767,"high_b":73.366835,"low_b":69.760967,"close_b":71.919662,"volume_b":222195200,"open_bench":293.818762,"close_bench":293.164093},{"date":"2020-02-25","open_a":115.244111,"high_a":116.690832,"low_a":113.611516,"close_a":114.039871,"volume_a":7585800,"open_b":72.587765,"high_b":72.968851,"low_b":69.013247,"close_b":69.483574,"volume_b":230673600,"open_bench":294.546148,"close_bench":284.280579},{"date":"2020-02-26","open_a":115.122908,"high_a":116.181675,"low_a":114.088387,"close_a":114.751122,"volume_a":6157200,"open_b":69.109746,"high_b":71.847316,"low_b":69.10251,"close_b":70.585861,"volume_b":198054800,"open_bench":285.671769,"close_bench":283.234955},{"date":"2020-02-27","open_a":114.290385,"high_a":115.688602,"low_a":110.168458,"close_a":110.216949,"volume_a":8605000,"open_b":67.800045,"high_b":68.981901,"low_b":65.83671,"close_b":65.971779,"volume_b":320605600,"open_bench":277.742985,"close_bench":270.514374},{"date":"2020-02-28","open_a":107.687257,"high_a":108.099444,"low_a":103.735061,"close_a":106.709305,"volume_a":14164000,"open_b":62.049927,"high_b":67.151208,"low_b":61.835259,"close_b":65.933167,"volume_b":426510000,"open_bench":262.503793,"close_bench":269.377808},{"date":"2020-03-02","open_a":106.685054,"high_a":111.348487,"low_a":106.458753,"close_a":111.194923,"volume_a":8504500,"open_b":68.084671,"high_b":72.705978,"low_b":66.98482,"close_b":72.071632,"volume_b":341397200,"open_bench":271.150841,"close_bench":281.04361},{"date":"2020-03-03","open_a":110.879711,"high_a":112.884093,"low_a":108.899574,"close_a":109.578476,"volume_a":8728300,"open_b":73.243833,"high_b":73.323425,"low_b":68.933664,"close_b":69.782677,"volume_b":319475600,"open_bench":281.416381,"close_bench":272.996613},{"date":"2020-03-04","open_a":110.071474,"high_a":115.341072,"low_a":109.925988,"close_a":115.082436,"volume_a":7635700,"open_b":71.49998,"high_b":73.178698,"low_b":70.701624,"close_b":73.019508,"volume_b":219178400,"open_bench":278.343114,"close_bench":284.471527},{"date":"2020-03-05","open_a":112.946539,"high_a":115.070238,"low_a":111.205271,"close_a":112.368835,"volume_a":6579100,"open_b":71.278088,"high_b":72.250106,"low_b":70.286778,"close_b":70.650986,"volume_b":187572800,"open_bench":277.306523,"close_bench":275.015167},{"date":"2020-03-06","open_a":107.047397,"high_a":112.425806,"low_a":106.957892,"close_a":111.685356,"volume_a":6754900,"open_b":68.017149,"high_b":70.144495,"low_b":67.831431,"close_b":69.712753,"volume_b":226176800,"open_bench":266.549988,"close_bench":270.468903},{"date":"2020-03-09","open_a":105.924529,"high_a":108.503897,"low_a":104.297174,"close_a":105.908264,"volume_a":8716300,"open_b":63.615324,"high_b":67.074067,"low_b":63.434427,"close_b":64.19902,"volume_b":286744800,"open_bench":250.319736,"close_bench":249.346848},{"date":"2020-03-10","open_a":107.446108,"high_a":109.496578,"low_a":103.22312,"close_a":109.14669,"volume_a":8258600,"open_b":66.844925,"high_b":69.08804,"low_b":64.970831,"close_b":68.822723,"volume_b":285290000,"open_bench":258.812139,"close_bench":262.249146},{"date":"2020-03-11","open_a":106.25,"high_a":107.844801,"low_a":103.849648,"close_a":105.574646,"volume_a":7465100,"open_b":66.905221,"high_b":67.828996,"low_b":65.571403,"close_b":66.432472,"volume_b":255598800,"open_bench":255.229708,"close_bench":249.464966},{"date":"2020-03-12","open_a":99.350017,"high_a":101.709683,"low_a":92.42562,"close_a":93.849556,"volume_a":12808300,"open_b":61.731579,"high_b":65.122787,"low_b":59.816486,"close_b":59.87196,"volume_b":418474000,"open_bench":232.770919,"close_bench":225.596848},{"date":"2020-03-13","open_a":98.194607,"high_a":103.817116,"low_a":93.629875,"close_a":103.703201,"volume_a":12007700,"open_b":63.890264,"high_b":67.515431,"low_b":61.010386,"close_b":67.045097,"volume_b":370732000,"open_bench":239.2176,"close_bench":244.882309},{"date":"2020-03-16","open_a":93.320665,"high_a":100.749537,"low_a":91.131871,"close_a":92.018776,"volume_a":10733300,"open_b":58.357247,"high_b":62.488923,"low_b":57.886916,"close_b":58.41996,"volume_b":322423600,"open_bench":219.295686,"close_bench":218.08638},{"date":"2020-03-17","open_a":94.956146,"high_a":104.036789,"low_a":94.703908,"close_a":103.922874,"volume_a":14242300,"open_b":59.6983,"high_b":62.134372,"low_b":57.501008,"close_b":60.988697,"volume_b":324056000,"open_bench":222.805435,"close_bench":229.861313},{"date":"2020-03-18","open_a":98.324751,"high_a":103.922854,"low_a":92.572054,"close_a":98.389847,"volume_a":15250800,"open_b":57.831442,"high_b":60.298871,"low_b":57.192272,"close_b":59.495689,"volume_b":300233600,"open_bench":214.813047,"close_bench":218.222778},{"date":"2020-03-19","open_a":98.804858,"high_a":99.341882,"low_a":94.435407,"close_a":95.476913,"volume_a":9393500,"open_b":59.669357,"high_b":60.983872,"low_b":58.516443,"close_b":59.039837,"volume_b":271857200,"open_bench":217.540813,"close_bench":218.686478},{"date":"2020-03-20","open_a":96.209221,"high_a":96.380092,"low_a":82.523166,"close_a":84.565498,"volume_a":13093200,"open_b":59.618688,"high_b":60.740249,"low_b":54.992561,"close_b":55.291645,"volume_b":401693200,"open_bench":221.81991,"close_bench":209.262344},{"date":"2020-03-23","open_a":86.249811,"high_a":91.839778,"low_a":83.703003,"close_a":85.582596,"volume_a":13926800,"open_b":55.011864,"high_b":55.113166,"low_b":51.28057,"close_b":54.117027,"volume_b":336752800,"open_bench":208.704432,"close_bench":203.91188},{"date":"2020-03-24","open_a":89.447575,"high_a":93.109124,"low_a":88.991918,"close_a":92.629051,"volume_a":11447000,"open_b":57.008967,"high_b":59.741713,"low_b":56.512105,"close_b":59.546345,"volume_b":287531200,"open_bench":214.402452,"close_bench":222.386978},{"date":"2020-03-25","open_a":91.733985,"high_a":94.679499,"low_a":90.350736,"close_a":91.538704,"volume_a":9485500,"open_b":60.47977,"high_b":62.288737,"low_b":58.92406,"close_b":59.218319,"volume_b":303602000,"open_bench":223.960049,"close_bench":225.716095},{"date":"2020-03-26","open_a":91.709585,"high_a":99.105916,"low_a":91.603809,"close_a":97.852852,"volume_a":10376800,"open_b":59.459518,"high_b":62.392453,"low_b":59.420926,"close_b":62.334568,"volume_b":252087200,"open_bench":228.213059,"close_bench":238.895691},{"date":"2020-03-27","open_a":96.477722,"high_a":101.424882,"low_a":96.005788,"close_a":98.015572,"volume_a":9852500,"open_b":60.962153,"high_b":61.714682,"low_b":59.58734,"close_b":59.753765,"volume_b":204216800,"open_bench":231.642844,"close_bench":231.780029},{"date":"2020-03-30","open_a":98.951287,"high_a":102.393138,"low_a":98.86992,"close_a":102.10022,"volume_a":6916800,"open_b":60.477366,"high_b":61.630281,"low_b":60.154161,"close_b":61.45903,"volume_b":167976400,"open_bench":233.865305,"close_bench":239.30722},{"date":"2020-03-31","open_a":101.660866,"high_a":102.279257,"low_a":97.258868,"close_a":97.722664,"volume_a":7091700,"open_b":61.649564,"high_b":63.311397,"low_b":60.781259,"close_b":61.333595,"volume_b":197002000,"open_bench":238.310296,"close_bench":235.74025},{"date":"2020-04-01","open_a":95.419942,"high_a":99.512744,"low_a":94.118059,"close_a":96.111572,"volume_a":6873000,"open_b":59.454681,"high_b":59.990136,"low_b":57.677072,"close_b":58.1064,"volume_b":176218400,"open_bench":226.804533,"close_bench":225.130798},{"date":"2020-04-02","open_a":95.379271,"high_a":101.856143,"low_a":94.777152,"close_a":100.782089,"volume_a":6181800,"open_b":57.968918,"high_b":59.129067,"low_b":57.139205,"close_b":59.076004,"volume_b":165934000,"open_bench":224.252778,"close_bench":230.325775},{"date":"2020-04-03","open_a":100.236932,"high_a":102.010749,"low_a":99.626674,"close_a":101.376076,"volume_a":5552000,"open_b":58.562263,"high_b":59.261728,"low_b":57.638484,"close_b":58.227001,"volume_b":129880000,"open_bench":229.347148,"close_bench":226.996613},{"date":"2020-04-06","open_a":104.687745,"high_a":107.511207,"low_a":101.94565,"close_a":106.721939,"volume_a":8242200,"open_b":60.51595,"high_b":63.460945,"low_b":60.149335,"close_b":63.306583,"volume_b":201820400,"open_bench":235.822564,"close_bench":242.243103},{"date":"2020-04-07","open_a":107.983119,"high_a":109.91153,"low_a":106.014009,"close_a":106.16861,"volume_a":6253900,"open_b":65.315733,"high_b":65.532815,"low_b":62.469629,"close_b":62.573341,"volume_b":202887200,"open_bench":250.794759,"close_bench":242.490128},{"date":"2020-04-08","open_a":105.485163,"high_a":108.455082,"low_a":105.070192,"close_a":107.901787,"volume_a":5241100,"open_b":63.371719,"high_b":64.488455,"low_b":63.007518,"close_b":64.174904,"volume_b":168895200,"open_bench":245.078403,"close_bench":250.630081},{"date":"2020-04-09","open_a":107.478635,"high_a":109.911534,"low_a":107.0067,"close_a":108.731705,"volume_a":5502400,"open_b":64.809216,"high_b":65.139653,"low_b":63.844435,"close_b":64.637962,"volume_b":161834800,"open_bench":253.876935,"close_bench":254.444016},{"date":"2020-04-13","open_a":107.153157,"high_a":108.503864,"low_a":105.314238,"close_a":106.502213,"volume_a":3785000,"open_b":64.715133,"high_b":66.01518,"low_b":64.116966,"close_b":65.906639,"volume_b":131022800,"open_bench":253.474556,"close_bench":252.120926},{"date":"2020-04-14","open_a":108.398117,"high_a":112.800116,"low_a":107.690222,"close_a":112.612961,"volume_a":6278700,"open_b":67.534736,"high_b":69.524598,"low_b":67.064401,"close_b":69.235161,"volume_b":194994800,"open_bench":256.986624,"close_bench":259.556671},{"date":"2020-04-15","open_a":111.587702,"high_a":111.929445,"low_a":108.78865,"close_a":109.870842,"volume_a":5773100,"open_b":68.113592,"high_b":69.061488,"low_b":67.686678,"close_b":68.603218,"volume_b":131154400,"open_bench":253.867757,"close_bench":254.041534},{"date":"2020-04-16","open_a":111.449395,"high_a":111.449395,"low_a":108.910726,"close_a":110.367203,"volume_a":4564000,"open_b":69.314756,"high_b":69.512538,"low_b":68.101543,"close_b":69.148331,"volume_b":157125200,"open_bench":255.312855,"close_bench":255.267136},{"date":"2020-04-17","open_a":112.067782,"high_a":112.287478,"low_a":109.773218,"close_a":111.921326,"volume_a":4866500,"open_b":68.665938,"high_b":69.211042,"low_b":66.777374,"close_b":68.210075,"volume_b":215250000,"open_bench":261.010945,"close_bench":262.163361},{"date":"2020-04-20","open_a":111.424989,"high_a":111.78301,"low_a":109.447747,"close_a":109.480301,"volume_a":3995400,"open_b":67.040298,"high_b":67.939952,"low_b":66.774981,"close_b":66.794273,"volume_b":130015200,"open_bench":258.477385,"close_bench":257.544495},{"date":"2020-04-21","open_a":107.519333,"high_a":108.170278,"low_a":106.258131,"close_a":106.542923,"volume_a":4359500,"open_b":66.637487,"high_b":66.871447,"low_b":64.020514,"close_b":64.72963,"volume_b":180991600,"open_bench":253.099547,"close_bench":249.72464},{"date":"2020-04-22","open_a":107.763441,"high_a":108.764261,"low_a":106.770753,"close_a":107.909897,"volume_a":4084600,"open_b":65.993474,"high_b":67.028204,"low_b":65.653395,"close_b":66.594055,"volume_b":116862400,"open_bench":254.58118,"close_bench":255.267136},{"date":"2020-04-23","open_a":107.812251,"high_a":109.146681,"low_a":105.802466,"close_a":105.989609,"volume_a":5527700,"open_b":66.538594,"high_b":67.956825,"low_b":66.297399,"close_b":66.335991,"volume_b":124814400,"open_bench":256.538441,"close_bench":255.24884},{"date":"2020-04-24","open_a":106.485938,"high_a":109.699973,"low_a":106.306928,"close_a":109.325676,"volume_a":4063100,"open_b":66.859403,"high_b":68.260749,"low_b":66.811161,"close_b":68.251099,"volume_b":126161200,"open_bench":256.757988,"close_bench":258.806702},{"date":"2020-04-27","open_a":109.374543,"high_a":110.017355,"low_a":108.943295,"close_a":109.407097,"volume_a":4009200,"open_b":67.968888,"high_b":68.629768,"low_b":67.522682,"close_b":68.299332,"volume_b":117087600,"open_bench":260.772991,"close_bench":262.538177},{"date":"2020-04-28","open_a":111.294794,"high_a":112.027099,"low_a":109.569802,"close_a":110.920509,"volume_a":6674200,"open_b":68.759993,"high_b":68.94089,"low_b":67.100575,"close_b":67.192223,"volume_b":112004800,"open_bench":266.169282,"close_bench":261.331024},{"date":"2020-04-29","open_a":111.286644,"high_a":111.457509,"low_a":108.137704,"close_a":108.512001,"volume_a":6706800,"open_b":68.675584,"high_b":69.86709,"low_b":68.47298,"close_b":69.39917,"volume_b":137280800,"open_bench":266.635706,"close_bench":268.172241},{"date":"2020-04-30","open_a":107.738999,"high_a":108.35739,"low_a":106.119777,"close_a":107.64135,"volume_a":6783000,"open_b":69.937046,"high_b":71.039311,"low_b":69.548724,"close_b":70.863235,"volume_b":183064000,"open_bench":266.800304,"close_bench":265.675354},{"date":"2020-05-01","open_a":106.689383,"high_a":107.592566,"low_a":105.387507,"close_a":105.891983,"volume_a":4488000,"open_b":69.042216,"high_b":72.117459,"low_b":68.945739,"close_b":69.722389,"volume_b":240616800,"open_bench":260.946924,"close_bench":258.64212},{"date":"2020-05-04","open_a":105.916369,"high_a":106.876514,"low_a":105.22475,"close_a":106.526627,"volume_a":5245300,"open_b":69.746502,"high_b":70.836703,"low_b":69.059094,"close_b":70.70887,"volume_b":133568000,"open_bench":256.767082,"close_bench":259.355438},{"date":"2020-05-05","open_a":107.112493,"high_a":108.487609,"low_a":106.404597,"close_a":107.136902,"volume_a":3580000,"open_b":71.167145,"high_b":72.599847,"low_b":71.022426,"close_b":71.770134,"volume_b":147751200,"open_bench":262.163386,"close_bench":261.751801},{"date":"2020-05-06","open_a":107.063679,"high_a":107.592565,"low_a":106.339507,"close_a":106.412735,"volume_a":4192200,"open_b":72.469583,"high_b":73.140106,"low_b":72.086083,"close_b":72.51059,"volume_b":142333600,"open_bench":263.443759,"close_bench":259.977386},{"date":"2020-05-07","open_a":107.039261,"high_a":107.405413,"low_a":105.875696,"close_a":107.039261,"volume_a":5528000,"open_b":73.1353,"high_b":73.605634,"low_b":72.833806,"close_b":73.260719,"volume_b":115215200,"open_bench":263.178501,"close_bench":263.114471},{"date":"2020-05-08","open_a":108.276053,"high_a":109.976636,"low_a":107.77971,"close_a":109.21991,"volume_a":4209100,"open_b":73.918536,"high_b":75.05764,"low_b":73.592039,"close_b":75.004433,"volume_b":133838400,"open_bench":266.233291,"close_bench":267.468018},{"date":"2020-05-11","open_a":108.568992,"high_a":110.822882,"low_a":108.35744,"close_a":109.341995,"volume_a":3591100,"open_b":74.513504,"high_b":76.678043,"low_b":74.30551,"close_b":76.184677,"volume_b":145946400,"open_bench":265.547422,"close_bench":267.52298},{"date":"2020-05-12","open_a":109.651159,"high_a":110.969325,"low_a":108.105177,"close_a":108.170273,"volume_a":3914600,"open_b":76.866665,"high_b":77.316507,"low_b":75.193079,"close_b":75.314003,"volume_b":162301200,"open_bench":268.702866,"close_bench":262.190857},{"date":"2020-05-13","open_a":108.365529,"high_a":109.415179,"low_a":106.95787,"close_a":108.186531,"volume_a":5168500,"open_b":75.492989,"high_b":76.412017,"low_b":73.330863,"close_b":74.404671,"volume_b":200622400,"open_bench":261.632763,"close_bench":257.553619},{"date":"2020-05-14","open_a":107.9831,"high_a":108.674718,"low_a":105.900076,"close_a":108.284157,"volume_a":5522200,"open_b":73.645266,"high_b":74.922225,"low_b":72.924555,"close_b":74.861763,"volume_b":158929200,"open_bench":255.130023,"close_bench":260.635956},{"date":"2020-05-15","open_a":108.84562,"high_a":112.010824,"low_a":108.048207,"close_a":110.619431,"volume_a":20338200,"open_b":72.639166,"high_b":74.465118,"low_b":72.605304,"close_b":74.419167,"volume_b":166348400,"open_bench":258.257864,"close_bench":261.833984},{"date":"2020-05-18","open_a":111.351739,"high_a":112.27933,"low_a":109.89525,"close_a":110.269547,"volume_a":8480300,"open_b":75.739666,"high_b":76.545018,"low_b":75.050397,"close_b":76.172569,"volume_b":135178400,"open_bench":268.025951,"close_bench":269.809448},{"date":"2020-05-19","open_a":109.382649,"high_a":109.651164,"low_a":106.917207,"close_a":107.185722,"volume_a":6640300,"open_b":76.189504,"high_b":77.033552,"low_b":75.700973,"close_b":75.732414,"volume_b":101729600,"open_bench":269.214857,"close_bench":267.038086},{"date":"2020-05-20","open_a":107.844797,"high_a":107.958712,"low_a":106.05471,"close_a":106.803291,"volume_a":8213100,"open_b":76.588551,"high_b":77.2754,"low_b":76.549854,"close_b":77.205269,"volume_b":111504800,"open_bench":270.559383,"close_bench":271.574585},{"date":"2020-05-21","open_a":106.575459,"high_a":106.949756,"low_a":105.460725,"close_a":105.948936,"volume_a":4886300,"open_b":77.067422,"high_b":77.606747,"low_b":76.392663,"close_b":76.629677,"volume_b":102688800,"open_bench":271.446551,"close_bench":269.699646},{"date":"2020-05-22","open_a":105.460714,"high_a":106.290669,"low_a":105.102694,"close_a":106.16861,"volume_a":3429200,"open_b":76.368481,"high_b":77.205282,"low_b":76.266909,"close_b":77.123055,"volume_b":81803200,"open_bench":269.416118,"close_bench":270.211823},{"date":"2020-05-26","open_a":107.356602,"high_a":107.446107,"low_a":105.289854,"close_a":105.574646,"volume_a":5579300,"open_b":78.237942,"high_b":78.416908,"low_b":76.545004,"close_b":76.600632,"volume_b":125522000,"open_bench":276.147716,"close_bench":273.541077},{"date":"2020-05-27","open_a":105.550213,"high_a":106.461528,"low_a":104.622622,"close_a":106.437119,"volume_a":5686500,"open_b":76.457961,"high_b":77.079505,"low_b":75.720318,"close_b":76.934395,"volume_b":112945200,"open_bench":276.321422,"close_bench":277.611023},{"date":"2020-05-28","open_a":106.762589,"high_a":107.99937,"low_a":105.867539,"close_a":107.64135,"volume_a":4974400,"open_b":76.610323,"high_b":78.223456,"low_b":76.33462,"close_b":76.968262,"volume_b":133560800,"open_bench":278.635474,"close_bench":277.098938},{"date":"2020-05-29","open_a":107.413546,"high_a":107.6007,"low_a":106.038441,"close_a":107.039261,"volume_a":13337900,"open_b":77.210093,"high_b":77.669604,"low_b":76.537755,"close_b":76.893272,"volume_b":153532400,"open_bench":276.632371,"close_bench":278.333557},{"date":"2020-06-01","open_a":107.079919,"high_a":108.992053,"low_a":106.746309,"close_a":108.21093,"volume_a":4049200,"open_b":76.847325,"high_b":77.95983,"low_b":76.716725,"close_b":77.838905,"volume_b":80791200,"open_bench":277.693339,"close_bench":279.458527},{"date":"2020-06-02","open_a":108.259773,"high_a":108.878163,"low_a":106.689369,"close_a":107.91803,"volume_a":5327500,"open_b":77.572863,"high_b":78.223436,"low_b":77.132697,"close_b":78.199249,"volume_b":87642800,"open_bench":280.373142,"close_bench":281.772491},{"date":"2020-06-03","open_a":108.414375,"high_a":109.260594,"low_a":107.893622,"close_a":108.072632,"volume_a":5421200,"open_b":78.51852,"high_b":78.890969,"low_b":77.947754,"close_b":78.629768,"volume_b":104491200,"open_bench":283.748118,"close_bench":285.522461},{"date":"2020-06-04","open_a":107.902905,"high_a":109.518292,"low_a":106.746711,"close_a":107.066521,"volume_a":5229700,"open_b":78.453204,"high_b":78.750672,"low_b":77.580127,"close_b":77.952576,"volume_b":87560400,"open_bench":284.543748,"close_bench":284.7724},{"date":"2020-06-05","open_a":107.4191,"high_a":109.485486,"low_a":107.230505,"close_a":108.599892,"volume_a":7075700,"open_b":78.201682,"high_b":80.233207,"low_b":78.172661,"close_b":80.172745,"volume_b":137250400,"open_bench":290.141204,"close_bench":292.071014},{"date":"2020-06-08","open_a":107.730712,"high_a":108.624502,"low_a":107.615914,"close_a":108.411308,"volume_a":5613000,"open_b":79.87042,"high_b":80.680614,"low_b":79.161806,"close_b":80.646751,"volume_b":95654400,"open_bench":292.875807,"close_bench":295.601349},{"date":"2020-06-09","open_a":108.099686,"high_a":109.18207,"low_a":107.460092,"close_a":109.034477,"volume_a":6446900,"open_b":80.327538,"high_b":83.585229,"low_b":80.296096,"close_b":83.193436,"volume_b":147712400,"open_bench":292.94904,"close_bench":293.397217},{"date":"2020-06-10","open_a":109.288696,"high_a":110.830287,"low_a":109.198496,"close_a":109.985695,"volume_a":7069900,"open_b":84.139039,"high_b":85.800536,"low_b":83.701294,"close_b":85.333771,"volume_b":166651600,"open_bench":293.973374,"close_bench":291.76001},{"date":"2020-06-11","open_a":110.059479,"high_a":110.813871,"low_a":104.516324,"close_a":104.827919,"volume_a":8541000,"open_b":84.480081,"high_b":84.903316,"low_b":81.13532,"close_b":81.236893,"volume_b":201662400,"open_bench":284.86381,"close_bench":274.940308},{"date":"2020-06-12","open_a":105.902115,"high_a":106.541709,"low_a":104.491732,"close_a":105.779121,"volume_a":7039500,"open_b":83.369986,"high_b":84.114876,"low_b":80.830578,"close_b":81.93824,"volume_b":200146000,"open_bench":281.91881,"close_bench":278.232941},{"date":"2020-06-15","open_a":105.196897,"high_a":107.386289,"low_a":103.753714,"close_a":106.992683,"volume_a":5481000,"open_b":80.596005,"high_b":83.60218,"low_b":80.433963,"close_b":82.951607,"volume_b":138808800,"open_bench":272.571563,"close_bench":280.830475},{"date":"2020-06-16","open_a":108.796687,"high_a":109.108288,"low_a":106.697506,"close_a":107.968498,"volume_a":5197700,"open_b":85.000026,"high_b":85.420847,"low_b":83.369971,"close_b":85.149971,"volume_b":165428800,"open_bench":288.54064,"close_bench":286.235809},{"date":"2020-06-17","open_a":108.148885,"high_a":109.091872,"low_a":107.820893,"close_a":108.042282,"volume_a":3309700,"open_b":85.892451,"high_b":85.952913,"low_b":84.910547,"close_b":85.031471,"volume_b":114406400,"open_bench":287.250992,"close_bench":285.046783},{"date":"2020-06-18","open_a":107.763479,"high_a":109.165672,"low_a":107.11569,"close_a":108.87867,"volume_a":3417200,"open_b":84.987952,"high_b":85.481325,"low_b":84.458304,"close_b":85.065346,"volume_b":96820400,"open_bench":283.537587,"close_bench":285.156433},{"date":"2020-06-19","open_a":110.215272,"high_a":110.551472,"low_a":107.599492,"close_a":107.648689,"volume_a":8310100,"open_b":85.769113,"high_b":86.233458,"low_b":83.473967,"close_b":84.579216,"volume_b":264476000,"open_bench":288.606956,"close_bench":283.526917},{"date":"2020-06-22","open_a":107.050094,"high_a":108.042279,"low_a":106.558093,"close_a":107.460091,"volume_a":3379900,"open_b":84.971011,"high_b":86.934819,"low_b":84.92506,"close_b":86.79213,"volume_b":135445200,"open_bench":282.929785,"close_bench":285.345795},{"date":"2020-06-23","open_a":107.927473,"high_a":108.796663,"low_a":107.402678,"close_a":107.714279,"volume_a":3625600,"open_b":88.032823,"high_b":90.059514,"low_b":87.614422,"close_b":88.644699,"volume_b":212155600,"open_bench":287.982196,"close_bench":286.659363},{"date":"2020-06-24","open_a":107.025495,"high_a":107.476494,"low_a":105.516699,"close_a":106.238297,"volume_a":4853500,"open_b":88.274665,"high_b":89.191272,"low_b":86.707485,"close_b":87.079933,"volume_b":192623200,"open_bench":284.629247,"close_bench":279.347107},{"date":"2020-06-25","open_a":105.893907,"high_a":108.017686,"low_a":105.000118,"close_a":107.755295,"volume_a":4442000,"open_b":87.234706,"high_b":88.274651,"low_b":86.477719,"close_b":88.235954,"volume_b":137522400,"open_bench":278.777557,"close_bench":282.341858},{"date":"2020-06-26","open_a":107.976707,"high_a":108.542504,"low_a":105.008335,"close_a":105.721725,"volume_a":7372700,"open_b":88.131998,"high_b":88.352081,"low_b":85.37734,"close_b":85.524872,"volume_b":205256800,"open_bench":281.248603,"close_bench":275.635742},{"date":"2020-06-29","open_a":106.746694,"high_a":107.747087,"low_a":106.131699,"close_a":107.484695,"volume_a":4173900,"open_b":85.432929,"high_b":87.59022,"low_b":84.956488,"close_b":87.495895,"volume_b":130646000,"open_bench":276.885162,"close_bench":279.686981},{"date":"2020-06-30","open_a":108.099687,"high_a":108.911485,"low_a":107.296097,"close_a":108.452278,"volume_a":4626700,"open_b":87.084756,"high_b":88.511667,"low_b":87.065411,"close_b":88.22628,"volume_b":140223200,"open_bench":279.255232,"close_bench":283.269653},{"date":"2020-07-01","open_a":108.673691,"high_a":109.567481,"low_a":108.189899,"close_a":108.534294,"volume_a":3633000,"open_b":88.303695,"high_b":88.845433,"low_b":88.01106,"close_b":88.059425,"volume_b":110737200,"open_bench":284.381222,"close_bench":285.253906},{"date":"2020-07-02","open_a":109.501889,"high_a":110.018488,"low_a":108.681895,"close_a":108.936104,"volume_a":2769200,"open_b":88.963943,"high_b":89.597585,"low_b":87.945764,"close_b":88.059425,"volume_b":114041600,"open_bench":288.671293,"close_bench":286.82486},{"date":"2020-07-06","open_a":109.12467,"high_a":109.838059,"low_a":108.493271,"close_a":109.305069,"volume_a":4221600,"open_b":89.483906,"high_b":90.881789,"low_b":89.452465,"close_b":90.415024,"volume_b":118655600,"open_bench":290.627991,"close_bench":291.252655},{"date":"2020-07-07","open_a":108.460496,"high_a":110.387472,"low_a":108.378491,"close_a":109.526489,"volume_a":4424500,"open_b":90.792319,"high_b":91.56865,"low_b":90.023243,"close_b":90.134491,"volume_b":112424400,"open_bench":289.718537,"close_bench":288.248718},{"date":"2020-07-08","open_a":110.010258,"high_a":110.444854,"low_a":109.190265,"close_a":110.059456,"volume_a":3798200,"open_b":91.109125,"high_b":92.26516,"low_b":91.022056,"close_b":92.233719,"volume_b":117092000,"open_bench":289.011109,"close_bench":290.453369},{"date":"2020-07-09","open_a":108.977093,"high_a":109.829892,"low_a":108.017702,"close_a":108.821304,"volume_a":4005200,"open_b":93.123721,"high_b":93.176928,"low_b":91.585569,"close_b":92.630356,"volume_b":125642800,"open_bench":291.059664,"close_bench":288.799835},{"date":"2020-07-10","open_a":108.542514,"high_a":110.608901,"low_a":107.648724,"close_a":110.25631,"volume_a":5932800,"open_b":92.226465,"high_b":92.850438,"low_b":91.617009,"close_b":92.792389,"volume_b":90257200,"open_bench":288.735572,"close_bench":291.748688},{"date":"2020-07-13","open_a":112.150455,"high_a":113.216448,"low_a":110.264468,"close_a":110.625267,"volume_a":8960000,"open_b":94.093526,"high_b":96.695816,"low_b":92.151483,"close_b":92.364311,"volume_b":191649200,"open_bench":294.081989,"close_bench":289.222412},{"date":"2020-07-14","open_a":110.36285,"high_a":112.298034,"low_a":110.084056,"close_a":111.12545,"volume_a":6766800,"open_b":91.747605,"high_b":94.083862,"low_b":90.816495,"close_b":93.892807,"volume_b":170989200,"open_bench":287.807748,"close_bench":292.97049},{"date":"2020-07-15","open_a":110.986062,"high_a":111.674853,"low_a":109.264071,"close_a":109.62487,"volume_a":5561300,"open_b":95.762288,"high_b":96.011392,"low_b":93.343804,"close_b":94.538536,"volume_b":153198000,"open_bench":296.176419,"close_bench":295.661987},{"date":"2020-07-16","open_a":109.928293,"high_a":110.174294,"low_a":109.100105,"close_a":109.780701,"volume_a":3788100,"open_b":93.413941,"high_b":94.228969,"low_b":92.777878,"close_b":93.375244,"volume_b":110577600,"open_bench":293.769691,"close_bench":294.688324},{"date":"2020-07-17","open_a":110.412088,"high_a":110.748288,"low_a":109.510103,"close_a":110.420296,"volume_a":3229300,"open_b":93.825083,"high_b":93.979862,"low_b":92.714992,"close_b":93.1866,"volume_b":92186800,"open_bench":295.689618,"close_bench":295.542633},{"date":"2020-07-20","open_a":110.379251,"high_a":110.379251,"low_a":108.386662,"close_a":109.149261,"volume_a":3620300,"open_b":93.27369,"high_b":95.288284,"low_b":92.930262,"close_b":95.150429,"volume_b":90318000,"open_bench":295.276168,"close_bench":297.93103},{"date":"2020-07-21","open_a":109.879083,"high_a":111.85527,"low_a":109.706879,"close_a":110.231674,"volume_a":5435400,"open_b":95.938844,"high_b":96.013817,"low_b":93.588078,"close_b":93.837181,"volume_b":103433200,"open_bench":299.887714,"close_bench":298.56488},{"date":"2020-07-22","open_a":111.158273,"high_a":111.72407,"low_a":109.616682,"close_a":111.527267,"volume_a":3522300,"open_b":93.539702,"high_b":94.780385,"low_b":93.45264,"close_b":94.100792,"volume_b":89001600,"open_bench":298.206736,"close_bench":300.264465},{"date":"2020-07-23","open_a":112.002832,"high_a":112.54403,"low_a":111.100847,"close_a":112.404633,"volume_a":5011400,"open_b":93.834748,"high_b":93.912141,"low_b":89.009877,"close_b":89.81765,"volume_b":197004400,"open_bench":299.906143,"close_bench":296.681732},{"date":"2020-07-24","open_a":112.33086,"high_a":113.683844,"low_a":111.018866,"close_a":111.56826,"volume_a":3300000,"open_b":88.020711,"high_b":89.938566,"low_b":86.238282,"close_b":89.595139,"volume_b":185438800,"open_bench":294.835246,"close_bench":294.770935},{"date":"2020-07-27","open_a":111.133658,"high_a":112.962239,"low_a":110.576069,"close_a":112.888443,"volume_a":3607700,"open_b":90.654443,"high_b":91.810478,"low_b":90.431947,"close_b":91.718575,"volume_b":121214000,"open_bench":295.459909,"close_bench":296.920532},{"date":"2020-07-28","open_a":113.142638,"high_a":113.642835,"low_a":112.412845,"close_a":112.65065,"volume_a":3787500,"open_b":91.29051,"high_b":91.467062,"low_b":90.207027,"close_b":90.211868,"volume_b":103625600,"open_bench":296.194904,"close_bench":295.037445},{"date":"2020-07-29","open_a":112.650637,"high_a":113.979022,"low_a":112.37183,"close_a":113.101624,"volume_a":3179000,"open_b":90.693155,"high_b":92.124901,"low_b":90.656879,"close_b":91.941093,"volume_b":90329200,"open_bench":295.910086,"close_bench":298.665985},{"date":"2020-07-30","open_a":112.535882,"high_a":113.339472,"low_a":111.912679,"close_a":112.904877,"volume_a":2817600,"open_b":91.116373,"high_b":93.157574,"low_b":90.71007,"close_b":93.053581,"volume_b":158130000,"open_bench":295.70793,"close_bench":297.600311},{"date":"2020-07-31","open_a":112.552252,"high_a":112.913051,"low_a":110.863068,"close_a":112.880257,"volume_a":4304100,"open_b":99.530296,"high_b":102.945195,"low_b":97.537461,"close_b":102.79525,"volume_b":374336800,"open_bench":299.382448,"close_bench":299.951996},{"date":"2020-08-03","open_a":112.429241,"high_a":112.749037,"low_a":111.519048,"close_a":112.09304,"volume_a":2819200,"open_b":104.671989,"high_b":107.997405,"low_b":104.37452,"close_b":105.385445,"volume_b":308151200,"open_bench":301.605595,"close_bench":302.037354},{"date":"2020-08-04","open_a":112.765451,"high_a":112.822844,"low_a":111.535462,"close_a":112.724449,"volume_a":2553500,"open_b":105.574095,"high_b":107.177551,"low_b":104.853384,"close_b":106.089233,"volume_b":173071600,"open_bench":301.183097,"close_bench":303.204102},{"date":"2020-08-05","open_a":113.282034,"high_a":113.282034,"low_a":111.330459,"close_a":111.724052,"volume_a":2924000,"open_b":105.811085,"high_b":106.792989,"low_b":105.346733,"close_b":106.473747,"volume_b":121776800,"open_bench":304.499309,"close_bench":305.087219},{"date":"2020-08-06","open_a":110.920458,"high_a":111.756842,"low_a":110.920458,"close_a":111.404251,"volume_a":2692900,"open_b":106.805096,"high_b":110.681926,"low_b":106.217406,"close_b":110.188553,"volume_b":202428800,"open_bench":304.508414,"close_bench":307.126495},{"date":"2020-08-07","open_a":111.469873,"high_a":112.142261,"low_a":111.002471,"close_a":112.12587,"volume_a":3692100,"open_b":109.711246,"high_b":110.166742,"low_b":106.888634,"close_b":107.683327,"volume_b":198045600,"open_bench":306.161971,"close_bench":307.347015},{"date":"2020-08-10","open_a":111.781477,"high_a":112.240672,"low_a":111.010682,"close_a":111.50267,"volume_a":3052000,"open_b":109.124914,"high_b":110.263654,"low_b":106.605158,"close_b":109.248482,"volume_b":212403600,"open_bench":307.797235,"close_bench":308.265747},{"date":"2020-08-11","open_a":112.314431,"high_a":112.708025,"low_a":110.387443,"close_a":110.79744,"volume_a":3301200,"open_b":108.514365,"high_b":109.011045,"low_b":105.740205,"close_b":105.999451,"volume_b":187902400,"open_bench":309.441488,"close_bench":305.721008},{"date":"2020-08-12","open_a":110.97786,"high_a":113.716634,"low_a":110.904064,"close_a":112.995049,"volume_a":4424600,"open_b":107.087299,"high_b":109.779083,"low_b":106.893475,"close_b":109.522263,"volume_b":165598000,"open_bench":308.146224,"close_bench":309.98349},{"date":"2020-08-13","open_a":112.42105,"high_a":113.659235,"low_a":112.42105,"close_a":113.241043,"volume_a":2393500,"open_b":110.898431,"high_b":112.461169,"low_b":110.411437,"close_b":111.460533,"volume_b":210082000,"open_bench":309.221087,"close_bench":309.423187},{"date":"2020-08-14","open_a":113.249239,"high_a":113.298437,"low_a":112.503043,"close_a":112.798241,"volume_a":2461700,"open_b":111.286106,"high_b":111.450858,"low_b":109.556191,"close_b":111.361214,"volume_b":165565200,"open_bench":309.037246,"close_bench":309.432251},{"date":"2020-08-17","open_a":112.749049,"high_a":113.888839,"low_a":112.380055,"close_a":113.093445,"volume_a":2862500,"open_b":112.48057,"high_b":112.5048,"low_b":110.445382,"close_b":111.070473,"volume_b":119561600,"open_bench":310.442933,"close_bench":310.415375},{"date":"2020-08-18","open_a":113.659257,"high_a":113.659257,"low_a":112.371861,"close_a":113.24926,"volume_a":2638600,"open_b":110.823346,"high_b":112.42,"low_b":110.488992,"close_b":111.996002,"volume_b":105633600,"open_bench":310.81027,"close_bench":311.085876},{"date":"2020-08-19","open_a":113.724858,"high_a":113.80685,"low_a":111.601079,"close_a":111.888069,"volume_a":3682200,"open_b":112.403035,"high_b":113.546618,"low_b":112.042033,"close_b":112.13652,"volume_b":145538000,"open_bench":311.46254,"close_bench":309.790649},{"date":"2020-08-20","open_a":111.240299,"high_a":112.462081,"low_a":111.2157,"close_a":112.109489,"volume_a":2263000,"open_b":112.177689,"high_b":114.738637,"low_b":112.160728,"close_b":114.624763,"volume_b":126907200,"open_bench":308.072676,"close_bench":310.755096},{"date":"2020-08-21","open_a":111.822462,"high_a":112.060267,"low_a":110.904074,"close_a":111.896271,"volume_a":4295600,"open_b":115.581793,"high_b":121.013813,"low_b":115.569681,"close_b":120.53167,"volume_b":338054800,"open_bench":310.424479,"close_bench":311.857544},{"date":"2020-08-24","open_a":111.896268,"high_a":112.896648,"low_a":111.609265,"close_a":112.863853,"volume_a":3177800,"open_b":124.725615,"high_b":124.810423,"low_b":120.112524,"close_b":121.973267,"volume_b":345937600,"open_bench":314.282746,"close_bench":315.01767},{"date":"2020-08-25","open_a":113.372262,"high_a":113.47067,"low_a":112.093074,"close_a":112.585075,"volume_a":2597400,"open_b":120.84905,"high_b":121.316658,"low_b":119.254815,"close_b":120.97261,"volume_b":211495600,"open_bench":315.577975,"close_bench":316.119965},{"date":"2020-08-26","open_a":111.994662,"high_a":113.70025,"low_a":111.830666,"close_a":113.544449,"volume_a":6019700,"open_b":122.285816,"high_b":123.07324,"low_b":121.222184,"close_b":122.617744,"volume_b":163022400,"open_bench":316.70792,"close_bench":319.289276},{"date":"2020-08-27","open_a":113.741261,"high_a":114.741642,"low_a":113.085264,"close_a":113.487053,"volume_a":4159300,"open_b":123.218603,"high_b":123.550531,"low_b":120.010751,"close_b":121.151917,"volume_b":155552400,"open_bench":320.152771,"close_bench":319.987396},{"date":"2020-08-28","open_a":113.585436,"high_a":114.766228,"low_a":112.535846,"close_a":114.749825,"volume_a":3355400,"open_b":122.123486,"high_b":122.540216,"low_b":120.732776,"close_b":120.955681,"volume_b":187630000,"open_bench":321.007125,"close_bench":322.054352},{"date":"2020-08-31","open_a":114.241442,"high_a":115.635427,"low_a":113.979038,"close_a":114.848228,"volume_a":3828700,"open_b":123.642608,"high_b":126.957058,"low_b":122.111368,"close_b":125.057541,"volume_b":225702700,"open_bench":321.843081,"close_bench":320.887695},{"date":"2020-09-01","open_a":115.004025,"high_a":115.19262,"low_a":113.577233,"close_a":114.134834,"volume_a":3421700,"open_b":128.662751,"high_b":130.639801,"low_b":126.501578,"close_b":130.038925,"volume_b":151948100,"open_bench":321.714397,"close_bench":323.909943},{"date":"2020-09-02","open_a":114.405431,"high_a":117.931403,"low_a":114.266034,"close_a":117.496796,"volume_a":4918200,"open_b":133.343685,"high_b":133.721648,"low_b":123.080518,"close_b":127.344719,"volume_b":200119000,"open_bench":325.811514,"close_bench":328.594971},{"date":"2020-09-03","open_a":117.912265,"high_a":118.961136,"low_a":114.071871,"close_a":115.104233,"volume_a":6642400,"open_b":122.993291,"high_b":124.86372,"low_b":116.781114,"close_b":117.149384,"volume_b":257599600,"open_bench":326.913939,"close_bench":317.286682},{"date":"2020-09-04","open_a":115.343726,"high_a":115.880549,"low_a":113.097306,"close_a":114.600418,"volume_a":4836400,"open_b":116.364374,"high_b":119.882341,"low_b":107.467689,"close_b":117.226906,"volume_b":332607200,"open_bench":317.966375,"close_bench":314.696045},{"date":"2020-09-08","open_a":114.228761,"high_a":114.493051,"low_a":111.007797,"close_a":111.94931,"volume_a":4901500,"open_b":110.433235,"high_b":115.31769,"low_b":109.202433,"close_b":109.338112,"volume_b":231366600,"open_bench":309.312933,"close_bench":306.097717},{"date":"2020-09-09","open_a":111.908019,"high_a":114.352657,"low_a":111.825425,"close_a":113.237701,"volume_a":4282300,"open_b":113.641111,"high_b":115.463088,"low_b":111.702835,"close_b":113.699257,"volume_b":176940500,"open_bench":310.084516,"close_bench":312.142273},{"date":"2020-09-10","open_a":112.816498,"high_a":113.601091,"low_a":110.570078,"close_a":111.296852,"volume_a":6050500,"open_b":116.645429,"high_b":116.781108,"low_b":109.028005,"close_b":109.98745,"volume_b":182274400,"open_bench":314.007134,"close_bench":306.722382},{"date":"2020-09-11","open_a":111.676775,"high_a":112.63481,"low_a":110.85914,"close_a":112.164047,"volume_a":3411900,"open_b":111.034118,"high_b":111.673753,"low_b":106.605159,"close_b":108.543434,"volume_b":180860300,"open_bench":308.495282,"close_bench":306.878479},{"date":"2020-09-14","open_a":112.758659,"high_a":113.559772,"low_a":112.155763,"close_a":112.874283,"volume_a":2779000,"open_b":111.179504,"high_b":112.35216,"low_b":109.318761,"close_b":111.799751,"volume_b":140150100,"open_bench":310.029427,"close_bench":310.920502},{"date":"2020-09-15","open_a":112.717391,"high_a":113.345075,"low_a":111.833696,"close_a":112.007126,"volume_a":4080800,"open_b":114.67807,"high_b":115.162639,"low_b":110.103738,"close_b":111.974174,"volume_b":184642000,"open_bench":313.364079,"close_bench":312.491394},{"date":"2020-09-16","open_a":112.552228,"high_a":112.816519,"low_a":111.313394,"close_a":111.478569,"volume_a":3353500,"open_b":111.673759,"high_b":112.419992,"low_b":108.582207,"close_b":108.669426,"volume_b":154679000,"open_bench":313.722346,"close_bench":311.251221},{"date":"2020-09-17","open_a":110.16539,"high_a":111.123424,"low_a":108.76138,"close_a":110.008469,"volume_a":5186400,"open_b":106.333778,"high_b":108.737236,"low_b":105.354947,"close_b":106.934639,"volume_b":178011000,"open_bench":306.419251,"close_bench":308.513733},{"date":"2020-09-18","open_a":110.429678,"high_a":111.321628,"low_a":107.373877,"close_a":108.579681,"volume_a":8473000,"open_b":106.992799,"high_b":107.457981,"low_b":102.81581,"close_b":103.542664,"volume_b":287104900,"open_bench":309.31522,"close_bench":304.961914},{"date":"2020-09-21","open_a":107.671215,"high_a":108.447553,"low_a":106.837071,"close_a":108.389748,"volume_a":4402700,"open_b":101.313684,"high_b":106.789315,"low_b":99.918123,"close_b":106.682709,"volume_b":195713800,"open_bench":300.396516,"close_bench":301.567841},{"date":"2020-09-22","open_a":108.191513,"high_a":109.752444,"low_a":107.737272,"close_a":109.141281,"volume_a":3951000,"open_b":109.202451,"high_b":109.376896,"low_b":105.791089,"close_b":108.359299,"volume_b":183055400,"open_bench":303.04349,"close_bench":304.639069},{"date":"2020-09-23","open_a":109.59553,"high_a":109.694646,"low_a":107.241753,"close_a":108.191521,"volume_a":5909900,"open_b":108.175177,"high_b":108.650053,"low_b":103.474852,"close_b":103.814056,"volume_b":150718700,"open_bench":305.192389,"close_bench":297.574127},{"date":"2020-09-24","open_a":108.554908,"high_a":109.273428,"low_a":107.299553,"close_a":108.670532,"volume_a":4316600,"open_b":101.924231,"high_b":106.847453,"low_b":101.759479,"close_b":104.880104,"volume_b":167743300,"open_bench":296.264535,"close_bench":298.367401},{"date":"2020-09-25","open_a":108.10067,"high_a":110.528787,"low_a":107.786828,"close_a":110.297539,"volume_a":4576000,"open_b":105.083602,"high_b":108.969847,"low_b":104.347056,"close_b":108.814781,"volume_b":149981400,"open_bench":297.51887,"close_bench":303.191101},{"date":"2020-09-28","open_a":112.354009,"high_a":114.451776,"low_a":112.048434,"close_a":113.947983,"volume_a":7487500,"open_b":111.460546,"high_b":111.760977,"low_b":109.299366,"close_b":111.412086,"volume_b":137672400,"open_bench":307.332347,"close_bench":308.22699},{"date":"2020-09-29","open_a":114.517833,"high_a":115.244608,"low_a":113.254224,"close_a":113.278999,"volume_a":4330500,"open_b":111.014757,"high_b":111.751296,"low_b":110.064998,"close_b":110.568947,"volume_b":99382200,"open_bench":308.023981,"close_bench":306.548279},{"date":"2020-09-30","open_a":113.964492,"high_a":114.955556,"low_a":113.064262,"close_a":114.468285,"volume_a":7060700,"open_b":110.278185,"high_b":113.641094,"low_b":110.113433,"close_b":112.23584,"volume_b":142675200,"open_bench":307.212353,"close_bench":308.872528},{"date":"2020-10-01","open_a":115.162038,"high_a":116.326545,"low_a":113.551555,"close_a":116.285248,"volume_a":6483900,"open_b":114.009366,"high_b":114.086899,"low_b":112.255229,"close_b":113.1856,"volume_b":116120400,"open_bench":311.454995,"close_bench":310.855499},{"date":"2020-10-02","open_a":115.029887,"high_a":115.550201,"low_a":112.486146,"close_a":114.022301,"volume_a":5130200,"open_b":109.405951,"high_b":111.809416,"low_b":108.756631,"close_b":109.531937,"volume_b":144712000,"open_bench":305.930293,"close_bench":307.904022},{"date":"2020-10-05","open_a":114.063568,"high_a":114.286562,"low_a":113.130309,"close_a":113.914902,"volume_a":4519800,"open_b":110.394485,"high_b":113.049921,"low_b":110.045595,"close_b":112.904549,"volume_b":106243800,"open_bench":309.951516,"close_bench":313.364075},{"date":"2020-10-06","open_a":113.972721,"high_a":114.649957,"low_a":111.618932,"close_a":112.073174,"volume_a":5031000,"open_b":112.129244,"high_b":112.536288,"low_b":108.785722,"close_b":109.667641,"volume_b":161498200,"open_bench":313.50254,"close_bench":308.909424},{"date":"2020-10-07","open_a":112.271395,"high_a":113.576302,"low_a":112.122729,"close_a":113.15509,"volume_a":4347300,"open_b":111.082586,"high_b":111.983885,"low_b":110.607704,"close_b":111.528389,"volume_b":96849000,"open_bench":311.851616,"close_bench":314.28653},{"date":"2020-10-08","open_a":113.378137,"high_a":114.080148,"low_a":112.94867,"close_a":113.8489,"volume_a":2977400,"open_b":112.662279,"high_b":112.807651,"low_b":111.053506,"close_b":111.421783,"volume_b":83477200,"open_bench":316.214036,"close_bench":317.071777},{"date":"2020-10-09","open_a":114.550847,"high_a":115.013343,"low_a":113.890133,"close_a":114.336121,"volume_a":3194000,"open_b":111.722201,"high_b":113.389119,"low_b":111.373311,"close_b":113.360046,"volume_b":100506900,"open_bench":318.713502,"close_bench":319.90329},{"date":"2020-10-12","open_a":116.549533,"high_a":118.5069,"low_a":115.442844,"close_a":117.38369,"volume_a":6167200,"open_b":116.354691,"high_b":121.316679,"low_b":115.598764,"close_b":120.560753,"volume_b":240226800,"open_bench":322.430447,"close_bench":325.049805},{"date":"2020-10-13","open_a":117.458004,"high_a":118.928087,"low_a":117.201981,"close_a":118.548172,"volume_a":4293200,"open_b":121.403874,"high_b":121.520173,"low_b":115.957325,"close_b":117.362572,"volume_b":262330500,"open_bench":324.911459,"close_bench":322.928497},{"date":"2020-10-14","open_a":118.556407,"high_a":119.018903,"low_a":116.970701,"close_a":117.705742,"volume_a":3641000,"open_b":117.265652,"high_b":119.233001,"low_b":115.928245,"close_b":117.449791,"volume_b":150712000,"open_bench":323.500336,"close_bench":320.899414},{"date":"2020-10-15","open_a":116.582553,"high_a":117.251522,"low_a":116.136566,"close_a":116.805534,"volume_a":3306800,"open_b":115.056037,"high_b":117.459495,"low_b":114.503629,"close_b":116.984619,"volume_b":112559200,"open_bench":317.007184,"close_bench":320.502747},{"date":"2020-10-16","open_a":117.069805,"high_a":117.953513,"low_a":116.557758,"close_a":117.053284,"volume_a":5076000,"open_b":117.53704,"high_b":117.798712,"low_b":115.143268,"close_b":115.346786,"volume_b":115393800,"open_bench":321.849324,"close_bench":320.309082},{"date":"2020-10-19","open_a":116.731209,"high_a":117.714019,"low_a":115.005103,"close_a":115.360229,"volume_a":3223200,"open_b":116.2578,"high_b":116.703603,"low_b":112.090511,"close_b":112.400635,"volume_b":120639300,"open_bench":321.563456,"close_bench":315.439331},{"date":"2020-10-20","open_a":116.450409,"high_a":116.632106,"low_a":115.137248,"close_a":115.385017,"volume_a":2750900,"open_b":112.613817,"high_b":115.308026,"low_b":112.061408,"close_b":113.883392,"volume_b":124423700,"open_bench":316.776684,"close_bench":316.702911},{"date":"2020-10-21","open_a":115.360227,"high_a":116.020941,"low_a":114.881222,"close_a":115.294167,"volume_a":2560100,"open_b":113.069291,"high_b":115.046332,"low_b":112.856079,"close_b":113.263123,"volume_b":89946000,"open_bench":316.656788,"close_bench":316.103424},{"date":"2020-10-22","open_a":115.459365,"high_a":115.789716,"low_a":114.806906,"close_a":115.302444,"volume_a":2909000,"open_b":113.825251,"high_b":114.397046,"low_b":111.053516,"close_b":112.177719,"volume_b":101988000,"open_bench":316.315535,"close_bench":317.837341},{"date":"2020-10-23","open_a":115.806191,"high_a":116.062214,"low_a":114.831635,"close_a":115.261101,"volume_a":2758400,"open_b":112.797951,"high_b":112.953017,"low_b":110.75307,"close_b":111.489616,"volume_b":82572600,"open_bench":319.054814,"close_bench":318.916473},{"date":"2020-10-26","open_a":114.583912,"high_a":114.864712,"low_a":112.659588,"close_a":114.022301,"volume_a":3431100,"open_b":110.491409,"high_b":112.95302,"low_b":109.396279,"close_b":111.499313,"volume_b":111850700,"open_bench":315.549948,"close_bench":313.022827},{"date":"2020-10-27","open_a":114.443522,"high_a":115.533689,"low_a":114.220528,"close_a":114.897751,"volume_a":3390800,"open_b":111.925721,"high_b":113.660479,"low_b":111.005043,"close_b":113.001465,"volume_b":92276800,"open_bench":313.364183,"close_bench":311.943817},{"date":"2020-10-28","open_a":113.27074,"high_a":113.733249,"low_a":110.619642,"close_a":110.809593,"volume_a":5196000,"open_b":111.49929,"high_b":111.86756,"low_b":107.671192,"close_b":107.768105,"volume_b":143937800,"open_bench":306.299261,"close_bench":301.281891},{"date":"2020-10-29","open_a":110.470978,"high_a":112.0154,"low_a":109.232144,"close_a":110.660942,"volume_a":3856500,"open_b":108.902025,"high_b":113.321291,"low_b":108.737266,"close_b":111.760979,"volume_b":146129200,"open_bench":301.512434,"close_bench":304.343933},{"date":"2020-10-30","open_a":109.785489,"high_a":110.537051,"low_a":108.777915,"close_a":110.082809,"volume_a":4303900,"open_b":107.632467,"high_b":108.533766,"low_b":104.395549,"close_b":105.500366,"volume_b":190272600,"open_bench":302.776107,"close_bench":301.171295},{"date":"2020-11-02","open_a":111.998849,"high_a":112.510897,"low_a":110.48747,"close_a":111.618935,"volume_a":3126400,"open_b":105.74262,"high_b":107.264165,"low_b":104.007862,"close_b":105.413109,"volume_b":122866900,"open_bench":304.546875,"close_bench":304.546875},{"date":"2020-11-03","open_a":112.832983,"high_a":114.468253,"low_a":112.552184,"close_a":113.179855,"volume_a":3293900,"open_b":106.275659,"high_b":108.049176,"low_b":105.374361,"close_b":107.031586,"volume_b":107624400,"open_bench":307.765687,"close_bench":309.923889},{"date":"2020-11-04","open_a":114.79863,"high_a":117.028529,"low_a":113.691941,"close_a":113.857117,"volume_a":4576300,"open_b":110.61739,"high_b":112.022637,"low_b":108.882633,"close_b":111.40239,"volume_b":138235500,"open_bench":314.378712,"close_bench":316.850525},{"date":"2020-11-05","open_a":115.087712,"high_a":115.847529,"low_a":113.535036,"close_a":114.014053,"volume_a":3667800,"open_b":114.309805,"high_b":115.928271,"low_b":113.263142,"close_b":115.356476,"volume_b":126387100,"open_bench":322.107567,"close_bench":323.029877},{"date":"2020-11-06","open_a":114.435245,"high_a":115.434576,"low_a":113.419404,"close_a":114.286591,"volume_a":2867000,"open_b":114.866216,"high_b":115.720526,"low_b":112.74014,"close_b":115.225418,"volume_b":114457900,"open_bench":322.744035,"close_bench":322.956177},{"date":"2020-11-09","open_a":117.276309,"high_a":118.639021,"low_a":114.11315,"close_a":114.344398,"volume_a":6111300,"open_b":116.98256,"high_b":118.429065,"low_b":112.66246,"close_b":112.924576,"volume_b":154515300,"open_bench":335.693287,"close_bench":327.014343},{"date":"2020-11-10","open_a":113.138605,"high_a":117.639701,"low_a":112.477891,"close_a":117.045059,"volume_a":5284500,"open_b":112.17707,"high_b":114.157515,"low_b":110.798515,"close_b":112.584808,"volume_b":138023400,"open_bench":326.027381,"close_bench":326.534668},{"date":"2020-11-11","open_a":117.986593,"high_a":119.126324,"low_a":116.954231,"close_a":118.845512,"volume_a":4389800,"open_b":113.769202,"high_b":116.137973,"low_b":113.041095,"close_b":116.00206,"volume_b":112295000,"open_bench":328.711377,"close_bench":328.960419},{"date":"2020-11-12","open_a":119.134531,"high_a":119.539222,"low_a":117.903952,"close_a":118.795914,"volume_a":2570100,"open_b":116.128291,"high_b":117.011724,"low_b":115.108937,"close_b":115.730255,"volume_b":103162300,"open_bench":327.955097,"close_bench":325.769226},{"date":"2020-11-13","open_a":118.73812,"high_a":119.663112,"low_a":118.325175,"close_a":119.514458,"volume_a":2580400,"open_b":115.953532,"high_b":116.176814,"low_b":114.429361,"close_b":115.778786,"volume_b":81581900,"open_bench":327.669143,"close_bench":330.279297},{"date":"2020-11-16","open_a":119.753959,"high_a":120.794575,"low_a":119.497935,"close_a":120.38163,"volume_a":3399000,"open_b":115.448714,"high_b":117.458291,"low_b":114.701194,"close_b":116.788437,"volume_b":91183000,"open_bench":332.935538,"close_bench":334.402008},{"date":"2020-11-17","open_a":119.15108,"high_a":120.66246,"low_a":119.018935,"close_a":119.952194,"volume_a":2924000,"open_b":116.060309,"high_b":117.147611,"low_b":115.487527,"close_b":115.904976,"volume_b":74271000,"open_bench":332.00402,"close_bench":332.603516},{"date":"2020-11-18","open_a":120.158674,"high_a":121.050636,"low_a":118.349973,"close_a":118.416046,"volume_a":4335200,"open_b":115.14775,"high_b":116.322429,"low_b":114.555556,"close_b":114.584679,"volume_b":76322100,"open_bench":332.87104,"close_bench":328.600739},{"date":"2020-11-19","open_a":118.0031,"high_a":118.680323,"low_a":117.557112,"close_a":118.366493,"volume_a":4263500,"open_b":114.157527,"high_b":115.584618,"low_b":113.400296,"close_b":115.17688,"volume_b":74113000,"open_bench":327.973471,"close_bench":329.9841},{"date":"2020-11-20","open_a":118.639025,"high_a":119.45666,"low_a":118.102202,"close_a":118.449074,"volume_a":3469100,"open_b":115.176889,"high_b":115.303092,"low_b":113.866297,"close_b":113.914833,"volume_b":73604300,"open_bench":329.725882,"close_bench":327.724457},{"date":"2020-11-23","open_a":118.358231,"high_a":118.63903,"low_a":117.177215,"close_a":118.118729,"volume_a":3443300,"open_b":113.759502,"high_b":114.18666,"low_b":110.429623,"close_b":110.526703,"volume_b":127959300,"open_bench":329.522986,"close_bench":329.688995},{"date":"2020-11-24","open_a":118.259121,"high_a":119.225411,"low_a":117.953546,"close_a":118.952866,"volume_a":4472300,"open_b":110.584936,"high_b":112.468302,"low_b":109.30346,"close_b":111.808151,"volume_b":113874200,"open_bench":332.225303,"close_bench":335.001465},{"date":"2020-11-25","open_a":119.299752,"high_a":119.687921,"low_a":118.680335,"close_a":119.060249,"volume_a":2278300,"open_b":112.177082,"high_b":113.342051,"low_b":111.808169,"close_b":112.643066,"volume_b":76499200,"open_bench":334.918472,"close_bench":334.484985},{"date":"2020-11-27","open_a":119.440131,"high_a":119.985221,"low_a":119.076737,"close_a":119.423622,"volume_a":1596000,"open_b":113.167316,"high_b":114.06046,"low_b":112.827534,"close_b":113.186729,"volume_b":46691300,"open_bench":335.573403,"close_bench":335.416626},{"date":"2020-11-30","open_a":119.307998,"high_a":119.448398,"low_a":118.003091,"close_a":119.118034,"volume_a":4906900,"open_b":113.555642,"high_b":117.438882,"low_b":113.400309,"close_b":115.574928,"volume_b":169410200,"open_bench":334.641809,"close_bench":333.931641},{"date":"2020-12-01","open_a":119.778733,"high_a":121.100161,"low_a":119.341012,"close_a":120.637665,"volume_a":3999200,"open_b":117.477688,"high_b":119.865879,"low_b":116.506878,"close_b":119.137772,"volume_b":127728200,"open_bench":337.168992,"close_bench":337.584015},{"date":"2020-12-02","open_a":120.480753,"high_a":121.356194,"low_a":118.837229,"close_a":119.440125,"volume_a":4432900,"open_b":118.458226,"high_b":119.768825,"low_b":117.361213,"close_b":119.487289,"volume_b":89004200,"open_bench":336.477237,"close_bench":338.294189},{"date":"2020-12-03","open_a":118.960213,"high_a":120.573855,"low_a":118.960213,"close_a":120.149643,"volume_a":4757900,"open_b":119.914444,"high_b":120.166856,"low_b":118.642685,"close_b":119.351379,"volume_b":78967600,"open_bench":338.192703,"close_bench":338.201935},{"date":"2020-12-04","open_a":119.667167,"high_a":121.422209,"low_a":119.650541,"close_a":121.314087,"volume_a":3592600,"open_b":119.021278,"high_b":119.27369,"low_b":117.972801,"close_b":118.681496,"volume_b":78260400,"open_bench":338.783011,"close_bench":341.116455},{"date":"2020-12-07","open_a":121.172734,"high_a":121.904699,"low_a":120.291056,"close_a":120.914886,"volume_a":4217300,"open_b":118.73975,"high_b":120.933782,"low_b":118.681503,"close_b":120.137718,"volume_b":86712000,"open_bench":340.350928,"close_bench":340.415497},{"date":"2020-12-08","open_a":120.881634,"high_a":121.572007,"low_a":120.016582,"close_a":121.039673,"volume_a":3623800,"open_b":120.739611,"high_b":121.331805,"low_b":119.496968,"close_b":120.749313,"volume_b":82225500,"open_bench":339.151889,"close_bench":341.41156},{"date":"2020-12-09","open_a":121.072873,"high_a":121.522037,"low_a":120.008204,"close_a":121.181007,"volume_a":3842000,"open_b":120.894939,"high_b":122.273487,"low_b":117.467982,"close_b":118.225212,"volume_b":115089200,"open_bench":342.066458,"close_bench":338.349548},{"date":"2020-12-10","open_a":121.472159,"high_a":121.680103,"low_a":120.124691,"close_a":120.332634,"volume_a":3985000,"open_b":116.982577,"high_b":120.254208,"low_b":116.642795,"close_b":119.642593,"volume_b":81312200,"open_bench":336.984504,"close_bench":338.238861},{"date":"2020-12-11","open_a":119.467565,"high_a":120.690273,"low_a":119.284574,"close_a":120.582138,"volume_a":2975500,"open_b":118.856258,"high_b":119.176627,"low_b":117.031138,"close_b":118.836845,"volume_b":86939800,"open_bench":336.550965,"close_bench":337.842194},{"date":"2020-12-14","open_a":120.815049,"high_a":121.829813,"low_a":119.816924,"close_a":119.966637,"volume_a":3952400,"open_b":119.021276,"high_b":119.749383,"low_b":117.99222,"close_b":118.225212,"volume_b":79184500,"open_bench":340.000332,"close_bench":336.329529},{"date":"2020-12-15","open_a":120.241116,"high_a":120.831681,"low_a":119.68383,"close_a":120.415794,"volume_a":3183300,"open_b":120.710503,"high_b":124.166591,"low_b":120.506634,"close_b":124.147171,"volume_b":157243700,"open_bench":338.856695,"close_bench":340.876556},{"date":"2020-12-16","open_a":119.75867,"high_a":120.998017,"low_a":119.692127,"close_a":120.515587,"volume_a":4915800,"open_b":123.690868,"high_b":124.622837,"low_b":122.865673,"close_b":124.079185,"volume_b":98208600,"open_bench":341.088746,"close_bench":341.41156},{"date":"2020-12-17","open_a":120.815041,"high_a":121.605223,"low_a":120.756811,"close_a":121.197662,"volume_a":3312500,"open_b":125.137373,"high_b":125.797532,"low_b":124.302476,"close_b":124.943214,"volume_b":94359800,"open_bench":343.044119,"close_bench":343.320801},{"date":"2020-12-18","open_a":121.713361,"high_a":122.35383,"low_a":121.022988,"close_a":122.212418,"volume_a":8980400,"open_b":125.195657,"high_b":125.33157,"low_b":122.438553,"close_b":122.962791,"volume_b":192541500,"open_bench":343.607901,"close_bench":341.949921},{"date":"2020-12-21","open_a":120.607146,"high_a":121.156119,"low_a":119.334533,"close_a":120.93985,"volume_a":3946900,"open_b":121.370647,"high_b":124.564613,"low_b":119.846476,"close_b":124.486946,"volume_b":121251600,"open_bench":338.050471,"close_bench":340.727295},{"date":"2020-12-22","open_a":120.274392,"high_a":120.607096,"low_a":119.342797,"close_a":119.791962,"volume_a":2653700,"open_b":127.768288,"high_b":130.486559,"low_b":125.865494,"close_b":128.030411,"volume_b":168904800,"open_bench":341.05141,"close_bench":340.152954},{"date":"2020-12-23","open_a":120.182907,"high_a":121.106176,"low_a":119.833564,"close_a":120.116364,"volume_a":2645100,"open_b":128.302245,"high_b":128.564353,"low_b":126.962522,"close_b":127.137276,"volume_b":88223700,"open_bench":341.116333,"close_bench":340.45871},{"date":"2020-12-24","open_a":120.465693,"high_a":120.698589,"low_a":119.891768,"close_a":120.656998,"volume_a":883300,"open_b":127.486732,"high_b":129.564264,"low_b":127.273152,"close_b":128.117752,"volume_b":54930100,"open_bench":340.93117,"close_bench":341.783325},{"date":"2020-12-28","open_a":121.255901,"high_a":122.769734,"low_a":121.05627,"close_a":122.195808,"volume_a":2493500,"open_b":130.078798,"high_b":133.331002,"low_b":129.612799,"close_b":132.699982,"volume_b":124486200,"open_bench":344.321116,"close_bench":344.719421},{"date":"2020-12-29","open_a":122.836259,"high_a":123.742902,"low_a":122.137573,"close_a":122.620003,"volume_a":3186700,"open_b":134.020288,"high_b":134.738678,"low_b":130.418577,"close_b":130.933105,"volume_b":121047300,"open_bench":346.238441,"close_bench":344.061768},{"date":"2020-12-30","open_a":122.661552,"high_a":123.094077,"low_a":122.278931,"close_a":122.528465,"volume_a":2224900,"open_b":131.622419,"high_b":132.020454,"low_b":129.506045,"close_b":129.816711,"volume_b":96452100,"open_bench":344.876894,"close_bench":344.552704},{"date":"2020-12-31","open_a":122.669899,"high_a":123.485046,"low_a":121.971213,"close_a":123.351959,"volume_a":2076800,"open_b":130.166198,"high_b":130.806936,"low_b":127.875086,"close_b":128.816772,"volume_b":99116600,"open_bench":344.358231,"close_bench":346.303345},{"date":"2021-01-04","open_a":122.278951,"high_a":122.91942,"low_a":118.061868,"close_a":119.999901,"volume_a":7486900,"open_b":129.622529,"high_b":129.709898,"low_b":123.059852,"close_b":125.6325,"volume_b":143301900,"open_bench":347.62777,"close_bench":341.588684},{"date":"2021-01-05","open_a":119.833596,"high_a":120.756853,"low_a":118.86041,"close_a":120.357605,"volume_a":4126000,"open_b":125.127686,"high_b":127.894501,"low_b":124.681107,"close_b":127.185799,"volume_b":97664900,"open_bench":340.949542,"close_bench":343.941284},{"date":"2021-01-06","open_a":117.837323,"high_a":119.442652,"low_a":117.77078,"close_a":118.885353,"volume_a":4843300,"open_b":123.991828,"high_b":127.224626,"low_b":122.690939,"close_b":122.904518,"volume_b":155088000,"open_bench":342.440824,"close_bench":345.997589},{"date":"2021-01-07","open_a":118.785508,"high_a":119.068308,"low_a":117.404762,"close_a":118.502708,"volume_a":4473200,"open_b":124.613133,"high_b":127.787685,"low_b":124.127729,"close_b":127.098404,"volume_b":109578200,"open_bench":348.359489,"close_bench":351.138214},{"date":"2021-01-08","open_a":118.502701,"high_a":120.108017,"low_a":117.828967,"close_a":119.925026,"volume_a":4312000,"open_b":128.564332,"high_b":128.758506,"low_b":126.428554,"close_b":128.195435,"volume_b":105158200,"open_bench":352.518352,"close_bench":353.138947},{"date":"2021-01-11","open_a":119.800295,"high_a":120.033191,"low_a":117.853949,"close_a":118.186653,"volume_a":3902700,"open_b":125.418896,"high_b":126.370285,"low_b":124.749035,"close_b":125.215019,"volume_b":100384500,"open_bench":349.980414,"close_bench":350.758453},{"date":"2021-01-12","open_a":117.80403,"high_a":118.111782,"low_a":115.990758,"close_a":117.637665,"volume_a":5150000,"open_b":124.749075,"high_b":125.904342,"low_b":123.156948,"close_b":125.040321,"volume_b":91951100,"open_bench":350.943753,"close_bench":350.832581},{"date":"2021-01-13","open_a":117.379831,"high_a":119.143199,"low_a":117.280023,"close_a":118.602539,"volume_a":4361600,"open_b":125.001498,"high_b":127.61298,"low_b":124.73939,"close_b":127.069328,"volume_b":88636800,"open_bench":350.758564,"close_bench":351.777435},{"date":"2021-01-14","open_a":117.679275,"high_a":118.477783,"low_a":117.338258,"close_a":117.91217,"volume_a":4724700,"open_b":126.981932,"high_b":127.176091,"low_b":125.001472,"close_b":125.147102,"volume_b":90221800,"open_bench":352.518279,"close_bench":350.54538},{"date":"2021-01-15","open_a":117.654305,"high_a":118.610852,"low_a":116.889075,"close_a":117.604401,"volume_a":4309100,"open_b":125.020892,"high_b":126.41886,"low_b":123.292852,"close_b":123.428764,"volume_b":111598500,"open_bench":348.933831,"close_bench":347.989075},{"date":"2021-01-19","open_a":118.427863,"high_a":118.594215,"low_a":117.645994,"close_a":118.16169,"volume_a":5158600,"open_b":124.050082,"high_b":124.952943,"low_b":123.234605,"close_b":124.098625,"volume_b":90757300,"open_bench":350.434181,"close_bench":350.721313},{"date":"2021-01-20","open_a":117.313283,"high_a":117.9288,"low_a":116.381701,"close_a":117.554504,"volume_a":5019300,"open_b":124.904416,"high_b":128.62262,"low_b":124.797626,"close_b":128.176041,"volume_b":104319500,"open_bench":353.000002,"close_bench":355.574982},{"date":"2021-01-21","open_a":116.689471,"high_a":117.105357,"low_a":115.400232,"close_a":116.123871,"volume_a":5684100,"open_b":129.894362,"high_b":135.593011,"low_b":129.690485,"close_b":132.874741,"volume_b":120150900,"open_bench":356.130639,"close_bench":355.899078},{"date":"2021-01-22","open_a":116.032345,"high_a":116.065611,"low_a":114.97599,"close_a":115.275429,"volume_a":4014900,"open_b":132.301976,"high_b":135.767774,"low_b":131.078761,"close_b":135.010544,"volume_b":114459400,"open_bench":354.055898,"close_bench":354.639435},{"date":"2021-01-25","open_a":114.834614,"high_a":116.722743,"low_a":114.435366,"close_a":116.597969,"volume_a":3829000,"open_b":138.893754,"high_b":140.854778,"low_b":132.554352,"close_b":138.748123,"volume_b":157611700,"open_bench":355.371191,"close_bench":356.038086},{"date":"2021-01-26","open_a":117.537869,"high_a":118.186651,"low_a":116.240305,"close_a":117.945442,"volume_a":4242500,"open_b":139.408284,"high_b":140.087847,"low_b":137.243367,"close_b":138.981125,"volume_b":98390600,"open_bench":356.982837,"close_bench":355.48233},{"date":"2021-01-27","open_a":116.872436,"high_a":118.386269,"low_a":114.410394,"close_a":114.817955,"volume_a":6659800,"open_b":139.243243,"high_b":140.087858,"low_b":136.311408,"close_b":137.913239,"volume_b":140843800,"open_bench":352.175621,"close_bench":346.794159},{"date":"2021-01-28","open_a":115.475059,"high_a":117.105327,"low_a":114.585056,"close_a":115.774498,"volume_a":5514000,"open_b":135.447378,"high_b":137.845279,"low_b":132.709687,"close_b":133.088303,"volume_b":142621100,"open_bench":348.600358,"close_bench":349.776703},{"date":"2021-01-29","open_a":114.876193,"high_a":115.167319,"low_a":113.129465,"close_a":113.595268,"volume_a":6778100,"open_b":131.865045,"high_b":132.748485,"low_b":126.4091,"close_b":128.108017,"volume_b":177523800,"open_bench":347.924194,"close_bench":342.774292},{"date":"2021-02-01","open_a":113.936279,"high_a":114.751426,"low_a":113.037963,"close_a":113.936279,"volume_a":4185700,"open_b":129.845796,"high_b":131.42822,"low_b":127.108105,"close_b":130.224411,"volume_b":106239800,"open_bench":346.155104,"close_bench":348.47998},{"date":"2021-02-02","open_a":114.327213,"high_a":116.057303,"low_a":113.620214,"close_a":115.100769,"volume_a":3581600,"open_b":131.767996,"high_b":132.331067,"low_b":130.680694,"close_b":131.049606,"volume_b":83305400,"open_bench":351.647708,"close_bench":353.407562},{"date":"2021-02-03","open_a":114.576771,"high_a":115.574897,"low_a":114.011171,"close_a":114.801353,"volume_a":3609600,"open_b":131.79714,"high_b":131.806857,"low_b":129.709904,"close_b":130.030273,"volume_b":89880900,"open_bench":354.231843,"close_bench":353.685364},{"date":"2021-02-04","open_a":114.576737,"high_a":116.248596,"low_a":114.576737,"close_a":116.182053,"volume_a":3860300,"open_b":132.321368,"high_b":133.38925,"low_b":130.661277,"close_b":133.379547,"volume_b":84183100,"open_bench":354.713551,"close_bench":357.705322},{"date":"2021-02-05","open_a":116.855808,"high_a":118.211589,"low_a":116.223652,"close_a":117.246742,"volume_a":4437900,"open_b":133.539984,"high_b":133.608035,"low_b":132.091311,"close_b":132.966339,"volume_b":75693800,"open_bench":359.567037,"close_bench":359.113159},{"date":"2021-02-08","open_a":117.438053,"high_a":117.953748,"low_a":116.032354,"close_a":116.780945,"volume_a":4112200,"open_b":132.256604,"high_b":133.160814,"low_b":131.177394,"close_b":133.112198,"volume_b":71297200,"open_bench":360.558136,"close_bench":361.706696},{"date":"2021-02-09","open_a":117.388151,"high_a":117.404778,"low_a":116.024044,"close_a":116.11554,"volume_a":3179300,"open_b":132.830247,"high_b":134.055305,"low_b":132.081617,"close_b":132.237167,"volume_b":76774200,"open_bench":360.873073,"close_bench":361.465881},{"date":"2021-02-10","open_a":117.379834,"high_a":117.379834,"low_a":113.753303,"close_a":114.535172,"volume_a":6966600,"open_b":132.694114,"high_b":133.189976,"low_b":130.67181,"close_b":131.634354,"volume_b":73046600,"open_bench":363.197891,"close_bench":361.30835},{"date":"2021-02-11","open_a":114.659922,"high_a":114.959361,"low_a":111.740391,"close_a":112.264412,"volume_a":7528000,"open_b":132.130222,"high_b":132.606635,"low_b":130.059317,"close_b":131.381592,"volume_b":64280000,"open_bench":362.382844,"close_bench":361.891937},{"date":"2021-02-12","open_a":112.505629,"high_a":112.788441,"low_a":111.000121,"close_a":111.349464,"volume_a":5659300,"open_b":130.623204,"high_b":131.770464,"low_b":129.981508,"close_b":131.614899,"volume_b":60145100,"open_bench":361.095343,"close_bench":363.679565},{"date":"2021-02-16","open_a":111.923402,"high_a":112.131346,"low_a":110.958542,"close_a":111.773689,"volume_a":5180100,"open_b":131.731594,"high_b":132.237159,"low_b":129.106479,"close_b":129.495392,"volume_b":80576300,"open_bench":364.902189,"close_bench":363.364624},{"date":"2021-02-17","open_a":111.507531,"high_a":112.34763,"low_a":111.341179,"close_a":111.840248,"volume_a":3353000,"open_b":127.609186,"high_b":128.55228,"low_b":125.878564,"close_b":127.210556,"volume_b":97918500,"open_bench":361.623331,"close_bench":363.448029},{"date":"2021-02-18","open_a":111.515827,"high_a":112.871609,"low_a":111.382741,"close_a":112.597122,"volume_a":3935200,"open_b":125.616053,"high_b":126.393865,"low_b":123.875713,"close_b":126.111916,"volume_b":96856700,"open_bench":360.854466,"close_bench":361.901123},{"date":"2021-02-19","open_a":112.297679,"high_a":112.389175,"low_a":110.035255,"close_a":110.218246,"volume_a":5414900,"open_b":126.627241,"high_b":127.084204,"low_b":125.227183,"close_b":126.267494,"volume_b":87668800,"open_bench":363.151626,"close_bench":361.262085},{"date":"2021-02-22","open_a":109.827315,"high_a":110.293118,"low_a":108.762646,"close_a":109.785736,"volume_a":5792900,"open_b":124.45906,"high_b":126.121632,"low_b":122.115916,"close_b":122.504822,"volume_b":103916400,"open_bench":358.511093,"close_bench":358.483307},{"date":"2021-02-23","open_a":110.334706,"high_a":111.740405,"low_a":109.21182,"close_a":110.442841,"volume_a":6264300,"open_b":120.32696,"high_b":123.195126,"low_b":115.105919,"close_b":122.368706,"volume_b":158273000,"open_bench":356.288148,"close_bench":358.918671},{"date":"2021-02-24","open_a":109.669294,"high_a":110.035264,"low_a":109.128634,"close_a":109.868912,"volume_a":4944000,"open_b":121.474233,"high_b":122.07703,"low_b":118.839408,"close_b":121.872856,"volume_b":111039900,"open_bench":357.834991,"close_bench":362.873749},{"date":"2021-02-25","open_a":109.253407,"high_a":109.868911,"low_a":107.964156,"close_a":108.130508,"volume_a":5702300,"open_b":121.22144,"high_b":122.952063,"low_b":117.196282,"close_b":117.633797,"volume_b":148199500,"open_bench":361.614054,"close_bench":354.130005},{"date":"2021-02-26","open_a":108.604623,"high_a":108.854145,"low_a":106.941077,"close_a":107.456772,"volume_a":8329000,"open_b":119.189417,"high_b":121.386728,"low_b":117.837976,"close_b":117.896317,"volume_b":164560400,"open_bench":356.00096,"close_bench":352.305237},{"date":"2021-03-01","open_a":108.637889,"high_a":109.769101,"low_a":108.413307,"close_a":108.646202,"volume_a":5287300,"open_b":120.317245,"high_b":124.381294,"low_b":119.383875,"close_b":124.245178,"volume_b":116307900,"open_bench":357.14955,"close_bench":360.845245},{"date":"2021-03-02","open_a":108.546399,"high_a":109.819011,"low_a":108.44659,"close_a":109.020515,"volume_a":5614000,"open_b":124.847969,"high_b":125.149367,"low_b":121.542282,"close_b":121.649231,"volume_b":102260900,"open_bench":361.067521,"close_bench":358.029449},{"date":"2021-03-03","open_a":107.964157,"high_a":108.571348,"low_a":107.282097,"close_a":107.415184,"volume_a":7505800,"open_b":121.347835,"high_b":122.22287,"low_b":118.46022,"close_b":118.674118,"volume_b":112966300,"open_bench":357.334812,"close_bench":353.28714},{"date":"2021-03-04","open_a":108.36511,"high_a":110.461147,"low_a":107.585391,"close_a":108.012978,"volume_a":9102800,"open_b":118.372726,"high_b":120.171406,"low_b":115.329553,"close_b":116.797661,"volume_b":178155000,"open_bench":353.101886,"close_bench":348.915283},{"date":"2021-03-05","open_a":108.692106,"high_a":111.852934,"low_a":108.314822,"close_a":111.534332,"volume_a":10186500,"open_b":117.624075,"high_b":118.557444,"low_b":114.308663,"close_b":118.051865,"volume_b":153766600,"open_bench":352.397908,"close_bench":355.334106},{"date":"2021-03-08","open_a":111.701988,"high_a":112.498479,"low_a":110.662362,"close_a":110.779739,"volume_a":8634900,"open_b":117.575455,"high_b":117.643513,"low_b":112.986384,"close_b":113.132225,"volume_b":154376600,"open_bench":356.288186,"close_bench":353.565033},{"date":"2021-03-09","open_a":111.458864,"high_a":112.272127,"low_a":110.670752,"close_a":110.880356,"volume_a":8930200,"open_b":115.728149,"high_b":118.674097,"low_b":115.494808,"close_b":117.731003,"volume_b":129525800,"open_bench":357.390422,"close_bench":358.613068},{"date":"2021-03-10","open_a":111.10675,"high_a":112.741657,"low_a":109.44668,"close_a":111.995468,"volume_a":7791700,"open_b":118.31439,"high_b":118.781071,"low_b":116.136521,"close_b":116.651825,"volume_b":111943300,"open_bench":360.947147,"close_bench":360.845245},{"date":"2021-03-11","open_a":111.802632,"high_a":112.833879,"low_a":111.593028,"close_a":111.693634,"volume_a":4458900,"open_b":119.140825,"high_b":119.792238,"low_b":117.896332,"close_b":118.576912,"volume_b":103026500,"open_bench":363.299773,"close_bench":364.503876},{"date":"2021-03-12","open_a":111.609769,"high_a":112.171505,"low_a":111.207333,"close_a":111.542694,"volume_a":5275100,"open_b":117.060162,"high_b":117.8088,"low_b":115.854561,"close_b":117.672684,"volume_b":88105100,"open_bench":363.151661,"close_bench":364.994873},{"date":"2021-03-15","open_a":111.978691,"high_a":112.037386,"low_a":110.645615,"close_a":111.534332,"volume_a":5389200,"open_b":118.042156,"high_b":120.560308,"low_b":117.079613,"close_b":120.550583,"volume_b":92403800,"open_bench":365.24491,"close_bench":367.171509},{"date":"2021-03-16","open_a":111.576245,"high_a":112.833867,"low_a":111.299567,"close_a":112.355965,"volume_a":5544700,"open_b":122.213153,"high_b":123.690993,"low_b":121.260341,"close_b":122.086761,"volume_b":115227900,"open_bench":367.782848,"close_bench":366.708405},{"date":"2021-03-17","open_a":112.649423,"high_a":112.817104,"low_a":111.82778,"close_a":112.028992,"volume_a":6199500,"open_b":120.608928,"high_b":122.368718,"low_b":118.946356,"close_b":121.299232,"volume_b":111932600,"open_bench":365.430212,"close_bench":367.958862},{"date":"2021-03-18","open_a":110.972583,"high_a":111.576243,"low_a":110.176092,"close_a":111.115112,"volume_a":6476700,"open_b":119.471372,"high_b":119.763053,"low_b":116.982388,"close_b":117.186562,"volume_b":121229700,"open_bench":365.383828,"close_bench":362.605103},{"date":"2021-03-19","open_a":110.930662,"high_a":113.194379,"low_a":110.821664,"close_a":112.766792,"volume_a":16114400,"open_b":116.574045,"high_b":118.061602,"low_b":116.360146,"close_b":116.661545,"volume_b":185549500,"open_bench":362.305878,"close_bench":361.934174},{"date":"2021-03-22","open_a":113.714203,"high_a":115.868923,"low_a":113.261452,"close_a":115.533562,"volume_a":7813400,"open_b":116.992109,"high_b":120.433912,"low_b":116.924051,"close_b":119.967224,"volume_b":111912300,"open_bench":362.445185,"close_bench":364.824127},{"date":"2021-03-23","open_a":116.078554,"high_a":117.520629,"low_a":115.852179,"close_a":116.925362,"volume_a":6978400,"open_b":119.908887,"high_b":120.79364,"low_b":118.751895,"close_b":119.1408,"volume_b":95467100,"open_bench":364.192347,"close_bench":361.952789},{"date":"2021-03-24","open_a":116.548044,"high_a":117.470305,"low_a":116.355212,"close_a":116.380363,"volume_a":5082300,"open_b":119.413025,"high_b":119.490808,"low_b":116.739309,"close_b":116.758751,"volume_b":88530500,"open_bench":363.346712,"close_bench":360.112823},{"date":"2021-03-25","open_a":116.690551,"high_a":117.587661,"low_a":115.986286,"close_a":117.067848,"volume_a":5827600,"open_b":116.224042,"high_b":118.285237,"low_b":115.69902,"close_b":117.244911,"volume_b":98844700,"open_bench":358.681675,"close_bench":362.13858},{"date":"2021-03-26","open_a":117.059428,"high_a":119.893273,"low_a":116.648613,"close_a":119.641747,"volume_a":6785600,"open_b":117.011557,"high_b":118.110217,"low_b":115.621224,"close_b":117.847702,"volume_b":94071200,"open_bench":363.281569,"close_bench":367.974426},{"date":"2021-03-29","open_a":119.784309,"high_a":121.536605,"low_a":119.407025,"close_a":121.410835,"volume_a":6105800,"open_b":118.275495,"high_b":119.179697,"low_b":117.381017,"close_b":118.022705,"volume_b":80819200,"open_bench":366.506231,"close_bench":367.788635},{"date":"2021-03-30","open_a":120.463422,"high_a":121.360532,"low_a":119.155497,"close_a":119.507629,"volume_a":4372700,"open_b":116.778194,"high_b":117.060151,"low_b":115.562869,"close_b":116.57402,"volume_b":85671900,"open_bench":366.524824,"close_bench":366.812897},{"date":"2021-03-31","open_a":120.161586,"high_a":120.169966,"low_a":118.358987,"close_a":118.593742,"volume_a":6257900,"open_b":118.275497,"high_b":120.093619,"low_b":117.789367,"close_b":118.761627,"volume_b":118323800,"open_bench":367.37974,"close_bench":368.299713},{"date":"2021-04-01","open_a":118.375794,"high_a":119.306435,"low_a":117.470304,"close_a":118.451248,"volume_a":4823400,"open_b":120.229723,"high_b":120.735296,"low_b":119.092173,"close_b":119.588028,"volume_b":75089100,"open_bench":370.223317,"close_bench":372.277008},{"date":"2021-04-05","open_a":118.929122,"high_a":121.016766,"low_a":118.484763,"close_a":120.027443,"volume_a":5980700,"open_b":120.43392,"high_b":122.660398,"low_b":119.656109,"close_b":122.407608,"volume_b":88651200,"open_bench":374.925498,"close_bench":377.620392},{"date":"2021-04-06","open_a":119.926835,"high_a":120.899399,"low_a":119.524399,"close_a":120.45504,"volume_a":4751900,"open_b":122.990948,"high_b":123.60347,"low_b":122.164529,"close_b":122.708992,"volume_b":80171300,"open_bench":377.062844,"close_bench":377.397369},{"date":"2021-04-07","open_a":120.731737,"high_a":120.815584,"low_a":119.910095,"close_a":120.237076,"volume_a":4849200,"open_b":122.339541,"high_b":124.371561,"low_b":121.668678,"close_b":124.352119,"volume_b":83466700,"open_bench":377.230053,"close_bench":377.834076},{"date":"2021-04-08","open_a":120.463422,"high_a":120.86587,"low_a":119.096814,"close_a":119.507629,"volume_a":6725100,"open_b":125.372975,"high_b":126.773033,"low_b":124.95491,"close_b":126.743866,"volume_b":88844600,"open_bench":379.079294,"close_bench":379.627563},{"date":"2021-04-09","open_a":119.256105,"high_a":120.010673,"low_a":118.7279,"close_a":119.532784,"volume_a":4950800,"open_b":126.199453,"high_b":129.349568,"low_b":125.878605,"close_b":129.310684,"volume_b":106686700,"open_bench":379.506842,"close_bench":382.387573},{"date":"2021-04-12","open_a":119.222574,"high_a":120.287364,"low_a":119.096817,"close_a":119.91008,"volume_a":4386700,"open_b":128.843972,"high_b":129.16482,"low_b":127.0064,"close_b":127.59948,"volume_b":91420000,"open_bench":381.792842,"close_bench":382.526978},{"date":"2021-04-13","open_a":119.113599,"high_a":120.706582,"low_a":118.945906,"close_a":119.935242,"volume_a":5283300,"open_b":128.766139,"high_b":130.924558,"low_b":128.270277,"close_b":130.700928,"volume_b":91266500,"open_bench":382.424751,"close_bench":383.660675},{"date":"2021-04-14","open_a":119.532809,"high_a":119.658566,"low_a":118.417716,"close_a":119.147133,"volume_a":4685300,"open_b":131.196851,"high_b":131.255184,"low_b":128.007837,"close_b":128.367569,"volume_b":87222800,"open_bench":383.632687,"close_bench":382.350311},{"date":"2021-04-15","open_a":118.778196,"high_a":119.977136,"low_a":118.568593,"close_a":119.314781,"volume_a":4777100,"open_b":130.107928,"high_b":131.255189,"low_b":129.932913,"close_b":130.769058,"volume_b":89347100,"open_bench":384.478402,"close_bench":386.457764},{"date":"2021-04-16","open_a":120.278995,"high_a":121.000032,"low_a":119.44058,"close_a":120.865883,"volume_a":6481400,"open_b":130.574562,"high_b":130.934293,"low_b":129.582852,"close_b":130.438446,"volume_b":84922400,"open_bench":387.740088,"close_bench":387.74939},{"date":"2021-04-19","open_a":121.142559,"high_a":122.0732,"low_a":120.1113,"close_a":121.360542,"volume_a":6621200,"open_b":129.806486,"high_b":131.712123,"low_b":129.641203,"close_b":131.099594,"volume_b":94264200,"open_bench":386.820269,"close_bench":385.844513},{"date":"2021-04-20","open_a":120.840709,"high_a":122.953504,"low_a":120.673015,"close_a":122.165405,"volume_a":4999700,"open_b":131.274608,"high_b":131.770456,"low_b":128.153646,"close_b":129.417587,"volume_b":94812300,"open_bench":384.63637,"close_bench":383.01944},{"date":"2021-04-21","open_a":123.540447,"high_a":123.917731,"low_a":122.584654,"close_a":123.230225,"volume_a":6316000,"open_b":128.688392,"high_b":130.039833,"low_b":127.657798,"close_b":129.796768,"volume_b":68847100,"open_bench":382.406152,"close_bench":386.643646},{"date":"2021-04-22","open_a":122.609784,"high_a":122.953524,"low_a":121.888746,"close_a":122.467255,"volume_a":4904400,"open_b":129.349538,"high_b":130.428747,"low_b":127.764763,"close_b":128.28006,"volume_b":84566500,"open_bench":386.476367,"close_bench":383.112366},{"date":"2021-04-23","open_a":121.997723,"high_a":122.693609,"low_a":121.620439,"close_a":122.266022,"volume_a":4354700,"open_b":128.493954,"high_b":131.371836,"low_b":128.493954,"close_b":130.59404,"volume_b":78657500,"open_bench":383.669883,"close_bench":387.266174},{"date":"2021-04-26","open_a":122.031275,"high_a":122.215727,"low_a":119.876543,"close_a":120.195145,"volume_a":5817700,"open_b":131.08989,"high_b":131.313506,"low_b":129.855115,"close_b":130.982941,"volume_b":66905100,"open_bench":387.916685,"close_bench":388.074646},{"date":"2021-04-27","open_a":119.574741,"high_a":120.044252,"low_a":119.163926,"close_a":119.801117,"volume_a":4693700,"open_b":131.264906,"high_b":131.653819,"low_b":130.389878,"close_b":130.662109,"volume_b":66015800,"open_bench":388.372095,"close_bench":387.991089},{"date":"2021-04-28","open_a":120.30414,"high_a":120.471821,"low_a":118.828522,"close_a":118.912369,"volume_a":4962700,"open_b":130.584319,"high_b":131.27463,"low_b":129.388442,"close_b":129.874573,"volume_b":107760100,"open_bench":388.260553,"close_bench":387.879547},{"date":"2021-04-29","open_a":119.138724,"high_a":121.092219,"low_a":119.029726,"close_a":120.681404,"volume_a":4699900,"open_b":132.684393,"high_b":133.267755,"low_b":128.775901,"close_b":129.777328,"volume_b":151101000,"open_bench":390.593061,"close_bench":390.35144},{"date":"2021-04-30","open_a":120.161617,"high_a":120.932958,"low_a":119.474111,"close_a":120.865883,"volume_a":5051100,"open_b":128.124524,"high_b":129.855147,"low_b":127.434227,"close_b":127.813408,"volume_b":109839500,"open_bench":388.093237,"close_bench":387.78656},{"date":"2021-05-03","open_a":121.595315,"high_a":123.062541,"low_a":120.815595,"close_a":122.232506,"volume_a":6109800,"open_b":128.377319,"high_b":130.351022,"low_b":128.173152,"close_b":128.863449,"volume_b":75135100,"open_bench":389.765946,"close_bench":388.622955},{"date":"2021-05-04","open_a":122.391785,"high_a":123.05414,"low_a":120.30414,"close_a":120.698196,"volume_a":6603800,"open_b":127.550843,"high_b":127.842524,"low_b":123.185389,"close_b":124.30349,"volume_b":137564700,"open_bench":386.643649,"close_bench":386.225464},{"date":"2021-05-05","open_a":120.496982,"high_a":121.20964,"low_a":120.31253,"close_a":120.639511,"volume_a":5305200,"open_b":125.616055,"high_b":126.831381,"low_b":124.420179,"close_b":124.546577,"volume_b":84000900,"open_bench":387.860976,"close_bench":386.346252},{"date":"2021-05-06","open_a":121.125796,"high_a":122.375039,"low_a":120.74012,"close_a":122.039665,"volume_a":4472200,"open_b":124.342425,"high_b":126.150831,"low_b":123.603505,"close_b":126.141113,"volume_b":78128300,"open_bench":386.420616,"close_bench":389.431488},{"date":"2021-05-07","open_a":121.762987,"high_a":122.40857,"low_a":121.209643,"close_a":122.039665,"volume_a":4181000,"open_b":127.436398,"high_b":127.835691,"low_b":126.102129,"close_b":126.813095,"volume_b":78973300,"open_bench":390.193481,"close_bench":392.265747},{"date":"2021-05-10","open_a":122.819383,"high_a":124.077005,"low_a":122.500781,"close_a":123.012215,"volume_a":5270600,"open_b":126.033961,"high_b":126.160559,"low_b":123.501784,"close_b":123.540741,"volume_b":88071200,"open_bench":392.618841,"close_bench":388.381348},{"date":"2021-05-11","open_a":123.389499,"high_a":123.783555,"low_a":121.586912,"close_a":122.115105,"volume_a":5266100,"open_b":120.278121,"high_b":122.975853,"low_b":119.567162,"close_b":122.625252,"volume_b":126142800,"open_bench":383.883649,"close_bench":384.915131},{"date":"2021-05-12","open_a":121.469538,"high_a":122.190575,"low_a":120.605972,"close_a":120.924561,"volume_a":5651700,"open_b":120.180731,"high_b":121.38838,"low_b":119.060731,"close_b":119.567162,"volume_b":112172300,"open_bench":382.146,"close_bench":376.73761},{"date":"2021-05-13","open_a":120.932966,"high_a":123.347592,"low_a":120.379609,"close_a":122.718781,"volume_a":4849400,"open_b":121.329968,"high_b":122.85901,"low_b":121.018316,"close_b":121.709793,"volume_b":105861300,"open_bench":378.280247,"close_bench":381.263214},{"date":"2021-05-14","open_a":123.414624,"high_a":124.001512,"low_a":122.760662,"close_a":122.903191,"volume_a":4224700,"open_b":122.956362,"high_b":124.553577,"low_b":122.566796,"close_b":124.125053,"volume_b":81918000,"open_bench":383.98593,"close_bench":387.117584},{"date":"2021-05-17","open_a":123.246957,"high_a":123.582318,"low_a":122.165408,"close_a":122.551071,"volume_a":3747100,"open_b":123.51153,"high_b":123.618661,"low_b":121.904574,"close_b":122.975876,"volume_b":74244600,"open_bench":386.011695,"close_bench":386.132477},{"date":"2021-05-18","open_a":122.945124,"high_a":123.20503,"low_a":121.838423,"close_a":121.997711,"volume_a":4527300,"open_b":123.258299,"high_b":123.677082,"low_b":121.524737,"close_b":121.592911,"volume_b":63342900,"open_bench":386.392653,"close_bench":382.805664},{"date":"2021-05-19","open_a":121.268307,"high_a":122.098342,"low_a":120.798797,"close_a":121.930649,"volume_a":4432700,"open_b":119.947004,"high_b":121.661084,"low_b":119.654827,"close_b":121.437088,"volume_b":92612000,"open_bench":378.140805,"close_bench":381.802124},{"date":"2021-05-20","open_a":121.779769,"high_a":123.859034,"low_a":121.662391,"close_a":123.439827,"volume_a":3964600,"open_b":121.963,"high_b":124.388039,"low_b":121.836387,"close_b":123.988731,"volume_b":76857100,"open_bench":382.675717,"close_bench":385.909607},{"date":"2021-05-21","open_a":123.976388,"high_a":124.345293,"low_a":122.844536,"close_a":123.330818,"volume_a":5194200,"open_b":124.485433,"high_b":124.660737,"low_b":121.943522,"close_b":122.157784,"volume_b":79295400,"open_bench":387.387031,"close_bench":385.593536},{"date":"2021-05-24","open_a":123.78356,"high_a":124.37884,"low_a":123.255355,"close_a":123.959633,"volume_a":3627800,"open_b":122.722627,"high_b":124.602277,"low_b":122.654454,"close_b":123.784187,"volume_b":63092900,"open_bench":387.823824,"close_bench":389.524414},{"date":"2021-05-25","open_a":123.95963,"high_a":124.395596,"low_a":123.255352,"close_a":124.336914,"volume_a":4649300,"open_b":124.485421,"high_b":124.972385,"low_b":123.024553,"close_b":123.589424,"volume_b":72009500,"open_bench":390.602308,"close_bench":388.660126},{"date":"2021-05-26","open_a":124.009933,"high_a":124.647123,"low_a":123.817088,"close_a":124.336914,"volume_a":4462600,"open_b":123.647872,"high_b":124.066654,"low_b":123.121958,"close_b":123.540741,"volume_b":56575900,"open_bench":389.245622,"close_bench":389.431488},{"date":"2021-05-27","open_a":124.554896,"high_a":124.730969,"low_a":122.777461,"close_a":122.836143,"volume_a":9203600,"open_b":123.141437,"high_b":124.310129,"low_b":121.816916,"close_b":122.011696,"volume_b":94625600,"open_bench":390.453662,"close_bench":389.635895},{"date":"2021-05-28","open_a":123.590721,"high_a":124.50459,"low_a":123.448192,"close_a":124.03508,"volume_a":4913700,"open_b":122.294139,"high_b":122.518142,"low_b":121.300752,"close_b":121.359184,"volume_b":71311100,"open_bench":391.197136,"close_bench":390.332916},{"date":"2021-06-01","open_a":124.739349,"high_a":124.798044,"low_a":123.322426,"close_a":123.775177,"volume_a":5530300,"open_b":121.816921,"high_b":122.079874,"low_b":120.706662,"close_b":121.037788,"volume_b":67637100,"open_bench":392.683968,"close_bench":389.989075},{"date":"2021-06-02","open_a":124.253049,"high_a":124.529728,"low_a":123.439786,"close_a":124.269821,"volume_a":4546900,"open_b":121.037774,"high_b":121.972729,"low_b":120.813779,"close_b":121.797424,"volume_b":59278900,"open_bench":390.639474,"close_bench":390.602295},{"date":"2021-06-03","open_a":123.902452,"high_a":124.789209,"low_a":123.437953,"close_a":124.713203,"volume_a":4651600,"open_b":121.42736,"high_b":121.592923,"low_b":119.917793,"close_b":120.317101,"volume_b":76229200,"open_bench":388.297701,"close_bench":389.152618},{"date":"2021-06-04","open_a":124.991926,"high_a":125.490201,"low_a":124.544321,"close_a":124.856796,"volume_a":4427400,"open_b":120.833254,"high_b":122.868734,"low_b":120.618993,"close_b":122.605774,"volume_b":75169300,"open_bench":390.992688,"close_bench":392.711853},{"date":"2021-06-07","open_a":124.789241,"high_a":125.20307,"low_a":123.902484,"close_a":124.7808,"volume_a":4724100,"open_b":122.87847,"high_b":123.024558,"low_b":121.573431,"close_b":122.615517,"volume_b":71057600,"open_bench":392.702454,"close_bench":392.33075},{"date":"2021-06-08","open_a":124.738573,"high_a":124.882144,"low_a":123.235293,"close_a":123.615334,"volume_a":5997200,"open_b":123.297247,"high_b":125.108731,"low_b":122.917422,"close_b":123.433594,"volume_b":74403800,"open_bench":393.185684,"close_bench":392.414398},{"date":"2021-06-09","open_a":124.358522,"high_a":124.645663,"low_a":123.319741,"close_a":123.387306,"volume_a":4008000,"open_b":123.891323,"high_b":124.417236,"low_b":123.219321,"close_b":123.813408,"volume_b":56877900,"open_bench":393.2508,"close_bench":391.82901},{"date":"2021-06-10","open_a":123.564635,"high_a":125.00035,"low_a":123.412624,"close_a":124.654099,"volume_a":4698200,"open_b":123.706286,"high_b":124.845769,"low_b":122.654467,"close_b":122.82003,"volume_b":71186400,"open_bench":393.046337,"close_bench":393.65036},{"date":"2021-06-11","open_a":125.034145,"high_a":125.093256,"low_a":123.970016,"close_a":124.73011,"volume_a":4364600,"open_b":123.22908,"high_b":124.115343,"low_b":122.810297,"close_b":124.027687,"volume_b":53522400,"open_bench":394.198697,"close_bench":394.300903},{"date":"2021-06-14","open_a":124.544291,"high_a":125.52396,"low_a":123.969997,"close_a":125.473289,"volume_a":3831900,"open_b":124.485418,"high_b":127.134452,"low_b":123.754984,"close_b":127.076019,"volume_b":96906500,"open_bench":394.41231,"close_bench":395.183624},{"date":"2021-06-15","open_a":125.75202,"high_a":125.980049,"low_a":125.051063,"close_a":125.481773,"volume_a":4256500,"open_b":126.550126,"high_b":127.192912,"low_b":126.014472,"close_b":126.25795,"volume_b":62746300,"open_bench":395.332322,"close_bench":394.458801},{"date":"2021-06-16","open_a":125.718236,"high_a":126.0645,"low_a":123.936255,"close_a":124.23185,"volume_a":5492100,"open_b":126.968855,"high_b":127.475293,"low_b":125.108695,"close_b":126.754593,"volume_b":91815000,"open_bench":394.598262,"close_bench":392.25647},{"date":"2021-06-17","open_a":124.358545,"high_a":125.83649,"low_a":124.071404,"close_a":125.431114,"volume_a":5316400,"open_b":126.413807,"high_b":129.092066,"low_b":126.267711,"close_b":128.351883,"volume_b":96721700,"open_bench":391.847571,"close_bench":392.126343},{"date":"2021-06-18","open_a":124.975016,"high_a":124.975016,"low_a":122.652531,"close_a":122.812996,"volume_a":10661700,"open_b":127.30005,"high_b":128.079168,"low_b":126.84231,"close_b":127.056572,"volume_b":108953300,"open_bench":388.859566,"close_bench":386.836456},{"date":"2021-06-21","open_a":123.091697,"high_a":123.927796,"low_a":122.838332,"close_a":123.775772,"volume_a":4904300,"open_b":126.900748,"high_b":128.955703,"low_b":125.839187,"close_b":128.848572,"volume_b":79663300,"open_bench":388.58916,"close_bench":392.374359},{"date":"2021-06-22","open_a":123.919379,"high_a":124.569679,"low_a":123.54778,"close_a":123.961609,"volume_a":3518000,"open_b":128.682998,"high_b":130.582123,"low_b":128.186293,"close_b":130.484726,"volume_b":74783600,"open_bench":392.365093,"close_bench":394.472107},{"date":"2021-06-23","open_a":123.353513,"high_a":123.412637,"low_a":122.314732,"close_a":122.331627,"volume_a":3901700,"open_b":130.280217,"high_b":130.815872,"low_b":129.754296,"close_b":130.212036,"volume_b":60214200,"open_bench":394.546645,"close_bench":393.996582},{"date":"2021-06-24","open_a":122.340087,"high_a":123.252192,"low_a":122.128952,"close_a":123.024162,"volume_a":3818900,"open_b":130.942473,"high_b":131.127519,"low_b":129.462123,"close_b":129.929611,"volume_b":68711000,"open_bench":396.131553,"close_bench":396.327332},{"date":"2021-06-25","open_a":123.100135,"high_a":123.97001,"low_a":122.660983,"close_a":123.649094,"volume_a":6387600,"open_b":129.978326,"high_b":130.397101,"low_b":129.345275,"close_b":129.637451,"volume_b":70783700,"open_bench":397.073293,"close_bench":397.735229},{"date":"2021-06-28","open_a":123.944713,"high_a":124.645683,"low_a":123.826478,"close_a":124.181183,"volume_a":4282800,"open_b":129.929615,"high_b":131.72161,"low_b":129.871183,"close_b":131.26387,"volume_b":62111300,"open_bench":398.257304,"close_bench":398.536987},{"date":"2021-06-29","open_a":124.527398,"high_a":124.738533,"low_a":123.640629,"close_a":124.096687,"volume_a":3197700,"open_b":131.283338,"high_b":132.929252,"low_b":130.845081,"close_b":132.773422,"volume_b":64556100,"open_bench":398.919122,"close_bench":398.751312},{"date":"2021-06-30","open_a":124.510511,"high_a":125.25371,"low_a":124.31627,"close_a":125.135475,"volume_a":4943700,"open_b":132.617579,"high_b":133.825236,"low_b":132.325403,"close_b":133.386978,"volume_b":63261400,"open_bench":398.294562,"close_bench":399.087036},{"date":"2021-07-01","open_a":125.059487,"high_a":125.904027,"low_a":124.941252,"close_a":125.160828,"volume_a":3997600,"open_b":133.036385,"high_b":133.747336,"low_b":132.218287,"close_b":133.688904,"volume_b":52485800,"open_bench":399.842251,"close_bench":401.296661},{"date":"2021-07-02","open_a":125.751976,"high_a":126.495175,"low_a":125.464835,"close_a":125.760429,"volume_a":3835500,"open_b":134.30243,"high_b":136.34765,"low_b":134.156349,"close_b":136.308701,"volume_b":78852600,"open_bench":402.452717,"close_bench":404.363953},{"date":"2021-07-06","open_a":125.895581,"high_a":126.191162,"low_a":124.797676,"close_a":125.946251,"volume_a":4313800,"open_b":136.415832,"high_b":139.415467,"low_b":136.415832,"close_b":138.314957,"volume_b":108181800,"open_bench":404.419885,"close_bench":403.627411},{"date":"2021-07-07","open_a":126.081346,"high_a":127.272151,"low_a":125.836436,"close_a":126.503616,"volume_a":4225000,"open_b":139.795349,"high_b":141.110137,"low_b":138.938317,"close_b":140.798492,"volume_b":104911600,"open_bench":404.307992,"close_bench":405.053833},{"date":"2021-07-08","open_a":126.191198,"high_a":127.246873,"low_a":125.996957,"close_a":126.562798,"volume_a":4216000,"open_b":137.886446,"high_b":140.301743,"low_b":137.000183,"close_b":139.503143,"volume_b":105575500,"open_bench":399.75834,"close_bench":401.75351},{"date":"2021-07-09","open_a":126.182733,"high_a":126.942814,"low_a":125.980039,"close_a":126.241844,"volume_a":4088600,"open_b":139.025923,"high_b":141.850262,"low_b":138.928526,"close_b":141.324356,"volume_b":99890800,"open_bench":403.254469,"close_bench":406.042084},{"date":"2021-07-12","open_a":126.250315,"high_a":126.503667,"low_a":125.650686,"close_a":126.267197,"volume_a":4873700,"open_b":142.395713,"high_b":142.502844,"low_b":140.24336,"close_b":140.730316,"volume_b":76299700,"open_bench":405.958175,"close_bench":407.49649},{"date":"2021-07-13","open_a":127.652204,"high_a":129.983142,"low_a":127.314393,"close_a":129.180832,"volume_a":8750800,"open_b":140.272562,"high_b":143.613088,"low_b":139.883003,"close_b":141.840561,"volume_b":100827100,"open_bench":406.713391,"close_bench":406.107391},{"date":"2021-07-14","open_a":129.121727,"high_a":131.021942,"low_a":129.045722,"close_a":130.515213,"volume_a":6390300,"open_b":144.236407,"high_b":145.668059,"low_b":143.827351,"close_b":145.259003,"volume_b":127050800,"open_bench":407.794868,"close_bench":406.713379},{"date":"2021-07-15","open_a":130.118302,"high_a":131.207753,"low_a":129.434213,"close_a":131.114853,"volume_a":4920600,"open_b":145.346643,"high_b":146.086811,"low_b":143.252723,"close_b":144.606461,"volume_b":106820300,"open_bench":405.380155,"close_bench":405.324219},{"date":"2021-07-16","open_a":131.478004,"high_a":132.457674,"low_a":130.996623,"close_a":131.596252,"volume_a":5852500,"open_b":144.586973,"high_b":145.853047,"low_b":142.074279,"close_b":142.570969,"volume_b":93251400,"open_bench":406.498915,"close_bench":402.144989},{"date":"2021-07-19","open_a":131.739795,"high_a":133.048822,"low_a":130.701002,"close_a":131.57933,"volume_a":5750200,"open_b":139.999853,"high_b":140.311512,"low_b":137.974114,"close_b":138.733765,"volume_b":121434600,"open_bench":397.343603,"close_bench":396.206177},{"date":"2021-07-20","open_a":131.917169,"high_a":132.871503,"low_a":131.376664,"close_a":131.596252,"volume_a":5213500,"open_b":139.717424,"high_b":143.262463,"low_b":139.230468,"close_b":142.337234,"volume_b":96350000,"open_bench":396.868079,"close_bench":401.883942},{"date":"2021-07-21","open_a":131.15707,"high_a":131.849586,"low_a":130.540559,"close_a":131.1064,"volume_a":4905900,"open_b":141.733425,"high_b":142.317778,"low_b":140.85691,"close_b":141.606812,"volume_b":74993500,"open_bench":403.077377,"close_bench":405.137787},{"date":"2021-07-22","open_a":131.064178,"high_a":131.54556,"low_a":130.033839,"close_a":131.064178,"volume_a":3333500,"open_b":142.132681,"high_b":144.333716,"low_b":142.006067,"close_b":142.970245,"volume_b":77338200,"open_bench":405.314877,"close_bench":405.986145},{"date":"2021-07-23","open_a":131.376646,"high_a":133.259967,"low_a":131.1064,"close_a":132.744797,"volume_a":3588400,"open_b":143.700681,"high_b":144.840156,"low_b":143.087112,"close_b":144.684326,"volume_b":71447400,"open_bench":407.906747,"close_bench":410.162964},{"date":"2021-07-26","open_a":132.415433,"high_a":132.989727,"low_a":131.773588,"close_a":132.651917,"volume_a":2999800,"open_b":144.401901,"high_b":145.921201,"low_b":143.846764,"close_b":145.103119,"volume_b":72434100,"open_bench":409.575609,"close_bench":411.169861},{"date":"2021-07-27","open_a":132.854639,"high_a":133.994774,"low_a":132.618169,"close_a":133.386703,"volume_a":4043900,"open_b":145.229755,"high_b":145.317418,"low_b":141.752897,"close_b":142.941071,"volume_b":104818600,"open_bench":410.134916,"close_bench":409.295837},{"date":"2021-07-28","open_a":133.099534,"high_a":133.403569,"low_a":131.672259,"close_a":132.162094,"volume_a":3556800,"open_b":141.032175,"high_b":143.135829,"low_b":138.821391,"close_b":141.197739,"volume_b":118931200,"open_bench":409.920587,"close_bench":409.128113},{"date":"2021-07-29","open_a":132.787017,"high_a":132.955923,"low_a":131.917142,"close_a":132.432312,"volume_a":3055600,"open_b":140.915347,"high_b":142.726825,"low_b":140.808216,"close_b":141.840561,"volume_b":56699500,"open_bench":410.051111,"close_bench":410.824921},{"date":"2021-07-30","open_a":132.905258,"high_a":133.099511,"low_a":132.271853,"close_a":132.550552,"volume_a":3757300,"open_b":140.613409,"high_b":142.512534,"low_b":140.350448,"close_b":142.054794,"volume_b":70440600,"open_bench":408.270317,"close_bench":408.829712},{"date":"2021-08-02","open_a":132.59279,"high_a":132.68569,"low_a":131.334421,"close_a":132.018509,"volume_a":3425900,"open_b":142.541761,"high_b":143.116366,"low_b":141.460718,"close_b":141.723679,"volume_b":62880000,"open_bench":410.535914,"close_bench":407.972046},{"date":"2021-08-03","open_a":132.592741,"high_a":132.719417,"low_a":132.010006,"close_a":132.314041,"volume_a":3359900,"open_b":142.006074,"high_b":144.177893,"low_b":141.392505,"close_b":143.51564,"volume_b":64786600,"open_bench":408.764418,"close_bench":411.290985},{"date":"2021-08-04","open_a":131.90871,"high_a":132.305645,"low_a":129.949371,"close_a":130.101395,"volume_a":4846300,"open_b":143.428038,"high_b":143.934461,"low_b":142.463859,"close_b":143.116379,"volume_b":56368300,"open_bench":410.013755,"close_bench":409.267914},{"date":"2021-08-05","open_a":130.886824,"high_a":131.064183,"low_a":129.949384,"close_a":130.320984,"volume_a":3733500,"open_b":143.14555,"high_b":143.983115,"low_b":142.356684,"close_b":143.223465,"volume_b":46397700,"open_bench":410.423998,"close_bench":411.859772},{"date":"2021-08-06","open_a":129.864965,"high_a":130.701052,"low_a":129.755171,"close_a":130.337906,"volume_a":3430900,"open_b":142.74556,"high_b":143.486837,"low_b":142.043292,"close_b":142.540726,"volume_b":54126800,"open_bench":412.176725,"close_bench":412.540314},{"date":"2021-08-09","open_a":130.599632,"high_a":130.726308,"low_a":129.814216,"close_a":130.354721,"volume_a":2636600,"open_b":142.599281,"high_b":143.086966,"low_b":141.936035,"close_b":142.491989,"volume_b":48908700,"open_bench":412.512333,"close_bench":412.204681},{"date":"2021-08-10","open_a":130.320946,"high_a":131.165486,"low_a":129.898676,"close_a":130.422287,"volume_a":2670200,"open_b":142.833346,"high_b":144.072072,"low_b":141.721424,"close_b":142.014038,"volume_b":69023100,"open_bench":412.65223,"close_bench":412.717499},{"date":"2021-08-11","open_a":130.658798,"high_a":131.765143,"low_a":130.456104,"close_a":130.937485,"volume_a":3058000,"open_b":142.452947,"high_b":143.106444,"low_b":141.94575,"close_b":142.267624,"volume_b":48493500,"open_bench":413.780282,"close_bench":413.742981},{"date":"2021-08-12","open_a":131.097946,"high_a":131.15707,"low_a":130.565895,"close_a":130.96283,"volume_a":1958700,"open_b":142.589523,"high_b":145.379085,"low_b":142.248137,"close_b":145.223022,"volume_b":72282600,"open_bench":413.593861,"close_bench":414.983002},{"date":"2021-08-13","open_a":130.988148,"high_a":132.314069,"low_a":130.81923,"close_a":132.187393,"volume_a":2113800,"open_b":145.301038,"high_b":145.759464,"low_b":144.618281,"close_b":145.427841,"volume_b":59375000,"open_bench":415.43051,"close_bench":415.73819},{"date":"2021-08-16","open_a":132.53366,"high_a":133.538664,"low_a":132.364754,"close_a":133.479553,"volume_a":3224000,"open_b":144.881612,"high_b":147.466354,"low_b":142.862602,"close_b":147.398071,"volume_b":103296000,"open_bench":414.442341,"close_bench":416.717194},{"date":"2021-08-17","open_a":133.361298,"high_a":133.817344,"low_a":132.795458,"close_a":133.352844,"volume_a":3736300,"open_b":146.530006,"high_b":147.944291,"low_b":145.418083,"close_b":146.490997,"volume_b":92229700,"open_bench":414.17192,"close_bench":413.985474},{"date":"2021-08-18","open_a":133.167062,"high_a":133.606226,"low_a":131.917146,"close_a":131.984711,"volume_a":3419200,"open_b":146.110605,"high_b":147.007944,"low_b":142.550491,"close_b":142.755325,"volume_b":86326000,"open_bench":412.978528,"close_bench":409.454376},{"date":"2021-08-19","open_a":131.596246,"high_a":134.636582,"low_a":131.562457,"close_a":134.205872,"volume_a":4745600,"open_b":141.458092,"high_b":144.354945,"low_b":140.941146,"close_b":143.08696,"volume_b":86960300,"open_bench":406.741399,"close_bench":410.088409},{"date":"2021-08-20","open_a":134.222717,"high_a":134.813905,"low_a":133.648436,"close_a":133.732895,"volume_a":3484100,"open_b":143.808694,"high_b":144.842585,"low_b":143.164946,"close_b":144.540222,"volume_b":60549600,"open_bench":410.433316,"close_bench":413.35144},{"date":"2021-08-23","open_a":133.293775,"high_a":133.876509,"low_a":132.863064,"close_a":133.310669,"volume_a":3102100,"open_b":144.657299,"high_b":146.491001,"low_b":144.247644,"close_b":146.022827,"volume_b":60131800,"open_bench":415.029619,"close_bench":416.987488},{"date":"2021-08-24","open_a":133.226195,"high_a":133.226195,"low_a":131.477991,"close_a":131.65535,"volume_a":3760800,"open_b":145.769232,"high_b":147.144509,"low_b":145.476618,"close_b":145.935043,"volume_b":48606400,"open_bench":417.649445,"close_bench":417.649445},{"date":"2021-08-25","open_a":131.207761,"high_a":131.469567,"low_a":130.194316,"close_a":131.072632,"volume_a":3346700,"open_b":146.120355,"high_b":146.617804,"low_b":144.159864,"close_b":144.70607,"volume_b":58991300,"open_bench":417.835883,"close_bench":418.525787},{"date":"2021-08-26","open_a":131.334403,"high_a":131.469533,"low_a":130.135158,"close_a":130.591217,"volume_a":3220700,"open_b":144.696328,"high_b":145.447354,"low_b":143.877005,"close_b":143.906265,"volume_b":48597200,"open_bench":418.246033,"close_bench":416.055115},{"date":"2021-08-27","open_a":130.768553,"high_a":131.249947,"low_a":130.236502,"close_a":130.853012,"volume_a":4166500,"open_b":143.847729,"high_b":145.086455,"low_b":143.213744,"close_b":144.940155,"volume_b":55802400,"open_bench":416.856963,"close_bench":419.775116},{"date":"2021-08-30","open_a":131.055701,"high_a":131.8918,"low_a":130.912131,"close_a":131.74823,"volume_a":2540300,"open_b":145.330322,"high_b":149.709744,"low_b":144.949927,"close_b":149.348846,"volume_b":90956700,"open_bench":420.446458,"close_bench":421.621185},{"date":"2021-08-31","open_a":131.824226,"high_a":132.145156,"low_a":131.190821,"close_a":132.077591,"volume_a":5291200,"open_b":148.900151,"high_b":149.036703,"low_b":147.563883,"close_b":148.090591,"volume_b":86453100,"open_bench":421.527826,"close_bench":420.996399},{"date":"2021-09-01","open_a":132.457664,"high_a":133.597798,"low_a":132.178964,"close_a":133.361328,"volume_a":4231400,"open_b":149.065956,"high_b":151.162998,"low_b":148.588019,"close_b":148.75383,"volume_b":80313700,"open_bench":421.928753,"close_bench":421.220184},{"date":"2021-09-02","open_a":133.323035,"high_a":133.7482,"low_a":132.540735,"close_a":133.459091,"volume_a":2862500,"open_b":150.080351,"high_b":150.909423,"low_b":148.646555,"close_b":149.865768,"volume_b":71115500,"open_bench":422.637258,"close_bench":422.516052},{"date":"2021-09-03","open_a":132.821333,"high_a":133.858735,"low_a":132.566232,"close_a":133.578125,"volume_a":2035100,"open_b":149.973062,"high_b":150.821645,"low_b":149.319565,"close_b":150.499771,"volume_b":57808700,"open_bench":421.387989,"close_bench":422.413513},{"date":"2021-09-07","open_a":133.059429,"high_a":133.29752,"low_a":130.678506,"close_a":131.401291,"volume_a":4609900,"open_b":151.153263,"high_b":153.386856,"low_b":150.587546,"close_b":152.830902,"volume_b":82278300,"open_bench":422.068623,"close_bench":420.903229},{"date":"2021-09-08","open_a":131.511844,"high_a":133.289028,"low_a":131.205725,"close_a":133.263519,"volume_a":3199100,"open_b":153.11378,"high_b":153.1723,"low_b":150.187666,"close_b":151.289841,"volume_b":74420200,"open_bench":420.371775,"close_bench":420.390411},{"date":"2021-09-09","open_a":133.399578,"high_a":133.476104,"low_a":132.141082,"close_a":132.421692,"volume_a":3560400,"open_b":151.660446,"high_b":152.265171,"low_b":150.158366,"close_b":150.275421,"volume_b":57305700,"open_bench":420.194648,"close_bench":418.591064},{"date":"2021-09-10","open_a":132.634285,"high_a":132.863877,"low_a":131.809465,"close_a":132.192123,"volume_a":3851500,"open_b":151.182527,"high_b":151.650701,"low_b":145.037686,"close_b":145.301041,"volume_b":140893200,"open_bench":420.511684,"close_bench":415.29071},{"date":"2021-09-13","open_a":132.600286,"high_a":133.841759,"low_a":132.132602,"close_a":132.44722,"volume_a":3367200,"open_b":146.920163,"high_b":147.6907,"low_b":145.08646,"close_b":145.86676,"volume_b":102404300,"open_bench":418.274112,"close_bench":416.353516},{"date":"2021-09-14","open_a":132.974426,"high_a":133.221016,"low_a":131.647915,"close_a":131.928513,"volume_a":2808600,"open_b":146.647058,"high_b":147.349326,"low_b":143.291779,"close_b":144.47197,"volume_b":109296300,"open_bench":417.789258,"close_bench":414.106628},{"date":"2021-09-15","open_a":131.988039,"high_a":132.974423,"low_a":131.613892,"close_a":132.668304,"volume_a":3702400,"open_b":144.901163,"high_b":145.759494,"low_b":142.765097,"close_b":145.359589,"volume_b":83281300,"open_bench":414.526103,"close_bench":417.56546},{"date":"2021-09-16","open_a":132.132622,"high_a":132.753358,"low_a":130.7891,"close_a":132.277176,"volume_a":4589900,"open_b":144.784117,"high_b":145.301063,"low_b":143.594163,"close_b":145.125488,"volume_b":68034100,"open_bench":417.043376,"close_bench":416.903534},{"date":"2021-09-17","open_a":131.996572,"high_a":132.226165,"low_a":130.806117,"close_a":131.061218,"volume_a":8127400,"open_b":145.154741,"high_b":145.154741,"low_b":142.170093,"close_b":142.462708,"volume_b":129868800,"open_bench":416.134843,"close_bench":412.84256},{"date":"2021-09-20","open_a":130.865639,"high_a":131.758487,"low_a":129.607155,"close_a":130.619049,"volume_a":5484600,"open_b":140.258375,"high_b":141.272755,"low_b":137.790688,"close_b":139.419556,"volume_b":123478900,"open_bench":406.74436,"close_bench":405.95871},{"date":"2021-09-21","open_a":131.469306,"high_a":132.753298,"low_a":130.074768,"close_a":130.559448,"volume_a":4479800,"open_b":140.385177,"high_b":141.038689,"low_b":139.263506,"close_b":139.897491,"volume_b":75834000,"open_bench":408.287659,"close_bench":405.575287},{"date":"2021-09-22","open_a":131.477838,"high_a":131.877494,"low_a":130.482955,"close_a":130.959137,"volume_a":3001000,"open_b":140.892375,"high_b":142.823606,"low_b":140.160847,"close_b":142.257904,"volume_b":76404300,"open_bench":407.838629,"close_bench":409.531525},{"date":"2021-09-23","open_a":130.687071,"high_a":131.741471,"low_a":130.559514,"close_a":131.061218,"volume_a":2712500,"open_b":143.038171,"high_b":143.457588,"low_b":142.053051,"close_b":143.213745,"volume_b":64838200,"open_bench":411.39284,"close_bench":414.507385},{"date":"2021-09-24","open_a":130.738052,"high_a":131.477834,"low_a":130.567988,"close_a":131.120697,"volume_a":2366100,"open_b":142.072551,"high_b":143.837971,"low_b":141.975008,"close_b":143.301514,"volume_b":53477900,"open_bench":412.879926,"close_bench":415.190125},{"date":"2021-09-27","open_a":130.780554,"high_a":131.078162,"low_a":129.581588,"close_a":129.862198,"volume_a":4482000,"open_b":141.887216,"high_b":142.365153,"low_b":140.27786,"close_b":141.789673,"volume_b":74150700,"open_bench":414.161305,"close_bench":414.002319},{"date":"2021-09-28","open_a":129.564581,"high_a":129.675128,"low_a":127.634344,"close_a":128.442154,"volume_a":5910900,"open_b":139.72188,"high_b":141.184936,"low_b":138.200304,"close_b":138.414886,"volume_b":108972300,"open_bench":411.243181,"close_bench":405.659424},{"date":"2021-09-29","open_a":128.688775,"high_a":130.244878,"low_a":128.136066,"close_a":129.505096,"volume_a":3326500,"open_b":138.961152,"high_b":140.892383,"low_b":138.531986,"close_b":139.312286,"volume_b":74602000,"open_bench":407.034247,"close_bench":406.342133},{"date":"2021-09-30","open_a":130.287379,"high_a":130.46594,"low_a":127.863936,"close_a":127.897957,"volume_a":4949200,"open_b":140.12182,"high_b":140.824088,"low_b":137.800432,"close_b":138.015015,"volume_b":89056700,"open_bench":407.810559,"close_bench":401.375702},{"date":"2021-10-01","open_a":128.697271,"high_a":128.918352,"low_a":127.141168,"close_a":128.357132,"volume_a":4274300,"open_b":138.40516,"high_b":139.400043,"low_b":135.683881,"close_b":139.136688,"volume_b":94639600,"open_bench":403.096625,"close_bench":406.145691},{"date":"2021-10-04","open_a":128.297606,"high_a":130.100299,"low_a":127.107138,"close_a":127.719376,"volume_a":5404900,"open_b":138.268633,"high_b":138.707562,"low_b":134.864597,"close_b":135.713165,"volume_b":98322000,"open_bench":404.985986,"close_bench":400.908081},{"date":"2021-10-05","open_a":128.705773,"high_a":129.581611,"low_a":127.608855,"close_a":128.476181,"volume_a":6103100,"open_b":136.054562,"high_b":138.736833,"low_b":135.927759,"close_b":137.634659,"volume_b":80861100,"open_bench":402.404488,"close_bench":405.079468},{"date":"2021-10-06","open_a":128.263589,"high_a":131.937016,"low_a":127.515309,"close_a":131.766953,"volume_a":5499200,"open_b":136.035013,"high_b":138.649001,"low_b":134.962099,"close_b":138.502701,"volume_b":83221100,"open_bench":401.497304,"close_bench":406.763062},{"date":"2021-10-07","open_a":132.243113,"high_a":134.139342,"low_a":132.243113,"close_a":132.982895,"volume_a":4605100,"open_b":139.536579,"high_b":140.668013,"low_b":139.204956,"close_b":139.76091,"volume_b":61732700,"open_bench":410.027233,"close_bench":410.279755},{"date":"2021-10-08","open_a":133.135973,"high_a":133.255019,"low_a":132.217617,"close_a":132.676788,"volume_a":4186000,"open_b":140.482698,"high_b":140.628997,"low_b":139.048901,"close_b":139.380524,"volume_b":58773200,"open_bench":411.046739,"close_bench":409.531525},{"date":"2021-10-11","open_a":132.982937,"high_a":133.69721,"low_a":132.3622,"close_a":132.855392,"volume_a":2738100,"open_b":138.766046,"high_b":141.243482,"low_b":138.317369,"close_b":139.29274,"volume_b":64452200,"open_bench":408.876818,"close_bench":406.56662},{"date":"2021-10-12","open_a":132.838352,"high_a":134.224391,"low_a":132.829853,"close_a":133.442078,"volume_a":3195400,"open_b":139.70242,"high_b":139.721932,"low_b":137.566355,"close_b":138.02478,"volume_b":73035900,"open_bench":407.483244,"close_bench":405.565857},{"date":"2021-10-13","open_a":133.892772,"high_a":134.394462,"low_a":132.668296,"close_a":134.343445,"volume_a":3599900,"open_b":137.761434,"high_b":137.917482,"low_b":135.771669,"close_b":137.43956,"volume_b":78762700,"open_bench":406.585309,"close_bench":407.024902},{"date":"2021-10-14","open_a":134.879115,"high_a":135.618909,"low_a":134.870616,"close_a":135.423325,"volume_a":3677100,"open_b":138.610004,"high_b":140.336415,"low_b":138.024775,"close_b":140.21936,"volume_b":69907100,"open_bench":410.672561,"close_bench":413.871307},{"date":"2021-10-15","open_a":135.76351,"high_a":135.76351,"low_a":133.935296,"close_a":135.040726,"volume_a":4313000,"open_b":140.229111,"high_b":141.331271,"low_b":139.975505,"close_b":141.272751,"volume_b":67940300,"open_bench":415.975751,"close_bench":417.023285},{"date":"2021-10-18","open_a":134.522019,"high_a":134.947183,"low_a":133.331564,"close_a":134.428482,"volume_a":3274200,"open_b":139.917002,"high_b":143.22351,"low_b":139.634151,"close_b":142.940659,"volume_b":85589200,"open_bench":415.246229,"close_bench":418.257904},{"date":"2021-10-19","open_a":134.896134,"high_a":136.188637,"low_a":133.960767,"close_a":136.137619,"volume_a":3125800,"open_b":143.389307,"high_b":145.496112,"low_b":142.940645,"close_b":145.096207,"volume_b":76378900,"open_bench":419.875959,"close_bench":421.48468},{"date":"2021-10-20","open_a":136.545798,"high_a":137.651228,"low_a":135.780494,"close_a":137.166534,"volume_a":3716500,"open_b":145.037687,"high_b":146.06183,"low_b":144.47197,"close_b":145.583893,"volume_b":58418800,"open_bench":421.943042,"close_bench":423.140228},{"date":"2021-10-21","open_a":137.234564,"high_a":137.268571,"low_a":135.185268,"close_a":135.355331,"volume_a":4164700,"open_b":145.144965,"high_b":145.954525,"low_b":144.228114,"close_b":145.798462,"volume_b":61421000,"open_bench":422.541578,"close_bench":424.243835},{"date":"2021-10-22","open_a":135.695487,"high_a":136.452279,"low_a":135.60195,"close_a":136.027115,"volume_a":3240300,"open_b":146.003325,"high_b":146.481248,"low_b":144.979183,"close_b":145.027954,"volume_b":58883400,"open_bench":423.813592,"close_bench":423.80423},{"date":"2021-10-25","open_a":136.01011,"high_a":136.01011,"low_a":135.023726,"close_a":135.389374,"volume_a":3281200,"open_b":145.01818,"high_b":145.691189,"low_b":143.984289,"close_b":144.979172,"volume_b":50720600,"open_bench":424.889203,"close_bench":426.077026},{"date":"2021-10-26","open_a":135.346813,"high_a":137.243042,"low_a":135.287296,"close_a":137.04747,"volume_a":3684700,"open_b":145.652189,"high_b":147.124994,"low_b":145.340063,"close_b":145.642441,"volume_b":60893400,"open_bench":427.620244,"close_bench":426.460449},{"date":"2021-10-27","open_a":137.540694,"high_a":137.61722,"low_a":135.678472,"close_a":136.57132,"volume_a":3751800,"open_b":145.681378,"high_b":146.042261,"low_b":144.832811,"close_b":145.183945,"volume_b":56094900,"open_bench":426.918846,"close_bench":424.571228},{"date":"2021-10-28","open_a":136.775386,"high_a":137.523667,"low_a":136.452257,"close_a":137.43013,"volume_a":3589600,"open_b":146.130122,"high_b":149.397607,"low_b":146.032579,"close_b":148.812393,"volume_b":100077900,"open_bench":425.992836,"close_bench":428.667816},{"date":"2021-10-29","open_a":136.928423,"high_a":137.974324,"low_a":136.826388,"close_a":137.413116,"volume_a":4558200,"open_b":143.594121,"high_b":146.247132,"low_b":142.804073,"close_b":146.11058,"volume_b":124953200,"open_bench":426.376301,"close_bench":429.537628},{"date":"2021-11-01","open_a":137.073021,"high_a":137.328122,"low_a":136.205682,"close_a":137.124039,"volume_a":3017100,"open_b":145.320581,"high_b":146.013086,"low_b":144.159886,"close_b":145.291321,"volume_b":74588300,"open_bench":430.519679,"close_bench":430.27652},{"date":"2021-11-02","open_a":137.370629,"high_a":138.442039,"low_a":136.851928,"close_a":138.382523,"volume_a":3464500,"open_b":144.998675,"high_b":147.837008,"low_b":144.988911,"close_b":146.32518,"volume_b":69122000,"open_bench":430.444873,"close_bench":432.016174},{"date":"2021-11-03","open_a":137.880815,"high_a":139.785557,"low_a":137.880815,"close_a":139.70903,"volume_a":3595000,"open_b":146.686088,"high_b":148.227177,"low_b":146.130134,"close_b":147.759003,"volume_b":54511500,"open_bench":431.45497,"close_bench":434.653717},{"date":"2021-11-04","open_a":139.640972,"high_a":139.930081,"low_a":138.901191,"close_a":139.717499,"volume_a":3310100,"open_b":147.84676,"high_b":148.675817,"low_b":146.929909,"close_b":147.242035,"volume_b":60394600,"open_bench":435.252412,"close_bench":436.702148},{"date":"2021-11-05","open_a":140.771957,"high_a":141.520237,"low_a":139.896119,"close_a":141.154602,"volume_a":3754300,"open_b":148.365331,"high_b":148.668135,"low_b":146.577795,"close_b":147.769485,"volume_b":65463900,"open_bench":438.918701,"close_bench":438.217224},{"date":"2021-11-08","open_a":140.993027,"high_a":141.23963,"low_a":137.030492,"close_a":138.118912,"volume_a":5766400,"open_b":147.896451,"high_b":148.052742,"low_b":146.675458,"close_b":146.948959,"volume_b":55020900,"open_bench":439.31151,"close_bench":438.591309},{"date":"2021-11-09","open_a":138.0169,"high_a":139.334913,"low_a":137.7533,"close_a":139.037292,"volume_a":3778600,"open_b":146.714564,"high_b":147.916018,"low_b":146.577814,"close_b":147.31041,"volume_b":56787900,"open_bench":438.956152,"close_bench":437.141663},{"date":"2021-11-10","open_a":140.083201,"high_a":140.304282,"low_a":138.629133,"close_a":139.487961,"volume_a":2683500,"open_b":146.538747,"high_b":146.646195,"low_b":144.419105,"close_b":144.487473,"volume_b":65187100,"open_bench":435.458124,"close_bench":433.624939},{"date":"2021-11-11","open_a":139.428401,"high_a":139.436899,"low_a":138.220936,"close_a":138.339981,"volume_a":2453000,"open_b":145.503362,"high_b":145.962442,"low_b":144.253051,"close_b":144.438644,"volume_b":41000000,"open_bench":435.111944,"close_bench":433.765106},{"date":"2021-11-12","open_a":138.612119,"high_a":139.39442,"low_a":137.821319,"close_a":138.306,"volume_a":3540900,"open_b":144.985603,"high_b":146.909889,"low_b":144.057651,"close_b":146.509415,"volume_b":63804000,"open_bench":435.027802,"close_bench":437.038696},{"date":"2021-11-15","open_a":138.186943,"high_a":139.351902,"low_a":138.169946,"close_a":139.283875,"volume_a":3357100,"open_b":146.880605,"high_b":148.355574,"low_b":145.962415,"close_b":146.519196,"volume_b":59222800,"open_bench":438.320152,"close_bench":437.188416},{"date":"2021-11-16","open_a":139.683524,"high_a":140.295762,"low_a":138.799175,"close_a":138.867203,"volume_a":2899400,"open_b":146.460587,"high_b":147.974622,"low_b":145.874504,"close_b":147.495987,"volume_b":59256200,"open_bench":436.926533,"close_bench":438.918732},{"date":"2021-11-17","open_a":138.280451,"high_a":139.173299,"low_a":137.821279,"close_a":138.918198,"volume_a":3332200,"open_b":147.495983,"high_b":151.403162,"low_b":147.486221,"close_b":149.928207,"volume_b":88807000,"open_bench":438.656855,"close_bench":437.852509},{"date":"2021-11-18","open_a":138.442038,"high_a":139.045777,"low_a":137.540692,"close_a":138.960739,"volume_a":3498900,"open_b":150.143089,"high_b":154.987982,"low_b":149.498401,"close_b":154.206543,"volume_b":137827700,"open_bench":438.881282,"close_bench":439.3396},{"date":"2021-11-19","open_a":139.879072,"high_a":140.329745,"low_a":138.977726,"close_a":139.292343,"volume_a":5242200,"open_b":153.991688,"high_b":157.283497,"low_b":152.897683,"close_b":156.824402,"volume_b":117305600,"open_bench":439.227345,"close_bench":438.553955},{"date":"2021-11-22","open_a":138.773642,"high_a":141.83483,"low_a":138.756632,"close_a":139.581451,"volume_a":4157200,"open_b":157.928166,"high_b":161.854885,"low_b":157.263953,"close_b":157.283493,"volume_b":117467900,"open_bench":440.424577,"close_bench":437.319366},{"date":"2021-11-23","open_a":139.079773,"high_a":140.916486,"low_a":138.867191,"close_a":140.51683,"volume_a":3878000,"open_b":157.38115,"high_b":158.045378,"low_b":155.368955,"close_b":157.664429,"volume_b":96041900,"open_bench":436.992017,"close_bench":437.899261},{"date":"2021-11-24","open_a":140.380801,"high_a":140.729439,"low_a":138.24648,"close_a":139.232864,"volume_a":3309500,"open_b":157.019756,"high_b":158.3775,"low_b":155.935513,"close_b":158.182144,"volume_b":69463600,"open_bench":435.907033,"close_bench":439.068359},{"date":"2021-11-26","open_a":138.356995,"high_a":139.164805,"low_a":136.80939,"close_a":137.021973,"volume_a":3402900,"open_b":155.867105,"high_b":156.726674,"low_b":152.731588,"close_b":153.171143,"volume_b":76959800,"open_bench":432.427721,"close_bench":429.275757},{"date":"2021-11-29","open_a":138.322998,"high_a":139.94713,"low_a":137.021996,"close_a":139.572983,"volume_a":5305800,"open_b":155.671739,"high_b":157.449512,"low_b":155.105197,"close_b":156.521561,"volume_b":88748200,"open_bench":434.045795,"close_bench":434.541504},{"date":"2021-11-30","open_a":139.309374,"high_a":139.453928,"low_a":134.683583,"close_a":135.86554,"volume_a":7610000,"open_b":156.277424,"high_b":161.679099,"low_b":156.209041,"close_b":161.464203,"volume_b":174048100,"open_bench":432.109715,"close_bench":426.086365},{"date":"2021-12-01","open_a":138.152917,"high_a":138.816172,"low_a":136.095123,"close_a":136.18866,"volume_a":5385600,"open_b":163.593547,"high_b":166.348114,"low_b":160.712006,"close_b":160.946442,"volume_b":152052500,"open_bench":431.773074,"close_bench":421.35379},{"date":"2021-12-02","open_a":137.083248,"high_a":138.418722,"low_a":136.826423,"close_a":137.502716,"volume_a":4619800,"open_b":155.056386,"high_b":160.389677,"low_b":154.138197,"close_b":159.959885,"volume_b":136739200,"open_bench":421.568891,"close_bench":427.807343},{"date":"2021-12-03","open_a":138.230417,"high_a":141.149636,"low_a":138.119123,"close_a":141.004105,"volume_a":6465900,"open_b":160.213862,"high_b":161.132052,"low_b":156.013642,"close_b":158.084442,"volume_b":118023100,"open_bench":429.462766,"close_bench":424.084778},{"date":"2021-12-06","open_a":142.151228,"high_a":143.212753,"low_a":141.92864,"close_a":142.467972,"volume_a":5403100,"open_b":160.477591,"high_b":163.984296,"low_b":160.467829,"close_b":161.483704,"volume_b":107497000,"open_bench":426.619544,"close_bench":429.107452},{"date":"2021-12-07","open_a":142.467969,"high_a":142.930257,"low_a":141.817357,"close_a":142.305313,"volume_a":5460100,"open_b":165.156443,"high_b":167.598429,"low_b":164.433609,"close_b":167.207703,"volume_b":120405400,"open_bench":434.363753,"close_bench":437.983368},{"date":"2021-12-08","open_a":142.228246,"high_a":142.699089,"low_a":140.421928,"close_a":142.553558,"volume_a":3891900,"open_b":168.135671,"high_b":171.876796,"low_b":166.738847,"close_b":171.017212,"volume_b":116998900,"open_bench":438.376201,"close_bench":439.143127},{"date":"2021-12-09","open_a":142.039941,"high_a":142.861779,"low_a":141.123948,"close_a":142.37381,"volume_a":3644500,"open_b":170.851116,"high_b":172.648414,"low_b":169.884084,"close_b":170.509232,"volume_b":108923700,"open_bench":437.861847,"close_bench":436.178314},{"date":"2021-12-10","open_a":143.426761,"high_a":144.779361,"low_a":142.878874,"close_a":144.650955,"volume_a":5025700,"open_b":171.144197,"high_b":175.461628,"low_b":170.63626,"close_b":175.285797,"volume_b":115402700,"open_bench":438.872045,"close_bench":440.284332},{"date":"2021-12-13","open_a":144.616691,"high_a":146.611347,"low_a":143.863354,"close_a":145.370041,"volume_a":5411800,"open_b":176.917041,"high_b":177.903613,"low_b":171.456762,"close_b":171.661896,"volume_b":153237000,"open_bench":439.769854,"close_bench":436.384064},{"date":"2021-12-14","open_a":144.976291,"high_a":145.866603,"low_a":144.07741,"close_a":145.010529,"volume_a":4985000,"open_b":171.183287,"high_b":173.615511,"low_b":168.213837,"close_b":170.284637,"volume_b":139380400,"open_bench":433.129192,"close_bench":433.381714},{"date":"2021-12-15","open_a":145.335802,"high_a":147.056508,"low_a":145.070421,"close_a":146.87674,"volume_a":5507800,"open_b":171.046555,"high_b":175.334684,"low_b":168.311526,"close_b":175.139328,"volume_b":131063300,"open_bench":433.437916,"close_bench":440.153381},{"date":"2021-12-16","open_a":146.876756,"high_a":148.580336,"low_a":146.405912,"close_a":147.090775,"volume_a":6909900,"open_b":175.119737,"high_b":176.936575,"low_b":166.78768,"close_b":168.262634,"volume_b":150185800,"open_bench":441.995866,"close_bench":436.27182},{"date":"2021-12-17","open_a":146.482976,"high_a":147.570195,"low_a":143.863389,"close_a":144.077408,"volume_a":10159600,"open_b":165.986697,"high_b":169.444558,"low_b":165.752276,"close_b":167.168625,"volume_b":195432700,"open_bench":433.205423,"close_bench":431.628601},{"date":"2021-12-20","open_a":143.101439,"high_a":145.104651,"low_a":142.750446,"close_a":145.036163,"volume_a":5263600,"open_b":164.375008,"high_b":166.621639,"low_b":163.574045,"close_b":165.810898,"volume_b":107499100,"open_bench":426.569585,"close_bench":427.038879},{"date":"2021-12-21","open_a":144.873505,"high_a":145.190261,"low_a":143.632199,"close_a":144.608124,"volume_a":3401400,"open_b":167.578884,"high_b":169.180827,"low_b":165.195503,"close_b":168.975708,"volume_b":91185900,"open_bench":430.445951,"close_bench":434.622681},{"date":"2021-12-22","open_a":144.642403,"high_a":145.524159,"low_a":144.05171,"close_a":145.507034,"volume_a":4362900,"open_b":169.024552,"high_b":171.77912,"low_b":168.155205,"close_b":171.564224,"volume_b":92135300,"open_bench":434.369337,"close_bench":438.968414},{"date":"2021-12-23","open_a":145.515619,"high_a":146.072075,"low_a":144.890687,"close_a":145.344406,"volume_a":3241700,"open_b":171.769347,"high_b":172.746142,"low_b":171.202804,"close_b":172.189362,"volume_b":68356600,"open_bench":439.963282,"close_bench":441.699677},{"date":"2021-12-27","open_a":145.524151,"high_a":146.868181,"low_a":145.335813,"close_a":146.791138,"volume_a":2868800,"open_b":172.980581,"high_b":176.233309,"low_b":172.961055,"close_b":176.145401,"volume_b":74919600,"open_bench":443.069983,"close_bench":447.950653},{"date":"2021-12-28","open_a":146.782608,"high_a":147.921176,"low_a":146.56002,"close_a":147.55307,"volume_a":2332100,"open_b":175.979323,"high_b":177.122171,"low_b":174.387143,"close_b":175.129501,"volume_b":79144300,"open_bench":448.382431,"close_bench":447.584625},{"date":"2021-12-29","open_a":147.921141,"high_a":148.494722,"low_a":147.184916,"close_a":148.075241,"volume_a":2299500,"open_b":175.16861,"high_b":176.438446,"low_b":174.006222,"close_b":175.217453,"volume_b":62348900,"open_bench":447.687932,"close_bench":448.157227},{"date":"2021-12-30","open_a":148.563247,"high_a":148.631735,"low_a":147.441791,"close_a":147.818466,"volume_a":1988900,"open_b":175.305322,"high_b":176.379802,"low_b":173.957341,"close_b":174.064789,"volume_b":59773000,"open_bench":448.579535,"close_bench":446.918243},{"date":"2021-12-31","open_a":147.638662,"high_a":148.974137,"low_a":147.339031,"close_a":148.708755,"volume_a":2914900,"open_b":173.957385,"high_b":175.070931,"low_b":173.146644,"close_b":173.449463,"volume_b":64062300,"open_bench":446.430193,"close_bench":445.791931},{"date":"2022-01-03","open_a":147.416079,"high_a":148.263598,"low_a":146.020686,"close_a":148.083817,"volume_a":5488900,"open_b":173.703368,"high_b":178.636183,"low_b":173.586157,"close_b":177.786362,"volume_b":104487900,"open_bench":447.049634,"close_bench":448.373047},{"date":"2022-01-04","open_a":147.809865,"high_a":149.342233,"low_a":147.416065,"close_a":148.297821,"volume_a":4717600,"open_b":178.392015,"high_b":178.694819,"low_b":174.963456,"close_b":175.529999,"volume_b":99310400,"open_bench":449.790294,"close_bench":448.222839},{"date":"2022-01-05","open_a":148.409148,"high_a":150.121298,"low_a":148.297854,"close_a":148.802948,"volume_a":6564600,"open_b":175.44207,"high_b":175.989072,"low_b":170.5874,"close_b":170.860901,"volume_b":94537600,"open_bench":447.856864,"close_bench":439.616058},{"date":"2022-01-06","open_a":148.879934,"high_a":150.326702,"low_a":148.297797,"close_a":148.837128,"volume_a":4288700,"open_b":168.692436,"high_b":171.232108,"low_b":167.657036,"close_b":168.008682,"volume_b":96904000,"open_bench":439.156177,"close_bench":439.203094},{"date":"2022-01-07","open_a":149.034053,"high_a":149.547704,"low_a":147.767066,"close_a":149.025497,"volume_a":4916000,"open_b":168.878022,"high_b":170.099015,"low_b":167.061183,"close_b":168.174728,"volume_b":86709100,"open_bench":439.212525,"close_bench":437.466736},{"date":"2022-01-10","open_a":149.333668,"high_a":150.532167,"low_a":148.63168,"close_a":149.102524,"volume_a":5055900,"open_b":165.156458,"high_b":168.497094,"low_b":164.267571,"close_b":168.19429,"volume_b":106765600,"open_bench":434.28481,"close_bench":436.922241},{"date":"2022-01-11","open_a":148.683033,"high_a":149.153877,"low_a":146.919522,"close_a":149.034027,"volume_a":6452300,"open_b":168.321264,"high_b":171.114882,"low_b":166.856072,"close_b":171.017212,"volume_b":76138300,"open_bench":436.659477,"close_bench":440.901886},{"date":"2022-01-12","open_a":148.871429,"high_a":149.239548,"low_a":148.058161,"close_a":148.871429,"volume_a":5268000,"open_b":172.033062,"high_b":173.068462,"low_b":170.763241,"close_b":171.456757,"volume_b":74805200,"open_bench":442.628966,"close_bench":442.093964},{"date":"2022-01-13","open_a":148.708771,"high_a":149.547721,"low_a":148.306415,"close_a":149.111115,"volume_a":4310300,"open_b":171.70098,"high_b":172.521484,"low_b":167.803563,"close_b":168.19429,"volume_b":84505800,"open_bench":443.192063,"close_bench":436.002472},{"date":"2022-01-14","open_a":149.444969,"high_a":150.617787,"low_a":148.44335,"close_a":150.360962,"volume_a":4474500,"open_b":167.363983,"high_b":169.747364,"low_b":167.119784,"close_b":169.053848,"volume_b":80440800,"open_bench":432.86754,"close_bench":436.180756},{"date":"2022-01-18","open_a":148.6317,"high_a":149.607625,"low_a":148.229344,"close_a":148.922775,"volume_a":5789500,"open_b":167.530021,"high_b":168.536118,"low_b":165.478761,"close_b":165.859711,"volume_b":90956700,"open_bench":431.506558,"close_bench":428.456146},{"date":"2022-01-19","open_a":148.922796,"high_a":150.634946,"low_a":148.588927,"close_a":149.992889,"volume_a":6424500,"open_b":166.055078,"high_b":167.110018,"low_b":162.089295,"close_b":162.372559,"volume_b":94815000,"open_bench":429.995523,"close_bench":424.007324},{"date":"2022-01-20","open_a":150.164086,"high_a":151.422517,"low_a":148.820042,"close_a":148.905655,"volume_a":6284200,"open_b":163.105158,"high_b":165.742501,"low_b":160.37013,"close_b":160.692474,"volume_b":91420500,"open_bench":425.884511,"close_bench":419.314392},{"date":"2022-01-21","open_a":150.095583,"high_a":151.730689,"low_a":149.051171,"close_a":149.14534,"volume_a":6753200,"open_b":160.604587,"high_b":162.470269,"low_b":158.533787,"close_b":158.641235,"volume_b":122848900,"open_bench":418.197496,"close_bench":411.083008},{"date":"2022-01-24","open_a":150.232563,"high_a":151.097194,"low_a":144.642396,"close_a":148.383438,"volume_a":9263900,"open_b":156.306645,"high_b":158.533735,"low_b":151.110091,"close_b":157.869507,"volume_b":162294600,"open_bench":405.498332,"close_bench":412.828705},{"date":"2022-01-25","open_a":147.31334,"high_a":147.655765,"low_a":145.310128,"close_a":146.67984,"volume_a":6536900,"open_b":155.290796,"high_b":158.983079,"low_b":153.376287,"close_b":156.072235,"volume_b":115798400,"open_bench":406.4651,"close_bench":407.788513},{"date":"2022-01-26","open_a":144.984855,"high_a":147.082236,"low_a":142.493674,"close_a":145.130386,"volume_a":6841800,"open_b":159.70593,"high_b":160.575276,"low_b":154.157743,"close_b":155.984344,"volume_b":108275300,"open_bench":413.654708,"close_bench":406.765472},{"date":"2022-01-27","open_a":146.200477,"high_a":148.340664,"low_a":144.488327,"close_a":144.993408,"volume_a":6543500,"open_b":158.680296,"high_b":160.03804,"low_b":154.607064,"close_b":155.525253,"volume_b":121954600,"open_bench":411.345714,"close_bench":404.756805},{"date":"2022-01-28","open_a":144.257191,"high_a":147.869829,"low_a":143.144304,"close_a":147.818466,"volume_a":5796000,"open_b":161.864655,"high_b":166.396981,"low_b":159.022179,"close_b":166.377441,"volume_b":179935700,"open_bench":406.10844,"close_bench":414.809174},{"date":"2022-01-31","open_a":146.191876,"high_a":149.051162,"low_a":146.037776,"close_a":148.546082,"volume_a":5908000,"open_b":166.211366,"high_b":170.939048,"low_b":165.57644,"close_b":170.724152,"volume_b":115541600,"open_bench":414.14276,"close_bench":422.280334},{"date":"2022-02-01","open_a":148.229338,"high_a":148.469038,"low_a":145.156032,"close_a":147.535919,"volume_a":5952700,"open_b":169.972084,"high_b":170.782825,"low_b":168.311535,"close_b":170.558167,"volume_b":86213900,"open_bench":423.002992,"close_bench":425.133606},{"date":"2022-02-02","open_a":147.895486,"high_a":150.360985,"low_a":147.732842,"close_a":150.215454,"volume_a":5767000,"open_b":170.694865,"high_b":171.798647,"low_b":169.307818,"close_b":171.759567,"volume_b":84914300,"open_bench":427.526973,"close_bench":429.263367},{"date":"2022-02-03","open_a":150.044237,"high_a":151.234181,"low_a":148.811488,"close_a":150.129837,"volume_a":4632700,"open_b":170.431124,"high_b":172.150292,"low_b":168.125888,"close_b":168.887787,"volume_b":89418100,"open_bench":423.256449,"close_bench":419.173584},{"date":"2022-02-04","open_a":148.186543,"high_a":149.718911,"low_a":146.551437,"close_a":147.664337,"volume_a":4592400,"open_b":167.909727,"high_b":170.276594,"low_b":166.931688,"close_b":168.604141,"volume_b":82465400,"open_bench":418.938965,"close_bench":421.144653},{"date":"2022-02-07","open_a":147.878355,"high_a":148.092374,"low_a":146.568562,"close_a":147.082199,"volume_a":4813300,"open_b":169.063872,"high_b":170.129932,"low_b":167.195814,"close_b":167.890228,"volume_b":77251200,"open_bench":421.904823,"close_bench":419.792999},{"date":"2022-02-08","open_a":147.784184,"high_a":147.79274,"low_a":145.798085,"close_a":147.261978,"volume_a":3957700,"open_b":167.958658,"high_b":171.49917,"low_b":167.665243,"close_b":170.990585,"volume_b":74829200,"open_bench":419.295621,"close_bench":423.24707},{"date":"2022-02-09","open_a":147.672912,"high_a":147.912612,"low_a":146.397356,"close_a":147.193512,"volume_a":6780200,"open_b":172.183792,"high_b":172.770607,"low_b":171.059038,"close_b":172.408737,"volume_b":71285000,"open_bench":427.264271,"close_bench":429.441803},{"date":"2022-02-10","open_a":144.385613,"high_a":146.774057,"low_a":143.221351,"close_a":144.137344,"volume_a":7899800,"open_b":170.315716,"high_b":171.626284,"low_b":167.782598,"close_b":168.340073,"volume_b":90865900,"open_bench":423.62247,"close_bench":421.726532},{"date":"2022-02-11","open_a":143.8719,"high_a":145.798068,"low_a":143.555156,"close_a":144.317062,"volume_a":7299600,"open_b":168.545506,"high_b":169.279035,"low_b":164.349709,"close_b":164.936539,"volume_b":98670700,"open_bench":421.810991,"close_bench":413.410614},{"date":"2022-02-14","open_a":145.036181,"high_a":145.258756,"low_a":141.337945,"close_a":142.707657,"volume_a":6822500,"open_b":163.694386,"high_b":165.855858,"low_b":162.902176,"close_b":165.171234,"volume_b":86185500,"open_bench":412.903896,"close_bench":412.059143},{"date":"2022-02-15","open_a":143.152838,"high_a":144.034594,"low_a":141.474925,"close_a":142.074188,"volume_a":4943800,"open_b":167.215338,"high_b":169.151851,"low_b":166.511149,"close_b":168.995361,"volume_b":62527400,"open_bench":416.479865,"close_bench":418.704315},{"date":"2022-02-16","open_a":141.791666,"high_a":143.135696,"low_a":140.858534,"close_a":142.365234,"volume_a":4161700,"open_b":168.076041,"high_b":169.53331,"low_b":166.315568,"close_b":168.760666,"volume_b":61177400,"open_bench":416.66754,"close_bench":419.173584},{"date":"2022-02-17","open_a":142.116958,"high_a":143.777745,"low_a":141.252327,"close_a":142.750458,"volume_a":5156800,"open_b":167.274012,"high_b":168.134691,"low_b":164.770235,"close_b":165.171234,"volume_b":69589300,"open_bench":416.00116,"close_bench":410.219452},{"date":"2022-02-18","open_a":143.058646,"high_a":144.026015,"low_a":142.29674,"close_a":143.572296,"volume_a":5181600,"open_b":166.090624,"high_b":166.794799,"low_b":162.540337,"close_b":163.625961,"volume_b":82772700,"open_bench":410.472833,"close_bench":407.563232},{"date":"2022-02-22","open_a":144.308531,"high_a":144.625274,"low_a":142.305306,"close_a":144.120193,"volume_a":7733900,"open_b":161.356905,"high_b":163.029359,"low_b":158.589052,"close_b":160.711411,"volume_b":91162800,"open_bench":405.366955,"close_bench":403.189423},{"date":"2022-02-23","open_a":144.779367,"high_a":144.779367,"low_a":142.553581,"close_a":142.699112,"volume_a":5812800,"open_b":161.904562,"high_b":162.501166,"low_b":156.241723,"close_b":156.554703,"volume_b":90009200,"open_bench":406.089658,"close_bench":396.037384},{"date":"2022-02-24","open_a":141.860151,"high_a":142.39947,"low_a":137.220215,"close_a":140.216476,"volume_a":8888800,"open_b":149.229211,"high_b":159.273677,"low_b":148.661946,"close_b":159.166092,"volume_b":141147500,"open_bench":385.778599,"close_bench":401.997406},{"date":"2022-02-25","open_a":140.755797,"high_a":144.214346,"low_a":140.618822,"close_a":144.145859,"volume_a":6535600,"open_b":160.241923,"high_b":161.493811,"low_b":157.337145,"close_b":161.229752,"volume_b":91974200,"open_bench":403.226972,"close_bench":410.867096},{"date":"2022-02-28","open_a":140.430503,"high_a":141.432122,"low_a":138.179035,"close_a":140.173691,"volume_a":7677000,"open_b":159.479077,"high_b":161.78725,"low_b":158.862907,"close_b":161.493835,"volume_b":95056600,"open_bench":405.498345,"close_bench":409.815857},{"date":"2022-03-01","open_a":139.60013,"high_a":140.738698,"low_a":138.084874,"close_a":138.915268,"volume_a":5107900,"open_b":161.083041,"high_b":162.941324,"low_b":158.412998,"close_b":159.615982,"volume_b":83474400,"open_bench":408.323473,"close_bench":403.574219},{"date":"2022-03-02","open_a":139.523068,"high_a":141.954331,"low_a":139.1036,"close_a":140.841431,"volume_a":5452100,"open_b":160.779833,"high_b":163.68461,"low_b":159.371454,"close_b":162.902176,"volume_b":79724800,"open_bench":405.817478,"close_bench":410.998505},{"date":"2022-03-03","open_a":141.293788,"high_a":142.34507,"low_a":140.285595,"close_a":140.690598,"volume_a":5352200,"open_b":164.770235,"high_b":165.200574,"low_b":161.914363,"close_b":162.579422,"volume_b":76678400,"open_bench":413.420052,"close_bench":408.952362},{"date":"2022-03-04","open_a":139.596242,"high_a":143.473916,"low_a":138.665599,"close_a":142.827637,"volume_a":6340300,"open_b":160.877643,"high_b":161.914362,"low_b":158.54013,"close_b":159.586624,"volume_b":83737200,"open_bench":405.235571,"close_bench":405.629791},{"date":"2022-03-07","open_a":141.37998,"high_a":142.31061,"low_a":139.70827,"close_a":139.984009,"volume_a":6319500,"open_b":159.772504,"high_b":161.396052,"low_b":155.547366,"close_b":155.801666,"volume_b":96418800,"open_bench":405.047869,"close_bench":393.67218},{"date":"2022-03-08","open_a":139.923714,"high_a":141.293822,"low_a":135.882314,"close_a":136.037415,"volume_a":6163700,"open_b":155.332182,"high_b":159.303018,"low_b":152.378499,"close_b":153.982483,"volume_b":131148300,"open_bench":393.850447,"close_bench":390.687408},{"date":"2022-03-09","open_a":137.709113,"high_a":137.92454,"low_a":134.770695,"close_a":135.632401,"volume_a":5764900,"open_b":157.933756,"high_b":159.82138,"low_b":155.909223,"close_b":159.371475,"volume_b":91454900,"open_bench":399.031463,"close_bench":401.162048},{"date":"2022-03-10","open_a":134.245102,"high_a":134.641493,"low_a":132.159777,"close_a":133.14212,"volume_a":6103700,"open_b":156.681871,"high_b":156.867701,"low_b":152.554545,"close_b":155.038773,"volume_b":105342000,"open_bench":396.572344,"close_bench":399.350586},{"date":"2022-03-11","open_a":133.95209,"high_a":135.192947,"low_a":132.254529,"close_a":132.469955,"volume_a":4762600,"open_b":155.439778,"high_b":155.782098,"low_b":151.107071,"close_b":151.332016,"volume_b":96970100,"open_bench":401.828514,"close_bench":394.272888},{"date":"2022-03-14","open_a":133.693556,"high_a":135.977083,"low_a":133.133453,"close_a":134.331223,"volume_a":6075600,"open_b":148.12403,"high_b":150.735393,"low_b":146.803686,"close_b":147.312256,"volume_b":108732100,"open_bench":395.042542,"close_bench":391.391418},{"date":"2022-03-15","open_a":135.56348,"high_a":137.441992,"low_a":135.14124,"close_a":137.011139,"volume_a":6203400,"open_b":147.586096,"high_b":152.153552,"low_b":147.077526,"close_b":151.684082,"volume_b":92964300,"open_bench":393.991151,"close_bench":399.998138},{"date":"2022-03-16","open_a":136.50274,"high_a":137.864236,"low_a":135.322195,"close_a":137.614334,"volume_a":6299000,"open_b":153.601053,"high_b":156.486266,"low_b":151.067935,"close_b":156.085266,"volume_b":102300200,"open_bench":403.489766,"close_bench":408.867859},{"date":"2022-03-17","open_a":137.726368,"high_a":139.27744,"low_a":137.269665,"close_a":138.682861,"volume_a":3587600,"open_b":155.126783,"high_b":157.464296,"low_b":154.168309,"close_b":157.092636,"volume_b":75615400,"open_bench":406.962563,"close_bench":413.983215},{"date":"2022-03-18","open_a":139.061968,"high_a":140.3976,"low_a":137.924511,"close_a":140.276962,"volume_a":12123000,"open_b":156.985063,"high_b":160.86788,"low_b":156.251534,"close_b":160.37886,"volume_b":123511700,"open_bench":412.378859,"close_bench":418.517456},{"date":"2022-03-21","open_a":139.828892,"high_a":141.155913,"low_a":138.794849,"close_a":140.078781,"volume_a":5122600,"open_b":159.91918,"high_b":162.696822,"low_b":159.43016,"close_b":161.748123,"volume_b":95811400,"open_bench":418.347958,"close_bench":418.39505},{"date":"2022-03-22","open_a":140.320082,"high_a":141.98318,"low_a":139.794442,"close_a":141.61264,"volume_a":5779200,"open_b":161.875272,"high_b":165.699409,"low_b":161.288458,"close_b":165.112595,"volume_b":81532000,"open_bench":419.779072,"close_bench":423.290894},{"date":"2022-03-23","open_a":141.535083,"high_a":142.422639,"low_a":140.139125,"close_a":140.871567,"volume_a":4533700,"open_b":164.300813,"high_b":168.848689,"low_b":163.968268,"close_b":166.472061,"volume_b":98062700,"open_bench":420.767662,"close_bench":417.839569},{"date":"2022-03-24","open_a":141.371353,"high_a":141.948694,"low_a":140.690612,"close_a":141.724655,"volume_a":3160300,"open_b":167.303362,"high_b":170.315724,"low_b":166.472037,"close_b":170.247269,"volume_b":90131400,"open_bench":419.854412,"close_bench":424.138245},{"date":"2022-03-25","open_a":141.690206,"high_a":142.956914,"low_a":141.690206,"close_a":142.388199,"volume_a":4146800,"open_b":170.061462,"high_b":171.43071,"low_b":168.956272,"close_b":170.883011,"volume_b":80546200,"open_bench":424.769003,"close_bench":426.209503},{"date":"2022-03-28","open_a":142.267529,"high_a":142.84487,"low_a":141.069759,"close_a":142.810394,"volume_a":4308000,"open_b":168.389016,"high_b":171.870833,"low_b":168.222751,"close_b":171.743698,"volume_b":90371900,"open_bench":425.616384,"close_bench":429.24118},{"date":"2022-03-29","open_a":144.154664,"high_a":145.266271,"low_a":143.361896,"close_a":144.930206,"volume_a":4440400,"open_b":172.809754,"high_b":175.078797,"low_b":172.467434,"close_b":175.029907,"volume_b":100589400,"open_bench":433.110769,"close_bench":434.55127},{"date":"2022-03-30","open_a":144.852694,"high_a":144.973332,"low_a":143.146508,"close_a":144.732056,"volume_a":3871500,"open_b":174.628867,"high_b":175.665585,"low_b":172.819488,"close_b":173.865997,"volume_b":92633200,"open_bench":433.412064,"close_bench":431.868011},{"date":"2022-03-31","open_a":145.085303,"high_a":145.886696,"low_a":144.137435,"close_a":144.232224,"volume_a":6076200,"open_b":173.93453,"high_b":174.12036,"low_b":170.570072,"close_b":170.775467,"volume_b":103049300,"open_bench":431.105379,"close_bench":425.220978},{"date":"2022-04-01","open_a":144.671712,"high_a":146.429585,"low_a":144.223621,"close_a":146.283096,"volume_a":3239600,"open_b":170.20816,"high_b":171.039499,"low_b":168.164062,"close_b":170.48201,"volume_b":78751300,"open_bench":426.793289,"close_bench":426.426117},{"date":"2022-04-04","open_a":145.722982,"high_a":145.964258,"low_a":144.292548,"close_a":145.903946,"volume_a":3759900,"open_b":170.73631,"high_b":174.570221,"low_b":170.60916,"close_b":174.521317,"volume_b":76468400,"open_bench":426.623828,"close_bench":430.079132},{"date":"2022-04-05","open_a":145.042246,"high_a":147.679062,"low_a":145.042246,"close_a":146.059052,"volume_a":4480700,"open_b":173.601948,"high_b":174.384383,"low_b":170.589586,"close_b":171.21553,"volume_b":73401800,"open_bench":428.59148,"close_bench":424.646576},{"date":"2022-04-06","open_a":146.420979,"high_a":148.816519,"low_a":146.050452,"close_a":148.549393,"volume_a":4294800,"open_b":168.574832,"high_b":169.816946,"low_b":166.393809,"close_b":168.056473,"volume_b":89058800,"open_bench":420.748832,"close_bench":420.400452},{"date":"2022-04-07","open_a":148.23915,"high_a":149.316282,"low_a":147.162019,"close_a":148.687241,"volume_a":5041600,"open_b":167.40116,"high_b":169.552843,"low_b":166.119932,"close_b":168.359634,"volume_b":77594700,"open_bench":419.524883,"close_bench":422.51886},{"date":"2022-04-08","open_a":149.066374,"high_a":149.669564,"low_a":147.859978,"close_a":149.187012,"volume_a":3883100,"open_b":168.007518,"high_b":168.007518,"low_b":165.484176,"close_b":166.35463,"volume_b":76575500,"open_bench":421.765603,"close_bench":421.389008},{"date":"2022-04-11","open_a":149.600634,"high_a":150.772553,"low_a":148.48904,"close_a":148.730316,"volume_a":3825100,"open_b":165.005003,"high_b":165.317968,"low_b":161.865491,"close_b":162.110001,"volume_b":72246700,"open_bench":418.13143,"close_bench":414.186554},{"date":"2022-04-12","open_a":149.747126,"high_a":150.496805,"low_a":148.75617,"close_a":149.333511,"volume_a":3924300,"open_b":164.330137,"high_b":166.139501,"low_b":162.980439,"close_b":163.978043,"volume_b":79265200,"open_bench":417.16167,"close_bench":412.651886},{"date":"2022-04-13","open_a":149.574803,"high_a":149.781617,"low_a":147.980643,"close_a":149.075012,"volume_a":4394000,"open_b":163.713945,"high_b":167.283781,"low_b":163.107566,"close_b":166.657837,"volume_b":70618900,"open_bench":412.407094,"close_bench":417.378235},{"date":"2022-04-14","open_a":149.488617,"high_a":150.160746,"low_a":148.006483,"close_a":148.127121,"volume_a":4834300,"open_b":166.873015,"high_b":167.508749,"low_b":161.415555,"close_b":161.660065,"volume_b":75329400,"open_bench":417.604135,"close_bench":412.181091},{"date":"2022-04-18","open_a":147.541158,"high_a":148.480414,"low_a":146.455414,"close_a":146.851791,"volume_a":3086000,"open_b":160.320162,"high_b":162.941314,"low_b":159.977857,"close_b":161.444916,"volume_b":69023900,"open_bench":411.258468,"close_bench":412.350616},{"date":"2022-04-19","open_a":146.352015,"high_a":149.221495,"low_a":146.352015,"close_a":148.988831,"volume_a":3642000,"open_b":161.396032,"high_b":164.134544,"low_b":160.310408,"close_b":163.723755,"volume_b":67723800,"open_bench":412.247057,"close_bench":419.00708},{"date":"2022-04-20","open_a":149.695461,"high_a":151.634298,"low_a":149.695461,"close_a":151.203445,"volume_a":4197600,"open_b":165.053894,"high_b":165.171269,"low_b":162.452321,"close_b":163.557495,"volume_b":67929800,"open_bench":420.777126,"close_bench":418.696381},{"date":"2022-04-21","open_a":151.45332,"high_a":152.728654,"low_a":150.496827,"close_a":150.660553,"volume_a":3701300,"open_b":165.200581,"high_b":167.763039,"low_b":162.266464,"close_b":162.765259,"volume_b":87227800,"open_bench":422.302339,"close_bench":412.435364},{"date":"2022-04-22","open_a":150.789844,"high_a":151.03112,"low_a":148.16164,"close_a":148.34259,"volume_a":5864000,"open_b":162.804378,"high_b":164.183402,"low_b":157.953298,"close_b":158.236923,"volume_b":84882400,"open_bench":411.352584,"close_bench":401.118439},{"date":"2022-04-25","open_a":149.255959,"high_a":149.867762,"low_a":147.351585,"close_a":149.712662,"volume_a":7369900,"open_b":157.581636,"high_b":159.586619,"low_b":154.980063,"close_b":159.302994,"volume_b":96046400,"open_bench":398.887123,"close_bench":403.444},{"date":"2022-04-26","open_a":149.910852,"high_a":151.10001,"low_a":148.842333,"close_a":149.333511,"volume_a":7386600,"open_b":158.686863,"high_b":158.774883,"low_b":153.278307,"close_b":153.356552,"volume_b":95623200,"open_bench":400.920767,"close_bench":391.759949},{"date":"2022-04-27","open_a":149.436929,"high_a":152.254709,"low_a":149.23874,"close_a":150.669174,"volume_a":5180300,"open_b":152.486082,"high_b":156.280864,"low_b":151.967723,"close_b":153.131592,"volume_b":88063200,"open_bench":392.833176,"close_bench":392.86142},{"date":"2022-04-28","open_a":150.393417,"high_a":153.056083,"low_a":149.781613,"close_a":152.952682,"volume_a":5235800,"open_b":155.752764,"high_b":160.907036,"low_b":155.439784,"close_b":160.046356,"volume_b":130216800,"open_bench":397.587819,"close_bench":402.784912},{"date":"2022-04-29","open_a":152.875142,"high_a":152.875142,"low_a":147.70491,"close_a":147.963425,"volume_a":6505600,"open_b":158.285837,"high_b":162.550088,"low_b":153.796641,"close_b":154.187851,"volume_b":131747600,"open_bench":398.811812,"close_bench":387.89978},{"date":"2022-05-02","open_a":149.075033,"high_a":149.471424,"low_a":142.414054,"close_a":144.559692,"volume_a":7692100,"open_b":153.26854,"high_b":154.755149,"low_b":149.904082,"close_b":154.491089,"volume_b":123055300,"open_bench":387.965652,"close_bench":390.23468},{"date":"2022-05-03","open_a":145.748829,"high_a":145.895318,"low_a":143.542866,"close_a":144.757874,"volume_a":5564900,"open_b":154.676867,"high_b":157.18066,"low_b":152.887069,"close_b":155.977661,"volume_b":88966500,"open_bench":390.733673,"close_bench":392.023529},{"date":"2022-05-04","open_a":144.430398,"high_a":150.040094,"low_a":144.430398,"close_a":149.816055,"volume_a":6231500,"open_b":156.163514,"high_b":162.823959,"low_b":155.762515,"close_b":162.374069,"volume_b":108256500,"open_bench":392.682561,"close_bench":403.961792},{"date":"2022-05-05","open_a":149.161201,"high_a":149.350777,"low_a":145.697142,"close_a":146.825974,"volume_a":6534300,"open_b":160.251729,"high_b":160.476674,"low_b":151.547171,"close_b":153.327209,"volume_b":130525300,"open_bench":399.715597,"close_bench":389.603851},{"date":"2022-05-06","open_a":145.835007,"high_a":147.179265,"low_a":145.386917,"close_a":146.843201,"volume_a":5824600,"open_b":152.808063,"high_b":156.167674,"low_b":151.01562,"close_b":154.052002,"volume_b":116124600,"open_bench":387.05243,"close_bench":387.278381},{"date":"2022-05-09","open_a":146.19693,"high_a":148.980235,"low_a":145.421388,"close_a":147.954803,"volume_a":7269100,"open_b":151.75024,"high_b":152.631777,"low_b":148.380854,"close_b":148.939148,"volume_b":131577900,"open_bench":381.403402,"close_bench":374.878784},{"date":"2022-05-10","open_a":149.178405,"high_a":150.402025,"low_a":147.136168,"close_a":147.773834,"volume_a":5936700,"open_b":152.328132,"high_b":153.523094,"low_b":149.791278,"close_b":151.338852,"volume_b":115366700,"open_bench":380.829051,"close_bench":375.744934},{"date":"2022-05-11","open_a":147.196531,"high_a":148.566639,"low_a":146.420988,"close_a":147.058655,"volume_a":5376900,"open_b":150.349557,"high_b":152.259532,"low_b":142.817384,"close_b":143.493225,"volume_b":142689800,"open_bench":374.784597,"close_bench":369.775787},{"date":"2022-05-12","open_a":147.567065,"high_a":148.204732,"low_a":144.904398,"close_a":146.83461,"volume_a":6473500,"open_b":139.839821,"high_b":143.199418,"low_b":135.9513,"close_b":139.634125,"volume_b":182602000,"open_bench":366.593471,"close_bench":369.38974},{"date":"2022-05-13","open_a":147.093066,"high_a":150.074572,"low_a":146.02456,"close_a":149.695419,"volume_a":4714600,"open_b":141.62246,"high_b":145.060431,"low_b":140.172839,"close_b":144.090744,"volume_b":113990900,"open_bench":373.504124,"close_bench":378.221069},{"date":"2022-05-16","open_a":149.686849,"high_a":151.918662,"low_a":149.436947,"close_a":151.203445,"volume_a":5317200,"open_b":142.562783,"high_b":144.492352,"low_b":141.22089,"close_b":142.552979,"volume_b":86643800,"open_bench":376.58285,"close_bench":376.686401},{"date":"2022-05-17","open_a":151.065556,"high_a":151.384383,"low_a":148.713104,"close_a":150.333115,"volume_a":5012200,"open_b":145.804812,"high_b":146.696138,"low_b":143.669546,"close_b":146.177017,"volume_b":78336300,"open_bench":382.749697,"close_bench":384.434998},{"date":"2022-05-18","open_a":149.669611,"high_a":149.824712,"low_a":140.38904,"close_a":141.018082,"volume_a":8875400,"open_b":143.83608,"high_b":144.335608,"low_b":137.028709,"close_b":137.92984,"volume_b":109742900,"open_bench":379.896959,"close_bench":368.937836},{"date":"2022-05-19","open_a":140.02711,"high_a":140.268387,"low_a":137.588483,"close_a":138.906891,"volume_a":6423800,"open_b":137.009109,"high_b":138.752576,"low_b":133.796429,"close_b":134.531036,"volume_b":136095600,"open_bench":365.887332,"close_bench":366.678192},{"date":"2022-05-20","open_a":139.803072,"high_a":140.509663,"low_a":137.545395,"close_a":139.777222,"volume_a":5768800,"open_b":136.235343,"high_b":137.8123,"low_b":129.888341,"close_b":134.766129,"volume_b":137426100,"open_bench":370.246559,"close_bench":366.838318},{"date":"2022-05-23","open_a":141.500655,"high_a":143.86172,"low_a":140.949164,"close_a":142.698425,"volume_a":4657800,"open_b":134.962031,"high_b":140.319767,"low_b":134.824904,"close_b":140.172852,"volume_b":117726300,"open_bench":369.851096,"close_bench":373.701874},{"date":"2022-05-24","open_a":143.060317,"high_a":144.990528,"low_a":142.31925,"close_a":144.611389,"volume_a":6327600,"open_b":137.92004,"high_b":139.056236,"low_b":134.511467,"close_b":137.479279,"volume_b":104132700,"open_bench":369.596965,"close_bench":370.849182},{"date":"2022-05-25","open_a":144.637235,"high_a":146.050431,"low_a":143.870318,"close_a":145.602341,"volume_a":5781000,"open_b":135.58887,"high_b":138.87991,"low_b":135.500721,"close_b":137.635986,"volume_b":92482700,"open_bench":369.36151,"close_bench":374.125519},{"date":"2022-05-26","open_a":145.59373,"high_a":147.429154,"low_a":145.59373,"close_a":146.584686,"volume_a":5193800,"open_b":134.570234,"high_b":141.377591,"low_b":134.325365,"close_b":140.829086,"volume_b":90601500,"open_bench":375.349501,"close_bench":381.601074},{"date":"2022-05-27","open_a":147.575659,"high_a":148.049599,"low_a":146.662253,"close_a":148.015137,"volume_a":6713500,"open_b":142.406043,"high_b":146.60799,"low_b":142.278707,"close_b":146.568817,"volume_b":90978500,"open_bench":384.048993,"close_bench":390.969055},{"date":"2022-05-31","open_a":147.334345,"high_a":147.334345,"low_a":142.500178,"close_a":144.551041,"volume_a":13610000,"open_b":146.010508,"high_b":147.567871,"low_b":143.826265,"close_b":145.785217,"volume_b":103718400,"open_bench":389.359088,"close_bench":388.77536},{"date":"2022-06-01","open_a":144.921603,"high_a":145.093955,"low_a":141.16458,"close_a":143.465332,"volume_a":5613900,"open_b":146.823477,"high_b":148.625725,"low_b":144.649039,"close_b":145.657913,"volume_b":74286600,"open_bench":390.884347,"close_bench":385.630737},{"date":"2022-06-02","open_a":143.708248,"high_a":144.688751,"low_a":141.131181,"close_a":144.628006,"volume_a":4677300,"open_b":144.795979,"high_b":148.165379,"low_b":143.845885,"close_b":148.106613,"volume_b":72348100,"open_bench":385.470607,"close_bench":392.974396},{"date":"2022-06-03","open_a":143.916548,"high_a":144.549964,"low_a":142.614999,"close_a":143.040176,"volume_a":4222000,"open_b":143.885049,"high_b":144.933095,"low_b":141.495139,"close_b":142.396255,"volume_b":88570300,"open_bench":388.276331,"close_bench":386.525146},{"date":"2022-06-06","open_a":144.43718,"high_a":145.443701,"low_a":142.936051,"close_a":143.638885,"volume_a":3447800,"open_b":144.012389,"high_b":145.520791,"low_b":141.9261,"close_b":143.140656,"volume_b":71598400,"open_bench":390.517145,"close_bench":387.702057},{"date":"2022-06-07","open_a":143.760347,"high_a":144.784238,"low_a":142.493514,"close_a":144.523926,"volume_a":4168300,"open_b":141.387396,"high_b":145.941955,"low_b":141.142527,"close_b":145.657913,"volume_b":67808200,"open_bench":384.227882,"close_bench":391.420959},{"date":"2022-06-08","open_a":143.812396,"high_a":144.740827,"low_a":143.265742,"close_a":143.803711,"volume_a":2646800,"open_b":145.530564,"high_b":146.794082,"low_b":144.433556,"close_b":144.923294,"volume_b":53950200,"open_bench":389.716782,"close_bench":387.165314},{"date":"2022-06-09","open_a":143.335145,"high_a":144.844932,"low_a":141.148541,"close_a":141.243988,"volume_a":3522900,"open_b":144.061339,"high_b":144.913479,"low_b":139.60472,"close_b":139.712463,"volume_b":69473000,"open_bench":385.395367,"close_bench":377.957489},{"date":"2022-06-10","open_a":139.673466,"high_a":142.233175,"low_a":138.354561,"close_a":141.018402,"volume_a":5203300,"open_b":137.400911,"high_b":137.871055,"low_b":134.246997,"close_b":134.315567,"volume_b":91437900,"open_bench":371.781117,"close_bench":366.99826},{"date":"2022-06-13","open_a":139.291677,"high_a":141.322089,"low_a":138.016159,"close_a":138.606201,"volume_a":6300700,"open_b":130.142996,"high_b":132.425178,"low_b":128.742353,"close_b":129.173325,"volume_b":122207100,"open_bench":357.630388,"close_bench":353.064087},{"date":"2022-06-14","open_a":138.831778,"high_a":138.875166,"low_a":134.371808,"close_a":135.465103,"volume_a":6159200,"open_b":130.397634,"high_b":131.14203,"low_b":128.781489,"close_b":130.035217,"volume_b":84784300,"open_bench":354.805876,"close_bench":352.000183},{"date":"2022-06-15","open_a":137.02699,"high_a":138.69297,"low_a":135.300264,"close_a":136.914185,"volume_a":5940100,"open_b":131.533849,"high_b":134.521255,"low_b":129.447575,"close_b":132.650452,"volume_b":91533000,"open_bench":355.286039,"close_bench":357.018433},{"date":"2022-06-16","open_a":135.907656,"high_a":137.495546,"low_a":134.58875,"close_a":136.25473,"volume_a":7207900,"open_b":129.369217,"high_b":129.672852,"low_b":126.391601,"close_b":127.390671,"volume_b":108123900,"open_bench":348.836767,"close_bench":345.202545},{"date":"2022-06-17","open_a":136.254762,"high_a":137.955458,"low_a":135.508541,"close_a":136.280792,"volume_a":9881800,"open_b":127.400469,"high_b":130.348686,"low_b":127.145795,"close_b":128.859879,"volume_b":134520300,"open_bench":345.615698,"close_bench":345.946625},{"date":"2022-06-21","open_a":137.243884,"high_a":139.66477,"low_a":136.532378,"close_a":139.447845,"volume_a":4449600,"open_b":130.681707,"high_b":134.247,"low_b":130.583769,"close_b":133.081421,"volume_b":81000500,"open_bench":351.648547,"close_bench":354.655457},{"date":"2022-06-22","open_a":139.118146,"high_a":141.426228,"low_a":138.450014,"close_a":140.593231,"volume_a":4706500,"open_b":132.023579,"high_b":134.932624,"low_b":131.16165,"close_b":132.572098,"volume_b":73409200,"open_bench":350.447629,"close_bench":354.012451},{"date":"2022-06-23","open_a":142.111719,"high_a":143.29179,"low_a":141.565065,"close_a":142.389389,"volume_a":5810700,"open_b":134.011922,"high_b":135.745584,"low_b":132.846343,"close_b":135.432159,"volume_b":72433800,"open_bench":356.13991,"close_bench":357.482605},{"date":"2022-06-24","open_a":143.725641,"high_a":144.202878,"low_a":142.250555,"close_a":144.150818,"volume_a":5951500,"open_b":137.028706,"high_b":138.997463,"low_b":136.901385,"close_b":138.752594,"volume_b":89116800,"open_bench":360.640838,"close_bench":368.848389},{"date":"2022-06-27","open_a":143.9252,"high_a":145.018509,"low_a":143.465321,"close_a":144.341705,"volume_a":4595800,"open_b":139.771243,"high_b":140.545037,"low_b":138.076753,"close_b":138.752594,"volume_b":70207900,"open_bench":369.765581,"close_bench":367.439484},{"date":"2022-06-28","open_a":145.035869,"high_a":146.16388,"low_a":142.172473,"close_a":142.311295,"volume_a":4372900,"open_b":139.212963,"high_b":140.476481,"low_b":134.501685,"close_b":134.619217,"volume_b":67083400,"open_bench":368.990228,"close_bench":359.931641},{"date":"2022-06-29","open_a":144.090091,"high_a":145.391639,"low_a":143.474032,"close_a":144.671448,"volume_a":3653600,"open_b":134.638805,"high_b":137.782915,"low_b":133.86501,"close_b":136.372467,"volume_b":66242400,"open_bench":360.480183,"close_bench":359.638611},{"date":"2022-06-30","open_a":145.018507,"high_a":145.408969,"low_a":143.716959,"close_a":144.610687,"volume_a":6295100,"open_b":134.433123,"high_b":135.530131,"low_b":131.024549,"close_b":133.914001,"volume_b":98964500,"open_bench":355.76173,"close_bench":356.716766},{"date":"2022-07-01","open_a":144.680111,"high_a":147.092313,"low_a":143.829756,"close_a":146.979507,"volume_a":3918700,"open_b":133.247928,"high_b":136.186356,"low_b":132.875737,"close_b":136.078613,"volume_b":71051600,"open_bench":356.064206,"close_bench":360.489471},{"date":"2022-07-05","open_a":146.320064,"high_a":146.97951,"low_a":143.456655,"close_a":146.64978,"volume_a":4971600,"open_b":134.942425,"high_b":138.703609,"low_b":134.119654,"close_b":138.654633,"volume_b":73353800,"open_bench":355.421349,"close_bench":361.17041},{"date":"2022-07-06","open_a":147.508779,"high_a":149.200789,"low_a":147.057572,"close_a":148.116165,"volume_a":4209100,"open_b":138.44894,"high_b":141.162078,"low_b":138.184477,"close_b":139.98671,"volume_b":74064300,"open_bench":361.312202,"close_bench":362.390167},{"date":"2022-07-07","open_a":147.092318,"high_a":148.411211,"low_a":146.970841,"close_a":147.821182,"volume_a":5150200,"open_b":140.349119,"high_b":143.542221,"low_b":140.33933,"close_b":143.346329,"volume_b":66253700,"open_bench":364.158423,"close_bench":367.81778},{"date":"2022-07-08","open_a":148.003392,"high_a":149.851582,"low_a":147.986035,"close_a":149.140076,"volume_a":3482900,"open_b":142.278705,"high_b":144.521713,"low_b":142.024046,"close_b":144.022171,"volume_b":64547800,"open_bench":366.191405,"close_bench":367.515228},{"date":"2022-07-11","open_a":148.73225,"high_a":149.93835,"low_a":147.526149,"close_a":147.916611,"volume_a":4921300,"open_b":142.680282,"high_b":143.630375,"low_b":140.829073,"close_b":141.896698,"volume_b":63141600,"open_bench":364.848654,"close_bench":363.316833},{"date":"2022-07-12","open_a":150.111897,"high_a":150.389567,"low_a":146.528297,"close_a":147.074951,"volume_a":5201600,"open_b":142.768425,"high_b":145.403218,"low_b":142.073006,"close_b":142.866379,"volume_b":77588800,"open_bench":362.768447,"close_bench":360.101929},{"date":"2022-07-13","open_a":146.24196,"high_a":148.758294,"low_a":145.75605,"close_a":147.595581,"volume_a":4533100,"open_b":140.055307,"high_b":143.444287,"low_b":139.203153,"close_b":142.503998,"volume_b":71185600,"open_bench":354.683762,"close_bench":358.210724},{"date":"2022-07-14","open_a":146.025034,"high_a":148.515339,"low_a":145.513095,"close_a":148.359146,"volume_a":4218300,"open_b":141.122921,"high_b":145.892965,"low_b":140.309954,"close_b":145.422821,"volume_b":78140700,"open_bench":353.274877,"close_bench":357.340851},{"date":"2022-07-15","open_a":148.671501,"high_a":150.042466,"low_a":147.526145,"close_a":148.480606,"volume_a":4659100,"open_b":146.705926,"high_b":147.763762,"low_b":145.158352,"close_b":147.087921,"volume_b":76259900,"open_bench":361.728287,"close_bench":364.167877},{"date":"2022-07-18","open_a":148.506642,"high_a":148.584745,"low_a":145.643246,"close_a":145.998993,"volume_a":3479700,"open_b":147.646235,"high_b":148.459202,"low_b":143.689143,"close_b":144.051559,"volume_b":81420900,"open_bench":367.240935,"close_bench":361.160919},{"date":"2022-07-19","open_a":147.725747,"high_a":148.159597,"low_a":146.320079,"close_a":147.560883,"volume_a":4201600,"open_b":144.884106,"high_b":148.126169,"low_b":143.89484,"close_b":147.900894,"volume_b":82982400,"open_bench":365.066072,"close_bench":370.919159},{"date":"2022-07-20","open_a":147.552181,"high_a":148.003388,"low_a":145.304844,"close_a":145.998993,"volume_a":3460700,"open_b":148.01841,"high_b":150.565054,"low_b":147.283803,"close_b":149.899002,"volume_b":64823400,"open_bench":371.10834,"close_bench":373.283142},{"date":"2022-07-21","open_a":145.764703,"high_a":146.996833,"low_a":145.391586,"close_a":146.215897,"volume_a":3088300,"open_b":151.32906,"high_b":152.377107,"low_b":148.821604,"close_b":152.161621,"volume_b":65086600,"open_bench":372.706291,"close_bench":377.08429},{"date":"2022-07-22","open_a":146.76256,"high_a":148.003376,"low_a":146.198561,"close_a":147.17038,"volume_a":3757800,"open_b":152.200812,"high_b":153.072546,"low_b":150.261454,"close_b":150.92749,"volume_b":66675400,"open_bench":377.207339,"close_bench":373.585785},{"date":"2022-07-25","open_a":147.777796,"high_a":148.255033,"low_a":146.675815,"close_a":147.378662,"volume_a":3357200,"open_b":150.849125,"high_b":151.857984,"low_b":149.154635,"close_b":149.810883,"volume_b":53623900,"open_bench":374.209838,"close_bench":374.039642},{"date":"2022-07-26","open_a":148.003369,"high_a":149.235499,"low_a":146.849328,"close_a":149.096664,"volume_a":3732100,"open_b":149.135014,"high_b":149.947981,"low_b":147.704987,"close_b":148.488571,"volume_b":55138700,"open_bench":372.403798,"close_bench":369.61438},{"date":"2022-07-27","open_a":147.933952,"high_a":149.200785,"low_a":146.693136,"close_a":148.888412,"volume_a":4479000,"open_b":149.448466,"high_b":154.100977,"low_b":149.037088,"close_b":153.572052,"volume_b":78620700,"open_bench":372.895476,"close_bench":379.211914},{"date":"2022-07-28","open_a":149.131399,"high_a":151.95142,"low_a":148.54137,"close_a":151.708466,"volume_a":3134200,"open_b":153.758159,"high_b":154.404616,"low_b":151.240913,"close_b":154.120575,"volume_b":81378700,"open_bench":380.015628,"close_bench":383.968109},{"date":"2022-07-29","open_a":150.684565,"high_a":152.26377,"low_a":150.450283,"close_a":151.812576,"volume_a":4215500,"open_b":157.930728,"high_b":160.271675,"low_b":156.226434,"close_b":159.174652,"volume_b":101786900,"open_bench":385.395975,"close_bench":389.565948},{"date":"2022-08-01","open_a":151.456835,"high_a":154.181409,"low_a":151.456835,"close_a":153.539307,"volume_a":4034600,"open_b":157.705452,"high_b":160.232503,"low_b":157.58792,"close_b":158.19519,"volume_b":67829400,"open_bench":386.880472,"close_bench":388.412292},{"date":"2022-08-02","open_a":153.530652,"high_a":153.712862,"low_a":152.038209,"close_a":152.272491,"volume_a":3781100,"open_b":156.814147,"high_b":159.076735,"low_b":156.353792,"close_b":156.725983,"volume_b":59907000,"open_bench":386.852096,"close_bench":385.849792},{"date":"2022-08-03","open_a":151.916708,"high_a":153.634748,"low_a":151.187844,"close_a":153.435181,"volume_a":4401300,"open_b":157.538931,"high_b":163.170918,"low_b":157.450781,"close_b":162.720367,"volume_b":82507500,"open_bench":387.967885,"close_bench":391.892029},{"date":"2022-08-04","open_a":153.07941,"high_a":153.35708,"low_a":151.482847,"close_a":152.602173,"volume_a":3822500,"open_b":162.602859,"high_b":163.758649,"low_b":161.055285,"close_b":162.406967,"volume_b":55474100,"open_bench":391.816325,"close_bench":391.627228},{"date":"2022-08-05","open_a":151.318013,"high_a":151.586998,"low_a":149.591288,"close_a":151.456848,"volume_a":3295600,"open_b":160.082376,"high_b":162.671784,"low_b":159.876394,"close_b":162.181366,"volume_b":56697000,"open_bench":387.362708,"close_bench":390.965332},{"date":"2022-08-08","open_a":152.532766,"high_a":152.688959,"low_a":150.389562,"close_a":150.849442,"volume_a":2846100,"open_b":163.181796,"high_b":164.594203,"low_b":161.053382,"close_b":161.710541,"volume_b":60276900,"open_bench":392.648414,"close_bench":390.511414},{"date":"2022-08-09","open_a":151.665048,"high_a":152.046838,"low_a":150.94487,"close_a":151.413422,"volume_a":3228800,"open_b":160.876835,"high_b":162.642345,"low_b":160.121587,"close_b":161.759583,"volume_b":63135500,"open_bench":389.783427,"close_bench":388.960785},{"date":"2022-08-10","open_a":152.550144,"high_a":153.088113,"low_a":151.604355,"close_a":152.662949,"volume_a":4041100,"open_b":164.466678,"high_b":166.09487,"low_b":163.701627,"close_b":165.996796,"volume_b":70170500,"open_bench":395.98633,"close_bench":397.130463},{"date":"2022-08-11","open_a":153.027342,"high_a":153.686788,"low_a":151.560929,"close_a":151.881973,"volume_a":2743800,"open_b":166.801091,"high_b":167.713276,"low_b":164.966931,"close_b":165.261185,"volume_b":57149200,"open_bench":399.967176,"close_bench":397.130463},{"date":"2022-08-12","open_a":152.654248,"high_a":153.912421,"low_a":152.177024,"close_a":153.869034,"volume_a":3756200,"open_b":166.565695,"high_b":168.870653,"low_b":166.153731,"close_b":168.802002,"volume_b":68039400,"open_bench":399.059402,"close_bench":403.853455},{"date":"2022-08-15","open_a":154.25083,"high_a":155.630468,"low_a":153.409161,"close_a":155.56105,"volume_a":3043100,"open_b":168.233129,"high_b":170.067289,"low_b":168.066388,"close_b":169.871124,"volume_b":54091700,"open_bench":401.650257,"close_bench":405.517639},{"date":"2022-08-16","open_a":155.994906,"high_a":156.793187,"low_a":155.005729,"close_a":156.46347,"volume_a":3395700,"open_b":169.468948,"high_b":170.381134,"low_b":168.370416,"close_b":169.714157,"volume_b":56377100,"open_bench":404.449174,"close_bench":406.311951},{"date":"2022-08-17","open_a":156.272544,"high_a":157.114212,"low_a":155.942814,"close_a":156.376663,"volume_a":2666200,"open_b":169.459175,"high_b":172.774393,"low_b":169.26301,"close_b":171.205063,"volume_b":79542000,"open_bench":402.728265,"close_bench":403.427979},{"date":"2022-08-18","open_a":156.897313,"high_a":156.992747,"low_a":156.133733,"close_a":156.532867,"volume_a":3010600,"open_b":170.42037,"high_b":171.548326,"low_b":169.802438,"close_b":170.812698,"volume_b":62290100,"open_bench":403.626498,"close_bench":404.600464},{"date":"2022-08-19","open_a":155.977553,"high_a":157.062177,"low_a":155.595777,"close_a":156.333313,"volume_a":3704000,"open_b":169.714182,"high_b":170.410583,"low_b":168.027141,"close_b":168.233124,"volume_b":70346300,"open_bench":401.84881,"close_bench":399.163391},{"date":"2022-08-22","open_a":156.307264,"high_a":156.593606,"low_a":154.467747,"close_a":154.840851,"volume_a":3385000,"open_b":166.438179,"high_b":166.60492,"low_b":163.937043,"close_b":164.35881,"volume_b":69026800,"open_bench":394.350464,"close_bench":390.851868},{"date":"2022-08-23","open_a":154.294215,"high_a":155.196616,"low_a":153.834322,"close_a":154.771439,"volume_a":2575000,"open_b":163.87818,"high_b":165.476948,"low_b":163.456412,"close_b":164.025299,"volume_b":54147100,"open_bench":390.426364,"close_bench":389.906311},{"date":"2022-08-24","open_a":154.771434,"high_a":155.751938,"low_a":154.545838,"close_a":155.543686,"volume_a":3340600,"open_b":164.113582,"high_b":164.888437,"low_b":163.06408,"close_b":164.31955,"volume_b":53841500,"open_bench":389.679362,"close_bench":391.15448},{"date":"2022-08-25","open_a":155.344131,"high_a":155.621801,"low_a":153.921105,"close_a":155.552383,"volume_a":3401800,"open_b":165.545606,"high_b":166.879545,"low_b":165.123854,"close_b":166.771652,"volume_b":51218200,"open_bench":392.638967,"close_bench":396.676575},{"date":"2022-08-26","open_a":156.151062,"high_a":156.168407,"low_a":151.560929,"close_a":151.881973,"volume_a":5246600,"open_b":167.301324,"high_b":167.772122,"low_b":160.425649,"close_b":160.484497,"volume_b":78961000,"open_bench":396.563109,"close_bench":383.249451},{"date":"2022-08-29","open_a":151.45686,"high_a":152.098949,"low_a":150.294147,"close_a":151.4048,"volume_a":4184500,"open_b":158.061811,"high_b":159.778275,"low_b":156.757312,"close_b":158.287415,"volume_b":73314000,"open_bench":380.308804,"close_bench":380.715393},{"date":"2022-08-30","open_a":151.274627,"high_a":151.578326,"low_a":149.530557,"close_a":150.103241,"volume_a":4026300,"open_b":159.023057,"high_b":159.444809,"low_b":154.697563,"close_b":155.864761,"volume_b":77906200,"open_bench":381.868924,"close_bench":376.535889},{"date":"2022-08-31","open_a":150.81472,"high_a":151.335345,"low_a":149.417725,"close_a":149.47847,"volume_a":5597900,"open_b":157.237904,"high_b":157.502734,"low_b":154.128654,"close_b":154.207123,"volume_b":87991100,"open_bench":378.162208,"close_bench":373.670746},{"date":"2022-09-01","open_a":149.303765,"high_a":151.103235,"low_a":148.709771,"close_a":150.989685,"volume_a":4846100,"open_b":153.638257,"high_b":155.384145,"low_b":151.706007,"close_b":154.932968,"volume_b":74229900,"open_bench":371.505488,"close_bench":374.843353},{"date":"2022-09-02","open_a":151.74966,"high_a":152.841573,"low_a":148.404038,"close_a":149.07666,"volume_a":5801500,"open_b":156.688625,"high_b":157.286936,"low_b":152.000228,"close_b":152.824127,"volume_b":76957800,"open_bench":378.493301,"close_bench":370.8909},{"date":"2022-09-06","open_a":149.356188,"high_a":150.762572,"low_a":147.740165,"close_a":148.072098,"volume_a":4900100,"open_b":153.471521,"high_b":154.079635,"low_b":150.744796,"close_b":151.568695,"volume_b":73714800,"open_bench":371.732485,"close_bench":369.491486},{"date":"2022-09-07","open_a":149.041693,"high_a":151.802051,"low_a":148.438955,"close_a":151.339081,"volume_a":6074600,"open_b":151.85313,"high_b":153.667669,"low_b":150.666311,"close_b":152.971283,"volume_b":87449600,"open_bench":369.179349,"close_bench":376.129303},{"date":"2022-09-08","open_a":150.928517,"high_a":151.496306,"low_a":148.989279,"close_a":150.832428,"volume_a":3888100,"open_b":151.676589,"high_b":153.36363,"low_b":149.754143,"close_b":151.500046,"volume_b":84923800,"open_bench":373.869438,"close_bench":378.58783},{"date":"2022-09-09","open_a":150.814963,"high_a":152.474678,"low_a":149.958907,"close_a":151.312881,"volume_a":3662000,"open_b":152.49068,"high_b":154.795652,"low_b":151.784476,"close_b":154.354263,"volume_b":68028800,"open_bench":380.819319,"close_bench":384.469238},{"date":"2022-09-12","open_a":151.312882,"high_a":152.378589,"low_a":150.902321,"close_a":151.906876,"volume_a":5953300,"open_b":156.531747,"high_b":161.112253,"low_b":156.247311,"close_b":160.298157,"volume_b":104956000,"open_bench":386.530576,"close_bench":388.601379},{"date":"2022-09-13","open_a":150.683904,"high_a":150.972171,"low_a":145.835811,"close_a":146.23764,"volume_a":6926200,"open_b":156.83579,"high_b":157.463525,"low_b":150.430928,"close_b":150.891922,"volume_b":122656600,"open_bench":379.95889,"close_bench":371.704071},{"date":"2022-09-14","open_a":146.857871,"high_a":147.792555,"low_a":146.37744,"close_a":147.347046,"volume_a":5156900,"open_b":151.823671,"high_b":154.089415,"low_b":150.666291,"close_b":152.33371,"volume_b":87965400,"open_bench":372.999508,"close_bench":373.122437},{"date":"2022-09-15","open_a":147.216018,"high_a":147.242223,"low_a":144.429455,"close_a":144.901169,"volume_a":5461000,"open_b":151.686349,"high_b":152.265053,"low_b":148.479024,"close_b":149.450043,"volume_b":90481100,"open_bench":371.571557,"close_bench":368.886139},{"date":"2022-09-16","open_a":145.320448,"high_a":146.028005,"low_a":144.350829,"close_a":145.853302,"volume_a":12012600,"open_b":148.31234,"high_b":148.449656,"low_b":145.526752,"close_b":147.812103,"volume_b":162278800,"open_bench":364.723872,"close_bench":366.072083},{"date":"2022-09-19","open_a":145.931968,"high_a":147.460646,"low_a":145.442794,"close_a":147.390762,"volume_a":3720900,"open_b":146.4487,"high_b":151.598092,"low_b":146.242732,"close_b":151.519623,"volume_b":81474200,"open_bench":362.938865,"close_bench":368.910919},{"date":"2022-09-20","open_a":146.386159,"high_a":147.757595,"low_a":145.757217,"close_a":147.556686,"volume_a":3872000,"open_b":150.460351,"high_b":155.050675,"low_b":150.146491,"close_b":153.89328,"volume_b":107689800,"open_bench":365.597395,"close_bench":364.676422},{"date":"2022-09-21","open_a":148.395322,"high_a":150.273406,"low_a":147.050078,"close_a":147.137436,"volume_a":5134500,"open_b":154.324855,"high_b":155.698036,"low_b":150.656535,"close_b":150.774231,"volume_b":101696800,"open_bench":366.594226,"close_bench":358.315002},{"date":"2022-09-22","open_a":147.102459,"high_a":148.404024,"low_a":146.796732,"close_a":147.277176,"volume_a":4290600,"open_b":149.459887,"high_b":151.509831,"low_b":148.018055,"close_b":149.812988,"volume_b":86652500,"open_bench":357.545938,"close_bench":355.305237},{"date":"2022-09-23","open_a":147.093697,"high_a":147.408168,"low_a":145.468931,"close_a":147.20726,"volume_a":4565100,"open_b":148.292687,"high_b":148.56732,"low_b":145.713082,"close_b":147.547241,"volume_b":96029900,"open_bench":351.849278,"close_bench":349.352234},{"date":"2022-09-26","open_a":147.041314,"high_a":148.508838,"low_a":146.342487,"close_a":147.146133,"volume_a":5326400,"open_b":146.792054,"high_b":150.823294,"low_b":146.772433,"close_b":147.880783,"volume_b":93339400,"open_bench":347.890011,"close_bench":345.896149},{"date":"2022-09-27","open_a":146.962706,"high_a":148.500115,"low_a":144.342116,"close_a":145.014725,"volume_a":5400700,"open_b":149.812992,"high_b":151.755044,"low_b":147.076449,"close_b":148.851761,"volume_b":84442700,"open_bench":349.418612,"close_bench":345.013153},{"date":"2022-09-28","open_a":145.80088,"high_a":148.168138,"low_a":144.342086,"close_a":147.338287,"volume_a":5480900,"open_b":144.810729,"high_b":147.753239,"low_b":142.064383,"close_b":146.968567,"volume_b":146691400,"open_bench":345.962612,"close_bench":351.801758},{"date":"2022-09-29","open_a":147.451882,"high_a":147.748879,"low_a":144.132465,"close_a":145.538849,"volume_a":5167700,"open_b":143.300227,"high_b":143.90834,"low_b":137.984079,"close_b":139.749588,"volume_b":128138200,"open_bench":348.269804,"close_bench":344.453003},{"date":"2022-09-30","open_a":146.482264,"high_a":146.962708,"low_a":142.394136,"close_a":142.612518,"volume_a":5971900,"open_b":138.572615,"high_b":140.357745,"low_b":135.355471,"close_b":135.551636,"volume_b":124925300,"open_bench":343.513037,"close_bench":339.126556},{"date":"2022-10-03","open_a":142.839672,"high_a":144.735231,"low_a":142.402907,"close_a":144.350876,"volume_a":5093400,"open_b":135.561446,"high_b":140.328313,"low_b":135.051407,"close_b":139.720184,"volume_b":114311700,"open_bench":342.829437,"close_bench":348.079926},{"date":"2022-10-04","open_a":146.220204,"high_a":147.390731,"low_a":144.612912,"close_a":145.975616,"volume_a":4596800,"open_b":142.250745,"high_b":143.417943,"low_b":141.495496,"close_b":143.300247,"volume_b":87830100,"open_bench":353.577248,"close_bench":358.865723},{"date":"2022-10-05","open_a":145.853337,"high_a":145.958157,"low_a":143.023095,"close_a":145.006012,"volume_a":3538600,"open_b":141.309131,"high_b":144.555698,"low_b":140.269432,"close_b":143.594467,"volume_b":79471000,"open_bench":354.517153,"close_bench":358.030121},{"date":"2022-10-06","open_a":144.822524,"high_a":145.023446,"low_a":141.948616,"close_a":142.210678,"volume_a":4623400,"open_b":143.015804,"high_b":144.712648,"low_b":142.437114,"close_b":142.643082,"volume_b":68402200,"open_bench":356.634489,"close_bench":354.336823},{"date":"2022-10-07","open_a":141.773942,"high_a":142.39414,"low_a":140.620875,"close_a":141.171204,"volume_a":4779200,"open_b":139.808427,"high_b":140.357708,"low_b":136.777645,"close_b":137.40538,"volume_b":85925600,"open_bench":350.320632,"close_bench":344.453003},{"date":"2022-10-10","open_a":141.503159,"high_a":142.420355,"low_a":140.804332,"close_a":141.35466,"volume_a":4839200,"open_b":137.729095,"high_b":139.170927,"low_b":135.914556,"close_b":137.729095,"volume_b":74899000,"open_bench":345.563855,"close_bench":341.822998},{"date":"2022-10-11","open_a":141.721525,"high_a":143.49479,"low_a":141.127517,"close_a":142.027252,"volume_a":5222600,"open_b":137.219049,"high_b":138.641274,"low_b":135.57125,"close_b":136.316681,"volume_b":77033700,"open_bench":340.1329,"close_bench":339.658173},{"date":"2022-10-12","open_a":147.390722,"high_a":149.111577,"low_a":145.530112,"close_a":147.967255,"volume_a":9795300,"open_b":136.463834,"high_b":137.670259,"low_b":135.512421,"close_b":135.688965,"volume_b":70433700,"open_bench":340.06648,"close_bench":338.537842},{"date":"2022-10-13","open_a":145.966878,"high_a":152.902696,"low_a":145.722291,"close_a":152.527084,"volume_a":8412700,"open_b":132.403154,"high_b":140.83834,"low_b":131.795025,"close_b":140.249847,"volume_b":113224000,"open_bench":331.559375,"close_bench":347.47226},{"date":"2022-10-14","open_a":153.723825,"high_a":153.968413,"low_a":148.491389,"close_a":148.666092,"volume_a":5937900,"open_b":141.544534,"high_b":141.750516,"low_b":135.541819,"close_b":135.72818,"volume_b":88598000,"open_bench":349.921747,"close_bench":339.553711},{"date":"2022-10-17","open_a":150.483041,"high_a":151.636094,"low_a":149.434807,"close_a":150.884857,"volume_a":6401800,"open_b":138.366652,"high_b":140.16157,"low_b":137.581979,"close_b":139.680969,"volume_b":85250900,"open_bench":345.611329,"close_bench":348.279297},{"date":"2022-10-18","open_a":152.710527,"high_a":154.536202,"low_a":151.478847,"close_a":152.920166,"volume_a":4547400,"open_b":142.7019,"high_b":143.888704,"low_b":137.915413,"close_b":140.995239,"volume_b":99136600,"open_bench":356.169251,"close_bench":352.371429},{"date":"2022-10-19","open_a":152.710556,"high_a":153.601547,"low_a":150.535461,"close_a":151.435196,"volume_a":5200200,"open_b":138.974736,"high_b":142.172258,"low_b":138.788375,"close_b":141.103149,"volume_b":61758300,"open_bench":350.339553,"close_bench":349.874329},{"date":"2022-10-20","open_a":152.168931,"high_a":152.308699,"low_a":149.469726,"close_a":149.775467,"volume_a":4314000,"open_b":140.279277,"high_b":143.094273,"low_b":139.916357,"close_b":140.642181,"volume_b":64522000,"open_bench":349.428121,"close_bench":346.940552},{"date":"2022-10-21","open_a":149.548329,"high_a":151.723424,"low_a":149.006745,"close_a":151.173096,"volume_a":5071500,"open_b":140.132125,"high_b":145.016702,"low_b":139.91634,"close_b":144.447815,"volume_b":86548600,"open_bench":346.665184,"close_bench":355.371704},{"date":"2022-10-24","open_a":152.701774,"high_a":155.470863,"low_a":152.579481,"close_a":155.208801,"volume_a":5316500,"open_b":144.369344,"high_b":147.35108,"low_b":143.202146,"close_b":146.586029,"volume_b":75981900,"open_bench":356.890886,"close_bench":359.720245},{"date":"2022-10-25","open_a":155.59316,"high_a":156.248307,"low_a":154.274134,"close_a":155.724197,"volume_a":5220000,"open_b":147.213772,"high_b":149.567789,"low_b":146.497765,"close_b":149.420654,"volume_b":74732300,"open_bench":359.64422,"close_bench":365.464386},{"date":"2022-10-26","open_a":156.588979,"high_a":158.021568,"low_a":155.767872,"close_a":156.423019,"volume_a":5136000,"open_b":148.067078,"high_b":149.077338,"low_b":145.203022,"close_b":146.48793,"volume_b":88194300,"open_bench":362.331192,"close_bench":362.710968},{"date":"2022-10-27","open_a":157.558616,"high_a":158.047791,"low_a":155.986273,"close_a":156.257065,"volume_a":5424100,"open_b":145.232486,"high_b":146.193702,"low_b":141.367988,"close_b":142.025146,"volume_b":109180200,"open_bench":363.707954,"close_bench":360.774139},{"date":"2022-10-28","open_a":156.97335,"high_a":159.794849,"low_a":156.781172,"close_a":159.18338,"volume_a":6715300,"open_b":145.360021,"high_b":154.481806,"low_b":144.987313,"close_b":152.755539,"volume_b":164762400,"open_bench":360.669725,"close_bench":369.357239},{"date":"2022-10-31","open_a":158.266181,"high_a":159.838537,"low_a":157.707122,"close_a":158.615601,"volume_a":4877200,"open_b":150.224946,"high_b":151.284251,"low_b":149.008703,"close_b":150.401489,"volume_b":97943200,"open_bench":366.907594,"close_bench":366.689209},{"date":"2022-11-01","open_a":158.056534,"high_a":159.690031,"low_a":157.139325,"close_a":157.899292,"volume_a":4084500,"open_b":152.108161,"high_b":152.471066,"low_b":146.272185,"close_b":147.763046,"volume_b":80379300,"open_bench":370.42059,"close_bench":365.084625},{"date":"2022-11-02","open_a":157.916743,"high_a":160.36263,"low_a":155.636829,"close_a":155.697983,"volume_a":4970600,"open_b":146.09561,"high_b":149.253905,"low_b":142.221309,"close_b":142.250732,"volume_b":93604600,"open_bench":364.495976,"close_bench":355.922394},{"date":"2022-11-03","open_a":154.422628,"high_a":156.274507,"low_a":154.204245,"close_a":155.296158,"volume_a":3753800,"open_b":139.337635,"high_b":140.06346,"low_b":136.091069,"close_b":136.218582,"volume_b":97918500,"open_bench":352.694218,"close_bench":352.257477},{"date":"2022-11-04","open_a":155.418441,"high_a":157.051938,"low_a":154.204235,"close_a":156.169678,"volume_a":3922400,"open_b":139.598271,"high_b":140.168102,"low_b":132.023484,"close_b":135.953339,"volume_b":140814800,"open_bench":357.944685,"close_bench":357.327545},{"date":"2022-11-07","open_a":156.816116,"high_a":157.776991,"low_a":156.213378,"close_a":157.418854,"volume_a":3246100,"open_b":134.705614,"high_b":136.709834,"low_b":133.290864,"close_b":136.483871,"volume_b":83374600,"open_bench":358.618866,"close_bench":360.745667},{"date":"2022-11-08","open_a":157.619764,"high_a":159.078558,"low_a":156.187176,"close_a":157.855621,"volume_a":3724700,"open_b":137.947741,"high_b":138.949844,"low_b":135.078949,"close_b":137.053696,"volume_b":89908500,"open_bench":361.846957,"close_bench":362.691986},{"date":"2022-11-09","open_a":157.191717,"high_a":158.17008,"low_a":155.837743,"close_a":155.968781,"volume_a":4723600,"open_b":136.071231,"high_b":136.120357,"low_b":132.229794,"close_b":132.504883,"volume_b":74917800,"open_bench":360.726587,"close_bench":355.219757},{"date":"2022-11-10","open_a":159.550249,"high_a":159.786106,"low_a":155.977514,"close_a":158.720398,"volume_a":6704100,"open_b":138.763203,"high_b":144.294464,"low_b":137.05371,"close_b":144.294464,"volume_b":118854000,"open_bench":368.436202,"close_bench":374.740601},{"date":"2022-11-11","open_a":158.222502,"high_a":158.711677,"low_a":153.252114,"close_a":155.532028,"volume_a":7390500,"open_b":143.262865,"high_b":147.379375,"low_b":141.83828,"close_b":147.074814,"volume_b":93979700,"open_bench":375.59507,"close_bench":378.367493},{"date":"2022-11-14","open_a":156.073602,"high_a":156.519098,"low_a":154.178044,"close_a":154.378952,"volume_a":5633000,"open_b":146.35762,"high_b":147.644645,"low_b":144.844618,"close_b":145.679718,"volume_b":73374100,"open_bench":376.611034,"close_bench":375.148865},{"date":"2022-11-15","open_a":155.523285,"high_a":156.099818,"low_a":153.23464,"close_a":154.780792,"volume_a":5153600,"open_b":149.550638,"high_b":150.896609,"low_b":145.954817,"close_b":147.408859,"volume_b":89868300,"open_bench":380.874066,"close_bench":378.348511},{"date":"2022-11-16","open_a":155.750422,"high_a":156.912218,"low_a":155.243773,"close_a":155.776627,"volume_a":4701100,"open_b":146.514851,"high_b":147.241864,"low_b":144.707105,"close_b":146.180801,"volume_b":64218300,"open_bench":376.724921,"close_bench":375.462158},{"date":"2022-11-17","open_a":155.042852,"high_a":157.375175,"low_a":154.990429,"close_a":157.261612,"volume_a":3920300,"open_b":143.862157,"high_b":148.823602,"low_b":143.587068,"close_b":148.076935,"volume_b":80389400,"open_bench":370.724383,"close_bench":374.313324},{"date":"2022-11-18","open_a":157.418883,"high_a":158.842728,"low_a":157.25291,"close_a":158.397232,"volume_a":4498100,"open_b":149.639034,"high_b":150.022194,"low_b":147.340073,"close_b":148.636917,"volume_b":74829600,"open_bench":377.636378,"close_bench":376.012817},{"date":"2022-11-21","open_a":159.218312,"high_a":161.716608,"low_a":159.06107,"close_a":161.445816,"volume_a":4397600,"open_b":147.52675,"high_b":147.733059,"low_b":145.129536,"close_b":145.414444,"volume_b":58724100,"open_bench":374.69309,"close_bench":374.645599},{"date":"2022-11-22","open_a":162.310597,"high_a":163.018154,"low_a":161.052712,"close_a":161.463272,"volume_a":4109600,"open_b":145.532333,"high_b":147.782168,"low_b":144.353365,"close_b":147.546371,"volume_b":51804100,"open_bench":376.582547,"close_bench":379.687256},{"date":"2022-11-23","open_a":161.603074,"high_a":162.546489,"low_a":161.166309,"close_a":161.672958,"volume_a":2511000,"open_b":146.829221,"high_b":149.167489,"low_b":146.721149,"close_b":148.420822,"volume_b":58301400,"open_bench":379.354933,"close_bench":382.079895},{"date":"2022-11-25","open_a":161.236198,"high_a":161.603079,"low_a":160.677139,"close_a":160.825638,"volume_a":1837000,"open_b":145.709216,"high_b":146.269227,"low_b":144.540081,"close_b":145.512726,"volume_b":35195900,"open_bench":381.519657,"close_bench":381.994385},{"date":"2022-11-28","open_a":161.078964,"high_a":162.031122,"low_a":160.301523,"close_a":160.633469,"volume_a":6411400,"open_b":142.594767,"high_b":144.068463,"low_b":140.865637,"close_b":141.690903,"volume_b":69246000,"open_bench":378.918187,"close_bench":375.898926},{"date":"2022-11-29","open_a":160.118107,"high_a":160.222926,"low_a":158.126459,"close_a":159.497894,"volume_a":4375400,"open_b":141.759694,"high_b":142.27058,"low_b":137.8888,"close_b":138.694412,"volume_b":83763800,"open_bench":376.031797,"close_bench":375.253265},{"date":"2022-11-30","open_a":158.895139,"high_a":162.214557,"low_a":157.445089,"close_a":162.048584,"volume_a":7915600,"open_b":138.920402,"high_b":146.112046,"low_b":138.085317,"close_b":145.434143,"volume_b":111380900,"open_bench":375.50014,"close_bench":387.074005},{"date":"2022-12-01","open_a":163.033037,"high_a":163.780174,"low_a":161.872791,"close_a":163.402206,"volume_a":4683300,"open_b":145.610961,"high_b":146.514825,"low_b":144.039013,"close_b":145.709198,"volume_b":71250400,"open_bench":388.108913,"close_bench":386.789185},{"date":"2022-12-02","open_a":161.732129,"high_a":163.762568,"low_a":161.591489,"close_a":163.217606,"volume_a":3547700,"open_b":143.400393,"high_b":145.404612,"low_b":143.095816,"close_b":145.217941,"volume_b":65447400,"open_bench":381.91846,"close_bench":386.342926},{"date":"2022-12-05","open_a":161.23113,"high_a":161.679418,"low_a":160.255476,"close_a":160.958649,"volume_a":4083200,"open_b":145.178694,"high_b":148.273449,"low_b":143.213766,"close_b":144.058685,"volume_b":68826400,"open_bench":383.532647,"close_bench":379.393005},{"date":"2022-12-06","open_a":161.512387,"high_a":161.98703,"low_a":158.831499,"close_a":159.648956,"volume_a":5835100,"open_b":144.490937,"high_b":144.7169,"low_b":139.43124,"close_b":140.403885,"volume_b":64727200,"open_bench":379.231523,"close_bench":373.924042},{"date":"2022-12-07","open_a":159.807152,"high_a":160.369698,"low_a":158.8227,"close_a":160.13237,"volume_a":4241700,"open_b":139.696525,"high_b":140.855825,"low_b":137.544926,"close_b":138.468445,"volume_b":69721100,"open_bench":373.079052,"close_bench":373.287933},{"date":"2022-12-08","open_a":159.807183,"high_a":161.714566,"low_a":159.323742,"close_a":161.538773,"volume_a":3800800,"open_b":139.86356,"high_b":141.003222,"low_b":138.625661,"close_b":140.148468,"volume_b":62128300,"open_bench":375.167841,"close_bench":376.212219},{"date":"2022-12-09","open_a":161.916707,"high_a":161.916707,"low_a":160.897088,"close_a":160.94104,"volume_a":3116100,"open_b":139.84389,"high_b":143.017258,"low_b":138.429139,"close_b":139.667053,"volume_b":76097000,"open_bench":374.977924,"close_bench":373.401825},{"date":"2022-12-12","open_a":161.890343,"high_a":161.890343,"low_a":160.290608,"close_a":161.705765,"volume_a":4691400,"open_b":140.197558,"high_b":141.965996,"low_b":138.586319,"close_b":141.956177,"volume_b":70462700,"open_bench":374.189949,"close_bench":378.785339},{"date":"2022-12-13","open_a":163.911979,"high_a":164.228412,"low_a":160.8004,"close_a":161.617844,"volume_a":5634900,"open_b":146.878334,"high_b":147.340093,"low_b":141.71058,"close_b":142.919006,"volume_b":93886200,"open_bench":389.485657,"close_bench":381.652649},{"date":"2022-12-14","open_a":161.793682,"high_a":163.384632,"low_a":160.088461,"close_a":161.169601,"volume_a":5392500,"open_b":142.801095,"high_b":144.08812,"low_b":138.68457,"close_b":140.698624,"volume_b":82291200,"open_bench":381.310813,"close_bench":379.212524},{"date":"2022-12-15","open_a":159.95662,"high_a":160.598284,"low_a":156.801088,"close_a":158.435989,"volume_a":6127800,"open_b":138.635467,"high_b":139.313369,"low_b":133.644549,"close_b":134.106308,"volume_b":98931900,"open_bench":374.370312,"close_bench":369.936371},{"date":"2022-12-16","open_a":158.286531,"high_a":159.262198,"low_a":156.880159,"close_a":158.51506,"volume_a":15708100,"open_b":134.292963,"high_b":135.236119,"low_b":131.384864,"close_b":132.151184,"volume_b":160156900,"open_bench":367.390547,"close_bench":365.568756},{"date":"2022-12-19","open_a":158.972167,"high_a":161.266303,"low_a":158.523893,"close_a":159.394073,"volume_a":4447800,"open_b":132.740693,"high_b":132.829111,"low_b":129.017162,"close_b":130.048737,"volume_b":79592600,"open_bench":365.759639,"close_bench":362.468964},{"date":"2022-12-20","open_a":159.815986,"high_a":160.193941,"low_a":158.778784,"close_a":159.297379,"volume_a":3249000,"open_b":129.085935,"high_b":130.913318,"low_b":127.612239,"close_b":129.97998,"volume_b":77432800,"open_bench":361.715409,"close_bench":362.964905},{"date":"2022-12-21","open_a":159.578638,"high_a":161.450855,"low_a":159.446784,"close_a":160.413666,"volume_a":4236900,"open_b":130.648035,"high_b":134.410874,"low_b":130.422073,"close_b":133.074722,"volume_b":85928000,"open_bench":365.54974,"close_bench":368.39212},{"date":"2022-12-22","open_a":159.851103,"high_a":160.018112,"low_a":157.266903,"close_a":159.174286,"volume_a":4304700,"open_b":131.993996,"high_b":132.200305,"low_b":128.015016,"close_b":129.911163,"volume_b":77852100,"open_bench":365.358974,"close_bench":363.136597},{"date":"2022-12-23","open_a":159.016115,"high_a":160.45764,"low_a":158.611778,"close_a":160.202728,"volume_a":2197800,"open_b":128.624165,"high_b":130.097861,"low_b":127.366613,"close_b":129.547684,"volume_b":63814900,"open_bench":362.116077,"close_bench":365.225525},{"date":"2022-12-27","open_a":161.099281,"high_a":161.389346,"low_a":160.211517,"close_a":160.914703,"volume_a":3045000,"open_b":129.076088,"high_b":129.105561,"low_b":126.462731,"close_b":127.749756,"volume_b":69007800,"open_bench":365.111065,"close_bench":363.785248},{"date":"2022-12-28","open_a":161.820046,"high_a":162.206786,"low_a":159.657752,"close_a":159.75444,"volume_a":2694300,"open_b":127.396068,"high_b":128.732219,"low_b":123.66271,"close_b":123.829727,"volume_b":85438400,"open_bench":363.718431,"close_bench":359.26413},{"date":"2022-12-29","open_a":159.903867,"high_a":160.73011,"low_a":159.877499,"close_a":159.956604,"volume_a":2549200,"open_b":125.745534,"high_b":128.191866,"low_b":125.490099,"close_b":127.337128,"volume_b":75703700,"open_bench":362.096953,"close_bench":365.730988},{"date":"2022-12-30","open_a":159.429222,"high_a":159.939032,"low_a":157.592146,"close_a":158.796356,"volume_a":3136200,"open_b":126.15817,"high_b":127.671158,"low_b":125.195353,"close_b":127.651505,"volume_b":77034200,"open_bench":363.060239,"close_bench":364.767548},{"date":"2023-01-03","open_a":158.734805,"high_a":158.875445,"low_a":155.20132,"close_a":157.697617,"volume_a":5137400,"open_b":127.995383,"high_b":128.604505,"low_b":121.992528,"close_b":122.876747,"volume_b":112117500,"open_bench":366.618028,"close_bench":363.231995},{"date":"2023-01-04","open_a":157.697651,"high_a":158.690888,"low_a":156.062749,"close_a":157.310898,"volume_a":4771300,"open_b":124.664816,"high_b":126.403781,"low_b":122.886559,"close_b":124.144112,"volume_b":89113600,"open_bench":365.482995,"close_bench":366.036224},{"date":"2023-01-05","open_a":156.423098,"high_a":157.135069,"low_a":155.456216,"close_a":155.667175,"volume_a":3647100,"open_b":124.900613,"high_b":125.529389,"low_b":122.572179,"close_b":122.827614,"volume_b":80962700,"open_bench":364.090415,"close_bench":361.85849},{"date":"2023-01-06","open_a":157.170251,"high_a":159.824772,"low_a":157.161466,"close_a":159.183121,"volume_a":3713200,"open_b":123.800252,"high_b":128.005188,"low_b":122.69989,"close_b":127.346939,"volume_b":87754700,"open_bench":364.939276,"close_bench":370.156647},{"date":"2023-01-09","open_a":158.602941,"high_a":160.650963,"low_a":157.170201,"close_a":157.627274,"volume_a":3319200,"open_b":128.182033,"high_b":131.070479,"low_b":127.612202,"close_b":127.867638,"volume_b":70790800,"open_bench":372.340863,"close_bench":369.946777},{"date":"2023-01-10","open_a":157.152645,"high_a":157.803094,"low_a":155.658383,"close_a":156.326416,"volume_a":3817300,"open_b":127.975726,"high_b":128.958189,"low_b":125.873254,"close_b":128.437485,"volume_b":63896200,"open_bench":369.365006,"close_bench":372.541199},{"date":"2023-01-11","open_a":157.680026,"high_a":157.961305,"low_a":154.419008,"close_a":156.502197,"volume_a":4369900,"open_b":128.948384,"high_b":131.168747,"low_b":128.172245,"close_b":131.149109,"volume_b":69458900,"open_bench":374.11505,"close_bench":377.253082},{"date":"2023-01-12","open_a":156.159397,"high_a":156.783464,"low_a":154.102589,"close_a":154.506912,"volume_a":4246600,"open_b":131.532239,"high_b":131.905565,"low_b":129.135026,"close_b":131.07048,"volume_b":71379600,"open_bench":378.349971,"close_bench":378.626556},{"date":"2023-01-13","open_a":154.172899,"high_a":154.462963,"low_a":153.091758,"close_a":154.032272,"volume_a":4720400,"open_b":129.714683,"high_b":132.554003,"low_b":129.351176,"close_b":132.396805,"volume_b":57809700,"open_bench":375.440805,"close_bench":380.095428},{"date":"2023-01-17","open_a":154.832126,"high_a":156.062705,"low_a":154.25201,"close_a":154.753021,"volume_a":5032600,"open_b":132.465602,"high_b":134.882454,"low_b":131.77788,"close_b":133.556137,"volume_b":63646600,"open_bench":380.076308,"close_bench":379.399078},{"date":"2023-01-18","open_a":154.032285,"high_a":154.471775,"low_a":150.595474,"close_a":150.850372,"volume_a":5032600,"open_b":134.42068,"high_b":136.179283,"low_b":132.662062,"close_b":132.838913,"volume_b":69672800,"open_bench":380.581828,"close_bench":373.409119},{"date":"2023-01-19","open_a":150.551506,"high_a":151.333797,"low_a":148.969341,"close_a":149.101196,"volume_a":4720400,"open_b":131.728739,"high_b":133.860683,"low_b":131.424177,"close_b":132.897873,"volume_b":58280400,"open_bench":371.377546,"close_bench":370.690826},{"date":"2023-01-20","open_a":148.503466,"high_a":149.329708,"low_a":147.193796,"close_a":149.320923,"volume_a":5779100,"open_b":132.907678,"high_b":135.599634,"low_b":131.866269,"close_b":135.452255,"volume_b":80223600,"open_bench":372.083384,"close_bench":377.596436},{"date":"2023-01-23","open_a":149.43523,"high_a":150.094464,"low_a":148.301353,"close_a":148.652939,"volume_a":5440400,"open_b":135.697895,"high_b":140.806719,"low_b":135.481752,"close_b":138.635468,"volume_b":81760300,"open_bench":378.397653,"close_bench":382.127075},{"date":"2023-01-24","open_a":149.083614,"high_a":150.191123,"low_a":148.063996,"close_a":150.032913,"volume_a":4125600,"open_b":137.849508,"high_b":140.649536,"low_b":137.839688,"close_b":140.030579,"volume_b":66435100,"open_bench":380.457906,"close_bench":381.716949},{"date":"2023-01-25","open_a":149.725275,"high_a":151.1668,"low_a":148.846296,"close_a":151.122849,"volume_a":3852800,"open_b":138.419308,"high_b":139.932296,"low_b":136.375782,"close_b":139.372299,"volume_b":65799300,"open_bench":377.663234,"close_bench":381.860016},{"date":"2023-01-26","open_a":150.867976,"high_a":150.894344,"low_a":149.461604,"close_a":149.619827,"volume_a":4558400,"open_b":140.659331,"high_b":141.720394,"low_b":139.411598,"close_b":141.435486,"volume_b":54105100,"open_bench":384.511678,"close_bench":386.056854},{"date":"2023-01-27","open_a":149.224247,"high_a":149.62857,"low_a":148.17826,"close_a":149.092392,"volume_a":5451400,"open_b":140.649527,"high_b":144.648147,"low_b":140.570928,"close_b":143.370941,"volume_b":70555800,"open_bench":385.017121,"close_bench":386.943817},{"date":"2023-01-30","open_a":149.233041,"high_a":150.709733,"low_a":148.670495,"close_a":148.969345,"volume_a":5191500,"open_b":142.417976,"high_b":142.997626,"low_b":140.344976,"close_b":140.49234,"volume_b":64015300,"open_bench":384.196883,"close_bench":382.088959},{"date":"2023-01-31","open_a":149.602214,"high_a":150.366934,"low_a":148.828721,"close_a":150.322983,"volume_a":4665500,"open_b":140.197545,"high_b":141.808785,"low_b":139.784912,"close_b":141.759659,"volume_b":65874500,"open_bench":382.604021,"close_bench":387.70694},{"date":"2023-02-01","open_a":150.199939,"high_a":151.465671,"low_a":148.995728,"close_a":150.797638,"volume_a":4147700,"open_b":141.445296,"high_b":144.038999,"low_b":138.841773,"close_b":142.879684,"volume_b":77663600,"open_bench":386.495567,"close_bench":391.827393},{"date":"2023-02-02","open_a":148.951811,"high_a":149.91868,"low_a":148.11677,"close_a":149.400085,"volume_a":5653800,"open_b":146.288828,"high_b":148.528843,"low_b":145.571633,"close_b":148.175171,"volume_b":118339000,"open_bench":395.699942,"close_bench":397.531281},{"date":"2023-02-03","open_a":150.270271,"high_a":150.560322,"low_a":146.833459,"close_a":148.652939,"volume_a":5308200,"open_b":145.434137,"high_b":154.620181,"low_b":145.237647,"close_b":151.79068,"volume_b":154357300,"open_bench":392.580877,"close_bench":393.305786},{"date":"2023-02-06","open_a":149.197891,"high_a":151.518395,"low_a":149.066036,"close_a":151.026169,"volume_a":5640000,"open_b":149.894495,"high_b":150.415199,"low_b":148.135876,"close_b":149.069214,"volume_b":69858300,"open_bench":390.864029,"close_bench":390.902161},{"date":"2023-02-07","open_a":149.681331,"high_a":151.210759,"low_a":148.775984,"close_a":150.797638,"volume_a":6044700,"open_b":147.998359,"high_b":152.507865,"low_b":147.998359,"close_b":151.938034,"volume_b":83322600,"open_bench":389.986498,"close_bench":396.014618},{"date":"2023-02-08","open_a":150.30542,"high_a":150.595485,"low_a":149.637401,"close_a":150.44606,"volume_a":5687300,"open_b":151.181525,"high_b":151.869247,"low_b":148.519042,"close_b":149.25589,"volume_b":64120100,"open_bench":394.049799,"close_bench":391.684326},{"date":"2023-02-09","open_a":154.102609,"high_a":155.131013,"low_a":151.219546,"close_a":151.869995,"volume_a":9619400,"open_b":151.083246,"high_b":151.623604,"low_b":147.782168,"close_b":148.224274,"volume_b":56007100,"open_bench":395.270632,"close_bench":388.288696},{"date":"2023-02-10","open_a":152.168827,"high_a":155.218885,"low_a":151.755706,"close_a":154.876083,"volume_a":6176000,"open_b":147.063217,"high_b":148.913058,"low_b":146.82706,"close_b":148.588348,"volume_b":57450700,"open_bench":387.115515,"close_bench":389.194855},{"date":"2023-02-13","open_a":154.928852,"high_a":156.660442,"low_a":154.735475,"close_a":156.256119,"volume_a":5205100,"open_b":148.529309,"high_b":151.786226,"low_b":148.499791,"close_b":151.382812,"volume_b":62199000,"open_bench":389.843353,"close_bench":393.763519},{"date":"2023-02-14","open_a":155.667197,"high_a":155.799039,"low_a":153.636745,"close_a":154.498154,"volume_a":5203000,"open_b":149.680573,"high_b":151.304122,"low_b":148.440784,"close_b":150.743256,"volume_b":61707600,"open_bench":392.247093,"close_bench":393.582458},{"date":"2023-02-15","open_a":153.821312,"high_a":154.744243,"low_a":153.460927,"close_a":154.709076,"volume_a":4125500,"open_b":150.654695,"high_b":153.006368,"low_b":150.428388,"close_b":152.839096,"volume_b":65573800,"open_bench":391.39818,"close_bench":394.860535},{"date":"2023-02-16","open_a":153.592773,"high_a":155.491357,"low_a":153.416966,"close_a":154.269577,"volume_a":4973200,"open_b":151.048267,"high_b":153.823052,"low_b":150.890844,"close_b":151.245071,"volume_b":68167900,"open_bench":389.910192,"close_bench":389.423737},{"date":"2023-02-17","open_a":153.961944,"high_a":155.069452,"low_a":153.671879,"close_a":154.946396,"volume_a":3952100,"open_b":149.906909,"high_b":150.546479,"low_b":148.430963,"close_b":150.103699,"volume_b":59144100,"open_bench":387.306307,"close_bench":388.450897},{"date":"2023-02-21","open_a":154.331126,"high_a":155.939659,"low_a":153.531252,"close_a":154.858505,"volume_a":5797300,"open_b":147.791357,"high_b":148.873723,"low_b":146.030068,"close_b":146.098938,"volume_b":58867200,"open_bench":384.444882,"close_bench":380.658234},{"date":"2023-02-22","open_a":155.333194,"high_a":155.957262,"low_a":154.498167,"close_a":154.805801,"volume_a":5299500,"open_b":146.482682,"high_b":147.545365,"low_b":144.800112,"close_b":146.522049,"volume_b":51011300,"open_bench":381.068357,"close_bench":380.133636},{"date":"2023-02-23","open_a":155.280411,"high_a":155.570476,"low_a":153.812518,"close_a":154.972763,"volume_a":3801400,"open_b":147.683118,"high_b":147.929109,"low_b":144.878831,"close_b":147.004181,"volume_b":48394200,"open_bench":383.014068,"close_bench":382.15564},{"date":"2023-02-24","open_a":154.225656,"high_a":154.990364,"low_a":153.197239,"close_a":154.665146,"volume_a":2959100,"open_b":144.75091,"high_b":144.829629,"low_b":143.383201,"close_b":144.35733,"volume_b":55469600,"open_bench":377.157679,"close_bench":378.073334},{"date":"2023-02-27","open_a":155.552885,"high_a":155.798998,"low_a":154.392627,"close_a":154.62117,"volume_a":3208500,"open_b":145.341289,"high_b":146.777867,"low_b":145.085448,"close_b":145.547913,"volume_b":44998500,"open_bench":381.402142,"close_bench":379.360992},{"date":"2023-02-28","open_a":153.715832,"high_a":153.874042,"low_a":152.239141,"close_a":152.529205,"volume_a":4317100,"open_b":144.6919,"high_b":146.689345,"low_b":144.475426,"close_b":145.046127,"volume_b":50547000,"open_bench":378.884095,"close_bench":377.958893},{"date":"2023-03-01","open_a":151.465668,"high_a":151.588725,"low_a":148.652925,"close_a":150.595474,"volume_a":5359100,"open_b":144.475396,"high_b":144.868975,"low_b":142.684575,"close_b":142.979767,"volume_b":55479000,"open_bench":377.148163,"close_bench":376.509094},{"date":"2023-03-02","open_a":150.887482,"high_a":153.763468,"low_a":150.241495,"close_a":153.374115,"volume_a":4476300,"open_b":142.064681,"high_b":144.357319,"low_b":141.592368,"close_b":143.570145,"volume_b":52238100,"open_bench":374.544179,"close_bench":379.437256},{"date":"2023-03-03","open_a":153.090916,"high_a":153.489126,"low_a":151.613103,"close_a":153.223648,"volume_a":3627100,"open_b":145.665975,"high_b":148.686751,"low_b":144.96737,"close_b":148.608032,"volume_b":70732300,"open_bench":381.249601,"close_bench":385.522705},{"date":"2023-03-06","open_a":152.400707,"high_a":153.674992,"low_a":152.400707,"close_a":153.533401,"volume_a":3611400,"open_b":151.323777,"high_b":153.793536,"low_b":150.999082,"close_b":151.363144,"volume_b":87558000,"open_bench":386.342903,"close_bench":385.789703},{"date":"2023-03-07","open_a":153.683819,"high_a":154.205918,"low_a":152.090963,"close_a":152.772354,"volume_a":4094900,"open_b":151.23523,"high_b":151.55994,"low_b":148.706451,"close_b":149.168915,"volume_b":56182000,"open_bench":385.741994,"close_bench":379.876007},{"date":"2023-03-08","open_a":152.807756,"high_a":153.409508,"low_a":152.011335,"close_a":153.188278,"volume_a":2920200,"open_b":150.359498,"high_b":151.008918,"low_b":149.395218,"close_b":150.418533,"volume_b":47204800,"open_bench":379.990481,"close_bench":380.496002},{"date":"2023-03-09","open_a":153.639552,"high_a":154.294396,"low_a":151.49805,"close_a":152.029007,"volume_a":4042100,"open_b":151.097468,"high_b":152.061748,"low_b":147.820867,"close_b":148.175095,"volume_b":53833600,"open_bench":381.278249,"close_bench":373.476044},{"date":"2023-03-10","open_a":152.675049,"high_a":154.188266,"low_a":151.285734,"close_a":152.23259,"volume_a":4129900,"open_b":147.801207,"high_b":148.519496,"low_b":145.242895,"close_b":146.118622,"volume_b":68572400,"open_bench":372.932284,"close_bench":368.086914},{"date":"2023-03-13","open_a":152.878555,"high_a":156.834142,"low_a":152.745822,"close_a":153.719238,"volume_a":8265100,"open_b":145.43969,"high_b":150.684219,"low_b":145.331453,"close_b":148.057037,"volume_b":84457100,"open_bench":364.176223,"close_bench":367.562256},{"date":"2023-03-14","open_a":152.639627,"high_a":153.807713,"low_a":151.825503,"close_a":153.559937,"volume_a":6480800,"open_b":148.854031,"high_b":150.940029,"low_b":147.692961,"close_b":150.143021,"volume_b":73695900,"open_bench":372.464889,"close_bench":373.638092},{"date":"2023-03-15","open_a":153.559924,"high_a":156.329723,"low_a":153.259055,"close_a":156.303177,"volume_a":7099800,"open_b":148.765498,"high_b":150.792462,"low_b":147.51586,"close_b":150.536636,"volume_b":77167900,"open_bench":368.06785,"close_bench":371.30127},{"date":"2023-03-16","open_a":156.285488,"high_a":156.630606,"low_a":155.143949,"close_a":156.196991,"volume_a":4734400,"open_b":149.71997,"high_b":153.951018,"low_b":149.208304,"close_b":153.3508,"volume_b":76161100,"open_bench":368.954902,"close_bench":377.815826},{"date":"2023-03-17","open_a":156.197024,"high_a":156.356317,"low_a":154.17941,"close_a":154.975845,"volume_a":8854600,"open_b":153.577072,"high_b":154.226491,"low_b":151.805934,"close_b":152.514389,"volume_b":98944600,"open_bench":376.490694,"close_bench":373.398102},{"date":"2023-03-20","open_a":156.188152,"high_a":157.232349,"low_a":155.312077,"close_a":157.15271,"volume_a":4367000,"open_b":152.583293,"high_b":155.289194,"low_b":151.678033,"close_b":154.875916,"volume_b":73641400,"open_bench":374.173664,"close_bench":376.988586},{"date":"2023-03-21","open_a":157.21467,"high_a":157.665974,"low_a":155.993478,"close_a":157.524384,"volume_a":3812000,"open_b":154.797193,"high_b":156.843825,"low_b":154.029688,"close_b":156.725754,"volume_b":73938300,"open_bench":380.339665,"close_bench":381.938629},{"date":"2023-03-22","open_a":157.373934,"high_a":158.727859,"low_a":156.117352,"close_a":156.196991,"volume_a":4380800,"open_b":156.74547,"high_b":159.539924,"low_b":155.279358,"close_b":155.299042,"volume_b":75701800,"open_bench":381.766329,"close_bench":375.427948},{"date":"2023-03-23","open_a":155.798794,"high_a":156.825303,"low_a":154.878484,"close_a":155.435974,"volume_a":4655600,"open_b":156.28296,"high_b":158.959343,"low_b":155.151393,"close_b":156.381348,"volume_b":67622100,"open_bench":378.28117,"close_bench":376.442871},{"date":"2023-03-24","open_a":156.214667,"high_a":158.701286,"low_a":156.090779,"close_a":158.480057,"volume_a":3713800,"open_b":156.312497,"high_b":157.768759,"low_b":155.318699,"close_b":157.680206,"volume_b":59196500,"open_bench":375.169401,"close_bench":378.913055},{"date":"2023-03-27","open_a":158.91367,"high_a":159.860525,"low_a":158.674738,"close_a":158.83403,"volume_a":4078600,"open_b":157.375143,"high_b":158.191835,"low_b":155.338332,"close_b":155.74176,"volume_b":52390300,"open_bench":381.18227,"close_bench":379.621613},{"date":"2023-03-28","open_a":158.692492,"high_a":159.692454,"low_a":158.692492,"close_a":158.780975,"volume_a":2850000,"open_b":155.436763,"high_b":155.948428,"low_b":153.478669,"close_b":155.121887,"volume_b":45992200,"open_bench":378.932191,"close_bench":378.76944},{"date":"2023-03-29","open_a":159.860594,"high_a":160.090681,"low_a":159.276558,"close_a":159.878296,"volume_a":4496000,"open_b":156.814306,"high_b":158.467373,"low_b":156.794637,"close_b":158.191864,"volume_b":51305700,"open_bench":382.915241,"close_bench":384.274841},{"date":"2023-03-30","open_a":160.170271,"high_a":160.462296,"low_a":159.356147,"close_a":160.019836,"volume_a":2743000,"open_b":158.939686,"high_b":159.864615,"low_b":158.683861,"close_b":159.756378,"volume_b":49501700,"open_bench":386.898234,"close_bench":386.524841},{"date":"2023-03-31","open_a":161.196806,"high_a":161.488831,"low_a":160.453477,"close_a":161.320694,"volume_a":4479800,"open_b":159.835073,"high_b":162.354018,"low_b":159.313573,"close_b":162.255615,"volume_b":68749800,"open_bench":387.444033,"close_bench":391.972809},{"date":"2023-04-03","open_a":160.966724,"high_a":161.727755,"low_a":159.639346,"close_a":161.497681,"volume_a":3902300,"open_b":161.635754,"high_b":163.62335,"low_b":161.586553,"close_b":163.50528,"volume_b":56976200,"open_bench":391.455707,"close_bench":393.46637},{"date":"2023-04-04","open_a":161.214497,"high_a":162.258708,"low_a":160.258796,"close_a":160.922485,"volume_a":3306400,"open_b":163.928384,"high_b":164.164526,"low_b":162.462273,"close_b":162.973938,"volume_b":46278300,"open_bench":394.107863,"close_bench":391.283386},{"date":"2023-04-05","open_a":162.161367,"high_a":163.674584,"low_a":161.99323,"close_a":162.506485,"volume_a":3826300,"open_b":162.098191,"high_b":162.403218,"low_b":159.205335,"close_b":161.133896,"volume_b":51511700,"open_bench":390.555781,"close_bench":390.258972},{"date":"2023-04-06","open_a":162.205619,"high_a":163.479904,"low_a":162.028638,"close_a":163.143631,"volume_a":3389700,"open_b":159.825265,"high_b":162.314708,"low_b":159.402168,"close_b":162.019516,"volume_b":45390100,"open_bench":389.464225,"close_bench":391.781281},{"date":"2023-04-10","open_a":162.67463,"high_a":162.736567,"low_a":160.780904,"close_a":162.117126,"volume_a":2408100,"open_b":158.831422,"high_b":159.431641,"low_b":157.512914,"close_b":159.431641,"volume_b":47716900,"open_bench":389.311013,"close_bench":392.18338},{"date":"2023-04-11","open_a":162.311798,"high_a":162.718867,"low_a":161.727749,"close_a":161.869339,"volume_a":3586700,"open_b":159.746537,"high_b":159.756371,"low_b":157.936032,"close_b":158.22139,"volume_b":47644200,"open_bench":392.805761,"close_bench":392.288727},{"date":"2023-04-12","open_a":161.665825,"high_a":162.630383,"low_a":161.276459,"close_a":161.550781,"volume_a":3620900,"open_b":158.634643,"high_b":159.461169,"low_b":157.217732,"close_b":157.532608,"volume_b":50133100,"open_bench":394.347246,"close_bench":390.689758},{"date":"2023-04-13","open_a":161.60385,"high_a":163.364843,"low_a":161.152547,"close_a":163.161316,"volume_a":4303600,"open_b":159.038089,"high_b":163.141217,"low_b":158.83145,"close_b":162.90506,"volume_b":68445600,"open_bench":391.771658,"close_bench":395.87915},{"date":"2023-04-14","open_a":162.240999,"high_a":162.842751,"low_a":161.869335,"close_a":162.391434,"volume_a":2572900,"open_b":161.950601,"high_b":163.652869,"low_b":161.19296,"close_b":162.560669,"volume_b":49386500,"open_bench":395.247225,"close_bench":394.912109},{"date":"2023-04-17","open_a":162.993177,"high_a":163.48873,"low_a":162.364879,"close_a":163.223251,"volume_a":3224400,"open_b":162.442583,"high_b":162.737775,"low_b":161.399584,"close_b":162.580338,"volume_b":41516200,"open_bench":394.825981,"close_bench":396.329193},{"date":"2023-04-18","open_a":163.506432,"high_a":163.904642,"low_a":162.594966,"close_a":163.559525,"volume_a":2438600,"open_b":163.436414,"high_b":164.725405,"low_b":162.993618,"close_b":163.800476,"volume_b":49923000,"open_bench":397.899448,"close_bench":396.587738},{"date":"2023-04-19","open_a":163.922335,"high_a":163.99313,"low_a":163.055105,"close_a":163.462173,"volume_a":2209900,"open_b":163.141223,"high_b":165.463379,"low_b":162.885383,"close_b":164.941879,"volume_b":47720200,"open_bench":394.682394,"close_bench":396.520721},{"date":"2023-04-20","open_a":163.568353,"high_a":164.630253,"low_a":163.002006,"close_a":164.001968,"volume_a":3868200,"open_b":163.426552,"high_b":165.178007,"low_b":162.905053,"close_b":163.97757,"volume_b":52456400,"open_bench":393.715273,"close_bench":394.356781},{"date":"2023-04-21","open_a":164.586024,"high_a":164.931141,"low_a":163.824993,"close_a":164.072769,"volume_a":3781800,"open_b":162.403221,"high_b":163.780764,"low_b":161.852203,"close_b":162.373703,"volume_b":58337300,"open_bench":394.653655,"close_bench":394.663239},{"date":"2023-04-24","open_a":165.037339,"high_a":165.09929,"low_a":163.727663,"close_a":164.15242,"volume_a":3540200,"open_b":162.354032,"high_b":162.944416,"low_b":161.261831,"close_b":162.678741,"volume_b":41949600,"open_bench":394.462135,"close_bench":395.074921},{"date":"2023-04-25","open_a":165.090458,"high_a":168.29386,"low_a":164.807291,"close_a":167.87796,"volume_a":6231600,"open_b":162.540964,"high_b":163.642999,"low_b":161.104371,"close_b":161.143738,"volume_b":48714100,"open_bench":393.112139,"close_bench":388.803589},{"date":"2023-04-26","open_a":166.789452,"high_a":167.621277,"low_a":166.249651,"close_a":166.842545,"volume_a":4481700,"open_b":160.445124,"high_b":162.629525,"low_b":160.189299,"close_b":161.133896,"volume_b":45498800,"open_bench":389.416318,"close_bench":387.156708},{"date":"2023-04-27","open_a":166.524024,"high_a":168.090334,"low_a":165.93113,"close_a":167.86026,"volume_a":4574500,"open_b":162.541001,"high_b":165.856954,"low_b":162.541001,"close_b":165.709366,"volume_b":64902300,"open_bench":389.68445,"close_bench":394.864288},{"date":"2023-04-28","open_a":168.04606,"high_a":169.568121,"low_a":167.815986,"close_a":168.922134,"volume_a":4608900,"open_b":165.78808,"high_b":167.126271,"low_b":165.187861,"close_b":166.958984,"volume_b":55275900,"open_bench":393.983362,"close_bench":398.234467},{"date":"2023-05-01","open_a":168.532785,"high_a":170.196436,"low_a":168.38235,"close_a":169.621231,"volume_a":3137800,"open_b":166.565381,"high_b":167.716617,"low_b":165.935645,"close_b":166.870407,"volume_b":52472900,"open_bench":397.794091,"close_bench":397.832397},{"date":"2023-05-02","open_a":169.338048,"high_a":170.346855,"low_a":167.913329,"close_a":170.125626,"volume_a":3712200,"open_b":167.362397,"high_b":167.618237,"low_b":164.853286,"close_b":165.83725,"volume_b":48425700,"open_bench":397.123878,"close_bench":393.361084},{"date":"2023-05-03","open_a":170.417666,"high_a":171.01056,"low_a":169.709728,"close_a":170.06369,"volume_a":3389600,"open_b":166.781853,"high_b":168.17908,"low_b":164.479382,"close_b":164.764725,"volume_b":65136000,"open_bench":393.85894,"close_bench":390.661041},{"date":"2023-05-04","open_a":170.860117,"high_a":171.629992,"low_a":170.045993,"close_a":171.125595,"volume_a":4267600,"open_b":162.245801,"high_b":164.361318,"low_b":161.675101,"close_b":163.131363,"volume_b":81235400,"open_bench":389.617452,"close_bench":387.894043},{"date":"2023-05-05","open_a":170.630033,"high_a":172.098988,"low_a":170.249511,"close_a":171.913162,"volume_a":4146500,"open_b":168.238128,"high_b":171.504895,"low_b":168.021655,"close_b":170.786606,"volume_b":113453200,"open_bench":391.513185,"close_bench":395.074921},{"date":"2023-05-08","open_a":171.143265,"high_a":171.674221,"low_a":170.479576,"close_a":171.09903,"volume_a":2249900,"open_b":169.714096,"high_b":171.062136,"low_b":169.350034,"close_b":170.717743,"volume_b":55962800,"open_bench":395.400462,"close_bench":395.180237},{"date":"2023-05-09","open_a":171.550335,"high_a":172.267117,"low_a":170.532684,"close_a":171.798111,"volume_a":3087600,"open_b":170.27493,"high_b":170.757062,"low_b":168.848185,"close_b":169.015457,"volume_b":45326900,"open_bench":393.638738,"close_bench":393.447235},{"date":"2023-05-10","open_a":171.771572,"high_a":173.072404,"low_a":170.497287,"close_a":171.913162,"volume_a":4352900,"open_b":170.245445,"high_b":171.239243,"low_b":169.143395,"close_b":170.776779,"volume_b":53724500,"open_bench":396.271762,"close_bench":395.285583},{"date":"2023-05-11","open_a":171.922004,"high_a":173.019308,"low_a":170.789309,"close_a":172.860016,"volume_a":3690200,"open_b":171.062086,"high_b":171.79021,"low_b":169.40902,"close_b":170.963684,"volume_b":49514700,"open_bench":394.423887,"close_bench":394.596222},{"date":"2023-05-12","open_a":172.860012,"high_a":173.638744,"low_a":172.514894,"close_a":173.550247,"volume_a":2741000,"open_b":171.0721,"high_b":171.505646,"low_b":168.490554,"close_b":170.037521,"volume_b":45533100,"open_bench":395.831415,"close_bench":394.079254},{"date":"2023-05-15","open_a":173.948478,"high_a":174.222801,"low_a":171.67423,"close_a":171.913162,"volume_a":2946400,"open_b":170.618846,"high_b":170.668115,"low_b":168.953645,"close_b":169.544846,"volume_b":37266700,"open_bench":394.682353,"close_bench":395.438751},{"date":"2023-05-16","open_a":172.099013,"high_a":172.099013,"low_a":170.382268,"close_a":171.169846,"volume_a":5803100,"open_b":169.466018,"high_b":170.599135,"low_b":169.278804,"close_b":169.544846,"volume_b":42110300,"open_bench":394.337663,"close_bench":392.796173},{"date":"2023-05-17","open_a":171.063637,"high_a":171.311414,"low_a":168.877887,"close_a":169.957489,"volume_a":4878800,"open_b":169.190132,"high_b":170.392215,"low_b":167.919055,"close_b":170.155746,"volume_b":57951600,"open_bench":394.806854,"close_bench":397.564331},{"date":"2023-05-18","open_a":169.302673,"high_a":169.647791,"low_a":168.497394,"close_a":169.515045,"volume_a":3709400,"open_b":170.461176,"high_b":172.668308,"low_b":170.047341,"close_b":172.481094,"volume_b":65496700,"open_bench":397.248305,"close_bench":401.394104},{"date":"2023-05-19","open_a":170.311446,"high_a":170.665408,"low_a":169.107956,"close_a":169.762787,"volume_a":3883000,"open_b":173.801423,"high_b":173.801423,"low_b":172.372705,"close_b":172.589478,"volume_b":55809500,"open_bench":402.294163,"close_bench":400.810089},{"date":"2023-05-22","open_a":169.187576,"high_a":169.364557,"low_a":164.957666,"close_a":165.161194,"volume_a":5787000,"open_b":171.426813,"high_b":172.14611,"low_b":170.904592,"close_b":171.643585,"volume_b":43570900,"open_bench":400.829257,"close_bench":400.97287},{"date":"2023-05-23","open_a":165.028509,"high_a":165.152397,"low_a":163.347156,"close_a":164.656845,"volume_a":6382300,"open_b":170.58928,"high_b":170.835611,"low_b":168.766423,"close_b":169.042313,"volume_b":50747300,"open_bench":399.335592,"close_bench":396.472809},{"date":"2023-05-24","open_a":164.833802,"high_a":165.249715,"low_a":163.152448,"close_a":163.61261,"volume_a":3721000,"open_b":168.579244,"high_b":169.889728,"low_b":168.017616,"close_b":169.318237,"volume_b":45143500,"open_bench":394.873866,"close_bench":393.600433},{"date":"2023-05-25","open_a":161.67464,"high_a":163.01972,"low_a":161.010951,"close_a":162.648056,"volume_a":4712700,"open_b":169.87985,"high_b":171.347974,"low_b":169.170414,"close_b":170.45134,"volume_b":56058300,"open_bench":397.09514,"close_bench":397.008972},{"date":"2023-05-26","open_a":162.568398,"high_a":163.046248,"low_a":161.409143,"close_a":162.453354,"volume_a":4449300,"open_b":170.776539,"high_b":173.190582,"low_b":170.569614,"close_b":172.85556,"volume_b":54835000,"open_bench":397.660043,"close_bench":402.150513},{"date":"2023-05-30","open_a":161.10834,"high_a":162.019805,"low_a":159.904849,"close_a":160.710129,"volume_a":4331800,"open_b":174.363083,"high_b":176.363291,"low_b":173.978807,"close_b":174.69809,"volume_b":55964400,"open_bench":404.074979,"close_bench":402.30368},{"date":"2023-05-31","open_a":161.816218,"high_a":163.373669,"low_a":160.577337,"close_a":161.364914,"volume_a":20414600,"open_b":174.727647,"high_b":176.718007,"low_b":174.166004,"close_b":174.648819,"volume_b":99625300,"open_bench":400.484575,"close_bench":400.072876},{"date":"2023-06-01","open_a":162.554574,"high_a":162.795175,"low_a":161.110995,"close_a":162.349625,"volume_a":5277400,"open_b":175.092203,"high_b":177.476687,"low_b":174.333499,"close_b":177.447128,"volume_b":68901800,"open_bench":400.302551,"close_bench":403.873871},{"date":"2023-06-02","open_a":162.198107,"high_a":164.220895,"low_a":161.912948,"close_a":164.015945,"volume_a":4288300,"open_b":178.373322,"high_b":179.112316,"low_b":176.629294,"close_b":178.294495,"volume_b":61996900,"open_bench":406.439996,"close_bench":409.714508},{"date":"2023-06-05","open_a":163.828819,"high_a":165.896165,"low_a":163.61495,"close_a":164.13179,"volume_a":4311700,"open_b":179.949869,"high_b":182.235815,"low_b":175.427217,"close_b":176.944626,"volume_b":121946500,"open_bench":410.059081,"close_bench":408.929291},{"date":"2023-06-06","open_a":164.835755,"high_a":164.960508,"low_a":160.781245,"close_a":161.788193,"volume_a":3975600,"open_b":177.328889,"high_b":177.476682,"low_b":174.826156,"close_b":176.580048,"volume_b":64848400,"open_bench":408.517607,"close_bench":409.819733},{"date":"2023-06-07","open_a":161.030775,"high_a":161.948607,"low_a":160.130755,"close_a":160.496109,"volume_a":5162700,"open_b":175.821361,"high_b":178.550715,"low_b":174.717802,"close_b":175.210464,"volume_b":61944600,"open_bench":410.212315,"close_bench":408.40271},{"date":"2023-06-08","open_a":160.13076,"high_a":162.447627,"low_a":159.801057,"close_a":162.269409,"volume_a":4036100,"open_b":175.28926,"high_b":178.186117,"low_b":174.855729,"close_b":177.92009,"volume_b":50214900,"open_bench":408.469763,"close_bench":410.872986},{"date":"2023-06-09","open_a":161.921853,"high_a":162.919881,"low_a":161.289166,"close_a":162.492157,"volume_a":3200100,"open_b":178.836482,"high_b":179.555765,"low_b":177.979254,"close_b":178.304413,"volume_b":48900000,"open_bench":411.667613,"close_bench":411.610168},{"date":"2023-06-12","open_a":162.616946,"high_a":162.830802,"low_a":161.012962,"close_a":162.091187,"volume_a":4207100,"open_b":178.609842,"high_b":181.191388,"low_b":178.314241,"close_b":181.09285,"volume_b":54274900,"open_bench":412.586791,"close_bench":415.344238},{"date":"2023-06-13","open_a":162.29613,"high_a":162.349594,"low_a":161.360472,"close_a":161.77037,"volume_a":2838700,"open_b":180.117378,"high_b":181.447558,"low_b":179.762661,"close_b":180.619888,"volume_b":54929100,"open_bench":416.799624,"close_bench":418.082611},{"date":"2023-06-14","open_a":162.180284,"high_a":163.650594,"low_a":161.68127,"close_a":163.22287,"volume_a":3559300,"open_b":180.679029,"high_b":181.684064,"low_b":179.348849,"close_b":181.250519,"volume_b":57462900,"open_bench":418.417693,"close_bench":418.580444},{"date":"2023-06-15","open_a":164.194161,"high_a":165.896166,"low_a":163.534755,"close_a":165.486267,"volume_a":4326100,"open_b":181.260385,"high_b":183.782814,"low_b":181.083018,"close_b":183.280289,"volume_b":65433200,"open_bench":417.766662,"close_bench":423.769928},{"date":"2023-06-16","open_a":165.869435,"high_a":167.161528,"low_a":165.477348,"close_a":165.780319,"volume_a":8925700,"open_b":183.989735,"high_b":184.245929,"low_b":181.565844,"close_b":182.206299,"volume_b":101256200,"open_bench":425.747679,"close_bench":422.326477},{"date":"2023-06-20","open_a":166.225856,"high_a":167.411021,"low_a":165.067437,"close_a":165.129807,"volume_a":4154800,"open_b":181.703766,"high_b":183.368968,"low_b":181.703766,"close_b":182.294952,"volume_b":49799100,"open_bench":420.394838,"close_bench":420.135345},{"date":"2023-06-21","open_a":165.495182,"high_a":166.528862,"low_a":164.131799,"close_a":165.236755,"volume_a":3832400,"open_b":182.186538,"high_b":182.689063,"low_b":179.91044,"close_b":181.260345,"volume_b":49515700,"open_bench":419.155041,"close_bench":417.982605},{"date":"2023-06-22","open_a":166.395186,"high_a":167.152627,"low_a":165.628839,"close_a":166.947678,"volume_a":3185800,"open_b":181.043584,"high_b":184.305007,"low_b":180.974604,"close_b":184.255737,"volume_b":51245300,"open_bench":417.031266,"close_bench":419.491455},{"date":"2023-06-23","open_a":167.526885,"high_a":167.526885,"low_a":165.700127,"close_a":165.807068,"volume_a":7592600,"open_b":182.827023,"high_b":184.80752,"low_b":182.294939,"close_b":183.94043,"volume_b":53117000,"open_bench":416.051017,"close_bench":416.320099},{"date":"2023-06-26","open_a":165.887256,"high_a":166.127843,"low_a":164.069403,"close_a":164.755554,"volume_a":3706100,"open_b":184.088252,"high_b":185.290349,"low_b":182.511726,"close_b":182.551147,"volume_b":48088700,"open_bench":415.753097,"close_bench":414.61911},{"date":"2023-06-27","open_a":164.559526,"high_a":166.288264,"low_a":163.926853,"close_a":165.940735,"volume_a":5763000,"open_b":183.162003,"high_b":185.625315,"low_b":182.94523,"close_b":185.300156,"volume_b":50730800,"open_bench":415.493599,"close_bench":419.164673},{"date":"2023-06-28","open_a":165.967469,"high_a":166.404113,"low_a":163.410015,"close_a":163.69516,"volume_a":5193900,"open_b":185.172051,"high_b":187.113142,"low_b":184.846907,"close_b":186.472687,"volume_b":51216800,"open_bench":418.088317,"close_bench":419.376099},{"date":"2023-06-29","open_a":163.294163,"high_a":163.864466,"low_a":161.636715,"close_a":163.85556,"volume_a":5770400,"open_b":186.305214,"high_b":187.280691,"low_b":186.167269,"close_b":186.807724,"volume_b":46347300,"open_bench":418.962913,"close_bench":421.029083},{"date":"2023-06-30","open_a":163.998116,"high_a":165.316954,"low_a":163.85555,"close_a":165.049622,"volume_a":7635100,"open_b":188.817817,"high_b":191.625984,"low_b":188.453237,"close_b":191.123474,"volume_b":85213200,"open_bench":424.229239,"close_bench":425.997498},{"date":"2023-07-03","open_a":164.229827,"high_a":165.423898,"low_a":163.490211,"close_a":165.38826,"volume_a":2389000,"open_b":190.936256,"high_b":191.034795,"low_b":188.945896,"close_b":189.635635,"volume_b":31458200,"open_bench":425.651595,"close_bench":426.487671},{"date":"2023-07-05","open_a":164.853595,"high_a":166.359557,"low_a":164.826863,"close_a":166.261536,"volume_a":5277100,"open_b":188.758685,"high_b":190.147981,"low_b":187.822614,"close_b":188.522202,"volume_b":46920300,"open_bench":424.680896,"close_bench":425.853333},{"date":"2023-07-06","open_a":165.575396,"high_a":166.413033,"low_a":164.755585,"close_a":166.279373,"volume_a":5592100,"open_b":187.054057,"high_b":189.202073,"low_b":186.423449,"close_b":188.995148,"volume_b":45094300,"open_bench":422.287982,"close_bench":422.518616},{"date":"2023-07-07","open_a":165.58429,"high_a":165.637754,"low_a":163.098112,"close_a":163.14267,"volume_a":6309900,"open_b":188.601012,"high_b":189.842516,"low_b":187.448184,"close_b":187.881714,"volume_b":46815000,"open_bench":421.528802,"close_bench":421.451904},{"date":"2023-07-10","open_a":164.024863,"high_a":165.664485,"low_a":163.766437,"close_a":164.229813,"volume_a":5508800,"open_b":186.482588,"high_b":187.201886,"low_b":184.295165,"close_b":185.842133,"volume_b":59922200,"open_bench":421.096307,"close_bench":422.518616},{"date":"2023-07-11","open_a":164.301074,"high_a":164.612951,"low_a":162.554512,"close_a":163.944626,"volume_a":3914300,"open_b":186.384053,"high_b":186.521998,"low_b":183.861624,"close_b":185.319901,"volume_b":46638100,"open_bench":423.277797,"close_bench":425.209412},{"date":"2023-07-12","open_a":163.944659,"high_a":164.506057,"low_a":163.053559,"close_a":163.22287,"volume_a":6080100,"open_b":186.896402,"high_b":188.886762,"low_b":185.704167,"close_b":186.985092,"volume_b":60750200,"open_bench":428.986275,"close_bench":428.630676},{"date":"2023-07-13","open_a":165.566478,"high_a":167.58036,"low_a":163.303075,"close_a":167.108078,"volume_a":8223100,"open_b":187.704399,"high_b":188.384275,"low_b":186.994963,"close_b":187.743805,"volume_b":41342300,"open_bench":430.4374,"close_bench":432.032684},{"date":"2023-07-14","open_a":167.40216,"high_a":168.088311,"low_a":166.716008,"close_a":167.71405,"volume_a":4467600,"open_b":187.438346,"high_b":188.374401,"low_b":186.84716,"close_b":187.891602,"volume_b":41616200,"open_bench":432.916776,"close_bench":431.76355},{"date":"2023-07-17","open_a":166.047682,"high_a":166.644718,"low_a":165.227871,"close_a":165.415009,"volume_a":5637400,"open_b":189.083818,"high_b":191.468317,"low_b":188.995142,"close_b":191.143158,"volume_b":50520200,"open_bench":431.619434,"close_bench":433.262756},{"date":"2023-07-18","open_a":165.245674,"high_a":166.475397,"low_a":163.802096,"close_a":164.36348,"volume_a":5684100,"open_b":190.51258,"high_b":191.478194,"low_b":189.59622,"close_b":190.886993,"volume_b":48288200,"open_bench":432.93607,"close_bench":436.482208},{"date":"2023-07-19","open_a":164.853598,"high_a":166.279364,"low_a":164.336758,"close_a":165.976379,"volume_a":4140800,"open_b":190.266228,"high_b":195.320934,"low_b":189.82282,"close_b":192.236877,"volume_b":80507300,"open_bench":437.270256,"close_bench":437.45285},{"date":"2023-07-20","open_a":167.081334,"high_a":169.300179,"low_a":166.466473,"close_a":168.685318,"volume_a":5056500,"open_b":192.227004,"high_b":193.586757,"low_b":189.675017,"close_b":190.295776,"volume_b":59581200,"open_bench":436.46291,"close_bench":434.550476},{"date":"2023-07-21","open_a":169.407102,"high_a":170.619,"low_a":168.685313,"close_a":169.45166,"volume_a":15022100,"open_b":191.251566,"high_b":192.108794,"low_b":188.423673,"close_b":189.12326,"volume_b":71951700,"open_bench":436.261077,"close_bench":434.550476},{"date":"2023-07-24","open_a":169.834841,"high_a":170.583363,"low_a":169.433849,"close_a":170.128906,"volume_a":4225800,"open_b":190.571674,"high_b":192.049661,"low_b":189.428694,"close_b":189.921356,"volume_b":45377800,"open_bench":435.694134,"close_bench":436.491791},{"date":"2023-07-25","open_a":169.674453,"high_a":171.100206,"low_a":169.362563,"close_a":170.520996,"volume_a":3521600,"open_b":190.492857,"high_b":191.586569,"low_b":190.088871,"close_b":190.778595,"volume_b":37283200,"open_bench":436.222683,"close_bench":437.683411},{"date":"2023-07-26","open_a":170.307125,"high_a":171.429921,"low_a":169.362561,"close_a":170.734863,"volume_a":3898900,"open_b":190.827869,"high_b":192.768961,"low_b":190.483015,"close_b":191.645691,"volume_b":47471900,"open_bench":436.75121,"close_bench":437.750671},{"date":"2023-07-27","open_a":171.091265,"high_a":171.349678,"low_a":167.883298,"close_a":167.999146,"volume_a":4141800,"open_b":193.143373,"high_b":194.306049,"low_b":189.724295,"close_b":190.38446,"volume_b":47460200,"open_bench":441.123766,"close_bench":434.848358},{"date":"2023-07-28","open_a":169.433826,"high_a":170.529876,"low_a":169.121936,"close_a":169.585312,"volume_a":5602300,"open_b":191.813181,"high_b":193.744424,"low_b":191.29096,"close_b":192.956161,"volume_b":48291400,"open_bench":438.106282,"close_bench":439.105743},{"date":"2023-07-31","open_a":168.818981,"high_a":168.943735,"low_a":166.502128,"close_a":167.0457,"volume_a":5687700,"open_b":193.182771,"high_b":193.606469,"low_b":192.394508,"close_b":193.567047,"volume_b":38824100,"open_bench":439.576644,"close_bench":439.941833},{"date":"2023-08-01","open_a":167.072454,"high_a":168.061576,"low_a":166.101144,"close_a":166.475418,"volume_a":2853900,"open_b":193.360139,"high_b":193.842939,"low_b":192.414221,"close_b":192.73938,"volume_b":35175100,"open_bench":438.481119,"close_bench":438.682953},{"date":"2023-08-02","open_a":166.270437,"high_a":169.282361,"low_a":166.101126,"close_a":167.803131,"volume_a":4285900,"open_b":192.17772,"high_b":192.315665,"low_b":189.034547,"close_b":189.75383,"volume_b":50389300,"open_bench":435.578767,"close_bench":432.580414},{"date":"2023-08-03","open_a":167.67838,"high_a":168.614038,"low_a":166.609062,"close_a":166.742722,"volume_a":2857400,"open_b":188.758671,"high_b":189.546919,"low_b":187.891581,"close_b":188.364532,"volume_b":61235200,"open_bench":430.5719,"close_bench":431.340698},{"date":"2023-08-04","open_a":166.742725,"high_a":167.535804,"low_a":164.274387,"close_a":164.497162,"volume_a":4651100,"open_b":182.797455,"high_b":184.63016,"low_b":179.25028,"close_b":179.31926,"volume_b":115956800,"open_bench":433.147423,"close_bench":429.389862},{"date":"2023-08-07","open_a":165.103108,"high_a":166.822939,"low_a":164.942717,"close_a":166.537781,"volume_a":3323400,"open_b":179.457206,"high_b":180.442531,"low_b":174.747355,"close_b":176.225342,"volume_b":97576100,"open_bench":431.215823,"close_bench":433.137848},{"date":"2023-08-08","open_a":166.814001,"high_a":167.063508,"low_a":163.989214,"close_a":164.354568,"volume_a":5017900,"open_b":177.053036,"high_b":177.624527,"low_b":174.974,"close_b":177.161423,"volume_b":67823000,"open_bench":430.61032,"close_bench":431.254211},{"date":"2023-08-09","open_a":163.855524,"high_a":164.853552,"low_a":162.741649,"close_a":163.944626,"volume_a":4413100,"open_b":178.215675,"high_b":178.274792,"low_b":174.412321,"close_b":175.575012,"volume_b":60378500,"open_bench":431.523273,"close_bench":428.371155},{"date":"2023-08-10","open_a":164.158519,"high_a":165.379322,"low_a":163.543657,"close_a":163.7397,"volume_a":3727200,"open_b":176.846081,"high_b":178.097447,"low_b":174.993681,"close_b":175.358246,"volume_b":54686900,"open_bench":430.716039,"close_bench":428.524933},{"date":"2023-08-11","open_a":163.962485,"high_a":164.211992,"low_a":163.320892,"close_a":163.998123,"volume_a":2873800,"open_b":174.953766,"high_b":176.236406,"low_b":174.194037,"close_b":175.41748,"volume_b":52036700,"open_bench":426.660562,"close_bench":428.275055},{"date":"2023-08-14","open_a":164.310037,"high_a":164.728856,"low_a":163.187256,"close_a":163.623886,"volume_a":3385600,"open_b":175.595033,"high_b":177.292082,"low_b":174.943837,"close_b":177.065155,"volume_b":43675600,"open_bench":427.362195,"close_bench":430.639221},{"date":"2023-08-15","open_a":163.20506,"high_a":163.481312,"low_a":161.405046,"close_a":161.601089,"volume_a":3331800,"open_b":176.492926,"high_b":177.08491,"low_b":174.687345,"close_b":175.082001,"volume_b":43622600,"open_bench":428.870908,"close_bench":425.622711},{"date":"2023-08-16","open_a":161.823862,"high_a":163.02684,"low_a":161.244639,"close_a":161.476334,"volume_a":2633700,"open_b":174.766254,"high_b":176.157427,"low_b":174.144657,"close_b":174.21373,"volume_b":46964900,"open_bench":425.209452,"close_bench":422.49942},{"date":"2023-08-17","open_a":161.476338,"high_a":161.654555,"low_a":158.901058,"close_a":159.016891,"volume_a":4503000,"open_b":174.776184,"high_b":175.141241,"low_b":171.16502,"close_b":171.678085,"volume_b":66062900,"open_bench":423.960185,"close_bench":419.28006},{"date":"2023-08-18","open_a":159.025781,"high_a":159.997077,"low_a":157.965369,"close_a":158.77626,"volume_a":6020300,"open_b":170.000752,"high_b":172.76339,"low_b":169.665292,"close_b":172.16153,"volume_b":61172200,"open_bench":416.473839,"close_bench":419.481812},{"date":"2023-08-21","open_a":158.375305,"high_a":158.580268,"low_a":157.038654,"close_a":157.769363,"volume_a":4284100,"open_b":172.733755,"high_b":173.779608,"low_b":171.421502,"close_b":173.493469,"volume_b":46311900,"open_bench":420.4909,"close_bench":422.211121},{"date":"2023-08-22","open_a":156.994085,"high_a":157.564402,"low_a":156.218832,"close_a":156.56636,"volume_a":3489300,"open_b":174.697184,"high_b":175.308906,"low_b":173.897996,"close_b":174.864914,"volume_b":42038900,"open_bench":423.97934,"close_bench":421.067474},{"date":"2023-08-23","open_a":157.947556,"high_a":158.865388,"low_a":157.163383,"close_a":158.722809,"volume_a":4042700,"open_b":176.137723,"high_b":179.127288,"low_b":175.950256,"close_b":178.703018,"volume_b":52722800,"open_bench":422.124638,"close_bench":425.757263},{"date":"2023-08-24","open_a":158.624793,"high_a":160.790174,"low_a":157.813902,"close_a":157.885178,"volume_a":5709800,"open_b":178.259036,"high_b":178.683306,"low_b":173.661218,"close_b":174.026291,"volume_b":54945800,"open_bench":427.352512,"close_bench":419.856628},{"date":"2023-08-25","open_a":158.669359,"high_a":160.139683,"low_a":158.473316,"close_a":159.881256,"volume_a":4710700,"open_b":175.012935,"high_b":176.759304,"low_b":173.473755,"close_b":176.226517,"volume_b":51449600,"open_bench":421.576784,"close_bench":422.816498},{"date":"2023-08-28","open_a":161.128829,"high_a":161.449612,"low_a":159.809991,"close_a":160.620895,"volume_a":3189800,"open_b":177.68679,"high_b":178.180118,"low_b":176.167347,"close_b":177.785461,"volume_b":43820700,"open_bench":424.998058,"close_bench":425.497803},{"date":"2023-08-29","open_a":160.665423,"high_a":161.485233,"low_a":159.524815,"close_a":161.226807,"volume_a":4368600,"open_b":177.301934,"high_b":182.432538,"low_b":177.104606,"close_b":181.662949,"volume_b":53003900,"open_bench":425.392055,"close_bench":431.648254},{"date":"2023-08-30","open_a":161.699096,"high_a":162.411966,"low_a":161.039677,"close_a":161.360474,"volume_a":3153400,"open_b":182.472031,"high_b":185.343202,"low_b":182.274703,"close_b":185.145859,"volume_b":60813900,"open_bench":431.984599,"close_bench":433.426117},{"date":"2023-08-31","open_a":160.934221,"high_a":161.194466,"low_a":159.606118,"close_a":159.659958,"volume_a":5566700,"open_b":185.333378,"high_b":186.596296,"low_b":184.978181,"close_b":185.362976,"volume_b":60794500,"open_bench":434.041144,"close_bench":432.79184},{"date":"2023-09-01","open_a":160.386832,"high_a":160.458623,"low_a":156.716594,"close_a":157.326813,"volume_a":4801400,"open_b":186.961361,"high_b":187.385616,"low_b":185.767501,"close_b":186.931763,"volume_b":45766500,"open_bench":435.501875,"close_bench":433.59906},{"date":"2023-09-05","open_a":157.281923,"high_a":157.829319,"low_a":155.828176,"close_a":156.285843,"volume_a":5081800,"open_b":185.767469,"high_b":187.44478,"low_b":185.106412,"close_b":187.168518,"volume_b":45280000,"open_bench":433.157056,"close_bench":431.725128},{"date":"2023-09-06","open_a":156.96785,"high_a":157.030673,"low_a":155.675636,"close_a":156.797348,"volume_a":5933800,"open_b":185.885844,"high_b":186.329851,"low_b":179.04833,"close_b":180.469116,"volume_b":81755800,"open_bench":430.917937,"close_bench":428.822937},{"date":"2023-09-07","open_a":157.802417,"high_a":158.888227,"low_a":157.228101,"close_a":158.295959,"volume_a":4934300,"open_b":172.842277,"high_b":175.831856,"low_b":171.224163,"close_b":175.190521,"volume_b":112488800,"open_bench":425.834105,"close_bench":427.506287},{"date":"2023-09-08","open_a":157.766495,"high_a":158.968966,"low_a":157.587024,"close_a":158.179291,"volume_a":2995000,"open_b":175.96996,"high_b":177.834738,"low_b":175.417421,"close_b":175.802216,"volume_b":65551300,"open_bench":427.554354,"close_bench":428.150177},{"date":"2023-09-11","open_a":158.681802,"high_a":161.014954,"low_a":158.681802,"close_a":160.566269,"volume_a":4512700,"open_b":177.667054,"high_b":177.893981,"low_b":174.973474,"close_b":176.966522,"volume_b":58953100,"open_bench":430.764139,"close_bench":430.965973},{"date":"2023-09-12","open_a":160.39581,"high_a":160.557329,"low_a":159.55228,"close_a":159.974045,"volume_a":3331100,"open_b":177.094775,"high_b":177.726234,"low_b":172.487096,"close_b":173.947342,"volume_b":90370200,"open_bench":429.52443,"close_bench":428.601837},{"date":"2023-09-13","open_a":160.027878,"high_a":161.652115,"low_a":159.920198,"close_a":161.239319,"volume_a":3781400,"open_b":174.154518,"high_b":174.933985,"low_b":171.658282,"close_b":171.885223,"volume_b":84267900,"open_bench":428.822861,"close_bench":429.101562},{"date":"2023-09-14","open_a":161.472679,"high_a":163.043076,"low_a":160.925283,"close_a":162.63028,"volume_a":4118300,"open_b":171.678066,"high_b":173.750048,"low_b":171.263672,"close_b":173.394852,"volume_b":60895800,"open_bench":431.561738,"close_bench":432.801422},{"date":"2023-09-15","open_a":162.791765,"high_a":163.15969,"low_a":161.176498,"close_a":161.382889,"volume_a":11661200,"open_b":174.12493,"high_b":174.144667,"low_b":171.500438,"close_b":172.674545,"volume_b":109259500,"open_bench":431.222728,"close_bench":427.586914},{"date":"2023-09-18","open_a":162.423857,"high_a":162.423857,"low_a":160.117611,"close_a":160.970123,"volume_a":3182600,"open_b":174.124925,"high_b":176.986235,"low_b":173.819064,"close_b":175.595047,"volume_b":67257600,"open_bench":427.278307,"close_bench":427.837677},{"date":"2023-09-19","open_a":160.503494,"high_a":160.745788,"low_a":159.1036,"close_a":160.018921,"volume_a":3369300,"open_b":175.151086,"high_b":177.232929,"low_b":174.766291,"close_b":176.680405,"volume_b":51826900,"open_bench":426.921448,"close_bench":426.950378},{"date":"2023-09-20","open_a":160.709833,"high_a":161.005966,"low_a":159.336859,"close_a":160.072708,"volume_a":3695800,"open_b":176.867812,"high_b":177.301942,"low_b":173.059322,"close_b":173.148132,"volume_b":58436200,"open_bench":428.204133,"close_bench":423.025299},{"date":"2023-09-21","open_a":159.821492,"high_a":160.027883,"low_a":157.156303,"close_a":157.380646,"volume_a":5351000,"open_b":172.220715,"high_b":173.947362,"low_b":171.53992,"close_b":171.608978,"volume_b":63149100,"open_bench":420.189956,"close_bench":416.033386},{"date":"2023-09-22","open_a":156.985803,"high_a":158.286986,"low_a":156.474296,"close_a":157.281937,"volume_a":4542300,"open_b":172.339095,"high_b":174.716938,"low_b":171.727374,"close_b":172.457489,"volume_b":56725400,"open_bench":417.055696,"close_bench":415.097961},{"date":"2023-09-25","open_a":156.752456,"high_a":157.434465,"low_a":155.406402,"close_a":156.438385,"volume_a":3553600,"open_b":171.875369,"high_b":174.608408,"low_b":171.826033,"close_b":173.730286,"volume_b":46172700,"open_bench":413.892408,"close_bench":416.843475},{"date":"2023-09-26","open_a":155.244912,"high_a":155.891019,"low_a":154.024489,"close_a":154.814178,"volume_a":5213700,"open_b":172.487103,"high_b":172.862022,"low_b":169.369268,"close_b":169.665268,"volume_b":64588900,"open_bench":413.815266,"close_bench":410.719543},{"date":"2023-09-27","open_a":154.62573,"high_a":154.966735,"low_a":151.682359,"close_a":152.166946,"volume_a":5729900,"open_b":170.316463,"high_b":170.730857,"low_b":166.794111,"close_b":168.155685,"volume_b":66921800,"open_bench":411.88647,"close_bench":410.883484},{"date":"2023-09-28","open_a":152.489958,"high_a":152.7143,"low_a":150.919561,"close_a":152.104095,"volume_a":4841600,"open_b":167.080225,"high_b":169.734331,"low_b":165.383177,"close_b":168.412216,"volume_b":56294400,"open_bench":410.333833,"close_bench":413.265594},{"date":"2023-09-29","open_a":153.010441,"high_a":153.306574,"low_a":150.793937,"close_a":152.050262,"volume_a":5194200,"open_b":169.724466,"high_b":170.760457,"low_b":168.066877,"close_b":168.925278,"volume_b":51861100,"open_bench":416.30342,"close_bench":412.262573},{"date":"2023-10-02","open_a":151.727208,"high_a":152.095133,"low_a":150.147843,"close_a":151.807968,"volume_a":3838800,"open_b":168.935144,"high_b":171.974045,"low_b":168.649006,"close_b":171.431381,"volume_b":52164500,"open_bench":411.433193,"close_bench":412.098633},{"date":"2023-10-03","open_a":151.529798,"high_a":152.552798,"low_a":149.88761,"close_a":150.488846,"volume_a":4873400,"open_b":169.961219,"high_b":171.312947,"low_b":168.540448,"close_b":170.09935,"volume_b":49594600,"open_bench":409.928752,"close_bench":406.582275},{"date":"2023-10-04","open_a":151.161835,"high_a":151.772053,"low_a":149.914492,"close_a":151.574631,"volume_a":4134000,"open_b":168.806879,"high_b":171.885254,"low_b":168.688485,"close_b":171.34259,"volume_b":53020300,"open_bench":407.045171,"close_bench":409.542969},{"date":"2023-10-05","open_a":151.449061,"high_a":151.48495,"low_a":143.462464,"close_a":143.668869,"volume_a":10556400,"open_b":171.470818,"high_b":173.108669,"low_b":170.37563,"close_b":172.575882,"volume_b":48527900,"open_bench":409.253611,"close_bench":409.388641},{"date":"2023-10-06","open_a":142.484287,"high_a":144.404657,"low_a":139.837051,"close_a":143.83931,"volume_a":12774700,"open_b":171.480714,"high_b":175.614803,"low_b":170.868978,"close_b":175.121475,"volume_b":57266700,"open_bench":406.948737,"close_bench":414.249268},{"date":"2023-10-09","open_a":143.579092,"high_a":144.853355,"low_a":142.358669,"close_a":144.799515,"volume_a":7244500,"open_b":174.450539,"high_b":176.660653,"low_b":173.454023,"close_b":176.601456,"volume_b":42390800,"open_bench":412.359012,"close_bench":416.901367},{"date":"2023-10-10","open_a":147.392912,"high_a":148.227459,"low_a":145.400751,"close_a":147.527512,"volume_a":11456400,"open_b":175.723337,"high_b":177.321714,"low_b":175.57533,"close_b":176.00946,"volume_b":43698000,"open_bench":417.528209,"close_bench":419.071259},{"date":"2023-10-11","open_a":148.074903,"high_a":148.460779,"low_a":145.014884,"close_a":145.930191,"volume_a":7371500,"open_b":175.821995,"high_b":177.449986,"low_b":175.230011,"close_b":177.40065,"volume_b":47551100,"open_bench":420.132147,"close_bench":420.787933},{"date":"2023-10-12","open_a":145.912269,"high_a":145.957126,"low_a":141.272857,"close_a":141.856155,"volume_a":9024100,"open_b":177.667049,"high_b":179.906746,"low_b":176.65078,"close_b":178.298508,"volume_b":56743100,"open_bench":421.395517,"close_bench":418.222626},{"date":"2023-10-13","open_a":142.340697,"high_a":143.794445,"low_a":141.739461,"close_a":143.579071,"volume_a":6757100,"open_b":178.999014,"high_b":179.502203,"low_b":175.762786,"close_b":176.463318,"volume_b":51427100,"open_bench":419.717419,"close_bench":416.139496},{"date":"2023-10-16","open_a":144.476457,"high_a":145.328955,"low_a":143.399617,"close_a":144.548248,"volume_a":5605000,"open_b":174.39134,"high_b":176.690249,"low_b":174.154537,"close_b":176.335052,"volume_b":52517000,"open_bench":418.376879,"close_bench":420.517853},{"date":"2023-10-17","open_a":143.579111,"high_a":145.041827,"low_a":142.825323,"close_a":143.911133,"volume_a":6411300,"open_b":174.292668,"high_b":176.039052,"low_b":172.467364,"close_b":174.785995,"volume_b":57549400,"open_bench":417.402874,"close_bench":420.498596},{"date":"2023-10-18","open_a":144.494419,"high_a":146.630148,"low_a":144.359805,"close_a":145.400757,"volume_a":6650600,"open_b":173.236944,"high_b":175.210255,"low_b":172.773215,"close_b":173.493469,"volume_b":54764400,"open_bench":418.733747,"close_bench":414.895416},{"date":"2023-10-19","open_a":145.284076,"high_a":146.019925,"low_a":143.947004,"close_a":144.081604,"volume_a":6336800,"open_b":173.690838,"high_b":175.466821,"low_b":172.85219,"close_b":173.118591,"volume_b":59302900,"open_bench":415.609055,"close_bench":411.249939},{"date":"2023-10-20","open_a":144.808462,"high_a":145.580201,"low_a":143.31884,"close_a":143.579071,"volume_a":5486500,"open_b":172.970548,"high_b":173.079081,"low_b":170.33618,"close_b":170.572983,"volume_b":64244000,"open_bench":410.815997,"close_bench":406.196503},{"date":"2023-10-23","open_a":143.220161,"high_a":145.014899,"low_a":142.735574,"close_a":143.650894,"volume_a":5795800,"open_b":168.629285,"high_b":171.687908,"low_b":167.662352,"close_b":170.691391,"volume_b":55980100,"open_bench":404.672745,"close_bench":405.492493},{"date":"2023-10-24","open_a":143.946995,"high_a":146.136578,"low_a":143.291906,"close_a":145.544312,"volume_a":5265200,"open_b":170.740709,"high_b":171.35243,"low_b":169.162054,"close_b":171.125504,"volume_b":43816600,"open_bench":407.604558,"close_bench":408.549683},{"date":"2023-10-25","open_a":145.418698,"high_a":146.719881,"low_a":144.898223,"close_a":145.687912,"volume_a":4534600,"open_b":169.586347,"high_b":170.750593,"low_b":168.37275,"close_b":168.816757,"volume_b":57157000,"open_bench":406.871566,"close_bench":402.686035},{"date":"2023-10-26","open_a":145.98404,"high_a":146.594258,"low_a":144.74568,"close_a":144.844391,"volume_a":5147000,"open_b":168.096459,"high_b":169.092991,"low_b":163.459182,"close_b":164.662903,"volume_b":70625300,"open_bench":401.625205,"close_bench":397.864014},{"date":"2023-10-27","open_a":145.391762,"high_a":145.499442,"low_a":142.582992,"close_a":143.238068,"volume_a":5078000,"open_b":164.682658,"high_b":166.705304,"low_b":164.603724,"close_b":165.975174,"volume_b":58499100,"open_bench":399.445699,"close_bench":396.060638},{"date":"2023-10-30","open_a":144.745704,"high_a":145.894336,"low_a":144.243179,"close_a":145.625122,"volume_a":4625900,"open_b":166.764461,"high_b":168.885763,"low_b":166.616453,"close_b":168.017502,"volume_b":51131000,"open_bench":398.838133,"close_bench":400.795868},{"date":"2023-10-31","open_a":146.226327,"high_a":146.585283,"low_a":145.068727,"close_a":146.522461,"volume_a":5947400,"open_b":167.090086,"high_b":168.61939,"low_b":165.659424,"close_b":168.491135,"volume_b":44846000,"open_bench":401.364847,"close_bench":403.312958},{"date":"2023-11-01","open_a":146.87247,"high_a":148.451835,"low_a":146.05586,"close_a":147.94931,"volume_a":6682500,"open_b":168.71807,"high_b":171.904962,"low_b":167.849808,"close_b":171.648438,"volume_b":56934900,"open_bench":404.277374,"close_bench":407.614197},{"date":"2023-11-02","open_a":147.832661,"high_a":150.004294,"low_a":147.769839,"close_a":149.70816,"volume_a":5413400,"open_b":173.177762,"high_b":175.407598,"low_b":173.118565,"close_b":175.200409,"volume_b":77334800,"open_bench":411.394589,"close_bench":415.425812},{"date":"2023-11-03","open_a":151.368253,"high_a":151.484916,"low_a":149.582497,"close_a":149.672226,"volume_a":4063300,"open_b":171.914821,"high_b":174.460393,"low_b":171.036698,"close_b":174.292648,"volume_b":79829200,"open_bench":417.721161,"close_bench":419.215973},{"date":"2023-11-06","open_a":148.478717,"high_a":150.309344,"low_a":148.254375,"close_a":149.591446,"volume_a":5784600,"open_b":174.026313,"high_b":177.0356,"low_b":173.858583,"close_b":176.838272,"volume_b":63841300,"open_bench":419.968159,"close_bench":420.180328},{"date":"2023-11-07","open_a":149.681241,"high_a":150.497851,"low_a":149.071023,"close_a":150.022232,"volume_a":4687200,"open_b":176.788907,"high_b":180.005414,"low_b":176.581718,"close_b":179.393692,"volume_b":70530000,"open_bench":420.180341,"close_bench":421.37619},{"date":"2023-11-08","open_a":150.910579,"high_a":151.06313,"low_a":149.555557,"close_a":150.210632,"volume_a":4926600,"open_b":179.91663,"high_b":181.001942,"low_b":179.166762,"close_b":180.449417,"volume_b":49340300,"open_bench":421.974093,"close_bench":421.684784},{"date":"2023-11-09","open_a":150.381166,"high_a":150.632428,"low_a":148.792818,"close_a":149.106903,"volume_a":5639300,"open_b":180.518478,"high_b":181.662987,"low_b":179.383816,"close_b":179.975815,"volume_b":53763500,"open_bench":422.82275,"close_bench":418.396149},{"date":"2023-11-10","open_a":150.138883,"high_a":150.560648,"low_a":147.940331,"close_a":149.78891,"volume_a":5552900,"open_b":181.754137,"high_b":184.322827,"low_b":181.319435,"close_b":184.154861,"volume_b":66133400,"open_bench":420.459954,"close_bench":424.92511},{"date":"2023-11-13","open_a":149.860673,"high_a":151.457989,"low_a":149.376099,"close_a":150.551651,"volume_a":5402300,"open_b":183.581846,"high_b":183.789308,"low_b":181.991237,"close_b":182.574127,"volume_b":43627500,"open_bench":423.594263,"close_bench":424.520081},{"date":"2023-11-14","open_a":150.713183,"high_a":151.152899,"low_a":149.761974,"close_a":150.856766,"volume_a":6238900,"open_b":185.439205,"high_b":185.844271,"low_b":184.056074,"close_b":185.182343,"volume_b":60108400,"open_bench":430.431922,"close_bench":432.756134},{"date":"2023-11-15","open_a":151.386189,"high_a":151.466949,"low_a":149.205589,"close_a":150.085007,"volume_a":5611200,"open_b":185.587361,"high_b":187.217481,"low_b":185.518197,"close_b":185.745422,"volume_b":53790500,"open_bench":434.086988,"close_bench":433.672302},{"date":"2023-11-16","open_a":151.108023,"high_a":152.552788,"low_a":149.322253,"close_a":150.497818,"volume_a":5386400,"open_b":187.28669,"high_b":188.659947,"low_b":186.377758,"close_b":187.425003,"volume_b":54412900,"open_bench":433.228673,"close_bench":434.202728},{"date":"2023-11-17","open_a":150.802924,"high_a":150.982395,"low_a":148.756923,"close_a":149.645309,"volume_a":5583100,"open_b":187.958465,"high_b":188.086904,"low_b":186.298707,"close_b":187.405212,"volume_b":50922700,"open_bench":434.212359,"close_bench":434.742798},{"date":"2023-11-20","open_a":148.963311,"high_a":149.654289,"low_a":148.712049,"close_a":149.429947,"volume_a":4886200,"open_b":187.602835,"high_b":189.598509,"low_b":187.592961,"close_b":189.144043,"volume_b":46505100,"open_bench":434.492034,"close_bench":438.089264},{"date":"2023-11-21","open_a":149.959381,"high_a":150.632407,"low_a":148.891509,"close_a":150.578568,"volume_a":3980000,"open_b":189.10451,"high_b":189.213186,"low_b":187.454627,"close_b":188.343781,"volume_b":38134500,"open_bench":437.047668,"close_bench":437.13446},{"date":"2023-11-22","open_a":151.072122,"high_a":152.166914,"low_a":151.027251,"close_a":151.852829,"volume_a":3235600,"open_b":189.183562,"high_b":190.606205,"low_b":188.531508,"close_b":189.005722,"volume_b":39617700,"open_bench":438.78365,"close_bench":438.822205},{"date":"2023-11-24","open_a":151.610579,"high_a":152.570771,"low_a":151.610579,"close_a":151.987473,"volume_a":1751700,"open_b":188.571008,"high_b":188.600645,"low_b":186.970525,"close_b":187.681854,"volume_b":24048300,"open_bench":438.8704,"close_bench":439.092194},{"date":"2023-11-27","open_a":152.283588,"high_a":152.283588,"low_a":150.838822,"close_a":151.045227,"volume_a":3664700,"open_b":187.632467,"high_b":188.373434,"low_b":186.624749,"close_b":187.504028,"volume_b":40552600,"open_bench":438.465385,"close_bench":438.301453},{"date":"2023-11-28","open_a":151.099067,"high_a":152.337427,"low_a":150.776013,"close_a":151.5298,"volume_a":4056100,"open_b":187.494147,"high_b":188.778492,"low_b":187.118719,"close_b":188.106674,"volume_b":38415400,"open_bench":437.915665,"close_bench":438.735413},{"date":"2023-11-29","open_a":151.745156,"high_a":151.763093,"low_a":149.54659,"close_a":150.004257,"volume_a":4722400,"open_b":188.60062,"high_b":189.776289,"low_b":186.693874,"close_b":187.08905,"volume_b":43014200,"open_bench":440.876347,"close_bench":438.426758},{"date":"2023-11-30","open_a":150.19867,"high_a":152.233147,"low_a":149.032242,"close_a":152.169846,"volume_a":7346700,"open_b":187.553449,"high_b":188.027678,"low_b":185.923329,"close_b":187.662125,"volume_b":48794400,"open_bench":439.265904,"close_bench":440.153137},{"date":"2023-12-01","open_a":152.24217,"high_a":152.676202,"low_a":151.347009,"close_a":152.531525,"volume_a":4897400,"open_b":188.037511,"high_b":189.252692,"low_b":186.950754,"close_b":188.936554,"volume_b":45704800,"open_bench":439.545483,"close_bench":442.756958},{"date":"2023-12-04","open_a":152.269324,"high_a":153.64373,"low_a":152.079447,"close_a":152.938446,"volume_a":5323300,"open_b":187.691755,"high_b":187.760919,"low_b":185.192229,"close_b":187.148376,"volume_b":43389500,"open_bench":439.381606,"close_bench":440.4328},{"date":"2023-12-05","open_a":152.947436,"high_a":153.327218,"low_a":151.645383,"close_a":151.853348,"volume_a":4553800,"open_b":187.918944,"high_b":192.058463,"low_b":187.889291,"close_b":191.090271,"volume_b":66628400,"open_bench":439.053683,"close_bench":440.345978},{"date":"2023-12-06","open_a":151.889535,"high_a":152.196977,"low_a":150.75927,"close_a":151.464554,"volume_a":4726600,"open_b":192.10789,"high_b":192.414153,"low_b":189.796078,"close_b":190.003555,"volume_b":41089700,"open_bench":442.47735,"close_bench":438.571533},{"date":"2023-12-07","open_a":151.907626,"high_a":152.178892,"low_a":150.732147,"close_a":151.419357,"volume_a":4934800,"open_b":191.297778,"high_b":192.651272,"low_b":191.258252,"close_b":191.930069,"volume_b":47477700,"open_bench":440.644921,"close_bench":441.917938},{"date":"2023-12-08","open_a":151.039589,"high_a":151.111928,"low_a":149.439144,"close_a":149.80986,"volume_a":5115200,"open_b":191.860915,"high_b":193.629364,"low_b":191.3373,"close_b":193.352737,"volume_b":53406400,"open_bench":441.175389,"close_bench":443.817871},{"date":"2023-12-11","open_a":151.401253,"high_a":152.025161,"low_a":150.515129,"close_a":151.744858,"volume_a":6912600,"open_b":190.784031,"high_b":191.159459,"low_b":189.114384,"close_b":190.85318,"volume_b":60943700,"open_bench":443.325955,"close_bench":445.544067},{"date":"2023-12-12","open_a":153.065021,"high_a":153.065021,"low_a":151.645415,"close_a":152.332611,"volume_a":4515800,"open_b":190.75439,"high_b":192.374635,"low_b":189.41077,"close_b":192.364761,"volume_b":52696900,"open_bench":445.196905,"close_bench":447.578979},{"date":"2023-12-13","open_a":152.395906,"high_a":155.5697,"low_a":152.016139,"close_a":155.533524,"volume_a":6082100,"open_b":192.740166,"high_b":195.615119,"low_b":192.503066,"close_b":195.575607,"volume_b":70404200,"open_bench":447.955064,"close_bench":453.751129},{"date":"2023-12-14","open_a":155.207967,"high_a":155.497321,"low_a":151.853348,"close_a":152.133652,"volume_a":7119300,"open_b":195.634904,"high_b":197.215623,"low_b":193.797306,"close_b":195.723816,"volume_b":66831600,"open_bench":455.679914,"close_bench":455.207367},{"date":"2023-12-15","open_a":151.410311,"high_a":152.368773,"low_a":150.506099,"close_a":151.003418,"volume_a":17734600,"open_b":195.150781,"high_b":196.010297,"low_b":194.627166,"close_b":195.190308,"volume_b":128538400,"open_bench":454.61285,"close_bench":454.457916},{"date":"2023-12-18","open_a":151.961868,"high_a":153.942094,"low_a":151.374135,"close_a":152.730453,"volume_a":5439500,"open_b":193.728136,"high_b":194.26164,"low_b":192.048615,"close_b":193.530548,"volume_b":55751900,"open_bench":456.055693,"close_bench":457.014313},{"date":"2023-12-19","open_a":151.853374,"high_a":152.658123,"low_a":151.283716,"close_a":151.862411,"volume_a":4855900,"open_b":193.797313,"high_b":194.577791,"low_b":193.530561,"close_b":194.567917,"volume_b":40714100,"open_bench":457.556506,"close_bench":459.793304},{"date":"2023-12-20","open_a":150.560355,"high_a":151.428405,"low_a":149.800821,"close_a":149.818909,"volume_a":5157800,"open_b":194.528366,"high_b":195.29897,"low_b":192.483307,"close_b":192.483307,"volume_b":52242800,"open_bench":458.941206,"close_bench":453.421844},{"date":"2023-12-21","open_a":150.551307,"high_a":151.09384,"low_a":149.547631,"close_a":150.976288,"volume_a":4374000,"open_b":193.738039,"high_b":194.706231,"low_b":191.169349,"close_b":192.335129,"volume_b":46482500,"open_bench":456.394548,"close_bench":457.721161},{"date":"2023-12-22","open_a":151.328915,"high_a":152.061324,"low_a":150.876809,"close_a":151.618256,"volume_a":3030400,"open_b":192.829099,"high_b":193.056339,"low_b":190.645726,"close_b":191.268143,"volume_b":37149600,"open_bench":458.84439,"close_bench":458.641052},{"date":"2023-12-26","open_a":151.591154,"high_a":153.191614,"low_a":151.229475,"close_a":152.685257,"volume_a":2779200,"open_b":191.278014,"high_b":191.554641,"low_b":190.507411,"close_b":190.724762,"volume_b":28919300,"open_bench":459.047717,"close_bench":460.577637},{"date":"2023-12-27","open_a":152.721402,"high_a":153.254897,"low_a":152.233133,"close_a":153.173508,"volume_a":3502500,"open_b":190.171508,"high_b":191.169338,"low_b":188.788362,"close_b":190.823547,"volume_b":48087700,"open_bench":460.374269,"close_bench":461.41037},{"date":"2023-12-28","open_a":153.173512,"high_a":153.300101,"low_a":152.540566,"close_a":153.164474,"volume_a":2911100,"open_b":191.80162,"high_b":192.315361,"low_b":190.843302,"close_b":191.248367,"volume_b":34049900,"open_bench":461.768698,"close_bench":461.584717},{"date":"2023-12-29","open_a":153.119259,"high_a":153.779344,"low_a":152.965545,"close_a":153.571365,"volume_a":3291500,"open_b":191.564508,"high_b":192.058485,"low_b":189.420647,"close_b":190.211014,"volume_b":42672100,"open_bench":461.390984,"close_bench":460.248383},{"date":"2024-01-02","open_a":153.282027,"high_a":156.419644,"low_a":152.983635,"close_a":156.347305,"volume_a":5743100,"open_b":184.895814,"high_b":186.170285,"low_b":181.675085,"close_b":183.404007,"volume_b":82488700,"open_bench":457.198205,"close_bench":457.672668},{"date":"2024-01-03","open_a":158.237121,"high_a":158.417961,"low_a":156.157431,"close_a":156.383484,"volume_a":5588200,"open_b":182.001124,"high_b":183.641134,"low_b":181.220631,"close_b":182.030762,"volume_b":58414500,"open_bench":455.523045,"close_bench":453.935028},{"date":"2024-01-04","open_a":155.171831,"high_a":156.907917,"low_a":154.240494,"close_a":155.045242,"volume_a":6283200,"open_b":179.956033,"high_b":180.884713,"low_b":178.70134,"close_b":179.718933,"volume_b":71983600,"open_bench":453.460568,"close_bench":452.4729},{"date":"2024-01-05","open_a":155.135681,"high_a":155.135681,"low_a":151.509782,"close_a":152.757599,"volume_a":5252500,"open_b":179.797983,"high_b":180.558698,"low_b":177.999897,"close_b":178.997726,"volume_b":62379700,"open_bench":452.676255,"close_bench":453.092651},{"date":"2024-01-08","open_a":152.757602,"high_a":153.471923,"low_a":151.717749,"close_a":152.911316,"volume_a":5735600,"open_b":179.896761,"high_b":183.364493,"low_b":179.313871,"close_b":183.324966,"volume_b":59144500,"open_bench":453.586385,"close_bench":459.560883},{"date":"2024-01-09","open_a":151.717727,"high_a":151.808154,"low_a":150.035892,"close_a":151.15712,"volume_a":6268800,"open_b":181.704712,"high_b":182.919893,"low_b":180.529043,"close_b":182.910019,"volume_b":42841800,"open_bench":456.917422,"close_bench":458.863739},{"date":"2024-01-10","open_a":150.6327,"high_a":151.87148,"low_a":150.280057,"close_a":150.931091,"volume_a":4394900,"open_b":182.129584,"high_b":184.15488,"low_b":181.704755,"close_b":183.947418,"volume_b":46792900,"open_bench":459.134889,"close_bench":461.458832},{"date":"2024-01-11","open_a":151.13,"high_a":151.13,"low_a":149.032236,"close_a":150.216751,"volume_a":4800700,"open_b":184.293184,"high_b":184.797051,"low_b":181.408356,"close_b":183.35463,"volume_b":49128400,"open_bench":462.456191,"close_bench":461.255493},{"date":"2024-01-12","open_a":150.406634,"high_a":151.347022,"low_a":149.710401,"close_a":151.247559,"volume_a":3546100,"open_b":183.818962,"high_b":184.490779,"low_b":182.959446,"close_b":183.680649,"volume_b":40477800,"open_bench":462.698288,"close_bench":461.575043},{"date":"2024-01-16","open_a":151.247569,"high_a":151.473622,"low_a":149.366806,"close_a":150.25293,"volume_a":4576600,"open_b":179.965918,"high_b":182.040614,"low_b":178.750722,"close_b":181.418213,"volume_b":65603000,"open_bench":460.199992,"close_bench":459.880432},{"date":"2024-01-17","open_a":150.090132,"high_a":150.75924,"low_a":149.592813,"close_a":150.497025,"volume_a":4392300,"open_b":179.086655,"high_b":180.726649,"low_b":178.128337,"close_b":180.47966,"volume_b":47317400,"open_bench":456.86905,"close_bench":457.324158},{"date":"2024-01-18","open_a":149.619962,"high_a":151.256584,"low_a":149.185944,"close_a":151.15712,"volume_a":4709400,"open_b":183.848572,"high_b":186.861838,"low_b":183.591709,"close_b":186.357986,"volume_b":78005800,"open_bench":458.989627,"close_bench":461.391022},{"date":"2024-01-19","open_a":151.129982,"high_a":151.220409,"low_a":149.710377,"close_a":149.900253,"volume_a":6528600,"open_b":187.049565,"high_b":189.638003,"low_b":186.545713,"close_b":189.252701,"volume_b":68903000,"open_bench":462.514291,"close_bench":467.142822},{"date":"2024-01-22","open_a":150.063004,"high_a":150.741163,"low_a":149.213043,"close_a":149.294418,"volume_a":5697400,"open_b":189.983797,"high_b":192.9773,"low_b":189.94427,"close_b":191.554642,"volume_b":60133900,"open_bench":468.672807,"close_bench":468.130554},{"date":"2024-01-23","open_a":148.797163,"high_a":151.844354,"low_a":148.616323,"close_a":151.582138,"volume_a":4932700,"open_b":192.67104,"high_b":193.392243,"low_b":191.495371,"close_b":192.829102,"volume_b":42355600,"open_bench":468.672716,"close_bench":469.495758},{"date":"2024-01-24","open_a":151.582105,"high_a":151.672532,"low_a":149.66518,"close_a":149.737518,"volume_a":4984700,"open_b":193.066205,"high_b":194.014648,"low_b":191.999211,"close_b":192.157288,"volume_b":53631300,"open_bench":472.352332,"close_bench":470.009033},{"date":"2024-01-25","open_a":149.927401,"high_a":150.641737,"low_a":149.140741,"close_a":150.60556,"volume_a":5373800,"open_b":192.86864,"high_b":193.905996,"low_b":190.784053,"close_b":191.831284,"volume_b":54822100,"open_bench":472.129616,"close_bench":472.565369},{"date":"2024-01-26","open_a":150.641734,"high_a":151.934751,"low_a":150.641734,"close_a":151.781036,"volume_a":4849000,"open_b":191.930064,"high_b":192.414152,"low_b":189.628126,"close_b":190.102341,"volume_b":44594000,"open_bench":472.139346,"close_bench":471.965057},{"date":"2024-01-29","open_a":151.482636,"high_a":152.540563,"low_a":151.356047,"close_a":152.043243,"volume_a":5713600,"open_b":189.697265,"high_b":189.884979,"low_b":187.296541,"close_b":189.420639,"volume_b":47145600,"open_bench":472.274803,"close_bench":475.702606},{"date":"2024-01-30","open_a":151.609211,"high_a":153.44476,"low_a":151.292745,"close_a":153.372421,"volume_a":4804200,"open_b":188.640211,"high_b":189.489853,"low_b":185.212004,"close_b":185.775131,"volume_b":55859400,"open_bench":475.015188,"close_bench":475.334747},{"date":"2024-01-31","open_a":153.679871,"high_a":154.059638,"low_a":151.799109,"close_a":152.386841,"volume_a":5769200,"open_b":184.787126,"high_b":184.846416,"low_b":182.129539,"close_b":182.178925,"volume_b":55467800,"open_bench":473.136685,"close_bench":467.578583},{"date":"2024-02-01","open_a":152.386854,"high_a":155.343631,"low_a":151.157125,"close_a":155.253204,"volume_a":4920800,"open_b":181.773902,"high_b":184.698242,"low_b":181.605952,"close_b":184.609329,"volume_b":64885400,"open_bench":469.27314,"close_bench":473.698334},{"date":"2024-02-02","open_a":154.837265,"high_a":155.488299,"low_a":153.806464,"close_a":154.593124,"volume_a":5307700,"open_b":177.693638,"high_b":185.073666,"low_b":177.090985,"close_b":183.611496,"volume_b":102551700,"open_bench":474.134041,"close_bench":478.68512},{"date":"2024-02-05","open_a":154.213359,"high_a":155.144696,"low_a":152.857041,"close_a":154.547913,"volume_a":5142700,"open_b":185.883758,"high_b":186.970515,"low_b":183.601584,"close_b":185.419418,"volume_b":69668800,"open_bench":478.055721,"close_bench":476.942139},{"date":"2024-02-06","open_a":155.027156,"high_a":155.370761,"low_a":153.661801,"close_a":155.000031,"volume_a":5182700,"open_b":184.60933,"high_b":187.029817,"low_b":184.520418,"close_b":187.019943,"volume_b":43490800,"open_bench":477.881368,"close_bench":478.326813},{"date":"2024-02-07","open_a":156.094132,"high_a":156.338259,"low_a":154.963867,"close_a":155.045242,"volume_a":5769700,"open_b":188.343782,"high_b":188.748847,"low_b":186.338234,"close_b":187.128601,"volume_b":53439000,"open_bench":480.563642,"close_bench":482.316284},{"date":"2024-02-08","open_a":155.27129,"high_a":157.423319,"low_a":154.990986,"close_a":157.197266,"volume_a":9269700,"open_b":187.108847,"high_b":187.257034,"low_b":185.093425,"close_b":186.051743,"volume_b":40962000,"open_bench":482.316267,"close_bench":482.529297},{"date":"2024-02-09","open_a":154.801105,"high_a":154.972908,"low_a":150.976292,"close_a":151.609238,"volume_a":12731700,"open_b":186.615576,"high_b":187.941136,"low_b":185.972591,"close_b":186.813431,"volume_b":45155200,"open_bench":483.032822,"close_bench":485.318054},{"date":"2024-02-12","open_a":151.781056,"high_a":154.683583,"low_a":151.184272,"close_a":154.267639,"volume_a":7867500,"open_b":186.388053,"high_b":186.635357,"low_b":184.775626,"close_b":185.131744,"volume_b":41781900,"open_bench":485.288962,"close_bench":485.10498},{"date":"2024-02-13","open_a":154.819207,"high_a":155.027172,"low_a":151.220446,"close_a":152.703354,"volume_a":7302000,"open_b":183.766633,"high_b":184.20189,"low_b":181.530996,"close_b":183.044495,"volume_b":56529500,"open_bench":478.859397,"close_bench":478.423645},{"date":"2024-02-14","open_a":152.215049,"high_a":152.486315,"low_a":150.153447,"close_a":151.473602,"volume_a":5731900,"open_b":183.32152,"high_b":183.529246,"low_b":180.472573,"close_b":182.164124,"volume_b":54630500,"open_bench":481.047798,"close_bench":482.771393},{"date":"2024-02-15","open_a":151.862412,"high_a":152.24218,"low_a":150.714059,"close_a":151.781036,"volume_a":4451400,"open_b":181.570573,"high_b":182.500439,"low_b":179.394301,"close_b":181.877228,"volume_b":65434500,"open_bench":483.468576,"close_bench":486.102386},{"date":"2024-02-16","open_a":151.193283,"high_a":151.374122,"low_a":149.809826,"close_a":150.388535,"volume_a":5246000,"open_b":181.441979,"high_b":182.856565,"low_b":179.710851,"close_b":180.343948,"volume_b":49752500,"open_bench":485.802216,"close_bench":483.68161},{"date":"2024-02-20","open_a":151.672524,"high_a":153.28202,"low_a":151.175204,"close_a":152.495346,"volume_a":6117900,"open_b":179.829551,"high_b":180.462649,"low_b":178.058861,"close_b":179.602036,"volume_b":53665600,"open_bench":481.948309,"close_bench":481.018738},{"date":"2024-02-21","open_a":153.670855,"high_a":153.87882,"low_a":151.781055,"close_a":152.658142,"volume_a":4192600,"open_b":179.977944,"high_b":180.917697,"low_b":178.711749,"close_b":180.353851,"volume_b":41371400,"open_bench":479.72118,"close_bench":481.454437},{"date":"2024-02-22","open_a":151.582082,"high_a":152.658096,"low_a":150.189602,"close_a":152.133652,"volume_a":7539200,"open_b":181.501328,"high_b":182.965378,"low_b":180.492338,"close_b":182.381729,"volume_b":52292200,"open_bench":488.038966,"close_bench":491.418365},{"date":"2024-02-23","open_a":151.853375,"high_a":153.942102,"low_a":151.726786,"close_a":153.35437,"volume_a":6409800,"open_b":183.014819,"high_b":183.044495,"low_b":180.2648,"close_b":180.551682,"volume_b":45119700,"open_bench":493.13228,"close_bench":491.757294},{"date":"2024-02-26","open_a":153.191589,"high_a":153.191589,"low_a":151.645395,"close_a":152.1427,"volume_a":3837000,"open_b":180.274715,"high_b":180.789096,"low_b":178.70185,"close_b":179.20636,"volume_b":40867400,"open_bench":492.193036,"close_bench":489.956238},{"date":"2024-02-27","open_a":151.744878,"high_a":152.106558,"low_a":151.319898,"close_a":152.052307,"volume_a":3567600,"open_b":179.146993,"high_b":181.936574,"low_b":177.623592,"close_b":180.660492,"volume_b":54318900,"open_bench":490.643762,"close_bench":490.866455},{"date":"2024-02-28","open_a":152.016121,"high_a":152.359726,"low_a":150.506088,"close_a":151.030533,"volume_a":5275300,"open_b":180.541773,"high_b":181.145195,"low_b":178.18745,"close_b":179.463531,"volume_b":48953900,"open_bench":489.31719,"close_bench":490.217743},{"date":"2024-02-29","open_a":151.435988,"high_a":151.864198,"low_a":150.251539,"close_a":150.643311,"volume_a":8489400,"open_b":179.315193,"high_b":180.601177,"low_b":177.593952,"close_b":178.800797,"volume_b":136682600,"open_bench":491.970287,"close_bench":491.97995},{"date":"2024-03-01","open_a":150.205996,"high_a":150.333551,"low_a":148.739104,"close_a":149.959991,"volume_a":6109900,"open_b":177.613735,"high_b":178.583163,"low_b":175.467138,"close_b":177.722549,"volume_b":73563100,"open_bench":492.851483,"close_bench":496.598816},{"date":"2024-03-04","open_a":149.832433,"high_a":150.925765,"low_a":149.057995,"close_a":150.670654,"volume_a":4303700,"open_b":174.250365,"high_b":174.992277,"low_b":171.915815,"close_b":173.2117,"volume_b":81510100,"open_bench":495.80488,"close_bench":496.066284},{"date":"2024-03-05","open_a":150.533971,"high_a":150.789082,"low_a":147.427075,"close_a":147.636627,"volume_a":6631300,"open_b":168.918498,"high_b":170.184693,"low_b":167.790792,"close_b":168.2854,"volume_b":95132400,"open_bench":494.071522,"close_bench":491.10849},{"date":"2024-03-06","open_a":147.654859,"high_a":149.413313,"low_a":147.636633,"close_a":148.511307,"volume_a":5302500,"open_b":169.215254,"high_b":169.393321,"low_b":166.860916,"close_b":167.296173,"volume_b":68587700,"open_bench":494.371684,"close_bench":493.597046},{"date":"2024-03-07","open_a":149.258422,"high_a":149.331299,"low_a":148.055747,"close_a":148.611526,"volume_a":5801600,"open_b":167.325897,"high_b":168.88886,"low_b":166.673026,"close_b":167.177521,"volume_b":71765100,"open_bench":496.87964,"close_bench":498.496704},{"date":"2024-03-08","open_a":147.891731,"high_a":149.604612,"low_a":147.682165,"close_a":148.556839,"volume_a":6065300,"open_b":167.177485,"high_b":171.826797,"low_b":167.118135,"close_b":168.888824,"volume_b":76267000,"open_bench":499.126207,"close_bench":495.5047},{"date":"2024-03-11","open_a":149.349526,"high_a":150.716194,"low_a":148.793747,"close_a":150.087524,"volume_a":5404800,"open_b":171.075023,"high_b":172.499497,"low_b":170.194621,"close_b":170.88707,"volume_b":60139500,"open_bench":494.303945,"close_bench":495.078583},{"date":"2024-03-12","open_a":150.461102,"high_a":150.716212,"low_a":149.385996,"close_a":149.914429,"volume_a":6158500,"open_b":171.282738,"high_b":172.153253,"low_b":169.165817,"close_b":171.361877,"volume_b":59825400,"open_bench":497.179862,"close_bench":500.404358},{"date":"2024-03-13","open_a":150.424653,"high_a":151.372204,"low_a":150.096652,"close_a":150.606873,"volume_a":4585800,"open_b":170.906829,"high_b":171.322298,"low_b":168.918495,"close_b":169.284515,"volume_b":52488700,"open_bench":500.723824,"close_bench":499.619934},{"date":"2024-03-14","open_a":151.48153,"high_a":151.48153,"low_a":149.60464,"close_a":150.133087,"volume_a":5155300,"open_b":171.045343,"high_b":172.430239,"low_b":170.194617,"close_b":171.134369,"volume_b":72913500,"open_bench":500.588244,"close_bench":498.632294},{"date":"2024-03-15","open_a":150.388213,"high_a":150.679775,"low_a":149.094435,"close_a":150.023773,"volume_a":21887300,"open_b":169.324078,"high_b":170.758438,"low_b":168.453563,"close_b":170.758438,"volume_b":121752700,"open_bench":495.577542,"close_bench":495.208435},{"date":"2024-03-18","open_a":152.201319,"high_a":156.875316,"low_a":152.110202,"close_a":156.037094,"volume_a":10478500,"open_b":173.676644,"high_b":175.793565,"low_b":171.648748,"close_b":171.846588,"volume_b":75604200,"open_bench":499.25887,"close_bench":498.15155},{"date":"2024-03-19","open_a":155.891323,"high_a":157.303536,"low_a":155.800206,"close_a":156.529099,"volume_a":7653700,"open_b":172.459881,"high_b":174.705405,"low_b":171.164011,"close_b":174.181122,"volume_b":55215200,"open_bench":497.461901,"close_bench":500.9198},{"date":"2024-03-20","open_a":156.984693,"high_a":157.713586,"low_a":155.909573,"close_a":156.583801,"volume_a":5046700,"open_b":173.825026,"high_b":176.74321,"low_b":173.201815,"close_b":176.74321,"volume_b":53423100,"open_bench":500.978096,"close_bench":505.552979},{"date":"2024-03-21","open_a":156.638449,"high_a":158.005118,"low_a":155.845786,"close_a":157.121338,"volume_a":4343000,"open_b":175.140681,"high_b":175.575939,"low_b":168.997644,"close_b":169.521927,"volume_b":106181300,"open_bench":508.379568,"close_bench":507.223694},{"date":"2024-03-22","open_a":157.640666,"high_a":157.713556,"low_a":156.492655,"close_a":156.729553,"volume_a":4106400,"open_b":169.907723,"high_b":171.18382,"low_b":168.226059,"close_b":170.422119,"volume_b":71160100,"open_bench":507.136269,"close_bench":506.262115},{"date":"2024-03-25","open_a":156.893542,"high_a":157.704432,"low_a":156.811546,"close_a":157.257996,"volume_a":3941900,"open_b":168.730574,"high_b":170.085795,"low_b":167.622642,"close_b":169.007553,"volume_b":54288300,"open_bench":504.892421,"close_bench":504.863312},{"date":"2024-03-26","open_a":156.866229,"high_a":157.595122,"low_a":156.547348,"close_a":157.37645,"volume_a":4593000,"open_b":168.166688,"high_b":169.571372,"low_b":167.751219,"close_b":167.879822,"volume_b":57388400,"open_bench":506.281457,"close_bench":503.930878},{"date":"2024-03-27","open_a":158.533526,"high_a":159.162195,"low_a":157.667972,"close_a":158.141754,"volume_a":4911300,"open_b":168.572289,"high_b":171.727891,"low_b":168.275522,"close_b":171.44101,"volume_b":60273300,"open_bench":506.747803,"close_bench":508.165894},{"date":"2024-03-28","open_a":159.226003,"high_a":160.182674,"low_a":158.87067,"close_a":159.453781,"volume_a":5710400,"open_b":169.89783,"high_b":170.372649,"low_b":168.671197,"close_b":169.630737,"volume_b":65672700,"open_bench":508.204694,"close_bench":508.068695},{"date":"2024-04-01","open_a":159.964009,"high_a":160.565333,"low_a":157.303549,"close_a":157.604218,"volume_a":4815400,"open_b":169.34389,"high_b":169.403241,"low_b":167.652324,"close_b":168.196396,"volume_b":46240500,"open_bench":508.806902,"close_bench":507.184753},{"date":"2024-04-02","open_a":157.567764,"high_a":158.415092,"low_a":156.128205,"close_a":156.201096,"volume_a":5158500,"open_b":167.256628,"high_b":167.513819,"low_b":166.415789,"close_b":167.019211,"volume_b":49329500,"open_bench":503.377194,"close_bench":503.960022},{"date":"2024-04-03","open_a":154.889112,"high_a":155.727334,"low_a":153.704663,"close_a":154.597549,"volume_a":5828000,"open_b":166.969754,"high_b":168.839372,"low_b":166.762027,"close_b":167.82048,"volume_b":47691700,"open_bench":502.872168,"close_bench":504.513702},{"date":"2024-04-04","open_a":155.727353,"high_a":156.51091,"low_a":154.278687,"close_a":154.570236,"volume_a":5422600,"open_b":168.453584,"high_b":170.066011,"low_b":166.999451,"close_b":166.999451,"volume_b":53704400,"open_bench":508.505812,"close_bench":498.355499},{"date":"2024-04-05","open_a":154.342445,"high_a":154.861785,"low_a":153.130677,"close_a":154.10556,"volume_a":4412200,"open_b":167.761123,"high_b":168.552498,"low_b":167.128025,"close_b":167.751236,"volume_b":42104800,"open_bench":499.705653,"close_bench":503.561768},{"date":"2024-04-08","open_a":154.114654,"high_a":154.934663,"low_a":153.741108,"close_a":154.506439,"volume_a":3755700,"open_b":167.20717,"high_b":167.375335,"low_b":166.425696,"close_b":166.633423,"volume_b":37425500,"open_bench":504.261226,"close_bench":503.843506},{"date":"2024-04-09","open_a":155.098643,"high_a":155.545093,"low_a":153.731974,"close_a":155.44487,"volume_a":3773900,"open_b":166.88073,"high_b":168.245853,"low_b":166.534514,"close_b":167.840271,"volume_b":42373800,"open_bench":505.572329,"close_bench":504.426178},{"date":"2024-04-10","open_a":154.497333,"high_a":154.551998,"low_a":152.857328,"close_a":153.932434,"volume_a":4581500,"open_b":166.979631,"high_b":167.266497,"low_b":165.307854,"close_b":165.970627,"volume_b":49709300,"open_bench":498.753737,"close_bench":499.375397},{"date":"2024-04-11","open_a":154.542863,"high_a":154.770642,"low_a":152.857314,"close_a":153.394867,"volume_a":4241500,"open_b":166.5246,"high_b":173.567828,"low_b":166.346549,"close_b":173.152344,"volume_b":91070300,"open_bench":500.890694,"close_bench":503.144165},{"date":"2024-04-12","open_a":152.784415,"high_a":153.53152,"low_a":152.447308,"close_a":153.157974,"volume_a":5453500,"open_b":172.380759,"high_b":176.436551,"low_b":172.331311,"close_b":174.646072,"volume_b":101670900,"open_bench":499.618196,"close_bench":496.199158},{"date":"2024-04-15","open_a":154.169322,"high_a":154.296877,"low_a":151.271977,"close_a":152.110199,"volume_a":5133200,"open_b":173.468887,"high_b":174.725195,"low_b":170.639729,"close_b":170.827682,"volume_b":73531800,"open_bench":500.356425,"close_bench":489.982727},{"date":"2024-04-16","open_a":152.183089,"high_a":153.531531,"low_a":152.055534,"close_a":152.611313,"volume_a":4569700,"open_b":169.897843,"high_b":171.886161,"low_b":166.455375,"close_b":167.553406,"volume_b":73711200,"open_bench":490.458646,"close_bench":489.089081},{"date":"2024-04-17","open_a":154.059965,"high_a":154.615744,"low_a":153.258196,"close_a":154.415298,"volume_a":4069200,"open_b":167.780932,"high_b":168.80971,"low_b":166.188293,"close_b":166.188293,"volume_b":50901200,"open_bench":491.536875,"close_bench":486.194611},{"date":"2024-04-18","open_a":154.980224,"high_a":157.330882,"low_a":154.980224,"close_a":156.957336,"volume_a":5574700,"open_b":166.217961,"high_b":166.821384,"low_b":164.753926,"close_b":165.238632,"volume_b":43122900,"open_bench":487.583562,"close_bench":485.194092},{"date":"2024-04-19","open_a":156.966429,"high_a":158.779548,"low_a":156.529099,"close_a":158.651993,"volume_a":6636700,"open_b":164.417585,"high_b":164.605523,"low_b":162.31055,"close_b":163.220627,"volume_b":68149400,"open_bench":485.11645,"close_bench":480.959198},{"date":"2024-04-22","open_a":158.907104,"high_a":161.449101,"low_a":158.560877,"close_a":160.774887,"volume_a":7288700,"open_b":163.735018,"high_b":165.456244,"low_b":162.993106,"close_b":164.051559,"volume_b":48116400,"open_bench":483.552556,"close_bench":485.388367},{"date":"2024-04-23","open_a":159.171323,"high_a":159.335331,"low_a":155.016653,"close_a":156.000656,"volume_a":9145000,"open_b":163.566871,"high_b":165.248535,"low_b":163.1415,"close_b":165.100143,"volume_b":49537800,"open_bench":487.389309,"close_bench":491.148315},{"date":"2024-04-24","open_a":153.522413,"high_a":162.259976,"low_a":152.747962,"close_a":161.640427,"volume_a":10016700,"open_b":164.743999,"high_b":167.474245,"low_b":164.417572,"close_b":167.197266,"volume_b":48251800,"open_bench":492.032235,"close_bench":490.915222},{"date":"2024-04-25","open_a":162.433127,"high_a":164.574247,"low_a":160.191798,"close_a":160.975342,"volume_a":7441500,"open_b":167.701758,"high_b":168.770113,"low_b":166.336636,"close_b":168.057877,"volume_b":50558300,"open_bench":484.863872,"close_bench":489.050262},{"date":"2024-04-26","open_a":158.934477,"high_a":162.697376,"low_a":158.84336,"close_a":159.973145,"volume_a":4472100,"open_b":168.048005,"high_b":169.492252,"low_b":167.355542,"close_b":167.474258,"volume_b":44838400,"open_bench":491.828246,"close_bench":493.683472},{"date":"2024-04-29","open_a":159.900226,"high_a":160.647345,"low_a":159.462896,"close_a":160.483337,"volume_a":3224500,"open_b":171.500349,"high_b":174.131667,"low_b":171.233272,"close_b":171.628952,"volume_b":68169400,"open_bench":495.460962,"close_bench":495.431824},{"date":"2024-04-30","open_a":160.601774,"high_a":160.783994,"low_a":159.298876,"close_a":160.273773,"volume_a":4465100,"open_b":171.460781,"high_b":173.102883,"low_b":168.166691,"close_b":168.493134,"volume_b":65934800,"open_bench":493.974834,"close_bench":487.583557},{"date":"2024-05-01","open_a":159.863786,"high_a":160.565332,"low_a":158.223781,"close_a":159.581329,"volume_a":4340500,"open_b":167.751238,"high_b":170.847488,"low_b":167.286305,"close_b":167.474258,"volume_b":50383100,"open_bench":487.000703,"close_bench":486.000244},{"date":"2024-05-02","open_a":159.973109,"high_a":160.647337,"low_a":159.034664,"close_a":159.85466,"volume_a":3964800,"open_b":170.649635,"high_b":171.549826,"low_b":169.04711,"close_b":171.164032,"volume_b":94214900,"open_bench":489.691345,"close_bench":490.546112},{"date":"2024-05-03","open_a":160.173566,"high_a":160.802236,"low_a":159.444673,"close_a":160.492447,"volume_a":3491900,"open_b":184.637161,"high_b":184.983392,"low_b":180.690199,"close_b":181.402435,"volume_b":163224100,"open_bench":496.50028,"close_bench":496.626556},{"date":"2024-05-06","open_a":161.048222,"high_a":161.048222,"low_a":159.563118,"close_a":160.191788,"volume_a":3576200,"open_b":180.383555,"high_b":182.213596,"low_b":178.474361,"close_b":179.750458,"volume_b":78569700,"open_bench":499.016026,"close_bench":501.755157},{"date":"2024-05-07","open_a":161.26689,"high_a":162.314663,"low_a":160.820439,"close_a":162.196228,"volume_a":3708000,"open_b":181.471663,"high_b":182.906023,"low_b":179.364643,"close_b":180.432983,"volume_b":77305800,"open_bench":502.716745,"close_bench":502.308807},{"date":"2024-05-08","open_a":162.824875,"high_a":162.979763,"low_a":161.439981,"close_a":161.640427,"volume_a":3400900,"open_b":180.878147,"high_b":181.095776,"low_b":179.493236,"close_b":180.769333,"volume_b":45057100,"open_bench":500.482719,"close_bench":502.357361},{"date":"2024-05-09","open_a":161.503789,"high_a":162.569788,"low_a":161.348901,"close_a":162.232681,"volume_a":2421400,"open_b":180.591256,"high_b":182.668616,"low_b":180.146112,"close_b":182.57959,"volume_b":48983000,"open_bench":502.541929,"close_bench":505.251892},{"date":"2024-05-10","open_a":162.13244,"high_a":164.146005,"low_a":161.822665,"close_a":163.808884,"volume_a":4201600,"open_b":183.154102,"high_b":183.34231,"low_b":180.410268,"close_b":181.321579,"volume_b":50759500,"open_bench":506.84474,"close_bench":505.902588},{"date":"2024-05-13","open_a":164.464888,"high_a":165.230233,"low_a":163.863564,"close_a":164.820221,"volume_a":4408100,"open_b":183.689008,"high_b":185.333337,"low_b":182.876743,"close_b":184.521072,"volume_b":72044800,"open_bench":507.573284,"close_bench":505.970581},{"date":"2024-05-14","open_a":164.956895,"high_a":165.166447,"low_a":162.724672,"close_a":163.881775,"volume_a":3508600,"open_b":185.739479,"high_b":186.522028,"low_b":184.530997,"close_b":185.660233,"volume_b":52393600,"open_bench":506.164889,"close_bench":508.292084},{"date":"2024-05-15","open_a":163.307779,"high_a":164.364658,"low_a":163.089107,"close_a":163.508224,"volume_a":4785100,"open_b":186.135697,"high_b":188.849816,"low_b":185.600788,"close_b":187.928604,"volume_b":70400000,"open_bench":510.749515,"close_bench":514.586243},{"date":"2024-05-16","open_a":163.663104,"high_a":167.107107,"low_a":163.508216,"close_a":166.833771,"volume_a":5984700,"open_b":188.671511,"high_b":189.295567,"low_b":187.869162,"close_b":188.047455,"volume_b":52845200,"open_bench":514.683401,"close_bench":513.527527},{"date":"2024-05-17","open_a":166.359999,"high_a":166.69712,"low_a":165.667559,"close_a":165.99556,"volume_a":4449100,"open_b":187.720562,"high_b":189.00829,"low_b":187.393677,"close_b":188.077164,"volume_b":41282900,"open_bench":513.644149,"close_bench":514.265808},{"date":"2024-05-20","open_a":165.904433,"high_a":165.940885,"low_a":163.918214,"close_a":164.282654,"volume_a":3221100,"open_b":187.542269,"high_b":190.10781,"low_b":187.225284,"close_b":189.236115,"volume_b":44361300,"open_bench":514.382217,"close_bench":514.858154},{"date":"2024-05-21","open_a":165.366903,"high_a":165.439794,"low_a":163.818015,"close_a":164.984238,"volume_a":3288100,"open_b":189.285655,"high_b":190.910169,"low_b":189.117262,"close_b":190.533768,"volume_b":42309400,"open_bench":514.100606,"close_bench":516.120911},{"date":"2024-05-22","open_a":164.410195,"high_a":165.940871,"low_a":163.981972,"close_a":165.904419,"volume_a":4047000,"open_b":190.454501,"high_b":190.999311,"low_b":188.473386,"close_b":189.097427,"volume_b":34648500,"open_bench":515.431439,"close_bench":514.634949},{"date":"2024-05-23","open_a":165.212018,"high_a":165.722239,"low_a":163.125563,"close_a":163.335129,"volume_a":4009000,"open_b":189.176664,"high_b":189.196479,"low_b":184.867748,"close_b":185.115387,"volume_b":51005900,"open_bench":517.675099,"close_bench":510.875854},{"date":"2024-05-24","open_a":163.42621,"high_a":163.471768,"low_a":162.059541,"close_a":162.168884,"volume_a":3447400,"open_b":187.037091,"high_b":188.780467,"low_b":186.264443,"close_b":188.186127,"volume_b":36327000,"open_bench":512.711495,"close_bench":514.25592},{"date":"2024-05-28","open_a":160.56532,"high_a":160.610879,"low_a":156.620202,"close_a":157.968658,"volume_a":6178100,"open_b":189.701697,"high_b":191.177633,"low_b":187.314464,"close_b":188.19606,"volume_b":52280100,"open_bench":515.062307,"close_bench":514.615479},{"date":"2024-05-29","open_a":157.103112,"high_a":157.786447,"low_a":155.545118,"close_a":155.93689,"volume_a":5352100,"open_b":187.819623,"high_b":190.434694,"low_b":187.720561,"close_b":188.493195,"volume_b":53068000,"open_bench":510.603933,"close_bench":511.011871},{"date":"2024-05-30","open_a":156.119108,"high_a":156.447109,"low_a":155.253541,"close_a":155.326431,"volume_a":3903100,"open_b":188.958755,"high_b":190.365345,"low_b":188.829993,"close_b":189.483749,"volume_b":49889100,"open_bench":509.477112,"close_bench":507.621857},{"date":"2024-05-31","open_a":154.788883,"high_a":157.595111,"low_a":154.451776,"close_a":157.531326,"volume_a":8155600,"open_b":189.632358,"high_b":190.751693,"low_b":188.116806,"close_b":190.434708,"volume_b":75158300,"open_bench":508.573861,"close_bench":512.245422},{"date":"2024-06-03","open_a":157.367297,"high_a":157.531291,"low_a":155.827515,"close_a":156.009735,"volume_a":5032400,"open_b":191.078556,"high_b":193.148833,"low_b":190.702155,"close_b":192.197891,"volume_b":50080500,"open_bench":513.848067,"close_bench":512.663025},{"date":"2024-06-04","open_a":155.99153,"high_a":158.88886,"low_a":155.636197,"close_a":158.433304,"volume_a":4619100,"open_b":192.802117,"high_b":193.475704,"low_b":191.207319,"close_b":192.514862,"volume_b":47471400,"open_bench":511.361625,"close_bench":513.236267},{"date":"2024-06-05","open_a":158.387805,"high_a":158.506254,"low_a":156.674923,"close_a":158.068924,"volume_a":5370700,"open_b":193.55496,"high_b":195.040796,"low_b":193.029965,"close_b":194.020523,"volume_b":54156800,"open_bench":515.547884,"close_bench":519.335999},{"date":"2024-06-06","open_a":158.533534,"high_a":158.897974,"low_a":157.330859,"close_a":157.804642,"volume_a":5536700,"open_b":193.842212,"high_b":194.644561,"low_b":192.336561,"close_b":192.643631,"volume_b":41181800,"open_bench":519.637185,"close_bench":519.326355},{"date":"2024-06-07","open_a":157.487868,"high_a":158.415342,"low_a":156.799145,"close_a":157.065445,"volume_a":3521100,"open_b":192.812044,"high_b":195.08043,"low_b":192.306865,"close_b":195.030899,"volume_b":53103900,"open_bench":518.354887,"close_bench":518.694885},{"date":"2024-06-10","open_a":156.211417,"high_a":156.450169,"low_a":152.088261,"close_a":152.345383,"volume_a":9429400,"open_b":195.0408,"high_b":195.437032,"low_b":190.335651,"close_b":191.296494,"volume_b":97010200,"open_bench":517.888751,"close_bench":520.297607},{"date":"2024-06-11","open_a":151.399536,"high_a":151.941341,"low_a":150.150652,"close_a":151.583206,"volume_a":8284500,"open_b":191.821501,"high_b":205.203946,"low_b":191.801701,"close_b":205.194031,"volume_b":172373300,"open_bench":518.753312,"close_bench":521.55072},{"date":"2024-06-12","open_a":151.408718,"high_a":151.821948,"low_a":149.168072,"close_a":150.444504,"volume_a":6281100,"open_b":205.411945,"high_b":218.120802,"low_b":204.946382,"close_b":211.058136,"volume_b":198134300,"open_bench":526.096443,"close_bench":525.834167},{"date":"2024-06-13","open_a":148.837511,"high_a":150.196586,"low_a":148.341623,"close_a":149.985382,"volume_a":5432600,"open_b":212.71235,"high_b":214.703366,"low_b":209.602,"close_b":212.217072,"volume_b":97862700,"open_bench":527.572881,"close_bench":526.892944},{"date":"2024-06-14","open_a":149.801728,"high_a":150.958776,"low_a":149.250744,"close_a":150.426163,"volume_a":3603300,"open_b":211.830725,"high_b":213.138253,"low_b":209.3048,"close_b":210.483566,"volume_b":70122700,"open_bench":525.367968,"close_bench":527.213501},{"date":"2024-06-17","open_a":150.132285,"high_a":152.813709,"low_a":149.489467,"close_a":152.565765,"volume_a":5412200,"open_b":211.355287,"high_b":216.882601,"low_b":210.71143,"close_b":214.62413,"volume_b":93728300,"open_bench":526.533556,"close_bench":531.409546},{"date":"2024-06-18","open_a":152.529048,"high_a":153.419796,"low_a":151.74849,"close_a":152.877991,"volume_a":3519500,"open_b":215.535445,"high_b":216.565633,"low_b":210.988789,"close_b":212.266602,"volume_b":79943300,"open_bench":531.467771,"close_bench":532.759644},{"date":"2024-06-20","open_a":152.189282,"high_a":153.943232,"low_a":152.189282,"close_a":153.061661,"volume_a":4942400,"open_b":211.910005,"high_b":212.21709,"low_b":206.877985,"close_b":207.700134,"volume_b":86172500,"open_bench":533.682464,"close_bench":531.312439},{"date":"2024-06-21","open_a":153.548367,"high_a":155.256399,"low_a":152.850453,"close_a":153.61264,"volume_a":10899100,"open_b":208.403403,"high_b":209.889239,"low_b":205.154376,"close_b":205.530792,"volume_b":241805100,"open_bench":530.492922,"close_bench":530.600098},{"date":"2024-06-24","open_a":153.814661,"high_a":155.752267,"low_a":153.438156,"close_a":154.347275,"volume_a":4476900,"open_b":205.75865,"high_b":210.691623,"low_b":204.639315,"close_b":206.174683,"volume_b":80727000,"open_bench":530.424714,"close_bench":528.875305},{"date":"2024-06-25","open_a":154.347278,"high_a":155.155384,"low_a":153.58509,"close_a":153.676926,"volume_a":4217300,"open_b":207.175148,"high_b":209.384103,"low_b":206.640254,"close_b":207.095917,"volume_b":55549700,"open_bench":530.093359,"close_bench":530.911926},{"date":"2024-06-26","open_a":152.336187,"high_a":153.364675,"low_a":151.381165,"close_a":153.116745,"volume_a":4778300,"open_b":209.502944,"high_b":212.831218,"low_b":208.651064,"close_b":211.23642,"volume_b":66213200,"open_bench":529.801071,"close_bench":531.574585},{"date":"2024-06-27","open_a":153.015728,"high_a":153.585068,"low_a":151.766845,"close_a":152.675949,"volume_a":4326500,"open_b":212.662825,"high_b":213.702914,"low_b":210.344924,"close_b":212.0784,"volume_b":49772700,"open_bench":531.438143,"close_bench":532.412598},{"date":"2024-06-28","open_a":151.794384,"high_a":152.639215,"low_a":151.169935,"close_a":151.454605,"volume_a":8755900,"open_b":213.732622,"high_b":214.029792,"low_b":208.31427,"close_b":208.631241,"volume_b":82542700,"open_bench":533.182342,"close_bench":530.317444},{"date":"2024-07-01","open_a":151.555661,"high_a":152.492328,"low_a":149.29666,"close_a":149.581329,"volume_a":4508000,"open_b":210.087352,"high_b":215.456172,"low_b":209.918959,"close_b":214.703354,"volume_b":60402900,"open_bench":531.691506,"close_bench":531.408936},{"date":"2024-07-02","open_a":150.141485,"high_a":151.032234,"low_a":149.057902,"close_a":150.214951,"volume_a":4650800,"open_b":214.109032,"high_b":218.299102,"low_b":213.068958,"close_b":218.19014,"volume_b":58046200,"open_bench":529.810818,"close_bench":534.985168},{"date":"2024-07-03","open_a":149.608865,"high_a":150.72001,"low_a":148.727308,"close_a":149.315018,"volume_a":2879600,"open_b":217.922686,"high_b":219.458054,"low_b":216.961844,"close_b":219.458054,"volume_b":37369800,"open_bench":534.673362,"close_bench":537.37262},{"date":"2024-07-05","open_a":149.351737,"high_a":151.078139,"low_a":149.195628,"close_a":150.958755,"volume_a":4208200,"open_b":219.557117,"high_b":224.311797,"low_b":219.557117,"close_b":224.202835,"volume_b":60412400,"open_bench":537.674635,"close_bench":540.471313},{"date":"2024-07-08","open_a":149.737438,"high_a":150.371065,"low_a":148.690581,"close_a":148.874237,"volume_a":5268800,"open_b":224.945736,"high_b":225.698569,"low_b":221.141998,"close_b":225.668854,"volume_b":59085900,"open_bench":541.250918,"close_bench":541.095032},{"date":"2024-07-09","open_a":148.938494,"high_a":149.278259,"low_a":148.304867,"close_a":148.67218,"volume_a":5801200,"open_b":225.777803,"high_b":227.233924,"low_b":224.232536,"close_b":226.520721,"volume_b":48076100,"open_bench":542.049978,"close_bench":541.621216},{"date":"2024-07-10","open_a":148.617102,"high_a":150.389422,"low_a":147.533519,"close_a":150.224121,"volume_a":7372300,"open_b":227.134876,"high_b":230.879183,"low_b":227.085345,"close_b":230.780121,"volume_b":62627700,"open_bench":542.839282,"close_bench":546.980713},{"date":"2024-07-11","open_a":146.100958,"high_a":151.417888,"low_a":145.118375,"close_a":150.554688,"volume_a":10258600,"open_b":229.205141,"high_b":230.195699,"low_b":223.638212,"close_b":225.421219,"volume_b":64710600,"open_bench":547.097658,"close_bench":542.264343},{"date":"2024-07-12","open_a":151.482229,"high_a":154.301393,"low_a":151.482229,"close_a":152.786209,"volume_a":6677000,"open_b":226.758445,"high_b":230.443321,"low_b":226.520706,"close_b":228.363144,"volume_b":53046500,"open_bench":543.384994,"close_bench":545.684692},{"date":"2024-07-15","open_a":152.446378,"high_a":152.712691,"low_a":150.279197,"close_a":150.472046,"volume_a":4551800,"open_b":234.247084,"high_b":234.990002,"low_b":230.889094,"close_b":232.186722,"volume_b":62631300,"open_bench":547.672469,"close_bench":547.185242},{"date":"2024-07-16","open_a":150.058846,"high_a":151.55566,"low_a":149.53541,"close_a":151.298538,"volume_a":4461700,"open_b":232.781031,"high_b":234.039043,"low_b":230.136244,"close_b":232.602737,"volume_b":43234300,"open_bench":548.491082,"close_bench":550.430237},{"date":"2024-07-17","open_a":152.473937,"high_a":156.174685,"low_a":152.336198,"close_a":156.009384,"volume_a":7671000,"open_b":227.283482,"high_b":229.274513,"low_b":224.500017,"close_b":226.718872,"volume_b":57345900,"open_bench":544.525056,"close_bench":542.712585},{"date":"2024-07-18","open_a":154.925818,"high_a":158.562265,"low_a":154.631957,"close_a":156.45018,"volume_a":6228600,"open_b":228.105594,"high_b":228.264087,"low_b":220.171233,"close_b":222.063187,"volume_b":66034600,"open_bench":544.242402,"close_bench":538.541809},{"date":"2024-07-19","open_a":156.367506,"high_a":156.560354,"low_a":154.705391,"close_a":155.522675,"volume_a":5332800,"open_b":222.697165,"high_b":224.658465,"low_b":221.171698,"close_b":222.191971,"volume_b":49151500,"open_bench":538.30807,"close_bench":534.965698},{"date":"2024-07-22","open_a":155.43086,"high_a":155.844104,"low_a":153.676924,"close_a":153.961594,"volume_a":5308400,"open_b":224.866495,"high_b":225.629229,"low_b":220.983511,"close_b":221.845306,"volume_b":48201800,"open_bench":538.873267,"close_bench":540.48114},{"date":"2024-07-23","open_a":154.080974,"high_a":154.60441,"low_a":152.602516,"close_a":152.694351,"volume_a":3344300,"open_b":222.25142,"high_b":224.79716,"low_b":220.577375,"close_b":222.885376,"volume_b":39960300,"open_bench":540.373897,"close_bench":539.633362},{"date":"2024-07-24","open_a":152.694367,"high_a":154.889095,"low_a":151.473031,"close_a":154.429947,"volume_a":5867000,"open_b":221.884907,"high_b":222.677356,"low_b":215.079781,"close_b":216.476456,"volume_b":61777600,"open_bench":534.838961,"close_bench":527.40387},{"date":"2024-07-25","open_a":155.642081,"high_a":159.315267,"low_a":155.06355,"close_a":157.047073,"volume_a":6270900,"open_b":216.862776,"high_b":218.76466,"low_b":212.593475,"close_b":215.436386,"volume_b":51391200,"open_bench":527.520843,"close_bench":524.655945},{"date":"2024-07-26","open_a":156.780734,"high_a":158.644874,"low_a":156.780734,"close_a":158.635696,"volume_a":5746300,"open_b":216.63495,"high_b":217.417499,"low_b":213.970348,"close_b":215.901947,"volume_b":41601300,"open_bench":528.427065,"close_bench":530.53186},{"date":"2024-07-29","open_a":158.213306,"high_a":159.315273,"low_a":156.890971,"close_a":159.058151,"volume_a":4152700,"open_b":214.911424,"high_b":217.229325,"low_b":213.712842,"close_b":216.179337,"volume_b":36311800,"open_bench":532.071572,"close_bench":530.84375},{"date":"2024-07-30","open_a":158.580622,"high_a":159.278523,"low_a":157.533765,"close_a":159.030579,"volume_a":4994600,"open_b":217.120326,"high_b":218.249561,"low_b":214.079307,"close_b":216.734009,"volume_b":41643800,"open_bench":532.305298,"close_bench":528.154114},{"date":"2024-07-31","open_a":159.710126,"high_a":160.417219,"low_a":158.185751,"close_a":158.562256,"volume_a":6245600,"open_b":219.349091,"high_b":221.706623,"low_b":218.546742,"close_b":219.983047,"volume_b":50036300,"open_bench":534.955929,"close_bench":536.739197},{"date":"2024-08-01","open_a":158.433693,"high_a":161.216131,"low_a":158.406146,"close_a":160.665161,"volume_a":5994500,"open_b":222.251418,"high_b":222.36038,"low_b":214.970828,"close_b":216.298172,"volume_b":62501000,"open_bench":538.454208,"close_bench":529.138428},{"date":"2024-08-02","open_a":164.228109,"high_a":165.247419,"low_a":159.636628,"close_a":163.493469,"volume_a":6856100,"open_b":217.080709,"high_b":223.469818,"low_b":215.654319,"close_b":217.784012,"volume_b":105568600,"open_bench":522.063976,"close_bench":519.286804},{"date":"2024-08-05","open_a":164.852603,"high_a":166.129035,"low_a":159.012236,"close_a":159.820328,"volume_a":7502300,"open_b":197.210121,"high_b":211.484061,"low_b":194.149302,"close_b":207.294006,"volume_b":119548600,"open_bench":498.569759,"close_bench":504.163116},{"date":"2024-08-06","open_a":159.122421,"high_a":161.813037,"low_a":158.27759,"close_a":158.396973,"volume_a":7287100,"open_b":203.361485,"high_b":208.007203,"low_b":199.171431,"close_b":205.273254,"volume_b":69660500,"open_bench":505.956198,"close_bench":508.811401},{"date":"2024-08-07","open_a":157.598042,"high_a":160.141727,"low_a":157.331742,"close_a":157.75415,"volume_a":5428900,"open_b":204.946365,"high_b":211.622728,"low_b":204.441186,"close_b":207.838806,"volume_b":63516400,"open_bench":514.969887,"close_bench":505.410492},{"date":"2024-08-08","open_a":156.303249,"high_a":158.86529,"low_a":155.935922,"close_a":158.286758,"volume_a":5112500,"open_b":211.097714,"high_b":212.177418,"low_b":206.858129,"close_b":211.295822,"volume_b":47161100,"open_bench":510.526306,"close_bench":517.094177},{"date":"2024-08-09","open_a":158.029653,"high_a":158.66328,"low_a":156.468552,"close_a":158.305145,"volume_a":3223000,"open_b":210.097318,"high_b":214.733121,"low_b":209.968541,"close_b":214.198227,"volume_b":42201600,"open_bench":516.275571,"close_bench":519.374329},{"date":"2024-08-12","open_a":158.167393,"high_a":158.74591,"low_a":156.7624,"close_a":157.414383,"volume_a":3802300,"open_b":214.277523,"high_b":217.688973,"low_b":213.811421,"close_b":215.725403,"volume_b":38028100,"open_bench":520.563267,"close_bench":519.647278},{"date":"2024-08-13","open_a":157.598036,"high_a":158.332676,"low_a":156.312427,"close_a":158.286758,"volume_a":3894700,"open_b":217.193141,"high_b":220.049254,"low_b":217.193141,"close_b":219.434402,"volume_b":44155300,"open_bench":522.823983,"close_bench":528.193176},{"date":"2024-08-14","open_a":158.158214,"high_a":160.095821,"low_a":157.882723,"close_a":159.517303,"volume_a":3555800,"open_b":218.740177,"high_b":221.17976,"low_b":217.877384,"close_b":219.88063,"volume_b":41960600,"open_bench":528.982525,"close_bench":529.859558},{"date":"2024-08-15","open_a":159.517274,"high_a":160.059066,"low_a":157.790873,"close_a":158.4245,"volume_a":4472100,"open_b":222.736759,"high_b":223.480537,"low_b":220.912012,"close_b":222.855759,"volume_b":46414000,"open_bench":535.462598,"close_bench":538.941406},{"date":"2024-08-16","open_a":158.424506,"high_a":158.663258,"low_a":157.129704,"close_a":158.516327,"volume_a":4614600,"open_b":222.062409,"high_b":224.948272,"low_b":221.794644,"close_b":224.174744,"volume_b":44340200,"open_bench":537.333654,"close_bench":540.149841},{"date":"2024-08-19","open_a":158.553089,"high_a":159.765233,"low_a":158.351063,"close_a":159.618317,"volume_a":3521700,"open_b":223.847464,"high_b":224.115229,"low_b":221.189689,"close_b":224.016052,"volume_b":40687800,"open_bench":540.55899,"close_bench":545.314331},{"date":"2024-08-20","open_a":159.351998,"high_a":161.647739,"low_a":158.406153,"close_a":161.482452,"volume_a":5330100,"open_b":223.897069,"high_b":225.285449,"low_b":223.579716,"close_b":224.63092,"volume_b":30299000,"open_bench":544.86619,"close_bench":544.427673},{"date":"2024-08-21","open_a":161.565082,"high_a":162.373187,"low_a":160.352924,"close_a":160.89473,"volume_a":4020700,"open_b":224.640865,"high_b":226.088745,"low_b":223.183059,"close_b":224.521851,"volume_b":34765500,"open_bench":545.470322,"close_bench":546.298584},{"date":"2024-08-22","open_a":161.464067,"high_a":161.693641,"low_a":160.426388,"close_a":161.381424,"volume_a":5265900,"open_b":225.900309,"high_b":226.445749,"low_b":222.04258,"close_b":222.667358,"volume_b":43695300,"open_bench":548.189053,"close_bench":542.010986},{"date":"2024-08-23","open_a":162.198708,"high_a":162.244611,"low_a":159.480544,"close_a":161.500793,"volume_a":5283600,"open_b":223.787957,"high_b":226.326717,"low_b":222.468988,"close_b":224.95816,"volume_b":38677300,"open_bench":545.236462,"close_bench":547.77002},{"date":"2024-08-26","open_a":161.914045,"high_a":162.39155,"low_a":161.37224,"close_a":161.592636,"volume_a":4146700,"open_b":224.878835,"high_b":225.394526,"low_b":222.032649,"close_b":225.295349,"volume_b":30602200,"open_bench":548.793071,"close_bench":546.464111},{"date":"2024-08-27","open_a":161.904846,"high_a":162.428281,"low_a":160.261101,"close_a":160.977371,"volume_a":3520600,"open_b":224.125162,"high_b":226.951526,"low_b":223.02437,"close_b":226.138321,"volume_b":35934600,"open_bench":545.197411,"close_bench":547.214539},{"date":"2024-08-28","open_a":161.206957,"high_a":161.381435,"low_a":159.186707,"close_a":159.710129,"volume_a":3915500,"open_b":226.029198,"high_b":227.953107,"low_b":223.807776,"close_b":224.611069,"volume_b":38052200,"open_bench":546.873476,"close_bench":544.037781},{"date":"2024-08-29","open_a":159.563201,"high_a":159.746872,"low_a":157.864361,"close_a":158.029648,"volume_a":4654300,"open_b":228.191154,"high_b":230.987752,"low_b":226.981273,"close_b":227.883713,"volume_b":51906300,"open_bench":545.9965,"close_bench":544.086548},{"date":"2024-08-30","open_a":158.002102,"high_a":159.379547,"low_a":157.258284,"close_a":158.755112,"volume_a":8677300,"open_b":228.280381,"high_b":228.48863,"low_b":225.592856,"close_b":227.10025,"volume_b":52990800,"open_bench":546.44476,"close_bench":549.280396},{"date":"2024-09-03","open_a":158.295963,"high_a":163.355772,"low_a":157.836815,"close_a":163.034363,"volume_a":7032100,"open_b":226.653985,"high_b":227.100249,"low_b":219.335203,"close_b":220.921936,"volume_b":50190600,"open_bench":546.152434,"close_bench":537.976807},{"date":"2024-09-04","open_a":163.364957,"high_a":164.971977,"low_a":161.886499,"close_a":163.934311,"volume_a":6188500,"open_b":219.821161,"high_b":219.940161,"low_b":215.67583,"close_b":219.017883,"volume_b":43840200,"open_bench":536.144709,"close_bench":536.875549},{"date":"2024-09-05","open_a":164.127138,"high_a":165.045435,"low_a":163.016007,"close_a":164.650574,"volume_a":5743100,"open_b":219.791409,"high_b":223.609461,"low_b":219.682321,"close_b":220.535187,"volume_b":36615400,"open_bench":536.817093,"close_bench":535.569763},{"date":"2024-09-06","open_a":164.183279,"high_a":165.238107,"low_a":163.776149,"close_a":164.090744,"volume_a":6395300,"open_b":222.092137,"high_b":223.371444,"low_b":217.946821,"close_b":218.988113,"volume_b":48423000,"open_bench":535.891442,"close_bench":526.556152},{"date":"2024-09-09","open_a":163.822429,"high_a":165.432423,"low_a":162.721333,"close_a":164.877258,"volume_a":4666200,"open_b":218.988128,"high_b":219.434391,"low_b":214.912223,"close_b":219.077377,"volume_b":67180000,"open_bench":530.736549,"close_bench":532.451538},{"date":"2024-09-10","open_a":165.302874,"high_a":165.719265,"low_a":164.368337,"close_a":164.525635,"volume_a":4315800,"open_b":217.103881,"high_b":219.642642,"low_b":214.932047,"close_b":218.284012,"volume_b":51591000,"open_bench":534.351744,"close_bench":534.770752},{"date":"2024-09-11","open_a":164.183292,"high_a":164.470129,"low_a":161.398162,"close_a":162.647308,"volume_a":4036000,"open_b":219.622824,"high_b":221.239292,"low_b":216.082433,"close_b":220.812866,"volume_b":44587100,"open_bench":534.683047,"close_bench":540.256897},{"date":"2024-09-12","open_a":162.647301,"high_a":164.414606,"low_a":160.95402,"close_a":164.007477,"volume_a":4222800,"open_b":220.654172,"high_b":221.695464,"low_b":217.996412,"close_b":220.921936,"volume_b":37455600,"open_bench":540.831888,"close_bench":544.807678},{"date":"2024-09-13","open_a":164.192526,"high_a":164.664432,"low_a":163.193212,"close_a":164.109253,"volume_a":3290400,"open_b":221.725217,"high_b":222.181393,"low_b":220.069073,"close_b":220.654175,"volume_b":36766600,"open_bench":545.411782,"close_bench":547.653015},{"date":"2024-09-16","open_a":165.626766,"high_a":166.024634,"low_a":163.174755,"close_a":163.970505,"volume_a":4940100,"open_b":214.743628,"high_b":215.417995,"low_b":212.145368,"close_b":214.525467,"volume_b":59357400,"open_bench":547.389979,"close_bench":548.461914},{"date":"2024-09-17","open_a":163.776145,"high_a":164.784707,"low_a":163.128446,"close_a":163.655853,"volume_a":6305100,"open_b":213.960181,"high_b":215.100635,"low_b":212.720551,"close_b":214.991547,"volume_b":45519300,"open_bench":550.664024,"close_bench":548.685913},{"date":"2024-09-18","open_a":163.646625,"high_a":164.756969,"low_a":161.398165,"close_a":162.119904,"volume_a":5307900,"open_b":215.745272,"high_b":220.862469,"low_b":215.735345,"close_b":218.859222,"volume_b":59894900,"open_bench":549.338906,"close_bench":547.058716},{"date":"2024-09-19","open_a":162.619539,"high_a":163.054426,"low_a":161.139071,"close_a":161.610977,"volume_a":5504000,"open_b":223.123512,"high_b":227.913444,"low_b":222.766498,"close_b":226.971313,"volume_b":66781300,"open_bench":556.423122,"close_bench":556.39386},{"date":"2024-09-20","open_a":159.677157,"high_a":160.787501,"low_a":157.502722,"close_a":158.390991,"volume_a":12198400,"open_b":228.062221,"high_b":231.156333,"low_b":225.73171,"close_b":226.3069,"volume_b":318679900,"open_bench":555.031342,"close_bench":555.432068},{"date":"2024-09-23","open_a":158.104147,"high_a":160.944791,"low_a":157.974607,"close_a":159.251511,"volume_a":5350300,"open_b":225.454041,"high_b":227.546537,"low_b":223.936734,"close_b":224.591263,"volume_b":54146000,"open_bench":556.497616,"close_bench":556.820129},{"date":"2024-09-24","open_a":158.215179,"high_a":158.779606,"low_a":156.836493,"close_a":157.225113,"volume_a":7837900,"open_b":226.75316,"high_b":227.447365,"low_b":223.857386,"close_b":225.48378,"volume_b":43556100,"open_bench":557.611758,"close_bench":558.413269},{"date":"2024-09-25","open_a":157.669247,"high_a":158.011613,"low_a":156.151773,"close_a":156.743958,"volume_a":4505100,"open_b":223.064009,"high_b":225.404431,"low_b":222.16157,"close_b":224.492065,"volume_b":42308700,"open_bench":558.256925,"close_bench":557.181702},{"date":"2024-09-26","open_a":156.577412,"high_a":158.224425,"low_a":156.077748,"close_a":157.012299,"volume_a":5673600,"open_b":225.414362,"high_b":226.604404,"low_b":223.540041,"close_b":225.632538,"volume_b":36636700,"open_bench":561.423846,"close_bench":559.390747},{"date":"2024-09-27","open_a":157.262158,"high_a":158.825885,"low_a":157.003064,"close_a":157.299164,"volume_a":7545800,"open_b":226.564754,"high_b":227.615958,"low_b":225.414374,"close_b":225.900299,"volume_b":34026000,"open_bench":560.456141,"close_bench":558.579407},{"date":"2024-09-30","open_a":157.835815,"high_a":158.909154,"low_a":156.910526,"close_a":157.345413,"volume_a":5054300,"open_b":228.131611,"high_b":231.067062,"low_b":227.744847,"close_b":231.067062,"volume_b":54541900,"open_bench":557.553184,"close_bench":560.817871},{"date":"2024-10-01","open_a":158.492748,"high_a":159.094194,"low_a":157.086319,"close_a":158.576035,"volume_a":7095900,"open_b":227.615958,"high_b":227.74487,"low_b":221.883909,"close_b":224.33342,"volume_b":63285000,"open_bench":560.465969,"close_bench":555.793762},{"date":"2024-10-02","open_a":158.687078,"high_a":158.909152,"low_a":155.92972,"close_a":156.882767,"volume_a":5786000,"open_b":224.016065,"high_b":225.483783,"low_b":221.169879,"close_b":224.898682,"volume_b":32880600,"open_bench":554.904357,"close_bench":556.028381},{"date":"2024-10-03","open_a":155.578103,"high_a":156.475634,"low_a":154.680571,"close_a":155.938965,"volume_a":5090300,"open_b":223.272265,"high_b":224.928409,"low_b":221.467371,"close_b":223.797867,"volume_b":34044200,"open_bench":554.562256,"close_bench":555.011902},{"date":"2024-10-04","open_a":154.662063,"high_a":155.911209,"low_a":154.35673,"close_a":155.420807,"volume_a":4390100,"open_b":226.009384,"high_b":226.10856,"low_b":222.27067,"close_b":224.918518,"volume_b":37245100,"open_bench":559.439565,"close_bench":560.055359},{"date":"2024-10-07","open_a":155.189498,"high_a":155.383801,"low_a":154.023639,"close_a":154.717606,"volume_a":5459500,"open_b":222.637593,"high_b":223.817723,"low_b":219.493892,"close_b":219.850906,"volume_b":39505400,"open_bench":558.413361,"close_bench":554.99231},{"date":"2024-10-08","open_a":155.189482,"high_a":157.900572,"low_a":154.384485,"close_a":157.687759,"volume_a":8297800,"open_b":222.439243,"high_b":224.105298,"low_b":221.39795,"close_b":223.897049,"volume_b":31855700,"open_bench":557.55312,"close_bench":560.241089},{"date":"2024-10-09","open_a":156.725496,"high_a":160.334118,"low_a":156.466416,"close_a":159.649399,"volume_a":7177500,"open_b":223.36153,"high_b":227.844037,"low_b":222.964854,"close_b":227.635773,"volume_b":33591100,"open_bench":560.231257,"close_bench":564.121521},{"date":"2024-10-10","open_a":159.214516,"high_a":160.130544,"low_a":158.428015,"close_a":160.065781,"volume_a":5292400,"open_b":225.890381,"high_b":227.596114,"low_b":225.285441,"close_b":227.139923,"volume_b":28183500,"open_bench":562.782412,"close_bench":563.134277},{"date":"2024-10-11","open_a":161.000289,"high_a":162.092124,"low_a":159.954708,"close_a":161.749771,"volume_a":4720300,"open_b":227.39779,"high_b":227.506878,"low_b":225.454043,"close_b":225.662308,"volume_b":31759200,"open_bench":563.056067,"close_bench":566.50647},{"date":"2024-10-14","open_a":161.888576,"high_a":163.100703,"low_a":161.407422,"close_a":162.943405,"volume_a":4535900,"open_b":226.802773,"high_b":229.807636,"low_b":226.703612,"close_b":229.38121,"volume_b":39882100,"open_bench":568.109539,"close_bench":571.139648},{"date":"2024-10-15","open_a":163.183986,"high_a":164.137032,"low_a":162.036622,"close_a":162.758347,"volume_a":5802300,"open_b":231.67202,"high_b":235.519837,"low_b":230.442301,"close_b":231.910034,"volume_b":64751400,"open_bench":571.403587,"close_bench":566.702087},{"date":"2024-10-16","open_a":161.934845,"high_a":162.230944,"low_a":160.657955,"close_a":161.444443,"volume_a":4400900,"open_b":229.678694,"high_b":230.194369,"low_b":227.933284,"close_b":229.857193,"volume_b":34082200,"open_bench":566.702045,"close_bench":569.165161},{"date":"2024-10-17","open_a":161.351912,"high_a":162.360473,"low_a":160.722708,"close_a":161.620239,"volume_a":4125900,"open_b":231.493501,"high_b":231.91003,"low_b":228.607654,"close_b":230.224121,"volume_b":32993800,"open_bench":572.693746,"close_bench":569.21405},{"date":"2024-10-18","open_a":159.427317,"high_a":162.230942,"low_a":159.260757,"close_a":161.98111,"volume_a":5097000,"open_b":234.22068,"high_b":234.22068,"low_b":232.068684,"close_b":233.050476,"volume_b":46431500,"open_bench":570.895275,"close_bench":571.403564},{"date":"2024-10-21","open_a":161.425917,"high_a":164.238789,"low_a":161.324134,"close_a":161.934814,"volume_a":6436300,"open_b":232.50503,"high_b":234.885129,"low_b":232.50503,"close_b":234.518188,"volume_b":36254500,"open_bench":570.680219,"close_bench":570.46521},{"date":"2024-10-22","open_a":161.472176,"high_a":161.879305,"low_a":160.546887,"close_a":161.342636,"volume_a":3790500,"open_b":231.949692,"high_b":234.260365,"low_b":230.670401,"close_b":233.903351,"volume_b":38846600,"open_bench":567.943355,"close_bench":570.16217},{"date":"2024-10-23","open_a":160.722714,"high_a":160.852254,"low_a":159.584613,"close_a":160.028748,"volume_a":3800700,"open_b":232.138107,"high_b":233.189311,"low_b":225.87053,"close_b":228.845642,"volume_b":52287000,"open_bench":568.14859,"close_bench":564.952332},{"date":"2024-10-24","open_a":160.00099,"high_a":160.574665,"low_a":158.872136,"close_a":159.297775,"volume_a":4244900,"open_b":228.072155,"high_b":228.905198,"low_b":226.515187,"close_b":228.657272,"volume_b":31109500,"open_bench":566.897554,"close_bench":566.174255},{"date":"2024-10-25","open_a":159.584587,"high_a":161.000278,"low_a":158.77959,"close_a":158.955383,"volume_a":4442600,"open_b":227.834136,"high_b":231.285262,"low_b":227.665548,"close_b":229.49028,"volume_b":38802300,"open_bench":568.392953,"close_bench":565.978638},{"date":"2024-10-28","open_a":160.056484,"high_a":160.056484,"low_a":157.049295,"close_a":157.151077,"volume_a":5186400,"open_b":231.384453,"high_b":232.782744,"low_b":230.620836,"close_b":231.463776,"volume_b":36087100,"open_bench":569.438797,"close_bench":567.728271},{"date":"2024-10-29","open_a":156.558889,"high_a":156.558889,"low_a":154.495499,"close_a":154.985901,"volume_a":5127900,"open_b":231.166273,"high_b":232.386065,"low_b":230.392745,"close_b":231.731537,"volume_b":35417200,"open_bench":566.770483,"close_bench":568.647217},{"date":"2024-10-30","open_a":154.523285,"high_a":155.115469,"low_a":153.672021,"close_a":153.792313,"volume_a":4801500,"open_b":230.680334,"high_b":231.533201,"low_b":227.645722,"close_b":228.191162,"volume_b":47070900,"open_bench":568.177976,"close_bench":566.92688},{"date":"2024-10-31","open_a":153.690537,"high_a":154.985937,"low_a":153.422196,"close_a":153.672028,"volume_a":5582400,"open_b":227.437458,"high_b":227.923398,"low_b":223.50039,"close_b":224.035919,"volume_b":64370100,"open_bench":562.577183,"close_bench":555.813293},{"date":"2024-11-01","open_a":153.681273,"high_a":153.829322,"low_a":152.867028,"close_a":153.218628,"volume_a":6339000,"open_b":219.136867,"high_b":223.480536,"low_b":218.442677,"close_b":221.060776,"volume_b":65276700,"open_bench":558.432892,"close_bench":558.15918},{"date":"2024-11-04","open_a":153.218631,"high_a":154.199435,"low_a":152.848521,"close_a":153.912598,"volume_a":4885500,"open_b":219.156731,"high_b":220.941787,"low_b":217.887351,"close_b":220.168259,"volume_b":44944500,"open_bench":558.296002,"close_bench":556.956909},{"date":"2024-11-05","open_a":153.551711,"high_a":155.411551,"low_a":153.292631,"close_a":155.309769,"volume_a":3678300,"open_b":219.960008,"high_b":222.092166,"low_b":219.305479,"close_b":221.596313,"volume_b":28111300,"open_bench":557.865945,"close_bench":563.691528},{"date":"2024-11-06","open_a":154.865662,"high_a":154.902682,"low_a":151.377332,"close_a":152.404404,"volume_a":7849300,"open_b":220.763257,"high_b":224.19456,"low_b":219.355039,"close_b":220.872345,"volume_b":54561100,"open_bench":575.909546,"close_bench":577.708008},{"date":"2024-11-07","open_a":152.210051,"high_a":152.580161,"low_a":151.257004,"close_a":151.747406,"volume_a":5333500,"open_b":222.766517,"high_b":225.989556,"low_b":222.707017,"close_b":225.592865,"volume_b":42137700,"open_bench":579.701966,"close_bench":582.174866},{"date":"2024-11-08","open_a":151.599354,"high_a":153.431437,"low_a":151.423547,"close_a":152.774475,"volume_a":4618300,"open_b":225.533308,"high_b":227.012578,"low_b":224.778789,"close_b":225.324829,"volume_b":38328800,"open_bench":582.722323,"close_bench":584.696777},{"date":"2024-11-11","open_a":152.126793,"high_a":153.736787,"low_a":151.728911,"close_a":151.987991,"volume_a":3762000,"open_b":223.378928,"high_b":224.073882,"low_b":219.904145,"close_b":222.614471,"volume_b":42005600,"open_bench":586.28021,"close_bench":585.253906},{"date":"2024-11-12","open_a":151.830674,"high_a":152.672691,"low_a":150.9609,"close_a":152.061996,"volume_a":5386600,"open_b":222.932173,"high_b":223.964674,"low_b":221.750744,"close_b":222.614471,"volume_b":40398300,"open_bench":585.17567,"close_bench":583.435852},{"date":"2024-11-13","open_a":151.275502,"high_a":152.487629,"low_a":150.239169,"close_a":152.432114,"volume_a":4145000,"open_b":222.396059,"high_b":225.017038,"low_b":221.155065,"close_b":223.498062,"volume_b":48566200,"open_bench":583.895232,"close_bench":583.719299},{"date":"2024-11-14","open_a":152.432135,"high_a":154.421501,"low_a":151.590119,"close_a":152.811493,"volume_a":5422700,"open_b":223.398803,"high_b":227.221056,"low_b":223.378943,"close_b":226.575745,"volume_b":44923900,"open_bench":583.846362,"close_bench":579.965881},{"date":"2024-11-15","open_a":152.099022,"high_a":152.099022,"low_a":146.241941,"close_a":146.769348,"volume_a":9108900,"open_b":224.768844,"high_b":225.285102,"low_b":222.6542,"close_b":223.378937,"volume_b":47923700,"open_bench":576.417836,"close_bench":572.537415},{"date":"2024-11-18","open_a":147.500336,"high_a":147.907466,"low_a":145.973601,"close_a":146.501022,"volume_a":6991300,"open_b":223.62712,"high_b":228.084776,"low_b":223.547694,"close_b":226.377167,"volume_b":44633700,"open_bench":572.996661,"close_bench":574.883179},{"date":"2024-11-19","open_a":145.964353,"high_a":146.075384,"low_a":144.20631,"close_a":145.011307,"volume_a":8248900,"open_b":225.344678,"high_b":228.501775,"low_b":225.026991,"close_b":226.635315,"volume_b":36211800,"open_bench":571.520806,"close_bench":576.98468},{"date":"2024-11-20","open_a":144.761482,"high_a":147.222754,"low_a":144.530159,"close_a":146.880402,"volume_a":5067300,"open_b":226.416883,"high_b":228.273405,"low_b":224.262519,"close_b":227.350113,"volume_b":35169600,"open_bench":577.062948,"close_bench":577.180237},{"date":"2024-11-21","open_a":146.556544,"high_a":148.508905,"low_a":146.149415,"close_a":148.360855,"volume_a":5977000,"open_b":227.230988,"high_b":228.501765,"low_b":224.083829,"close_b":226.873581,"volume_b":42108300,"open_bench":580.014817,"close_bench":580.278687},{"date":"2024-11-22","open_a":148.879012,"high_a":149.95235,"low_a":147.944475,"close_a":149.896835,"volume_a":5846300,"open_b":226.416895,"high_b":229.057734,"low_b":226.416895,"close_b":228.213852,"volume_b":38168300,"open_bench":580.268965,"close_bench":582.077271},{"date":"2024-11-25","open_a":150.544521,"high_a":151.368028,"low_a":149.896821,"close_a":150.868378,"volume_a":8580400,"open_b":229.792385,"high_b":231.569482,"low_b":228.084776,"close_b":231.192215,"volume_b":90152800,"open_bench":585.996799,"close_bench":584.051697},{"date":"2024-11-26","open_a":151.071932,"high_a":151.45129,"low_a":149.804277,"close_a":150.044861,"volume_a":4653600,"open_b":231.648923,"high_b":233.87279,"low_b":231.648923,"close_b":233.366455,"volume_b":45986200,"open_bench":585.292952,"close_bench":587.101257},{"date":"2024-11-27","open_a":149.841325,"high_a":152.793,"low_a":149.600741,"close_a":150.563049,"volume_a":3622900,"open_b":232.780704,"high_b":233.991915,"low_b":232.125455,"close_b":233.237381,"volume_b":33498400,"open_bench":586.915564,"close_bench":585.322327},{"date":"2024-11-29","open_a":149.369407,"high_a":151.368035,"low_a":148.925272,"close_a":151.238495,"volume_a":2799300,"open_b":233.118254,"high_b":236.096639,"low_b":232.284309,"close_b":235.620102,"volume_b":28481400,"open_bench":586.13361,"close_bench":588.958435},{"date":"2024-12-02","open_a":151.11821,"high_a":151.423543,"low_a":149.674747,"close_a":150.868378,"volume_a":4967400,"open_b":235.560523,"high_b":239.055151,"low_b":235.451315,"close_b":237.8638,"volume_b":48137100,"open_bench":589.368892,"close_bench":590.014038},{"date":"2024-12-03","open_a":150.674076,"high_a":150.868393,"low_a":149.360168,"close_a":149.619247,"volume_a":4277400,"open_b":238.082229,"high_b":241.010972,"low_b":237.178782,"close_b":240.901764,"volume_b":38861000,"open_bench":589.779429,"close_bench":590.287659},{"date":"2024-12-04","open_a":148.601448,"high_a":148.999316,"low_a":147.500352,"close_a":148.212814,"volume_a":6964600,"open_b":241.120195,"high_b":242.351267,"low_b":239.511871,"close_b":241.259186,"volume_b":44383900,"open_bench":591.968946,"close_bench":593.953125},{"date":"2024-12-05","open_a":148.906764,"high_a":148.980789,"low_a":146.963666,"close_a":148.499649,"volume_a":5229600,"open_b":242.232108,"high_b":242.778133,"low_b":240.385508,"close_b":241.28894,"volume_b":40033900,"open_bench":593.95309,"close_bench":592.975647},{"date":"2024-12-06","open_a":148.700276,"high_a":150.053366,"low_a":147.151214,"close_a":147.244522,"volume_a":8359200,"open_b":241.159896,"high_b":242.867505,"low_b":240.335874,"close_b":241.090393,"volume_b":36870600,"open_bench":593.738082,"close_bench":594.099731},{"date":"2024-12-09","open_a":147.356486,"high_a":150.706554,"low_a":146.637943,"close_a":148.812225,"volume_a":8380400,"open_b":240.087663,"high_b":245.458689,"low_b":240.008238,"close_b":244.972214,"volume_b":44649200,"open_bench":593.982519,"close_bench":591.040405},{"date":"2024-12-10","open_a":148.373659,"high_a":149.698755,"low_a":146.833938,"close_a":149.157516,"volume_a":5329400,"open_b":245.111214,"high_b":246.421711,"low_b":243.572378,"close_b":245.984879,"volume_b":36914800,"open_bench":591.714795,"close_bench":589.202759},{"date":"2024-12-11","open_a":149.502809,"high_a":149.782748,"low_a":145.816785,"close_a":146.292709,"volume_a":7925900,"open_b":246.173522,"high_b":248.993057,"low_b":244.485758,"close_b":244.714111,"volume_b":45205800,"open_bench":592.115532,"close_bench":593.757629},{"date":"2024-12-12","open_a":148.457639,"high_a":148.746918,"low_a":147.412495,"close_a":148.280334,"volume_a":6849500,"open_b":245.111225,"high_b":246.947902,"low_b":243.909936,"close_b":246.173523,"volume_b":32777500,"open_bench":592.897489,"close_bench":590.698242},{"date":"2024-12-13","open_a":147.00191,"high_a":148.90558,"low_a":146.56332,"close_a":147.412506,"volume_a":4353900,"open_b":246.034536,"high_b":247.493931,"low_b":244.465918,"close_b":246.3423,"volume_b":33155300,"open_bench":592.721597,"close_bench":590.580994},{"date":"2024-12-16","open_a":147.300518,"high_a":148.205691,"low_a":145.462176,"close_a":145.592819,"volume_a":6696500,"open_b":246.20331,"high_b":249.568885,"low_b":245.865748,"close_b":249.231323,"volume_b":51694800,"open_bench":592.330624,"close_bench":593.102783},{"date":"2024-12-17","open_a":145.331557,"high_a":147.907095,"low_a":144.864973,"close_a":145.704819,"volume_a":6572600,"open_b":248.27822,"high_b":252.001201,"low_b":247.980378,"close_b":251.653717,"volume_b":51356400,"open_bench":590.561337,"close_bench":590.659058},{"date":"2024-12-18","open_a":144.669007,"high_a":145.415544,"low_a":143.810494,"close_a":144.1091,"volume_a":7620900,"open_b":250.343252,"high_b":252.447973,"low_b":245.955099,"close_b":246.262863,"volume_b":56774100,"open_bench":590.356242,"close_bench":573.055542},{"date":"2024-12-19","open_a":142.858636,"high_a":143.782477,"low_a":141.290921,"close_a":141.346909,"volume_a":9829500,"open_b":245.716847,"high_b":250.184426,"low_b":245.309797,"close_b":247.990341,"volume_b":60882300,"open_bench":578.020755,"close_bench":572.879395},{"date":"2024-12-20","open_a":140.777698,"high_a":143.231934,"low_a":140.451083,"close_a":142.578705,"volume_a":19205200,"open_b":246.252952,"high_b":253.162814,"low_b":243.919892,"close_b":252.656494,"volume_b":147495300,"open_bench":570.561035,"close_bench":579.760315},{"date":"2024-12-23","open_a":142.224097,"high_a":142.224097,"low_a":139.704547,"close_a":141.17894,"volume_a":6822200,"open_b":252.934465,"high_b":253.808114,"low_b":251.623968,"close_b":253.430862,"volume_b":40858800,"open_bench":579.505283,"close_bench":583.232056},{"date":"2024-12-24","open_a":140.441751,"high_a":142.616034,"low_a":140.357769,"close_a":142.597366,"volume_a":3399800,"open_b":253.649301,"high_b":256.34969,"low_b":253.450729,"close_b":256.339783,"volume_b":23234700,"open_bench":584.575691,"close_bench":589.714722},{"date":"2024-12-26","open_a":142.336057,"high_a":142.94261,"low_a":142.065445,"close_a":142.252075,"volume_a":5528500,"open_b":256.329802,"high_b":258.226044,"low_b":255.773839,"close_b":257.153809,"volume_b":27237100,"open_bench":587.949392,"close_bench":589.753967},{"date":"2024-12-27","open_a":141.178953,"high_a":143.213267,"low_a":141.150959,"close_a":142.672028,"volume_a":4287000,"open_b":255.972387,"high_b":256.836144,"low_b":251.236764,"close_b":253.748535,"volume_b":42355300,"open_bench":586.027122,"close_bench":583.545898},{"date":"2024-12-30","open_a":142.457395,"high_a":142.858651,"low_a":140.992315,"close_a":141.580215,"volume_a":5397000,"open_b":250.412717,"high_b":251.673571,"low_b":248.943385,"close_b":250.382935,"volume_b":35557500,"open_bench":576.563059,"close_bench":576.886658},{"date":"2024-12-31","open_a":141.654858,"high_a":142.914626,"low_a":140.852333,"close_a":141.897476,"volume_a":4274800,"open_b":250.621234,"high_b":251.455179,"low_b":247.632911,"close_b":248.615784,"volume_b":39480700,"open_bench":578.544251,"close_bench":574.788086},{"date":"2025-01-02","open_a":142.662683,"high_a":143.390553,"low_a":139.872507,"close_a":140.171127,"volume_a":4970200,"open_b":247.136511,"high_b":247.305299,"low_b":240.077751,"close_b":242.093124,"volume_b":55740700,"open_bench":578.034214,"close_bench":573.375732},{"date":"2025-01-03","open_a":140.871006,"high_a":140.871006,"low_a":139.564563,"close_a":139.648544,"volume_a":5218100,"open_b":241.606659,"high_b":242.420743,"low_b":240.147249,"close_b":241.606659,"volume_b":40244100,"open_bench":576.210038,"close_bench":580.544861},{"date":"2025-01-06","open_a":139.601896,"high_a":139.928496,"low_a":136.335809,"close_a":136.494446,"volume_a":9470600,"open_b":242.549802,"high_b":245.548047,"low_b":241.447798,"close_b":243.234833,"volume_b":45045600,"open_bench":584.7816,"close_bench":583.889099},{"date":"2025-01-07","open_a":136.671741,"high_a":138.080819,"low_a":135.626584,"close_a":135.682571,"volume_a":7473300,"open_b":241.229392,"high_b":243.780883,"low_b":239.611146,"close_b":240.464951,"volume_b":40856000,"open_bench":585.909439,"close_bench":577.288818},{"date":"2025-01-08","open_a":135.682573,"high_a":136.8677,"low_a":134.898716,"close_a":136.746384,"volume_a":6791800,"open_b":240.177052,"high_b":241.954164,"low_b":238.32053,"close_b":240.951431,"volume_b":37628900,"open_bench":577.357506,"close_bench":578.132263},{"date":"2025-01-10","open_a":135.309317,"high_a":135.841229,"low_a":132.052556,"close_a":133.10704,"volume_a":9632700,"open_b":238.280792,"high_b":238.42972,"low_b":231.321302,"close_b":235.14357,"volume_b":61710900,"open_bench":574.591767,"close_bench":569.305603},{"date":"2025-01-13","open_a":133.247005,"high_a":135.439954,"low_a":132.640451,"close_a":134.842728,"volume_a":8101400,"open_b":231.84747,"high_b":232.979256,"low_b":228.064922,"close_b":232.711197,"volume_b":49630700,"open_bench":564.67658,"close_bench":570.188293},{"date":"2025-01-14","open_a":134.553462,"high_a":135.439969,"low_a":133.965562,"close_a":135.262665,"volume_a":6996400,"open_b":233.058699,"high_b":234.418824,"low_b":230.795127,"close_b":231.599289,"volume_b":39435300,"open_bench":573.101074,"close_bench":570.9729},{"date":"2025-01-15","open_a":135.766575,"high_a":136.429116,"low_a":134.534786,"close_a":135.104019,"volume_a":7141600,"open_b":232.94948,"high_b":237.238363,"low_b":232.740986,"close_b":236.156204,"volume_b":39832000,"open_bench":578.956079,"close_bench":581.358887},{"date":"2025-01-16","open_a":134.422805,"high_a":136.979676,"low_a":134.086864,"close_a":136.746384,"volume_a":5068000,"open_b":235.639953,"high_b":236.295186,"low_b":226.387094,"close_b":226.615433,"volume_b":71759100,"open_bench":582.722068,"close_bench":580.240845},{"date":"2025-01-17","open_a":137.847555,"high_a":138.6874,"low_a":137.33431,"close_a":138.342133,"volume_a":7338900,"open_b":230.447625,"high_b":230.616398,"low_b":226.833851,"close_b":228.323044,"volume_b":68488300,"open_bench":585.458357,"close_bench":586.066406},{"date":"2025-01-21","open_a":138.416772,"high_a":139.769863,"low_a":137.894201,"close_a":138.62207,"volume_a":7456200,"open_b":222.386121,"high_b":222.803093,"low_b":217.799412,"close_b":221.035919,"volume_b":98070400,"open_bench":589.09682,"close_bench":591.430969},{"date":"2025-01-22","open_a":138.323453,"high_a":139.461919,"low_a":137.912857,"close_a":138.19281,"volume_a":5905100,"open_b":218.206475,"high_b":222.505281,"low_b":218.206475,"close_b":222.217377,"volume_b":64126500,"open_bench":594.245737,"close_bench":594.755737},{"date":"2025-01-23","open_a":138.566076,"high_a":138.985998,"low_a":136.811716,"close_a":138.687378,"volume_a":4759100,"open_b":223.120805,"high_b":225.394299,"low_b":220.698382,"close_b":222.048584,"volume_b":60234800,"open_bench":594.127985,"close_bench":598.001892},{"date":"2025-01-24","open_a":138.874022,"high_a":139.331265,"low_a":138.230134,"close_a":139.153961,"volume_a":4509600,"open_b":223.160533,"high_b":224.004415,"low_b":219.814817,"close_b":221.174942,"volume_b":54697900,"open_bench":598.060739,"close_bench":596.256165},{"date":"2025-01-27","open_a":140.833678,"high_a":145.079594,"low_a":140.81501,"close_a":144.277069,"volume_a":9820600,"open_b":222.406013,"high_b":230.477428,"low_b":222.366292,"close_b":228.203934,"volume_b":94863400,"open_bench":583.34976,"close_bench":587.821899},{"date":"2025-01-28","open_a":143.847783,"high_a":144.277046,"low_a":140.087119,"close_a":140.208435,"volume_a":6349700,"open_b":229.186809,"high_b":238.459514,"low_b":229.147089,"close_b":236.543411,"volume_b":75707600,"open_bench":589.047798,"close_bench":592.872681},{"date":"2025-01-29","open_a":141.029651,"high_a":141.841503,"low_a":140.189792,"close_a":140.320435,"volume_a":4874100,"open_b":232.433215,"high_b":238.131865,"low_b":232.324007,"close_b":237.635468,"volume_b":45486100,"open_bench":592.088017,"close_bench":590.214844},{"date":"2025-01-30","open_a":141.169625,"high_a":142.84933,"low_a":140.563071,"close_a":141.748184,"volume_a":4671300,"open_b":236.950456,"high_b":239.055177,"low_b":235.500983,"close_b":235.878235,"volume_b":55658300,"open_bench":592.323481,"close_bench":593.382629},{"date":"2025-01-31","open_a":140.684363,"high_a":141.552217,"low_a":140.22712,"close_a":140.619049,"volume_a":4905500,"open_b":245.409064,"high_b":245.409064,"low_b":231.758129,"close_b":234.299683,"volume_b":100959800,"open_bench":595.795226,"close_bench":590.22467},{"date":"2025-02-03","open_a":139.751172,"high_a":141.029621,"low_a":138.883333,"close_a":140.227097,"volume_a":7196200,"open_b":228.332994,"high_b":230.159734,"low_b":224.073894,"close_b":226.367249,"volume_b":73063300,"open_bench":581.251033,"close_bench":586.252808},{"date":"2025-02-04","open_a":136.335806,"high_a":137.875527,"low_a":133.442981,"close_a":133.900238,"volume_a":12804900,"open_b":225.612725,"high_b":231.450366,"low_b":225.017042,"close_b":231.122742,"volume_b":45067300,"open_bench":586.311579,"close_bench":590.185486},{"date":"2025-02-05","open_a":133.806922,"high_a":136.046519,"low_a":133.163034,"close_a":135.925217,"volume_a":9248100,"open_b":226.883504,"high_b":230.993676,"low_b":226.625383,"close_b":230.79512,"volume_b":39620300,"open_bench":589.067389,"close_bench":592.578369},{"date":"2025-02-06","open_a":136.485097,"high_a":137.082324,"low_a":134.338823,"close_a":135.635925,"volume_a":5604900,"open_b":229.623601,"high_b":232.115526,"low_b":228.769796,"close_b":231.539703,"volume_b":29925300,"open_bench":594.314403,"close_bench":594.638062},{"date":"2025-02-07","open_a":135.318629,"high_a":135.449272,"low_a":134.282812,"close_a":134.917374,"volume_a":5872300,"open_b":230.924199,"high_b":232.314107,"low_b":225.622661,"close_b":225.990005,"volume_b":39707200,"open_bench":595.197027,"close_bench":589.194946},{"date":"2025-02-10","open_a":134.702759,"high_a":134.861396,"low_a":133.452303,"close_a":134.702759,"volume_a":8715400,"open_b":228.166607,"high_b":229.180361,"low_b":225.811085,"close_b":226.258331,"volume_b":33115600,"open_bench":592.392141,"close_bench":593.196289},{"date":"2025-02-11","open_a":134.721413,"high_a":135.981195,"low_a":133.816239,"close_a":135.90654,"volume_a":6354000,"open_b":226.804974,"high_b":233.791998,"low_b":226.73541,"close_b":231.197952,"volume_b":53718400,"open_bench":590.940628,"close_bench":593.647461},{"date":"2025-02-12","open_a":134.413468,"high_a":135.104018,"low_a":133.470974,"close_a":134.366821,"volume_a":6658200,"open_b":229.786615,"high_b":235.511413,"low_b":229.26979,"close_b":235.421951,"volume_b":45243300,"open_bench":587.655163,"close_bench":591.734985},{"date":"2025-02-13","open_a":134.590759,"high_a":135.253315,"low_a":134.002873,"close_a":134.917374,"volume_a":7175300,"open_b":235.461745,"high_b":240.858543,"low_b":234.12994,"close_b":240.053497,"volume_b":53614100,"open_bench":592.833452,"close_bench":597.9823},{"date":"2025-02-14","open_a":135.001353,"high_a":136.401105,"low_a":133.750912,"close_a":133.8069,"volume_a":5001800,"open_b":239.775179,"high_b":244.048895,"low_b":239.516774,"close_b":243.104706,"volume_b":40896200,"open_bench":598.188186,"close_bench":597.95282},{"date":"2025-02-18","open_a":133.349655,"high_a":134.516114,"low_a":132.136548,"close_a":134.469467,"volume_a":8738700,"open_b":242.657459,"high_b":243.681161,"low_b":240.361582,"close_b":242.97551,"volume_b":48822500,"open_bench":599.110141,"close_bench":599.708374},{"date":"2025-02-19","open_a":134.590779,"high_a":136.177161,"low_a":134.516124,"close_a":136.065186,"volume_a":5777300,"open_b":243.164354,"high_b":244.506093,"low_b":241.673524,"close_b":243.373062,"volume_b":32204200,"open_bench":598.325541,"close_bench":601.120605},{"date":"2025-02-20","open_a":135.411956,"high_a":139.219281,"low_a":135.309307,"close_a":139.1353,"volume_a":6789800,"open_b":243.442651,"high_b":245.271399,"low_b":242.796615,"close_b":244.327209,"volume_b":32316900,"open_bench":599.757497,"close_bench":598.619873},{"date":"2025-02-21","open_a":139.965828,"high_a":144.22107,"low_a":138.948664,"close_a":143.241241,"volume_a":10604400,"open_b":244.446438,"high_b":247.169693,"low_b":243.720905,"close_b":244.048889,"volume_b":53197400,"open_bench":598.404044,"close_bench":588.380981},{"date":"2025-02-24","open_a":142.158748,"high_a":145.872766,"low_a":141.95345,"close_a":143.567841,"volume_a":7187000,"open_b":243.432684,"high_b":247.338667,"low_b":242.925807,"close_b":245.589432,"volume_b":51326400,"open_bench":590.420752,"close_bench":585.70343},{"date":"2025-02-25","open_a":144.445012,"high_a":147.86975,"low_a":143.707816,"close_a":145.96608,"volume_a":9474600,"open_b":246.483954,"high_b":248.471728,"low_b":243.412847,"close_b":245.529816,"volume_b":48013300,"open_bench":585.644615,"close_bench":582.790649},{"date":"2025-02-26","open_a":145.172904,"high_a":145.210225,"low_a":141.038964,"close_a":141.384232,"volume_a":9076500,"open_b":242.836357,"high_b":243.482377,"low_b":237.668149,"close_b":238.890625,"volume_b":44433600,"open_bench":584.448194,"close_bench":583.084961},{"date":"2025-02-27","open_a":140.814981,"high_a":142.485359,"low_a":139.909807,"close_a":141.860138,"volume_a":6731000,"open_b":237.946436,"high_b":240.977794,"low_b":235.610796,"close_b":235.849335,"volume_b":41153600,"open_bench":585.35047,"close_bench":573.777832},{"date":"2025-02-28","open_a":143.427866,"high_a":144.734309,"low_a":142.168098,"close_a":143.213242,"volume_a":8455500,"open_b":235.501482,"high_b":240.610059,"low_b":228.792745,"close_b":240.361588,"volume_b":56833400,"open_bench":574.27802,"close_bench":582.731934},{"date":"2025-03-03","open_a":141.234922,"high_a":145.714115,"low_a":140.441724,"close_a":145.564819,"volume_a":8321700,"open_b":240.311884,"high_b":242.538196,"low_b":234.666614,"close_b":236.574875,"volume_b":47184000,"open_bench":584.69333,"close_bench":572.522461},{"date":"2025-03-04","open_a":146.824589,"high_a":149.446788,"low_a":143.670492,"close_a":143.885117,"volume_a":10274200,"open_b":236.256849,"high_b":238.602423,"low_b":233.245358,"close_b":234.487717,"volume_b":53798100,"open_bench":568.54073,"close_bench":565.745605},{"date":"2025-03-05","open_a":142.70932,"high_a":144.958257,"low_a":142.522689,"close_a":143.59584,"volume_a":5578700,"open_b":233.980823,"high_b":235.10392,"low_b":227.828661,"close_b":234.298874,"volume_b":47227600,"open_bench":565.578847,"close_bench":571.826111},{"date":"2025-03-06","open_a":143.866473,"high_a":144.61301,"low_a":142.336078,"close_a":144.277069,"volume_a":8421000,"open_b":233.006829,"high_b":236.40592,"low_b":231.734655,"close_b":233.891388,"volume_b":45170400,"open_bench":564.392188,"close_bench":561.675598},{"date":"2025-03-07","open_a":144.272349,"high_a":148.358101,"low_a":143.820472,"close_a":145.392639,"volume_a":8997700,"open_b":233.672737,"high_b":239.894463,"low_b":233.324871,"close_b":237.608536,"volume_b":46273600,"open_bench":559.900492,"close_bench":564.82373},{"date":"2025-03-10","open_a":146.748268,"high_a":150.702221,"low_a":146.268136,"close_a":147.096588,"volume_a":11156900,"open_b":234.100096,"high_b":234.716316,"low_b":222.849305,"close_b":226.089371,"volume_b":72071200,"open_bench":556.654121,"close_bench":549.779175},{"date":"2025-03-11","open_a":146.663534,"high_a":146.908312,"low_a":142.916693,"close_a":143.406235,"volume_a":7981600,"open_b":222.44182,"high_b":224.459409,"low_b":216.120699,"close_b":219.489975,"volume_b":76137400,"open_bench":548.622037,"close_bench":545.209045},{"date":"2025-03-12","open_a":138.896831,"high_a":140.770258,"low_a":137.927172,"close_a":139.489929,"volume_a":8330200,"open_b":218.794251,"high_b":220.394409,"low_b":213.596227,"close_b":215.653564,"volume_b":62547500,"open_bench":551.338641,"close_bench":548.102234},{"date":"2025-03-13","open_a":139.67822,"high_a":141.664613,"low_a":139.386383,"close_a":139.649979,"volume_a":5351100,"open_b":214.629868,"high_b":215.514427,"low_b":207.145901,"close_b":208.398193,"volume_b":61368300,"open_bench":547.729504,"close_bench":540.795715},{"date":"2025-03-14","open_a":138.510868,"high_a":140.421945,"low_a":138.463794,"close_a":139.88533,"volume_a":5277500,"open_b":209.958594,"high_b":212.642085,"low_b":208.298805,"close_b":212.184906,"volume_b":60107600,"open_bench":545.395386,"close_bench":551.966309},{"date":"2025-03-17","open_a":140.48785,"high_a":142.784913,"low_a":140.393717,"close_a":142.474243,"volume_a":5449500,"open_b":212.006003,"high_b":213.904331,"low_b":208.686425,"close_b":212.691788,"volume_b":48073400,"open_bench":551.946737,"close_bench":556.222778},{"date":"2025-03-18","open_a":141.212729,"high_a":142.436576,"low_a":140.035956,"close_a":140.261902,"volume_a":8158600,"open_b":212.850801,"high_b":213.834739,"low_b":210.197125,"close_b":211.389786,"volume_b":42432400,"open_bench":553.918016,"close_bench":550.210876},{"date":"2025-03-19","open_a":139.48053,"high_a":140.158352,"low_a":138.190777,"close_a":139.433456,"volume_a":6080100,"open_b":212.910426,"high_b":217.422665,"low_b":212.443298,"close_b":213.924194,"volume_b":54385400,"open_bench":551.985864,"close_bench":556.203003},{"date":"2025-03-20","open_a":138.915679,"high_a":139.395811,"low_a":138.049575,"close_a":138.529694,"volume_a":4536400,"open_b":212.681871,"high_b":216.160475,"low_b":210.922686,"close_b":212.791199,"volume_b":48862900,"open_bench":552.476369,"close_bench":554.594727},{"date":"2025-03-21","open_a":138.388492,"high_a":139.047482,"low_a":136.270301,"close_a":136.929291,"volume_a":11574800,"open_b":210.266682,"high_b":217.502177,"low_b":209.988395,"close_b":216.935669,"volume_b":94127800,"open_bench":550.154386,"close_bench":554.777649},{"date":"2025-03-24","open_a":136.966928,"high_a":138.124869,"low_a":136.345589,"close_a":137.870682,"volume_a":5968900,"open_b":219.648999,"high_b":220.126061,"low_b":217.243795,"close_b":219.380646,"volume_b":44299500,"open_bench":561.48632,"close_bench":564.71283},{"date":"2025-03-25","open_a":137.588273,"high_a":137.729495,"low_a":136.345609,"close_a":137.014008,"volume_a":4763300,"open_b":219.420423,"high_b":222.730069,"low_b":218.734639,"close_b":222.382202,"volume_b":34493600,"open_bench":565.912951,"close_bench":566.070374},{"date":"2025-03-26","open_a":137.447045,"high_a":140.581969,"low_a":137.211691,"close_a":139.932388,"volume_a":5882200,"open_b":222.143627,"high_b":223.644406,"low_b":219.122218,"close_b":220.175735,"volume_b":34466100,"open_bench":565.804724,"close_bench":559.312439},{"date":"2025-03-27","open_a":140.826754,"high_a":141.598724,"low_a":140.356045,"close_a":140.902069,"volume_a":5397500,"open_b":220.036584,"high_b":223.614582,"low_b":219.211656,"close_b":222.481552,"volume_b":37094800,"open_bench":557.925371,"close_bench":557.827026},{"date":"2025-03-28","open_a":141.626955,"high_a":142.51189,"low_a":138.492031,"close_a":140.525497,"volume_a":8550900,"open_b":220.314905,"high_b":222.441823,"low_b":216.349291,"close_b":216.567947,"volume_b":39818600,"open_bench":556.302268,"close_bench":546.593262},{"date":"2025-03-31","open_a":141.401029,"high_a":143.651018,"low_a":140.977393,"close_a":141.156265,"volume_a":9985700,"open_b":215.683369,"high_b":224.240735,"low_b":214.908138,"close_b":220.772079,"volume_b":65299300,"open_bench":540.858564,"close_bench":550.262573},{"date":"2025-04-01","open_a":141.674039,"high_a":141.956453,"low_a":139.960651,"close_a":140.902069,"volume_a":6072800,"open_b":218.466265,"high_b":222.312602,"low_b":217.561824,"close_b":221.825607,"volume_b":36412700,"open_bench":548.354248,"close_bench":551.816772},{"date":"2025-04-02","open_a":141.429267,"high_a":141.683454,"low_a":139.395815,"close_a":140.384293,"volume_a":5866200,"open_b":219.967035,"high_b":223.813372,"low_b":219.668866,"close_b":222.521317,"volume_b":35905900,"open_bench":545.993327,"close_bench":555.308838},{"date":"2025-04-03","open_a":142.803733,"high_a":145.02548,"low_a":142.022354,"close_a":142.502472,"volume_a":8216200,"open_b":204.283486,"high_b":206.221577,"low_b":200.019718,"close_b":201.947861,"volume_b":103419000,"open_bench":536.21556,"close_bench":527.94281},{"date":"2025-04-04","open_a":144.027569,"high_a":145.844499,"low_a":137.861277,"close_a":138.021317,"volume_a":12911400,"open_b":192.70472,"high_b":198.658108,"low_b":186.194758,"close_b":187.228409,"volume_b":125910900,"open_bench":515.125349,"close_bench":497.035431},{"date":"2025-04-07","open_a":133.041249,"high_a":137.541228,"low_a":132.429318,"close_a":134.801697,"volume_a":14423200,"open_b":176.11674,"high_b":192.963119,"low_b":173.552511,"close_b":180.350708,"volume_b":160466300,"open_bench":481.207996,"close_bench":496.150146},{"date":"2025-04-08","open_a":135.517149,"high_a":136.882203,"low_a":130.998339,"close_a":132.080978,"volume_a":9488900,"open_b":185.55867,"high_b":189.176417,"low_b":168.175599,"close_b":171.365967,"volume_b":120859500,"open_bench":513.344914,"close_bench":488.379059},{"date":"2025-04-09","open_a":132.033896,"high_a":137.399977,"low_a":130.226375,"close_a":137.061066,"volume_a":14067000,"open_b":170.898835,"high_b":199.383635,"low_b":170.839204,"close_b":197.634399,"volume_b":184395900,"open_bench":485.388642,"close_bench":539.668274},{"date":"2025-04-10","open_a":136.901044,"high_a":138.520284,"low_a":134.547498,"close_a":135.69603,"volume_a":10980300,"open_b":187.914213,"high_b":193.589299,"low_b":181.881312,"close_b":189.255951,"volume_b":121880000,"open_bench":523.486691,"close_bench":516.020569},{"date":"2025-04-11","open_a":136.646847,"high_a":136.778645,"low_a":133.794351,"close_a":135.969025,"volume_a":8563300,"open_b":184.96235,"high_b":198.320177,"low_b":184.922586,"close_b":196.938675,"volume_b":87435900,"open_bench":514.476193,"close_bench":525.227844},{"date":"2025-04-14","open_a":135.714852,"high_a":138.557925,"low_a":135.093513,"close_a":138.153122,"volume_a":5393800,"open_b":210.147452,"high_b":211.638282,"low_b":199.930296,"close_b":201.281982,"volume_b":101352900,"open_bench":535.172854,"close_bench":530.323303},{"date":"2025-04-15","open_a":137.296438,"high_a":137.428236,"low_a":134.274493,"close_a":134.472183,"volume_a":5459600,"open_b":200.625995,"high_b":202.265902,"low_b":198.57859,"close_b":200.904282,"volume_b":51343900,"open_bench":530.864276,"close_bench":528.837891},{"date":"2025-04-16","open_a":135.667775,"high_a":135.94078,"low_a":131.582024,"close_a":131.88327,"volume_a":7878600,"open_b":197.147391,"high_b":199.473082,"low_b":191.194003,"close_b":193.082397,"volume_b":59732400,"open_bench":523.004678,"close_bench":517.082886},{"date":"2025-04-17","open_a":132.118638,"high_a":135.159429,"low_a":131.864465,"close_a":134.472183,"volume_a":6131400,"open_b":195.994474,"high_b":197.614515,"low_b":193.23147,"close_b":195.775818,"volume_b":52164700,"open_bench":519.03065,"close_bench":517.820679},{"date":"2025-04-21","open_a":134.152096,"high_a":134.462766,"low_a":132.335166,"close_a":133.4272,"volume_a":7412800,"open_b":192.088485,"high_b":192.615243,"low_b":188.64963,"close_b":191.979156,"volume_b":46742500,"open_bench":512.6563,"close_bench":505.495117},{"date":"2025-04-22","open_a":133.944997,"high_a":135.827833,"low_a":133.634327,"close_a":135.055878,"volume_a":7144700,"open_b":194.921086,"high_b":200.357648,"low_b":194.772009,"close_b":198.518967,"volume_b":52976400,"open_bench":511.652999,"close_bench":518.646973},{"date":"2025-04-23","open_a":134.877004,"high_a":135.573645,"low_a":132.598774,"close_a":133.926163,"volume_a":7760000,"open_b":204.740709,"high_b":206.728483,"low_b":201.560274,"close_b":203.349274,"volume_b":52929200,"open_bench":531.611917,"close_bench":526.683655},{"date":"2025-04-24","open_a":133.163616,"high_a":133.540192,"low_a":126.649003,"close_a":127.383308,"volume_a":15577300,"open_b":203.637485,"high_b":207.553401,"low_b":201.699408,"close_b":207.096207,"volume_b":47311000,"open_bench":527.962423,"close_bench":537.769775},{"date":"2025-04-25","open_a":127.533929,"high_a":127.665727,"low_a":124.078926,"close_a":125.566368,"volume_a":11544700,"open_b":205.108426,"high_b":208.467769,"low_b":204.939468,"close_b":208.000641,"volume_b":38222300,"open_bench":537.730448,"close_bench":541.655334},{"date":"2025-04-28","open_a":126.055908,"high_a":127.477459,"low_a":125.302785,"close_a":125.92411,"volume_a":8498600,"open_b":208.716234,"high_b":210.207064,"low_b":206.191768,"close_b":208.855377,"volume_b":38743100,"open_bench":542.393105,"close_bench":541.861877},{"date":"2025-04-29","open_a":124.71909,"high_a":126.583108,"low_a":123.815336,"close_a":126.441887,"volume_a":7010700,"open_b":207.414224,"high_b":210.942525,"low_b":207.096173,"close_b":209.918823,"volume_b":36827600,"open_bench":539.953569,"close_bench":545.27533},{"date":"2025-04-30","open_a":127.684563,"high_a":128.202332,"low_a":125.302776,"close_a":127.637489,"volume_a":7923500,"open_b":208.020523,"high_b":212.274358,"low_b":205.406596,"close_b":211.200958,"volume_b":52286500,"open_bench":538.635456,"close_bench":545.491699},{"date":"2025-05-01","open_a":126.592498,"high_a":126.912576,"low_a":125.434557,"close_a":125.726395,"volume_a":6807000,"open_b":207.801875,"high_b":213.248371,"low_b":207.622967,"close_b":212.015961,"volume_b":57365700,"open_bench":551.226505,"close_bench":549.357483},{"date":"2025-05-02","open_a":126.291234,"high_a":126.526589,"low_a":125.349816,"close_a":125.914673,"volume_a":7169700,"open_b":204.830147,"high_b":205.724655,"low_b":200.924179,"close_b":204.094681,"volume_b":101010600,"open_bench":555.515362,"close_bench":557.512268},{"date":"2025-05-05","open_a":126.187693,"high_a":126.187693,"low_a":123.984778,"close_a":124.257797,"volume_a":8972200,"open_b":201.858419,"high_b":202.852306,"low_b":196.998313,"close_b":197.674149,"volume_b":69018500,"open_bench":553.390704,"close_bench":554.315369},{"date":"2025-05-06","open_a":124.173066,"high_a":124.173066,"low_a":122.535007,"close_a":123.081032,"volume_a":6889900,"open_b":196.998309,"high_b":199.42338,"low_b":195.815581,"close_b":197.296463,"volume_b":51216500,"open_bench":548.826329,"close_bench":549.682129},{"date":"2025-05-07","open_a":123.278725,"high_a":124.897966,"low_a":122.638568,"close_a":124.191902,"volume_a":6313200,"open_b":197.952441,"high_b":198.220794,"low_b":192.068632,"close_b":195.050293,"volume_b":68536700,"open_bench":551.010152,"close_bench":551.993835},{"date":"2025-05-08","open_a":124.163657,"high_a":125.1898,"low_a":123.570559,"close_a":123.730598,"volume_a":6392400,"open_b":196.511305,"high_b":198.827063,"low_b":193.48988,"close_b":196.282715,"volume_b":50478900,"open_bench":556.017083,"close_bench":555.840027},{"date":"2025-05-09","open_a":123.467006,"high_a":123.937715,"low_a":122.553829,"close_a":122.798607,"volume_a":4770500,"open_b":197.783473,"high_b":199.314052,"low_b":196.332392,"close_b":197.316345,"volume_b":36453900,"open_bench":557.23675,"close_bench":555.131714},{"date":"2025-05-12","open_a":123.175155,"high_a":124.577873,"low_a":122.440849,"close_a":123.965942,"volume_a":8392700,"open_b":209.95528,"high_b":210.253841,"low_b":205.755577,"close_b":209.776138,"volume_b":63775800,"open_bench":571.982199,"close_bench":573.477417},{"date":"2025-05-13","open_a":123.485836,"high_a":123.928304,"low_a":122.149024,"close_a":122.676216,"volume_a":8650200,"open_b":209.417863,"high_b":212.373579,"low_b":207.994748,"close_b":211.905838,"volume_b":51909300,"open_bench":573.890623,"close_bench":577.264709},{"date":"2025-05-14","open_a":122.450278,"high_a":122.544411,"low_a":120.379154,"close_a":120.925171,"volume_a":10463000,"open_b":211.408241,"high_b":212.910988,"low_b":209.567148,"close_b":211.308731,"volume_b":49325800,"open_bench":578.218823,"close_bench":578.002441},{"date":"2025-05-15","open_a":121.621842,"high_a":123.84359,"low_a":121.264099,"close_a":123.796516,"volume_a":9165100,"open_b":209.935373,"high_b":211.935715,"low_b":208.532151,"close_b":210.432968,"volume_b":45029500,"open_bench":576.00549,"close_bench":580.825562},{"date":"2025-05-16","open_a":124.351932,"high_a":124.511971,"low_a":122.996287,"close_a":124.248375,"volume_a":8896400,"open_b":211.338597,"high_b":211.547593,"low_b":208.761058,"close_b":210.243881,"volume_b":54737900,"open_bench":581.6027,"close_bench":584.504578},{"date":"2025-05-19","open_a":124.389605,"high_a":124.511979,"low_a":123.429354,"close_a":124.069511,"volume_a":9406200,"open_b":206.909992,"high_b":208.472433,"low_b":203.277539,"close_b":207.775803,"volume_b":46140500,"open_bench":578.503999,"close_bench":585.14386},{"date":"2025-05-20","open_a":124.03186,"high_a":124.427254,"low_a":123.655298,"close_a":124.078934,"volume_a":7624200,"open_b":206.671153,"high_b":207.467308,"low_b":204.043851,"close_b":205.865051,"volume_b":42496600,"open_bench":583.412709,"close_bench":583.176575},{"date":"2025-05-21","open_a":124.078938,"high_a":124.173071,"low_a":122.440865,"close_a":122.525589,"volume_a":9243700,"open_b":204.183169,"high_b":206.044169,"low_b":199.744629,"close_b":201.117981,"volume_b":59211800,"open_bench":578.838517,"close_bench":573.349548},{"date":"2025-05-22","open_a":122.403204,"high_a":123.043362,"low_a":121.273505,"close_a":122.497337,"volume_a":7140200,"open_b":199.744597,"high_b":201.774778,"low_b":198.739445,"close_b":200.391464,"volume_b":46742400,"open_bench":573.15283,"close_bench":573.575867},{"date":"2025-05-23","open_a":121.970144,"high_a":122.158425,"low_a":120.266179,"close_a":121.763031,"volume_a":8122500,"open_b":192.738485,"high_b":196.7491,"low_b":192.529503,"close_b":194.330795,"volume_b":78432900,"open_bench":566.581833,"close_bench":569.660767},{"date":"2025-05-27","open_a":122.459695,"high_a":123.758857,"low_a":121.56535,"close_a":123.674118,"volume_a":13974800,"open_b":197.346223,"high_b":199.774489,"low_b":196.480397,"close_b":199.24704,"volume_b":56288500,"open_bench":576.507267,"close_bench":581.504395},{"date":"2025-05-28","open_a":123.598789,"high_a":124.446074,"low_a":122.826833,"close_a":123.015114,"volume_a":9431500,"open_b":199.625189,"high_b":201.754896,"low_b":198.938506,"close_b":199.456009,"volume_b":45339700,"open_bench":581.907599,"close_bench":578.140076},{"date":"2025-05-29","open_a":123.194001,"high_a":124.333124,"low_a":122.619735,"close_a":124.191902,"volume_a":6656900,"open_b":202.600827,"high_b":202.829716,"low_b":197.555205,"close_b":198.988281,"volume_b":51396800,"open_bench":583.383137,"close_bench":580.422241},{"date":"2025-05-30","open_a":124.408418,"high_a":124.945033,"low_a":123.598797,"close_a":123.749428,"volume_a":12992100,"open_b":198.411064,"high_b":200.988619,"low_b":195.833526,"close_b":199.883957,"volume_b":70819900,"open_bench":579.320494,"close_bench":579.77301},{"date":"2025-06-02","open_a":123.231635,"high_a":123.354024,"low_a":121.508852,"close_a":123.241058,"volume_a":6723700,"open_b":199.31669,"high_b":201.157798,"low_b":199.157456,"close_b":200.729858,"volume_b":35423300,"open_bench":578.169696,"close_bench":583.03894},{"date":"2025-06-03","open_a":122.977468,"high_a":124.464911,"low_a":122.205498,"close_a":124.125999,"volume_a":7339500,"open_b":200.381549,"high_b":202.789908,"low_b":199.993426,"close_b":202.292313,"volume_b":46381600,"open_bench":582.674957,"close_bench":586.36377},{"date":"2025-06-04","open_a":124.493147,"high_a":124.963856,"low_a":123.419931,"close_a":124.022438,"volume_a":6519700,"open_b":201.934046,"high_b":205.248031,"low_b":201.127944,"close_b":201.844482,"volume_b":43604000,"open_bench":587.219582,"close_bench":586.20636},{"date":"2025-06-05","open_a":123.514076,"high_a":123.71178,"low_a":122.582081,"close_a":123.429352,"volume_a":9560800,"open_b":202.521199,"high_b":203.765187,"low_b":199.187306,"close_b":199.665009,"volume_b":55126100,"open_bench":587.878515,"close_bench":583.37323},{"date":"2025-06-06","open_a":123.965184,"high_a":124.364917,"low_a":123.09909,"close_a":123.755798,"volume_a":7628200,"open_b":202.023605,"high_b":204.710616,"low_b":201.078178,"close_b":202.939178,"volume_b":46607700,"open_bench":588.891744,"close_bench":589.363953},{"date":"2025-06-09","open_a":123.565447,"high_a":124.345872,"low_a":122.946802,"close_a":123.689178,"volume_a":8453100,"open_b":203.406925,"high_b":205.009182,"low_b":199.057949,"close_b":200.481064,"volume_b":72862600,"open_bench":589.934406,"close_bench":589.895081},{"date":"2025-06-10","open_a":123.917579,"high_a":125.744934,"low_a":123.213295,"close_a":125.468933,"volume_a":11852900,"open_b":199.63514,"high_b":203.367102,"low_b":199.605285,"close_b":201.695175,"volume_b":54672600,"open_bench":590.426245,"close_bench":593.239624},{"date":"2025-06-11","open_a":125.573639,"high_a":125.60219,"low_a":123.527372,"close_a":123.632065,"volume_a":9168400,"open_b":202.521181,"high_b":203.516372,"low_b":197.455668,"close_b":197.823883,"volume_b":60989900,"open_bench":594.331509,"close_bench":591.547668},{"date":"2025-06-12","open_a":123.622544,"high_a":125.944811,"low_a":123.451236,"close_a":125.91626,"volume_a":11444900,"open_b":198.122464,"high_b":198.719569,"low_b":196.410736,"close_b":198.241882,"volume_b":43904600,"open_bench":590.219716,"close_bench":593.898682},{"date":"2025-06-13","open_a":125.402312,"high_a":126.639587,"low_a":124.288768,"close_a":124.536232,"volume_a":9581000,"open_b":198.769334,"high_b":199.406255,"low_b":194.758719,"close_b":195.505112,"volume_b":51447300,"open_bench":588.734436,"close_bench":587.258911},{"date":"2025-06-16","open_a":124.974033,"high_a":125.716396,"low_a":123.946144,"close_a":125.069214,"volume_a":11183700,"open_b":196.351029,"high_b":197.734343,"low_b":195.614583,"close_b":197.465637,"volume_b":43020700,"open_bench":590.603486,"close_bench":592.846252},{"date":"2025-06-17","open_a":124.831269,"high_a":125.335695,"low_a":122.899222,"close_a":123.051491,"volume_a":10235200,"open_b":196.251484,"high_b":197.435763,"low_b":194.271065,"close_b":194.69899,"volume_b":38856200,"open_bench":590.416537,"close_bench":587.780273},{"date":"2025-06-18","open_a":123.375102,"high_a":123.422677,"low_a":122.385275,"close_a":122.842125,"volume_a":8785800,"open_b":194.997555,"high_b":196.619719,"low_b":194.131744,"close_b":195.634476,"volume_b":45394700,"open_bench":588.675334,"close_bench":587.69165},{"date":"2025-06-20","open_a":123.279921,"high_a":124.126976,"low_a":122.680315,"close_a":122.842125,"volume_a":23001000,"open_b":197.286484,"high_b":200.729833,"low_b":195.913117,"close_b":200.033203,"volume_b":96813500,"open_bench":590.356462,"close_bench":586.311462},{"date":"2025-06-23","open_a":123.051492,"high_a":123.251365,"low_a":121.652428,"close_a":122.861145,"volume_a":12840400,"open_b":200.660202,"high_b":201.326978,"low_b":198.003046,"close_b":200.530823,"volume_b":55814300,"open_bench":587.061195,"close_bench":592.102722},{"date":"2025-06-24","open_a":122.908728,"high_a":124.878853,"low_a":122.823074,"close_a":124.72657,"volume_a":9226800,"open_b":201.615572,"high_b":202.46149,"low_b":199.237069,"close_b":199.336594,"volume_b":54064000,"open_bench":596.22676,"close_bench":598.643921},{"date":"2025-06-25","open_a":123.974695,"high_a":124.288774,"low_a":121.4716,"close_a":121.842789,"volume_a":10922600,"open_b":200.481059,"high_b":202.690382,"low_b":199.655049,"close_b":200.59053,"volume_b":39525700,"open_bench":599.758695,"close_bench":598.979309},{"date":"2025-06-26","open_a":121.547733,"high_a":122.927772,"low_a":121.44304,"close_a":122.033127,"volume_a":11538100,"open_b":200.461128,"high_b":201.665314,"low_b":198.500617,"close_b":200.033203,"volume_b":50799100,"open_bench":600.824201,"close_bench":603.665588},{"date":"2025-06-27","open_a":122.099727,"high_a":125.002561,"low_a":121.928419,"close_a":124.717033,"volume_a":26536500,"open_b":200.91894,"high_b":202.242545,"low_b":199.038032,"close_b":200.112839,"volume_b":73188600,"open_bench":604.662046,"close_bench":606.664795},{"date":"2025-06-30","open_a":124.869328,"high_a":125.973359,"low_a":124.288761,"close_a":125.668793,"volume_a":11402400,"open_b":201.038362,"high_b":206.39249,"low_b":198.301589,"close_b":204.183167,"volume_b":91912800,"open_bench":609.1017,"close_bench":609.565369},{"date":"2025-07-01","open_a":126.192279,"high_a":129.808924,"low_a":126.078073,"close_a":128.733444,"volume_a":12846700,"open_b":205.675942,"high_b":209.179015,"low_b":205.148492,"close_b":206.820419,"volume_b":78788900,"open_bench":608.095423,"close_bench":609.368164},{"date":"2025-07-02","open_a":128.723916,"high_a":130.389475,"low_a":128.371774,"close_a":129.894562,"volume_a":9388700,"open_b":207.905178,"high_b":212.313863,"low_b":207.138877,"close_b":211.418198,"volume_b":67941800,"open_bench":608.963575,"close_bench":612.130554},{"date":"2025-07-03","open_a":130.103972,"high_a":130.227703,"low_a":128.714427,"close_a":128.847672,"volume_a":4941400,"open_b":211.129598,"high_b":213.617573,"low_b":210.791236,"close_b":212.522873,"volume_b":34955800,"open_bench":614.103693,"close_bench":616.954956},{"date":"2025-07-07","open_a":128.990399,"high_a":129.171233,"low_a":127.229659,"close_a":127.962509,"volume_a":7773700,"open_b":211.657035,"high_b":215.189963,"low_b":207.795708,"close_b":208.94017,"volume_b":50229000,"open_bench":615.001479,"close_bench":612.357422},{"date":"2025-07-08","open_a":126.972693,"high_a":129.123638,"low_a":126.848961,"close_a":128.524033,"volume_a":7718300,"open_b":209.089472,"high_b":210.413061,"low_b":207.447399,"close_b":208.999893,"volume_b":42848900,"open_bench":613.01844,"close_bench":612.022034},{"date":"2025-07-09","open_a":128.390806,"high_a":128.847656,"low_a":126.544427,"close_a":127.991074,"volume_a":7762600,"open_b":208.522209,"high_b":210.313555,"low_b":206.223322,"close_b":210.124466,"volume_b":48749400,"open_bench":614.419458,"close_bench":615.692139},{"date":"2025-07-10","open_a":127.943492,"high_a":130.084926,"low_a":127.182106,"close_a":129.513885,"volume_a":8869700,"open_b":209.497466,"high_b":212.453182,"low_b":209.019779,"close_b":211.388336,"volume_b":44443600,"open_bench":615.830316,"close_bench":617.428589},{"date":"2025-07-11","open_a":128.333712,"high_a":128.857176,"low_a":127.353412,"close_a":128.733444,"volume_a":7963600,"open_b":209.55719,"high_b":211.109684,"low_b":208.850598,"close_b":210.144348,"volume_b":39765800,"open_bench":614.389791,"close_bench":615.257996},{"date":"2025-07-14","open_a":128.248048,"high_a":129.323528,"low_a":127.400993,"close_a":129.028488,"volume_a":8480300,"open_b":208.920259,"high_b":209.895556,"low_b":206.541755,"close_b":207.616562,"volume_b":38840100,"open_bench":614.804168,"close_bench":616.432068},{"date":"2025-07-15","open_a":128.895228,"high_a":128.97137,"low_a":127.296299,"close_a":127.353401,"volume_a":8933500,"open_b":208.213704,"high_b":210.87086,"low_b":207.915144,"close_b":208.104233,"volume_b":42296300,"open_bench":619.105778,"close_bench":613.797913},{"date":"2025-07-16","open_a":127.486651,"high_a":128.971376,"low_a":127.296304,"close_a":128.819107,"volume_a":10941600,"open_b":209.288511,"high_b":211.378402,"low_b":207.636492,"close_b":209.149185,"volume_b":47490500,"open_bench":615.376431,"close_bench":615.849976},{"date":"2025-07-17","open_a":134.862683,"high_a":138.917139,"low_a":134.805581,"close_a":138.422226,"volume_a":26161000,"open_b":209.5572,"high_b":210.781279,"low_b":208.581902,"close_b":209.009842,"volume_b":48068100,"open_bench":616.027567,"close_bench":619.618713},{"date":"2025-07-18","open_a":139.202668,"high_a":139.288337,"low_a":136.042872,"close_a":136.3284,"volume_a":13752800,"open_b":209.855739,"high_b":210.771312,"low_b":208.691368,"close_b":210.164246,"volume_b":48974600,"open_bench":620.861886,"close_bench":619.164978},{"date":"2025-07-21","open_a":136.195142,"high_a":136.661489,"low_a":133.682528,"close_a":134.862686,"volume_a":9025600,"open_b":211.079831,"high_b":214.742123,"low_b":210.61209,"close_b":211.457993,"volume_b":51377400,"open_bench":620.338989,"close_bench":620.338989},{"date":"2025-07-22","open_a":134.415365,"high_a":139.335901,"low_a":134.415365,"close_a":138.993271,"volume_a":8281500,"open_b":212.11484,"high_b":213.916132,"low_b":211.209213,"close_b":213.368774,"volume_b":46404100,"open_bench":620.664568,"close_bench":620.427795},{"date":"2025-07-23","open_a":138.821969,"high_a":139.935512,"low_a":138.089119,"close_a":138.631622,"volume_a":6166000,"open_b":213.965898,"high_b":214.11517,"low_b":211.388359,"close_b":213.11998,"volume_b":46989300,"open_bench":623.081627,"close_bench":625.705994},{"date":"2025-07-24","open_a":138.783896,"high_a":139.402527,"low_a":137.032669,"close_a":137.537094,"volume_a":9749400,"open_b":212.871153,"high_b":214.652552,"low_b":212.502937,"close_b":212.731827,"volume_b":46022600,"open_bench":626.090848,"close_bench":625.913269},{"date":"2025-07-25","open_a":137.822637,"high_a":137.832164,"low_a":135.814448,"close_a":136.528259,"volume_a":7021600,"open_b":213.667321,"high_b":214.204732,"low_b":212.37357,"close_b":212.851273,"volume_b":40268800,"open_bench":626.574253,"close_bench":628.557251},{"date":"2025-07-28","open_a":135.814449,"high_a":136.195157,"low_a":134.63429,"close_a":134.738983,"volume_a":10639100,"open_b":213.000569,"high_b":213.816633,"low_b":212.035233,"close_b":213.020477,"volume_b":37858000,"open_bench":628.932091,"close_bench":628.399353},{"date":"2025-07-29","open_a":134.738974,"high_a":137.042202,"low_a":134.320202,"close_a":136.947021,"volume_a":6282900,"open_b":213.149822,"high_b":213.776796,"low_b":209.805997,"close_b":210.25383,"volume_b":51411700,"open_bench":629.790476,"close_bench":626.741943},{"date":"2025-07-30","open_a":136.832815,"high_a":137.432406,"low_a":135.424217,"close_a":135.966721,"volume_a":9738400,"open_b":210.880777,"high_b":211.368426,"low_b":206.72089,"close_b":208.044495,"volume_b":45512500,"open_bench":627.393082,"close_bench":625.952698},{"date":"2025-07-31","open_a":134.177422,"high_a":134.738964,"low_a":130.998587,"close_a":131.265076,"volume_a":11689700,"open_b":207.487213,"high_b":208.830711,"low_b":206.163608,"close_b":206.57164,"volume_b":80698400,"open_bench":630.885601,"close_bench":623.604553},{"date":"2025-08-01","open_a":132.388156,"high_a":133.063889,"low_a":131.617228,"close_a":132.559464,"volume_a":7630500,"open_b":209.855756,"high_b":212.552728,"low_b":200.530829,"close_b":201.406601,"volume_b":104434500,"open_bench":617.902027,"close_bench":613.383423},{"date":"2025-08-04","open_a":132.568967,"high_a":133.549281,"low_a":132.055029,"close_a":132.825943,"volume_a":8384700,"open_b":203.526342,"high_b":206.880143,"low_b":200.709952,"close_b":202.371933,"volume_b":75109300,"open_bench":617.280475,"close_bench":622.706726},{"date":"2025-08-05","open_a":132.864008,"high_a":133.577819,"low_a":132.540403,"close_a":132.721237,"volume_a":5535100,"open_b":202.421672,"high_b":204.352343,"low_b":201.187646,"close_b":201.943985,"volume_b":44155100,"open_bench":623.318468,"close_bench":619.549683},{"date":"2025-08-06","open_a":133.149537,"high_a":134.748481,"low_a":132.635599,"close_a":134.006119,"volume_a":6747400,"open_b":204.640975,"high_b":214.34408,"low_b":204.601159,"close_b":212.224319,"volume_b":108483100,"open_bench":620.615201,"close_bench":624.295227},{"date":"2025-08-07","open_a":134.424908,"high_a":137.346767,"low_a":133.70157,"close_a":136.975586,"volume_a":7674600,"open_b":217.827232,"high_b":219.787758,"low_b":215.538292,"close_b":218.971695,"volume_b":90224800,"open_bench":627.708828,"close_bench":623.772339},{"date":"2025-08-08","open_a":136.97556,"high_a":138.355592,"low_a":136.861355,"close_a":138.203323,"volume_a":5315100,"open_b":219.767847,"high_b":229.88893,"low_b":218.195445,"close_b":228.246872,"volume_b":113854000,"open_bench":625.55807,"close_bench":628.63623},{"date":"2025-08-11","open_a":138.308028,"high_a":138.574517,"low_a":136.813777,"close_a":137.87973,"volume_a":5773400,"open_b":227.081154,"high_b":228.715118,"low_b":223.932781,"close_b":226.343872,"volume_b":61806100,"open_bench":628.912394,"close_bench":627.393005},{"date":"2025-08-12","open_a":138.526949,"high_a":139.916493,"low_a":137.736996,"close_a":139.783249,"volume_a":7668700,"open_b":227.170831,"high_b":229.950571,"low_b":226.234303,"close_b":228.804794,"volume_b":55626200,"open_bench":629.731301,"close_bench":634.072327},{"date":"2025-08-13","open_a":139.802278,"high_a":142.324405,"low_a":139.764215,"close_a":142.143585,"volume_a":6147000,"open_b":230.219576,"high_b":234.135104,"low_b":229.581917,"close_b":232.471252,"volume_b":69878500,"open_bench":636.262489,"close_bench":636.242798},{"date":"2025-08-14","open_a":141.848524,"high_a":142.286334,"low_a":140.906303,"close_a":141.448792,"volume_a":6892500,"open_b":233.198567,"high_b":234.254663,"low_b":230.000389,"close_b":231.923279,"volume_b":51916300,"open_bench":634.170992,"close_bench":636.302063},{"date":"2025-08-15","open_a":142.181638,"high_a":143.809134,"low_a":141.325071,"close_a":143.142899,"volume_a":9162200,"open_b":233.138774,"high_b":233.417742,"low_b":228.495921,"close_b":230.73764,"volume_b":56038700,"open_bench":637.328051,"close_bench":634.812256},{"date":"2025-08-18","open_a":143.285687,"high_a":143.285687,"low_a":141.734333,"close_a":142.333939,"volume_a":9901000,"open_b":230.847254,"high_b":232.262026,"low_b":229.263109,"close_b":230.040237,"volume_b":37476200,"open_bench":634.240031,"close_bench":634.674133},{"date":"2025-08-19","open_a":142.762199,"high_a":144.970261,"low_a":142.628955,"close_a":144.817978,"volume_a":8596900,"open_b":230.428792,"high_b":232.012937,"low_b":228.505903,"close_b":229.711441,"volume_b":39402600,"open_bench":634.496572,"close_bench":631.230957},{"date":"2025-08-20","open_a":145.817331,"high_a":147.149773,"low_a":141.515426,"close_a":143.456985,"volume_a":10023500,"open_b":229.133582,"high_b":229.621784,"low_b":224.939085,"close_b":225.178192,"volume_b":42263900,"open_bench":630.826452,"close_bench":629.553711},{"date":"2025-08-21","open_a":143.066766,"high_a":143.066766,"low_a":141.429758,"close_a":141.791412,"volume_a":8153600,"open_b":225.437249,"high_b":225.686329,"low_b":222.956408,"close_b":224.072281,"volume_b":30621200,"open_bench":627.748329,"close_bench":627.028076},{"date":"2025-08-22","open_a":142.429098,"high_a":143.885272,"low_a":141.896122,"close_a":142.419586,"volume_a":5218000,"open_b":225.337608,"high_b":228.24686,"low_b":224.580411,"close_b":226.921753,"volume_b":42477800,"open_bench":629.208414,"close_bench":636.657166},{"date":"2025-08-25","open_a":141.267924,"high_a":141.63912,"low_a":140.458947,"close_a":141.049026,"volume_a":6624300,"open_b":225.646454,"high_b":228.456083,"low_b":225.397374,"close_b":226.323959,"volume_b":30983100,"open_bench":635.40418,"close_bench":633.855225},{"date":"2025-08-26","open_a":140.382826,"high_a":141.096637,"low_a":138.774369,"close_a":139.906952,"volume_a":11208100,"open_b":226.035027,"high_b":228.645395,"low_b":223.863057,"close_b":228.466049,"volume_b":54575100,"open_bench":633.588884,"close_bench":636.509155},{"date":"2025-08-27","open_a":139.906949,"high_a":140.658824,"low_a":139.250241,"close_a":140.516068,"volume_a":8092900,"open_b":227.768623,"high_b":230.050188,"low_b":227.419905,"close_b":229.641708,"volume_b":31259500,"open_bench":635.927097,"close_bench":637.959473},{"date":"2025-08-28","open_a":140.820648,"high_a":140.896775,"low_a":139.174113,"close_a":139.887924,"volume_a":9265100,"open_b":229.970515,"high_b":232.550979,"low_b":228.495951,"close_b":231.704102,"volume_b":38074700,"open_bench":638.561284,"close_bench":640.21875},{"date":"2025-08-29","open_a":140.002158,"high_a":142.019859,"low_a":139.783245,"close_a":141.477356,"volume_a":7760700,"open_b":231.654263,"high_b":232.521071,"low_b":230.518459,"close_b":231.285629,"volume_b":39418400,"open_bench":638.78823,"close_bench":636.400696},{"date":"2025-09-02","open_a":149.843206,"high_a":150.185835,"low_a":142.962072,"close_a":143.028687,"volume_a":17348600,"open_b":228.406286,"high_b":230.000404,"low_b":226.134678,"close_b":228.874557,"volume_b":44075600,"open_bench":628.951877,"close_bench":631.684753},{"date":"2025-09-03","open_a":142.714607,"high_a":143.200008,"low_a":140.011646,"close_a":141.467819,"volume_a":11259200,"open_b":236.336989,"high_b":237.970953,"low_b":233.497472,"close_b":237.592346,"volume_b":66427800,"open_bench":634.052556,"close_bench":635.108215},{"date":"2025-09-04","open_a":141.829489,"high_a":142.257773,"low_a":139.38349,"close_a":139.802261,"volume_a":6342200,"open_b":237.5724,"high_b":239.01706,"low_b":235.868702,"close_b":238.897507,"volume_b":47549400,"open_bench":635.779086,"close_bench":640.416077},{"date":"2025-09-05","open_a":140.314519,"high_a":142.121313,"low_a":139.622556,"close_a":140.689331,"volume_a":7186300,"open_b":239.116703,"high_b":240.431852,"low_b":237.612266,"close_b":238.807846,"volume_b":54870400,"open_bench":642.744416,"close_bench":638.561279},{"date":"2025-09-08","open_a":140.045412,"high_a":140.045412,"low_a":134.884534,"close_a":136.191574,"volume_a":11127600,"open_b":238.419289,"high_b":239.266151,"low_b":235.470176,"close_b":237.004517,"volume_b":48999500,"open_bench":639.92286,"close_bench":640.130066},{"date":"2025-09-09","open_a":135.682206,"high_a":138.03681,"low_a":135.605334,"close_a":137.527451,"volume_a":5866400,"open_b":236.127744,"high_b":237.901192,"low_b":232.501141,"close_b":233.487503,"volume_b":66313900,"open_bench":640.268114,"close_bench":641.609924},{"date":"2025-09-10","open_a":136.691321,"high_a":137.537057,"low_a":135.912866,"close_a":137.104584,"volume_a":6015100,"open_b":231.335457,"high_b":231.564607,"low_b":225.118417,"close_b":225.955322,"volume_b":83440800,"open_bench":644.855789,"close_bench":643.464722},{"date":"2025-09-11","open_a":137.325619,"high_a":139.113188,"low_a":136.950807,"close_a":138.613434,"volume_a":6144400,"open_b":226.044995,"high_b":229.601848,"low_b":225.81583,"close_b":229.183395,"volume_b":50208600,"open_bench":645.408199,"close_bench":648.811951},{"date":"2025-09-12","open_a":138.459684,"high_a":138.632668,"low_a":137.335233,"close_a":137.940704,"volume_a":5713700,"open_b":228.376386,"high_b":233.64691,"low_b":228.177125,"close_b":233.208542,"volume_b":55824200,"open_bench":648.782364,"close_bench":648.59491},{"date":"2025-09-15","open_a":137.815754,"high_a":137.940696,"low_a":135.057522,"close_a":135.163239,"volume_a":7279600,"open_b":236.127741,"high_b":237.313364,"low_b":234.16499,"close_b":235.828842,"volume_b":42699500,"open_bench":650.795044,"close_bench":652.047974},{"date":"2025-09-16","open_a":135.211298,"high_a":135.807149,"low_a":134.461674,"close_a":134.576996,"volume_a":9283500,"open_b":236.307083,"high_b":240.332222,"low_b":235.450262,"close_b":237.273514,"volume_b":63421100,"open_bench":652.600468,"close_bench":651.150208},{"date":"2025-09-17","open_a":134.711521,"high_a":136.508695,"low_a":134.692295,"close_a":135.73024,"volume_a":6092900,"open_b":238.090497,"high_b":239.216343,"low_b":236.855055,"close_b":238.110428,"volume_b":46508000,"open_bench":651.160135,"close_bench":650.341248},{"date":"2025-09-18","open_a":135.163249,"high_a":136.114702,"low_a":134.922984,"close_a":135.249741,"volume_a":6106600,"open_b":239.086821,"high_b":240.31229,"low_b":235.779033,"close_b":237.004517,"volume_b":44249600,"open_bench":653.014849,"close_bench":653.379883},{"date":"2025-09-19","open_a":135.951303,"high_a":136.547154,"low_a":134.990245,"close_a":136.239609,"volume_a":12365500,"open_b":240.342162,"high_b":245.393509,"low_b":239.325927,"close_b":244.596451,"volume_b":163741300,"open_bench":655.260655,"close_bench":656.616028},{"date":"2025-09-22","open_a":135.586111,"high_a":136.008979,"low_a":134.490505,"close_a":135.538055,"volume_a":5797600,"open_b":247.386162,"high_b":255.695479,"low_b":247.206817,"close_b":255.137512,"volume_b":105517400,"open_bench":655.132105,"close_bench":659.722595},{"date":"2025-09-23","open_a":135.547635,"high_a":136.691296,"low_a":134.509705,"close_a":136.374146,"volume_a":6356500,"open_b":254.938267,"high_b":256.392885,"low_b":252.646729,"close_b":253.493591,"volume_b":60275200,"open_bench":659.603711,"close_bench":656.131226},{"date":"2025-09-24","open_a":136.28767,"high_a":136.79703,"low_a":135.518821,"close_a":136.633652,"volume_a":5949100,"open_b":254.280687,"high_b":254.798778,"low_b":250.116063,"close_b":251.381393,"volume_b":42303700,"open_bench":657.417338,"close_bench":654.043701},{"date":"2025-09-25","open_a":137.431329,"high_a":137.834972,"low_a":134.413607,"close_a":134.471268,"volume_a":7101700,"open_b":252.278088,"high_b":256.22352,"low_b":250.783609,"close_b":255.924606,"volume_b":55202100,"open_bench":650.917495,"close_bench":651.026306},{"date":"2025-09-26","open_a":134.49051,"high_a":135.499624,"low_a":134.404018,"close_a":134.971039,"volume_a":6224700,"open_b":253.164811,"high_b":256.651929,"low_b":252.845981,"close_b":254.519806,"volume_b":46076300,"open_bench":652.470701,"close_bench":654.756042},{"date":"2025-09-29","open_a":134.971038,"high_a":135.192077,"low_a":133.702434,"close_a":134.711548,"volume_a":11930200,"open_b":253.623118,"high_b":254.061501,"low_b":252.078819,"close_b":253.493591,"volume_b":40127700,"open_bench":657.268926,"close_bench":656.596191},{"date":"2025-09-30","open_a":134.711549,"high_a":135.53806,"low_a":134.384793,"close_a":134.971039,"volume_a":7583400,"open_b":253.92202,"high_b":254.978117,"low_b":252.178461,"close_b":253.692871,"volume_b":37704300,"open_bench":655.854147,"close_bench":659.069458},{"date":"2025-10-01","open_a":135.326605,"high_a":137.786911,"low_a":135.009454,"close_a":137.565872,"volume_a":5880800,"open_b":254.101362,"high_b":257.837576,"low_b":253.991767,"close_b":254.509857,"volume_b":48713900,"open_bench":656.091635,"close_bench":661.315308},{"date":"2025-10-02","open_a":136.364561,"high_a":137.671601,"low_a":135.864807,"close_a":136.768204,"volume_a":4497300,"open_b":255.635661,"high_b":257.229778,"low_b":253.214612,"close_b":256.183655,"volume_b":42630200,"open_bench":663.293939,"close_bench":662.077026},{"date":"2025-10-03","open_a":136.720145,"high_a":137.777315,"low_a":136.335728,"close_a":136.45105,"volume_a":5425700,"open_b":253.732712,"high_b":258.285885,"low_b":253.015361,"close_b":257.070374,"volume_b":49155600,"open_bench":662.838843,"close_bench":662.0672},{"date":"2025-10-06","open_a":136.249224,"high_a":136.326111,"low_a":133.471767,"close_a":134.259827,"volume_a":6750700,"open_b":257.040473,"high_b":258.116515,"low_b":254.111306,"close_b":255.74527,"volume_b":44664100,"open_bench":664.451431,"close_bench":664.441528},{"date":"2025-10-07","open_a":134.903761,"high_a":136.316518,"low_a":134.250248,"close_a":135.307404,"volume_a":6110300,"open_b":255.864844,"high_b":256.452669,"low_b":254.489918,"close_b":255.536072,"volume_b":31955800,"open_bench":665.361629,"close_bench":661.978149},{"date":"2025-10-08","open_a":135.30738,"high_a":135.489983,"low_a":133.289166,"close_a":133.433319,"volume_a":8256700,"open_b":255.575919,"high_b":257.568559,"low_b":255.167424,"close_b":257.11026,"volume_b":36496900,"open_bench":663.096078,"close_bench":665.925537},{"date":"2025-10-09","open_a":134.99027,"high_a":139.103599,"low_a":133.193081,"close_a":139.074768,"volume_a":13706600,"open_b":256.861155,"high_b":257.050459,"low_b":252.208345,"close_b":253.105026,"volume_b":38322000,"open_bench":666.341158,"close_bench":663.996399},{"date":"2025-10-10","open_a":139.95892,"high_a":144.946815,"low_a":139.353448,"close_a":144.235626,"volume_a":17545200,"open_b":254.001718,"high_b":255.436421,"low_b":243.101979,"close_b":244.36731,"volume_b":61999100,"open_bench":664.956002,"close_bench":646.049988},{"date":"2025-10-13","open_a":142.861305,"high_a":144.139515,"low_a":141.602321,"close_a":143.091965,"volume_a":7272000,"open_b":248.462191,"high_b":248.771048,"low_b":244.656243,"close_b":246.74852,"volume_b":38142900,"open_bench":653.598568,"close_bench":655.963013},{"date":"2025-10-14","open_a":142.899771,"high_a":145.715665,"low_a":142.717168,"close_a":145.638779,"volume_a":8371100,"open_b":245.692416,"high_b":247.934135,"low_b":243.7994,"close_b":246.858109,"volume_b":35478000,"open_bench":650.155692,"close_bench":655.161682},{"date":"2025-10-15","open_a":145.436953,"high_a":146.455672,"low_a":143.361064,"close_a":145.273575,"volume_a":6101100,"open_b":248.57179,"high_b":250.893216,"low_b":246.55922,"close_b":248.422333,"volume_b":33893600,"open_bench":659.702725,"close_bench":658.070312},{"date":"2025-10-16","open_a":146.446077,"high_a":148.368193,"low_a":146.35957,"close_a":146.724777,"volume_a":8532900,"open_b":247.33635,"high_b":248.123436,"low_b":244.227838,"close_b":246.539291,"volume_b":39777000,"open_bench":659.702715,"close_bench":653.588684},{"date":"2025-10-17","open_a":147.781935,"high_a":147.916482,"low_a":145.830988,"close_a":147.724274,"volume_a":6710100,"open_b":247.107192,"high_b":252.447465,"low_b":246.359952,"close_b":251.361465,"volume_b":49147000,"open_bench":652.460824,"close_bench":657.298645},{"date":"2025-10-20","open_a":148.406622,"high_a":148.406622,"low_a":146.772826,"close_a":147.656998,"volume_a":4810600,"open_b":254.948221,"high_b":263.40698,"low_b":254.689183,"close_b":261.274841,"volume_b":90483000,"open_bench":660.197387,"close_bench":664.134888},{"date":"2025-10-21","open_a":148.675733,"high_a":149.146657,"low_a":146.888164,"close_a":147.21492,"volume_a":5614300,"open_b":260.916148,"high_b":264.313601,"low_b":260.866314,"close_b":261.802856,"volume_b":46695900,"open_bench":664.273362,"close_bench":664.124939},{"date":"2025-10-22","open_a":147.186062,"high_a":149.137009,"low_a":146.763209,"close_a":147.07074,"volume_a":6486000,"open_b":261.683334,"high_b":261.88261,"low_b":254.489905,"close_b":257.49881,"volume_b":45015300,"open_bench":664.827425,"close_bench":660.672241},{"date":"2025-10-23","open_a":147.2053,"high_a":147.426339,"low_a":145.148636,"close_a":145.60994,"volume_a":6091800,"open_b":258.983294,"high_b":259.660784,"low_b":257.060405,"close_b":258.624603,"volume_b":32754900,"open_bench":660.988803,"close_bench":664.589966},{"date":"2025-10-24","open_a":146.119301,"high_a":146.484508,"low_a":145.379297,"close_a":145.648392,"volume_a":3580200,"open_b":260.228747,"high_b":263.157929,"low_b":258.226134,"close_b":261.852753,"volume_b":38253700,"open_bench":669.239755,"close_bench":670.021301},{"date":"2025-10-27","open_a":145.129416,"high_a":146.916985,"low_a":145.081375,"close_a":146.68634,"volume_a":4744600,"open_b":263.90515,"high_b":268.129535,"low_b":263.675985,"close_b":267.820679,"volume_b":44888200,"open_bench":675.442867,"close_bench":677.926086},{"date":"2025-10-28","open_a":145.513852,"high_a":147.24376,"low_a":144.08187,"close_a":144.274078,"volume_a":5626500,"open_b":268.000006,"high_b":268.896718,"low_b":267.163101,"close_b":268.009979,"volume_b":41534800,"open_bench":679.716721,"close_bench":679.726624},{"date":"2025-10-29","open_a":142.284676,"high_a":142.870923,"low_a":138.075237,"close_a":140.468277,"volume_a":7767600,"open_b":268.28893,"high_b":270.411095,"low_b":266.126903,"close_b":268.707397,"volume_b":51086700,"open_bench":681.368861,"close_bench":680.053101},{"date":"2025-10-30","open_a":140.641266,"high_a":142.323118,"low_a":140.554775,"close_a":141.804153,"volume_a":5752700,"open_b":270.988948,"high_b":273.131059,"low_b":267.491887,"close_b":270.401123,"volume_b":69886500,"open_bench":676.60048,"close_bench":672.573914},{"date":"2025-10-31","open_a":140.506733,"high_a":141.275583,"low_a":139.641787,"close_a":140.401016,"volume_a":6313600,"open_b":275.97057,"high_b":276.299373,"low_b":268.169401,"close_b":269.374939,"volume_b":86167100,"open_bench":677.728203,"close_bench":674.780029},{"date":"2025-11-03","open_a":140.179982,"high_a":140.228038,"low_a":137.671619,"close_a":138.007996,"volume_a":8331500,"open_b":269.424737,"high_b":269.853147,"low_b":265.270071,"close_b":268.059753,"volume_b":50194600,"open_bench":678.351474,"close_bench":676.046387},{"date":"2025-11-04","open_a":139.065144,"high_a":139.286183,"low_a":136.52795,"close_a":137.335236,"volume_a":8302900,"open_b":267.342445,"high_b":270.490818,"low_b":266.635066,"close_b":269.046173,"volume_b":49274800,"open_bench":668.893485,"close_bench":668.032776},{"date":"2025-11-05","open_a":137.335219,"high_a":137.700426,"low_a":136.06663,"close_a":137.296783,"volume_a":7136400,"open_b":267.621415,"high_b":270.700069,"low_b":265.947605,"close_b":269.145813,"volume_b":43683100,"open_bench":667.775554,"close_bench":670.347839},{"date":"2025-11-06","open_a":136.8643,"high_a":137.633149,"low_a":135.797525,"close_a":136.06662,"volume_a":6449400,"open_b":266.904075,"high_b":272.393775,"low_b":266.904075,"close_b":268.77713,"volume_b":51204000,"open_bench":669.249681,"close_bench":663.155457},{"date":"2025-11-07","open_a":137.364045,"high_a":138.44044,"low_a":136.133892,"close_a":137.38327,"volume_a":7592300,"open_b":268.807025,"high_b":271.287882,"low_b":265.788178,"close_b":267.481934,"volume_b":48227400,"open_bench":660.781013,"close_bench":663.80835},{"date":"2025-11-10","open_a":136.941204,"high_a":137.450564,"low_a":135.759092,"close_a":137.066132,"volume_a":6617500,"open_b":268.229883,"high_b":272.986954,"low_b":266.733955,"close_b":268.698608,"volume_b":41312400,"open_bench":670.011443,"close_bench":674.166626},{"date":"2025-11-11","open_a":137.700421,"high_a":140.083841,"low_a":137.12378,"close_a":139.430328,"volume_a":7683600,"open_b":269.077573,"high_b":275.16102,"low_b":269.06759,"close_b":274.502808,"volume_b":46208300,"open_bench":672.692527,"close_bench":675.709961},{"date":"2025-11-12","open_a":138.93059,"high_a":139.833987,"low_a":138.450061,"close_a":138.767212,"volume_a":5790100,"open_b":274.253476,"high_b":274.981505,"low_b":270.962446,"close_b":272.727631,"volume_b":48398000,"open_bench":677.480861,"close_bench":676.085938},{"date":"2025-11-13","open_a":138.488509,"high_a":140.689326,"low_a":138.056021,"close_a":139.372681,"volume_a":6663300,"open_b":273.3659,"high_b":275.948896,"low_b":271.351395,"close_b":272.209076,"volume_b":49602800,"open_bench":673.236729,"close_bench":664.867004},{"date":"2025-11-14","open_a":140.2088,"high_a":141.121803,"low_a":138.325119,"close_a":140.170364,"volume_a":5770500,"open_b":270.314178,"high_b":275.210852,"low_b":268.868132,"close_b":271.670502,"volume_b":47431300,"open_bench":658.278041,"close_bench":664.758118},{"date":"2025-11-17","open_a":140.429841,"high_a":142.342353,"low_a":139.564895,"close_a":142.073257,"volume_a":7320200,"open_b":268.090272,"high_b":269.755721,"low_b":265.008663,"close_b":266.733948,"volume_b":45018300,"open_bench":662.551956,"close_bench":658.564941},{"date":"2025-11-18","open_a":142.524925,"high_a":143.995343,"low_a":141.70802,"close_a":142.976624,"volume_a":10821500,"open_b":269.257116,"high_b":269.975163,"low_b":264.59981,"close_b":266.71405,"volume_b":45677300,"open_bench":655.033066,"close_bench":653.034668},{"date":"2025-11-19","open_a":142.524948,"high_a":143.005477,"low_a":140.977644,"close_a":141.381287,"volume_a":8123200,"open_b":264.80919,"high_b":271.471049,"low_b":264.779272,"close_b":267.830963,"volume_b":40424500,"open_bench":653.727143,"close_bench":655.557373},{"date":"2025-11-20","open_a":140.891143,"high_a":141.352447,"low_a":139.257332,"close_a":140.410614,"volume_a":6878900,"open_b":270.094806,"high_b":274.682325,"low_b":265.198161,"close_b":265.527252,"volume_b":45823600,"open_bench":665.727665,"close_bench":645.565247},{"date":"2025-11-21","open_a":140.583612,"high_a":143.140031,"low_a":140.295292,"close_a":140.631668,"volume_a":7911800,"open_b":265.228052,"high_b":272.587992,"low_b":264.948813,"close_b":270.752991,"volume_b":59030800,"open_bench":648.058289,"close_bench":651.99585},{"date":"2025-11-24","open_a":140.151151,"high_a":140.641285,"low_a":139.122811,"close_a":139.834,"volume_a":9391400,"open_b":270.164628,"high_b":276.248075,"low_b":270.164628,"close_b":275.171021,"volume_b":65585800,"open_bench":655.616775,"close_bench":661.592285},{"date":"2025-11-25","open_a":140.247229,"high_a":141.390891,"low_a":139.901262,"close_a":140.420227,"volume_a":8954000,"open_b":274.522773,"high_b":279.618917,"low_b":274.502838,"close_b":276.21817,"volume_b":46914200,"open_bench":661.493314,"close_bench":667.815125},{"date":"2025-11-26","open_a":140.622062,"high_a":142.688331,"low_a":140.535555,"close_a":142.217407,"volume_a":5180600,"open_b":276.208175,"high_b":278.771206,"low_b":275.879084,"close_b":276.79657,"volume_b":33431400,"open_bench":670.397308,"close_bench":672.425415},{"date":"2025-11-28","open_a":141.496592,"high_a":143.024686,"low_a":141.496592,"close_a":142.9478,"volume_a":2969300,"open_b":276.507368,"high_b":278.242635,"low_b":275.240796,"close_b":278.093048,"volume_b":20135600,"open_bench":673.592861,"close_bench":676.095886},{"date":"2025-12-01","open_a":142.842114,"high_a":143.985776,"low_a":142.726777,"close_a":143.687836,"volume_a":7623300,"open_b":277.255333,"high_b":282.650651,"low_b":275.390414,"close_b":282.331512,"volume_b":46587700,"open_bench":671.564717,"close_bench":673.009155},{"date":"2025-12-02","open_a":143.639781,"high_a":143.687823,"low_a":140.823873,"close_a":142.822876,"volume_a":6830300,"open_b":282.231774,"high_b":286.619823,"low_b":281.862783,"close_b":285.413116,"volume_b":53669500,"open_bench":674.641468,"close_bench":674.255676},{"date":"2025-12-03","open_a":142.899752,"high_a":144.831474,"low_a":142.150128,"close_a":142.294281,"volume_a":5109200,"open_b":285.423116,"high_b":287.836529,"low_b":282.530963,"close_b":283.378662,"volume_b":43538700,"open_bench":673.305944,"close_bench":676.590515},{"date":"2025-12-04","open_a":141.996372,"high_a":142.476902,"low_a":140.439463,"close_a":141.189087,"volume_a":6803000,"open_b":283.328783,"high_b":283.957078,"low_b":277.833731,"close_b":279.938019,"volume_b":43989100,"open_bench":677.985405,"close_bench":677.085144},{"date":"2025-12-05","open_a":141.318136,"high_a":141.492827,"low_a":140.240927,"close_a":140.73587,"volume_a":5971300,"open_b":279.778487,"high_b":280.376864,"low_b":277.295225,"close_b":278.023254,"volume_b":47265800,"open_bench":678.153594,"close_bench":678.371277},{"date":"2025-12-08","open_a":140.726171,"high_a":142.230385,"low_a":139.940102,"close_a":141.327866,"volume_a":6327400,"open_b":277.374989,"high_b":278.910817,"low_b":275.400353,"close_b":277.135651,"volume_b":38211800,"open_bench":679.261741,"close_bench":676.333313},{"date":"2025-12-09","open_a":142.026586,"high_a":143.200833,"low_a":139.658665,"close_a":140.367096,"volume_a":10035200,"open_b":277.404933,"high_b":279.269851,"low_b":276.168308,"close_b":276.427582,"volume_b":32193300,"open_bench":675.858445,"close_bench":675.749573},{"date":"2025-12-10","open_a":143.404641,"high_a":145.355257,"low_a":142.657384,"close_a":145.277618,"volume_a":18526400,"open_b":276.996052,"high_b":278.990623,"low_b":275.68961,"close_b":278.023254,"volume_b":33038300,"open_bench":675.274666,"close_bench":680.231201},{"date":"2025-12-11","open_a":145.71431,"high_a":146.432454,"low_a":143.753994,"close_a":144.6371,"volume_a":7854000,"open_b":278.342371,"high_b":278.831031,"low_b":273.066723,"close_b":277.275269,"volume_b":33248000,"open_bench":677.827193,"close_bench":681.814148},{"date":"2025-12-12","open_a":145.326133,"high_a":146.218952,"low_a":144.345975,"close_a":146.199539,"volume_a":8143100,"open_b":277.145654,"high_b":278.462079,"low_b":276.068599,"close_b":277.524628,"volume_b":39532900,"open_bench":680.824833,"close_bench":674.483276},{"date":"2025-12-15","open_a":146.480989,"high_a":147.218532,"low_a":145.840483,"close_a":146.752716,"volume_a":8409600,"open_b":279.389487,"high_b":279.389487,"low_b":272.099334,"close_b":273.365875,"volume_b":50409100,"open_bench":678.420708,"close_bench":673.464172},{"date":"2025-12-16","open_a":147.499979,"high_a":147.849344,"low_a":145.559062,"close_a":145.927826,"volume_a":7520300,"open_b":272.079413,"high_b":274.752131,"low_b":271.052211,"close_b":273.864532,"volume_b":37648600,"open_bench":671.980228,"close_bench":671.624084},{"date":"2025-12-17","open_a":145.937533,"high_a":146.772128,"low_a":145.064112,"close_a":145.646393,"volume_a":8067100,"open_b":274.263459,"high_b":275.410331,"low_b":270.902612,"close_b":271.102051,"volume_b":50138700,"open_bench":672.633198,"close_bench":664.233826},{"date":"2025-12-18","open_a":145.374672,"high_a":145.927825,"low_a":144.840917,"close_a":144.957367,"volume_a":6643200,"open_b":272.86727,"high_b":272.887235,"low_b":266.225375,"close_b":271.451141,"volume_b":51630700,"open_bench":670.367639,"close_bench":669.249695},{"date":"2025-12-19","open_a":144.743856,"high_a":145.462,"low_a":143.530783,"close_a":143.783112,"volume_a":17543500,"open_b":271.411231,"high_b":273.854592,"low_b":269.167338,"close_b":272.927124,"volume_b":144632000,"open_bench":671.346369,"close_bench":675.315369},{"date":"2025-12-22","open_a":143.171733,"high_a":144.578895,"low_a":142.084825,"close_a":142.705917,"volume_a":8818700,"open_b":272.11929,"high_b":273.13654,"low_b":269.775693,"close_b":270.234436,"volume_b":36571800,"open_bench":678.639344,"close_bench":679.522461},{"date":"2025-12-23","open_a":142.841774,"high_a":142.841774,"low_a":139.319005,"close_a":139.435455,"volume_a":7254800,"open_b":270.104803,"high_b":271.760301,"low_b":268.828279,"close_b":271.620667,"volume_b":29642000,"open_bench":678.619507,"close_bench":682.628235},{"date":"2025-12-24","open_a":139.542209,"high_a":139.736313,"low_a":138.727028,"close_a":139.493698,"volume_a":2618700,"open_b":271.600701,"high_b":274.682309,"low_b":271.461097,"close_b":273.066711,"volume_b":17910600,"open_bench":682.61832,"close_bench":685.02948},{"date":"2025-12-26","open_a":139.318999,"high_a":139.687778,"low_a":138.862883,"close_a":139.532501,"volume_a":4975500,"open_b":273.415753,"high_b":274.62246,"low_b":272.119264,"close_b":272.657806,"volume_b":21521800,"open_bench":685.287481,"close_bench":684.960022},{"date":"2025-12-29","open_a":139.493698,"high_a":140.764997,"low_a":139.357835,"close_a":139.978928,"volume_a":7201900,"open_b":271.949773,"high_b":273.615223,"low_b":271.6107,"close_b":273.016876,"volume_b":23715200,"open_bench":682.211448,"close_bench":682.519043},{"date":"2025-12-30","open_a":139.716901,"high_a":140.9785,"low_a":139.668376,"close_a":139.901291,"volume_a":4713800,"open_b":272.069428,"high_b":273.335969,"low_b":271.540868,"close_b":272.338684,"volume_b":22139600,"open_bench":682.122139,"close_bench":681.685547},{"date":"2025-12-31","open_a":139.581037,"high_a":139.775126,"low_a":138.930832,"close_a":139.280197,"volume_a":4380300,"open_b":272.318764,"high_b":272.937076,"low_b":271.012322,"close_b":271.122009,"volume_b":27293600,"open_bench":681.814586,"close_bench":676.63501}]')
df = pd.DataFrame(DF_RECORDS)
csv_bytes = df.to_csv(index=False).encode('utf-8')
slice_sha256 = hashlib.sha256(csv_bytes).hexdigest()
assert slice_sha256 == EXPECTED_SNAPSHOT['aligned_slice_sha256'], (
    f'Data slice SHA-256 mismatch: {slice_sha256} != {EXPECTED_SNAPSHOT["aligned_slice_sha256"]}'
)
assert df['date'].is_monotonic_increasing, 'Dates must be strictly monotonic increasing!'
assert not df.isna().any().any(), 'Aligned DataFrame contains unexpected NaN values!'

n_bars = len(df)
split_idx = max(25, min(n_bars - 25, int(round(n_bars * spec.execution.train_split_ratio))))
split_date = str(df['date'].iloc[split_idx])
assert split_date == EXPECTED_SNAPSHOT['split_date'], f'Split date mismatch: {split_date}'

print(f'Verified {n_bars} aligned trading days ({df["date"].iloc[0]} to {df["date"].iloc[-1]})')
print(f'Formation Split [0:{split_idx}]: {df["date"].iloc[0]} to {df["date"].iloc[split_idx - 1]} ({split_idx} days)')
print(f'Out-of-Sample Split [{split_idx}:{n_bars}]: {split_date} to {df["date"].iloc[-1]} ({n_bars - split_idx} days)')
display(df.head(5))

Verified 2011 aligned trading days (2018-01-02 to 2025-12-31)
Formation Split [0:1006]: 2018-01-02 to 2021-12-29 (1006 days)
Out-of-Sample Split [1006:2011]: 2021-12-30 to 2025-12-31 (1005 days)


,date,open_a,high_a,low_a,close_a,volume_a,open_b,high_b,low_b,close_b,volume_b,open_bench,close_bench
0,2018-01-02,91.111547,91.301474,89.592135,89.690895,4165800,39.741914,40.241724,39.531711,40.232380,102223600,234.555390,235.369812
1,2018-01-03,89.576947,90.002380,89.371823,89.455391,4224000,40.295440,40.767224,40.162315,40.225372,118071600,235.536252,236.858612
2,2018-01-04,89.493391,90.488604,89.341446,89.896034,3147000,40.297769,40.514978,40.190335,40.412212,89738400,237.497832,237.856857
3,2018-01-05,89.956831,90.473426,89.835275,90.154350,2714200,40.507983,40.958744,40.416896,40.872330,94640000,238.645069,239.441986
4,2018-01-08,89.804870,90.207513,89.394628,89.637733,5730400,40.720509,41.014788,40.622412,40.720509,82271200,239.345582,239.879791


## Cell 4 — Strictly Causal Signal Construction & Prefix-Invariance Proof
Computes the signal series at bar $t$ using only prices $[0..t]$. In `ols_log` pairs mode, OLS parameters $(\hat{\alpha}_t, \hat{\beta}_t)$ are estimated on expanding windows $[0..t]$ inside the Formation split and frozen at `split_idx` for Out-of-Sample evaluation. We prove zero look-ahead bias by mutating future prices both inside Formation ($m = \lfloor k/2 \rfloor$) and at the split boundary ($m = k$) and asserting that all prior signals $[0..m)$ are 100% invariant.

In [4]:
# Cell 4: Compute Causal Signals & Prove Prefix Invariance
precomputed = compute_causal_signals(df, spec, split_idx)
signal_values, indicators, pairs_diag, ols_alpha, ols_beta = precomputed

# Prefix-Invariance Test 1: Mutate prices in second half of Formation [m_form : split_idx)
m_form = max(35, split_idx // 2)
df_mut_form = df.copy(deep=True)
df_mut_form.loc[m_form:, 'close_a'] *= np.linspace(1.0, 1.45, n_bars - m_form)
sig_mut_form, _, _, _, _ = compute_causal_signals(df_mut_form, spec, split_idx)
mask_form = np.isfinite(signal_values[:m_form])
assert np.allclose(signal_values[:m_form][mask_form], sig_mut_form[:m_form][mask_form], atol=1e-12), (
    'Look-ahead violation detected inside Formation window!'
)

# Prefix-Invariance Test 2: Mutate Out-of-Sample prices [split_idx : n_bars)
df_mut_oos = df.copy(deep=True)
df_mut_oos.loc[split_idx:, 'close_a'] *= 1.75
sig_mut_oos, _, _, _, _ = compute_causal_signals(df_mut_oos, spec, split_idx)
mask_oos = np.isfinite(signal_values[:split_idx])
assert np.allclose(signal_values[:split_idx][mask_oos], sig_mut_oos[:split_idx][mask_oos], atol=1e-12), (
    'Look-ahead violation detected across split boundary!'
)
print(f'Causal prefix-invariance verified at m_form={m_form} ({df["date"].iloc[m_form]}) and split_idx={split_idx} ({split_date}).')

Causal prefix-invariance verified at m_form=503 (2020-01-02) and split_idx=1006 (2021-12-30).


## Cell 5 — Walk-Forward Trade Simulation & Exact Compounding Reconciliation
Executes signals triggered at the close of bar $t$ on bar $t+1$ (`next_open` or `next_close`) with unit gross portfolio exposure ($|w_A| + |w_B| = 1.0$) and explicit commission + slippage fees. Verifies that compounded per-trade net returns reconcile with terminal portfolio net equity.

In [5]:
# Cell 5: Run Bar-by-Bar Walk-Forward Simulation & Verify Trade Ledger
sim = simulate_strategy_core(df, spec, split_idx, precomputed_signals=precomputed)
trades = sim['trades']
equity_net = sim['equity_net']
equity_gross = sim['equity_gross']
daily_net_ret = sim['daily_net_ret']
daily_gross_ret = sim['daily_gross_ret']
daily_bench_ret = sim['daily_bench_ret']
daily_pos = sim['daily_position']

# Reconcile compounded daily returns vs terminal equity
initial_cap = spec.execution.initial_capital
assert math.isclose(initial_cap * float(np.prod(1.0 + daily_net_ret)), float(equity_net[-1]), rel_tol=1e-9)

# Reconcile compounded trade net returns vs terminal equity
if trades:
    compounded_trades_eq = initial_cap * float(np.prod([1.0 + t.net_return_pct / 100.0 for t in trades]))
    assert math.isclose(compounded_trades_eq, float(equity_net[-1]), rel_tol=1e-3), (
        f'Trade ledger vs daily equity mismatch: {compounded_trades_eq:.2f} vs {equity_net[-1]:.2f}'
    )
    trades_df = pd.DataFrame([t.model_dump() for t in trades])
    print(f'Executed {len(trades)} trades. Terminal Net Equity: ${equity_net[-1]:,.2f} (Compounded Trade Ledger: ${compounded_trades_eq:,.2f})')
    display(trades_df[['trade_id', 'split', 'direction', 'signal_date', 'entry_date', 'exit_date', 'holding_days', 'leg_weight_a', 'leg_weight_b', 'gross_return_pct', 'cost_pct', 'net_return_pct', 'exit_reason', 'cumulative_equity_after']].head(15))
else:
    print('Zero trades triggered across sample.')

Executed 12 trades. Terminal Net Equity: $75,617.32 (Compounded Trade Ledger: $75,617.35)


,trade_id,split,direction,signal_date,entry_date,exit_date,holding_days,leg_weight_a,leg_weight_b,gross_return_pct,cost_pct,net_return_pct,exit_reason,cumulative_equity_after
0,1,in_sample,long_spread,2018-02-06,2018-02-07,2018-03-08,20,0.8349,-0.1651,-3.5406,0.2,-3.7406,spread_convergence (z=0.04 >= -0.25),96259.39
1,2,in_sample,short_spread,2018-03-13,2018-03-14,2018-03-20,4,-0.9524,0.0476,2.2620,0.2,2.0620,spread_convergence (z=0.04 <= +0.25),98244.29
2,3,in_sample,long_spread,2018-03-23,2018-03-26,2018-03-29,3,0.9524,-0.0476,3.0718,0.2,2.8718,spread_convergence (z=-0.23 >= -0.25),101065.64
3,4,in_sample,long_spread,2018-04-19,2018-04-20,2018-05-17,19,0.9524,-0.0476,-7.5990,0.2,-7.7990,spread_convergence (z=0.30 >= -0.25),93183.53
4,5,in_sample,short_spread,2018-05-21,2018-05-22,2018-06-20,20,-0.6716,-0.3284,-4.1699,0.2,-4.3699,holding_period_reached (20d),89111.47
5,6,in_sample,short_spread,2018-07-10,2018-07-11,2018-08-08,20,-0.6922,-0.3078,-4.0332,0.2,-4.2332,spread_convergence (z=0.23 <= +0.25),85339.19
6,7,in_sample,long_spread,2018-10-02,2018-10-03,2018-10-19,12,0.8884,-0.1116,0.0715,0.2,-0.1285,spread_convergence (z=-0.21 >= -0.25),85229.55
7,8,in_sample,short_spread,2018-10-24,2018-10-25,2018-11-23,20,-0.9114,0.0886,-3.5123,0.2,-3.7123,holding_period_reached (20d),82065.61
8,9,in_sample,short_spread,2018-11-30,2018-12-03,2018-12-10,4,-0.9095,0.0905,1.4053,0.2,1.2053,spread_convergence (z=0.18 <= +0.25),83054.75
9,10,in_sample,long_spread,2018-12-18,2018-12-19,2019-01-09,13,0.9344,-0.0656,-1.3439,0.2,-1.5439,spread_convergence (z=-0.19 >= -0.25),81772.43


## Cell 6 — Running High-Water Mark ($E_0$ Included) & Maximum Drawdown Verification
Verifies the exact drawdown equations:
$$E_0 = C_0, \quad H_t = \max\!\left(E_0, \max_{1 \le s \le t} E_s\right), \quad D_t = 1 - \frac{E_t}{H_t}, \quad \mathrm{MDD} = \max_{1 \le t \le T} D_t$$
where $D_t \in [0, 1]$ is reported as a positive loss percentage in tables and $U_t = -D_t \le 0\%$ is plotted on underwater charts.

In [6]:
# Cell 6: Verify Running High-Water Mark Including Initial Capital E_0
running_peak = np.maximum(initial_cap, np.maximum.accumulate(equity_net))
drawdown_loss_pct = ((running_peak - equity_net) / running_peak) * 100.0
underwater_series_pct = -drawdown_loss_pct
max_dd_full = float(np.max(drawdown_loss_pct))

print(f'Full-Sample Max Drawdown (positive loss magnitude): {max_dd_full:.2f}%')
print(f'Underwater Chart Minimum (signed trough): {float(np.min(underwater_series_pct)):.2f}%')
if sim['circuit_breaker_tripped']:
    print(f'Portfolio Circuit Breaker Triggered on: {sim["circuit_breaker_date"]} (Limit: {spec.risk_controls.max_drawdown_circuit_breaker_pct}%)')

Full-Sample Max Drawdown (positive loss magnitude): 25.36%
Underwater Chart Minimum (signed trough): -25.36%
Portfolio Circuit Breaker Triggered on: 2019-04-15 (Limit: 25.0%)


## Cell 7 — Split Performance & Risk Metrics Table
Computes `PeriodMetrics` across In-Sample Formation, Held-Out Out-of-Sample, and Full Sample, explicitly flagging zero-exposure / prior-halt splits.

In [7]:
# Cell 7: Compute PeriodMetrics for Full Sample, In-Sample, and Out-of-Sample
dates = sim['dates']
is_trades = [t for t in trades if t.split == 'in_sample']
oos_trades = [t for t in trades if t.split == 'out_of_sample']
cb_tripped = bool(sim['circuit_breaker_tripped'])
cb_date = sim['circuit_breaker_date']
cb_in_is = bool(cb_tripped and cb_date is not None and cb_date < split_date)
cb_in_oos = bool(cb_tripped and cb_date is not None and cb_date >= split_date)

full_metrics = _compute_period_metrics('Full Sample', dates, daily_gross_ret, daily_net_ret, daily_bench_ret, daily_pos, trades, cb_tripped, cb_date)
is_metrics = _compute_period_metrics('In-Sample Formation', dates[:split_idx], daily_gross_ret[:split_idx], daily_net_ret[:split_idx], daily_bench_ret[:split_idx], daily_pos[:split_idx], is_trades, cb_in_is, cb_date if cb_in_is else None, halted_prior_to_period=False)
oos_metrics = _compute_period_metrics('Out-of-Sample (Held-Out)', dates[split_idx:], daily_gross_ret[split_idx:], daily_net_ret[split_idx:], daily_bench_ret[split_idx:], daily_pos[split_idx:], oos_trades, cb_in_oos, cb_date if cb_tripped else None, halted_prior_to_period=cb_in_is)

def _fmt_sr(m):
    if m.is_zero_exposure:
        return 'N/A (0 trades — Halted in Formation)' if m.halted_prior_to_period else 'N/A (0 trades)'
    return f'{m.sharpe_ratio:.3f} (PSR {m.probabilistic_sharpe_psr_pct:.1f}%)'

period_summary_df = pd.DataFrame([
    {
        'Period': m.period_label,
        'Date Range': f'{m.start_date} -> {m.end_date}',
        'Days': m.trading_days,
        'Trades (N)': m.trade_count,
        'Exposure (%)': f'{m.exposure_pct:.1f}%',
        'Net Return (%)': f'{m.total_net_return_pct:+.2f}%',
        'Benchmark (%)': f'{m.benchmark_return_pct:+.2f}%',
        'Excess (pp)': f'{m.excess_return_pct:+.2f} pp',
        'Max DD (%)': f'{m.max_drawdown_pct:.2f}%',
        'Sharpe / PSR': _fmt_sr(m),
        'Circuit Breaker': m.circuit_breaker_date or ('Halted Prior' if m.halted_prior_to_period else 'Active'),
    }
    for m in (oos_metrics, is_metrics, full_metrics)
])
display(period_summary_df)

,Period,Date Range,Days,Trades (N),Exposure (%),Net Return (%),Benchmark (%),Excess (pp),Max DD (%),Sharpe / PSR,Circuit Breaker
0,Out-of-Sample (Held-Out),2021-12-30 -> 2025-12-31,1005,0,0.0%,+0.00%,+50.98%,-50.98 pp,0.00%,N/A (0 trades — Halted in Formation),2019-04-15
1,In-Sample Formation,2018-01-02 -> 2021-12-29,1006,12,17.3%,-24.38%,+90.41%,-114.79 pp,25.36%,-1.106 (PSR 1.0%),2019-04-15
2,Full Sample,2018-01-02 -> 2025-12-31,2011,12,8.7%,-24.38%,+187.48%,-211.86 pp,25.36%,-0.782 (PSR 1.0%),2019-04-15


## Cell 8 — Econometric & Non-IID Statistical Inference
Computes Lo (2002) / Mertens (2002) non-normality-adjusted Sharpe standard errors, Newey-West (1987) HAC factor regression against the benchmark, Politis & Romano (1994) Stationary Bootstrap confidence intervals, and (for pairs) Engle-Granger / ADF cointegration and discrete Ornstein-Uhlenbeck half-life $t_{1/2} = -\ln(2)/\ln(1 + \hat{\lambda})$.

In [8]:
# Cell 8: Econometric Diagnostics (Newey-West HAC, Stationary Bootstrap, Pairs Cointegration)
factor_reg = newey_west_factor_regression(daily_net_ret, daily_bench_ret)
boot_cis = stationary_bootstrap_cis(daily_net_ret, daily_bench_ret, [t.net_return_pct for t in trades], replications=1000, seed=BOOTSTRAP_SEED)

print(f'Newey-West HAC Annualized Alpha: {factor_reg.annualized_alpha_pct:+.2f}% (t_HAC = {factor_reg.alpha_t_stat_hac:.2f}, p = {factor_reg.alpha_p_value_hac:.4f}), Beta = {factor_reg.beta:.3f}')
boot_df = pd.DataFrame([b.model_dump() for b in boot_cis])
display(boot_df[['metric_name', 'point_estimate', 'ci_lower_95', 'ci_upper_95', 'bootstrap_std_error', 'block_length', 'replications']])

if pairs_diag is not None:
    print('\nPairs Cointegration & Co-Movement Diagnostics (Formation Window):')
    pairs_df = pd.DataFrame([{
        'Daily Return Corr (r)': round(pairs_diag.return_correlation, 4),
        'Log-Level Corr': round(pairs_diag.level_correlation, 4),
        'OLS Beta': round(pairs_diag.ols_hedge_ratio_beta, 4),
        'OLS Alpha': round(pairs_diag.ols_alpha, 4),
        'ADF Stat': round(pairs_diag.adf_statistic, 3),
        'ADF p-value': round(pairs_diag.adf_p_value, 4),
        'OU Lambda': round(pairs_diag.ou_lambda, 5),
        'Half-Life (days)': pairs_diag.half_life_days,
        'Hurst H': round(pairs_diag.hurst_exponent, 3),
    }])
    display(pairs_df)

Newey-West HAC Annualized Alpha: -3.59% (t_HAC = -2.34, p = 0.0191), Beta = 0.012


,metric_name,point_estimate,ci_lower_95,ci_upper_95,bootstrap_std_error,block_length,replications
0,Annualized Return (Mean),-3.410,-6.820,-0.440,1.620,13,1000
1,Annualized Sharpe Ratio,-0.782,-1.366,-0.112,0.312,13,1000
2,Annualized Excess Return vs. Benchmark,-18.540,-30.090,-6.960,6.010,13,1000
3,Mean Net Return per Trade,-2.255,-3.862,-0.567,0.875,1,1000



Pairs Cointegration & Co-Movement Diagnostics (Formation Window):


,Daily Return Corr (r),Log-Level Corr,OLS Beta,OLS Alpha,ADF Stat,ADF p-value,OU Lambda,Half-Life (days),Hurst H
0,0.5079,0.8924,0.2899,3.4155,-1.981,0.2948,-0.01364,50.5,0.492


## Cell 9 — Standard Literature Threshold Bracket & 25-Trial Parameter Surface (DSR)
Evaluates Conservative, Baseline, and Aggressive literature profiles alongside the $5 \times 5$ parameter sensitivity grid ($K=25$ trials) and Bailey & López de Prado (2014) Deflated Sharpe Ratio.

In [9]:
# Cell 9: Compute Standard Bracket Profiles & 5x5 Parameter Sensitivity Surface
bracket_profiles = compute_standard_bracket_profiles(df, spec, split_idx)
param_surface = compute_parameter_sensitivity_surface(df, spec, split_idx) if 'parameter_sensitivity_grid' in spec.selected_diagnostics else None

bracket_df = pd.DataFrame([
    {
        'Profile': bp.profile_label,
        'Entry': bp.entry_threshold,
        'Exit': bp.exit_threshold,
        'IS Net (%)': bp.in_sample_metrics.total_net_return_pct,
        'OOS Net (%)': bp.out_of_sample_metrics.total_net_return_pct,
        'Full Net (%)': bp.full_sample_metrics.total_net_return_pct,
        'Full Sharpe': bp.full_sample_metrics.sharpe_ratio,
        'Max DD (%)': bp.full_sample_metrics.max_drawdown_pct,
    }
    for bp in bracket_profiles
])
display(bracket_df)
if param_surface is not None:
    print(f'25-Cell Grid Expected Max SR under Null: {param_surface.expected_max_sr_under_null:.3f} | Selected Spec DSR: {param_surface.selected_spec_deflated_sharpe_psr_pct:.1f}%')

,Profile,Entry,Exit,IS Net (%),OOS Net (%),Full Net (%),Full Sharpe,Max DD (%)
0,Aggressive (1.5σ Entry),1.5,0.50,-25.10,0.00,-25.10,-0.746,25.40
1,Baseline Literature (2.0σ Entry),2.0,0.25,-24.38,0.00,-24.38,-0.782,25.36
2,Conservative Tail (2.5σ Entry),2.5,0.00,0.15,-3.42,-3.27,-0.028,18.66


25-Cell Grid Expected Max SR under Null: 0.784 | Selected Spec DSR: 0.0%


## Cell 10 — Five Adversarial Stress Tests & Non-Overlapping Monte Carlo Null
Runs all 5 adversarial stress tests: (1) Right-Tail Outlier Trade Removal, (2) $t+2$ Execution Latency & $2\times/3\times$ Slippage Shock, (3) Realized Volatility & Bull/Bear Macro Regime Split, (4) 500-Run Matched Non-Overlapping Monte Carlo Random-Entry Benchmark, and (5) Out-of-Sample Cointegration Structural Break / Parameter Cliff-Edge.

In [10]:
# Cell 10: Execute Adversarial Stress Test Suite
def _stress_sim_cb(d_frame, s_spec, s_idx, cost_override, delay_bars):
    return simulate_strategy_core(
        d_frame, s_spec, s_idx,
        override_cost_bps_per_leg=cost_override,
        entry_delay_bars=delay_bars,
        precomputed_signals=precomputed,
    )

stress_suite = run_adversarial_stress_suite(
    df=df, spec=spec, split_idx=split_idx, base_sim=sim,
    full_metrics=full_metrics, oos_metrics=oos_metrics,
    pairs_diag=pairs_diag, parameter_surface=param_surface,
    simulate_fn=_stress_sim_cb,
)
stress_df = pd.DataFrame([
    {
        'Test ID': t.test_id,
        'Title': t.title,
        'Outcome': t.outcome_badge,
        'Baseline': f'{t.baseline_label}: {t.baseline_value}',
        'Stressed': f'{t.stressed_label}: {t.stressed_value}',
    }
    for t in stress_suite.tests
])
display(stress_df)

,Test ID,Title,Outcome,Baseline,Stressed
0,outlier_removal,Right-Tail Outlier Concentration Test (Excl. T...,HIGH_VULNERABILITY,All Trades Compounded: -24.38% (N=12),Excl. Top 3 Trades: -28.84% (N=9)
1,execution_latency_shock,Execution Latency (t+2 Fill Delay) & Liquidity...,HIGH_VULNERABILITY,"Baseline (t+1, 10 bps): -24.38% (SR -0.78)",t+2 Delay + 2× Cost: -25.37% (SR -0.88)
2,regime_vulnerability,Realized Volatility Terciles & Bull/Bear Macro...,HIGH_VULNERABILITY,Positive Regimes: 0 / 5 Regimes,High-Vol vs Low-Vol SR: -0.39 vs -1.38
3,monte_carlo_random_entry,Monte Carlo Random-Entry Null Benchmark (500 N...,HIGH_VULNERABILITY,Actual Signal Return: -24.38% (6th pct),Random-Entry Median: -5.78% (p=0.944±0.010)
4,pairs_structural_break,Out-of-Sample Hedge Ratio Drift & Cointegratio...,HIGH_VULNERABILITY,In-Sample β / ADF p: β=0.290 (p=0.295),OOS β Drift / ADF p: Δβ=138.5% (p=0.721)


## Cell 11 — Publication Figures (Equity Curve, Underwater Drawdown & Causal Signal Trajectory)
Generates the primary visual diagnostics with explicit Out-of-Sample split and circuit-breaker annotations.

In [11]:
# Cell 11: Plot Equity Curve, Underwater Drawdown, and Causal Signal Trajectory
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

bench_equity = initial_cap * np.cumprod(1.0 + daily_bench_ret)
x_idx = np.arange(n_bars)
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(x_idx, equity_net, color='#059669', lw=1.8, label='Confirmed Rule (Net)')
axes[0].plot(x_idx, bench_equity, color='#78716c', lw=1.3, ls='--', label=f'Benchmark ({spec.benchmark_ticker})')
axes[0].axvline(split_idx, color='#d97706', ls=':', lw=1.2, label=f'OOS Split ({split_date})')
axes[0].set_ylabel('Portfolio Equity ($)')
axes[0].set_title(f'{spec.title} — Walk-Forward Equity, Underwater Drawdown & Causal Signal')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].grid(True, alpha=0.25)

axes[1].fill_between(x_idx, underwater_series_pct, 0.0, color='#e11d48', alpha=0.22)
axes[1].plot(x_idx, underwater_series_pct, color='#e11d48', lw=1.2, label='Underwater Drawdown (%)')
axes[1].axvline(split_idx, color='#d97706', ls=':', lw=1.2)
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend(loc='lower left', fontsize=8)
axes[1].grid(True, alpha=0.25)

axes[2].plot(x_idx, signal_values, color='#0284c7', lw=1.1, label=f'Causal Signal ({spec.signal_variant})')
axes[2].axvline(split_idx, color='#d97706', ls=':', lw=1.2)
axes[2].set_ylabel('Signal Value')
axes[2].set_xlabel('Trading Bar Index')
axes[2].legend(loc='upper left', fontsize=8)
axes[2].grid(True, alpha=0.25)

plt.tight_layout()
plt.close(fig)
print('Generated 3-panel Matplotlib diagnostic figure successfully.')

Generated 3-panel Matplotlib diagnostic figure successfully.


## Cell 12 — Empirical Hypothesis Scorecard, Multi-Question Breakdown & Formula Ledger
Synthesizes the 3-layer hypothesis comparison (separating return co-movement, cointegration stationarity, and trading rule P&L), the 5 scorecard dimensions, and the step-by-step substituted formula ledger.

In [12]:
# Cell 12: Build Empirical Hypothesis Scorecard & Formula Calculation Ledger
if len(trades) > 0 and full_metrics.total_gross_return_pct > 0:
    avg_gross_bps = float(np.mean([t.gross_return_pct for t in trades])) * 100.0
    break_even_bps = round(max(0.0, avg_gross_bps / 2.0), 1)
else:
    break_even_bps = 0.0

scorecard = build_hypothesis_scorecard_and_ledger(
    spec=spec,
    full_metrics=full_metrics,
    in_metrics=is_metrics,
    oos_metrics=oos_metrics,
    factor_regression=factor_reg,
    bootstrap_cis=boot_cis,
    break_even_cost_bps=break_even_bps,
    parameter_surface=param_surface,
    pairs_diag=pairs_diag,
    stress_suite=stress_suite,
)

print(f'Historical Verdict: {scorecard.historical_verdict_label} ({scorecard.supported_checks_count}/{scorecard.evaluated_checks_count} evaluated checks supported)')
print(scorecard.executive_summary)

if scorecard.hypothesis_comparison_breakdown:
    display(pd.DataFrame(scorecard.hypothesis_comparison_breakdown))

display(pd.DataFrame([d.model_dump() for d in scorecard.dimensions]))
display(pd.DataFrame([s.model_dump() for s in scorecard.calculation_steps]))

Historical Verdict: CONTRADICTED BY HISTORICAL DATA (0/4 evaluated checks supported)
Historical Audit Summary for 'I think that pepsi and apple stock move together': Across 2011 trading days (2018-01-02 to 2025-12-31), the specified rule triggered 12 trades (In-Sample N = 12, Out-of-Sample N = 0), generating -24.38% net return (Sharpe -0.78, Max Drawdown 25.36%) versus +187.48% (Sharpe 0.78) for passive SPY Buy-and-Hold (-211.86 percentage points excess return). The 25.0% portfolio drawdown circuit breaker tripped on 2019-04-15, halting all subsequent entries and resulting in 0 Out-of-Sample trades (2021-12-30 to 2025-12-31). Econometrically, PEP and AAPL exhibit daily return correlation r = +0.508, while the log-price spread ADF stationarity test yields p = 0.2948 (non-stationary drifting spread). The trading rule satisfied 0 of 4 evaluated empirical checks (1 check had NO_EVIDENCE due to 0 holdout trades) (CONTRADICTED BY HISTORICAL DATA). Note: This scorecard summarizes historical s

,aspect,lens_title,question_answered,evidence,primary_metric,verdict,status,explanation,interpretation
0,1. Return Co-Movement (Daily Correlation),1. Return Co-Movement (Daily Correlation),Do PEP and AAPL daily returns move in the same...,Daily Return r = +0.508 | Log-Price R² = 0.784...,Daily Return r = +0.508 | Log-Price R² = 0.784...,POSITIVE CO-MOVEMENT,SUPPORTED,PEP and AAPL daily returns exhibit r = +0.508 ...,PEP and AAPL daily returns exhibit r = +0.508 ...
1,2. Long-Run Equilibrium Cointegration (Station...,2. Long-Run Equilibrium Cointegration (Station...,Is the log-price spread between PEP and AAPL s...,ADF Stat = -1.981 (p = 0.2948) | Half-Life = 5...,ADF Stat = -1.981 (p = 0.2948) | Half-Life = 5...,NON-STATIONARY / DRIFTING SPREAD,CONTRADICTED,Formation spread does NOT reject the unit-root...,Formation spread does NOT reject the unit-root...
2,3. Active Spread Mean-Reversion Trading Rule,3. Active Spread Mean-Reversion Trading Rule,Did trading ±2.00σ spread divergences generate...,"Net Return: -24.38% (N = 12 trades, IS N = 12,...","Net Return: -24.38% (N = 12 trades, IS N = 12,...",HALTED IN FORMATION (2019-04-15),NO_EVIDENCE,Trading ±2.00σ log-spread divergences produced...,Trading ±2.00σ log-spread divergences produced...


,dimension_id,name,question,status,evidence_metric,explanation
0,post_signal_direction,1. Post-Signal Direction & Net Trade Expectancy,Did historical occurrences of the signal produ...,CONTRADICTED,Mean Net Trade: -2.255% | Win Rate: 25.0% (N =...,Across 12 historical signals from 2018-01-02 t...
1,statistical_significance,2. Statistical Significance vs. Noise (Lo 2002...,Is the observed return/Sharpe ratio statistica...,CONTRADICTED,Sharpe: -0.78 ± 0.33 (PSR 1.0%) | HAC Alpha t ...,After adjusting for sample skewness (-2.82) an...
2,out_of_sample_persistence,3. Held-Out Out-of-Sample Persistence,Did the hypothesis continue to hold in the uns...,NO_EVIDENCE,"In-Sample SR: -1.11 (-24.4%, N=12) → Out-of-Sa...",No Out-of-Sample trades were executed between ...
3,transaction_cost_survival,4. Transaction Cost & Microstructure Friction ...,Do historical returns survive realistic commis...,CONTRADICTED,Break-Even Cost: 0.0 bps/notional (Configured:...,Deducting 10.0 bps per unit of portfolio notio...
4,stress_test_resilience,5. Adversarial Stress-Test & Null-Benchmark Re...,"Did the signal survive outlier removal, regime...",CONTRADICTED,0/5 Resilient | 0/5 Moderate | 5/5 High Vulner...,"Out of 5 adversarial stress tests in Step 5, t..."


,step_id,title,formula,substituted_inputs,output_value,interpretation
0,calc_net_return,"Step A — Compounded Net Trade Return, Unit Gro...","R_{net,i} = R_{gross,i} − 2·c, |w_A| + |w_B|...","Initial Capital E_0 = $100,000.00, Unit gross ...",Gross = -22.50% → Net = -24.38% (Mean Net/Trad...,Every trade deducts round-trip commission + sl...
1,calc_lo_sharpe,Step B — Annualized Sharpe Ratio & Lo (2002) /...,"SR = √252 × (μ_d / σ_d), SE(SR_d) = √[(1 − γ...","T = 2011 days, Ann. Vol = 4.36%, Skewness γ₃ =...","Annualized SR = -0.782, Lo (2002) SE = 0.334, ...",Adjusts standard Sharpe inference for non-norm...
2,calc_dsr,Step C — Bailey & López de Prado (2014) Multip...,E[max SR₀] ≈ √V[{SR_k}] × [(1 − γ)·Φ⁻¹(1 − 1/K...,"K = 25 parameter grid trials, Null Hurdle E[ma...",Selected Spec DSR = 0.0% (Grid Best DSR = 5.9%),Penalizes the observed Sharpe ratio for the nu...
3,calc_newey_west,Step D — Newey-West (1987) HAC Benchmark Facto...,"R_strat,t = α + β · R_bench,t + ε_t, Var_HAC...","Benchmark = SPY, Bartlett kernel lags L = 7, S...","Annualized α = -3.59% (t_HAC = -2.34, p = 0.01...",Separates passive correlation with the benchma...
4,calc_pairs_coint,Step E — Cointegrated Spread & Exact Discrete ...,"S_t = log(P_A,t) − (hat(α) + hat(β)·log(P_B,t)...",Formation window: 2018-01-02 to 2021-12-29 | h...,"ADF Stat = -1.981 (p = 0.2948), Half-Life t_{1...",Formation spread does NOT reject the unit-root...


## Cell 13 — Automated End-to-End Reconciliation Assertions
Programmatically asserts that every metric, trade count, drawdown, econometric statistic, and verdict recomputed in this notebook matches the API `AuditReport` payload.

In [13]:
# Cell 13: Assert Exact Reconciliation Between Notebook Execution and AuditReport
assert len(trades) == EXPECTED_SNAPSHOT['trade_count'], f'Trade count mismatch: {len(trades)} != {EXPECTED_SNAPSHOT["trade_count"]}'

for split_key, comp_metrics in [
    ('full_sample', full_metrics),
    ('in_sample', is_metrics),
    ('out_of_sample', oos_metrics),
]:
    exp_m = EXPECTED_SNAPSHOT[split_key]
    for field_name in (
        'trading_days', 'trade_count', 'exposure_pct', 'total_gross_return_pct',
        'total_net_return_pct', 'cagr_net_pct', 'benchmark_return_pct',
        'max_drawdown_pct', 'benchmark_max_drawdown_pct', 'sharpe_ratio',
        'probabilistic_sharpe_psr_pct', 'win_rate_pct', 'is_zero_exposure',
        'circuit_breaker_triggered', 'halted_prior_to_period'
    ):
        actual_val = getattr(comp_metrics, field_name)
        expected_val = exp_m[field_name]
        if isinstance(actual_val, float):
            assert math.isclose(actual_val, float(expected_val), abs_tol=1e-6), (
                f'Mismatch in {split_key}.{field_name}: {actual_val} != {expected_val}'
            )
        else:
            assert actual_val == expected_val, (
                f'Mismatch in {split_key}.{field_name}: {actual_val} != {expected_val}'
            )

for exp_b, act_b in zip(EXPECTED_SNAPSHOT['bootstrap_cis'], boot_cis):
    assert exp_b['metric_name'] == act_b.metric_name, f'Bootstrap metric name mismatch: {act_b.metric_name}'
    assert math.isclose(float(exp_b['point_estimate']), act_b.point_estimate, abs_tol=1e-6), f'Bootstrap point estimate mismatch for {act_b.metric_name}'
    assert math.isclose(float(exp_b['ci_lower_95']), act_b.ci_lower_95, abs_tol=1e-6), f'Bootstrap CI lower mismatch for {act_b.metric_name}'
    assert math.isclose(float(exp_b['ci_upper_95']), act_b.ci_upper_95, abs_tol=1e-6), f'Bootstrap CI upper mismatch for {act_b.metric_name}'

for exp_st, act_st in zip(EXPECTED_SNAPSHOT['stress_test_outcomes'], stress_suite.tests):
    assert exp_st['test_id'] == act_st.test_id, f'Stress test ID mismatch: {act_st.test_id}'
    assert exp_st['outcome_badge'] == act_st.outcome_badge, f'Stress test outcome mismatch for {act_st.test_id}'
    assert exp_st['baseline_value'] == act_st.baseline_value, f'Stress test baseline mismatch for {act_st.test_id}'
    assert exp_st['stressed_value'] == act_st.stressed_value, f'Stress test stressed mismatch for {act_st.test_id}'

assert scorecard.historical_verdict_label == EXPECTED_SNAPSHOT['verdict_label'], 'Verdict label mismatch!'
assert scorecard.supported_checks_count == EXPECTED_SNAPSHOT['supported_checks_count'], 'Supported checks count mismatch!'
assert scorecard.evaluated_checks_count == EXPECTED_SNAPSHOT['evaluated_checks_count'], 'Evaluated checks count mismatch!'
for d in scorecard.dimensions:
    assert d.status == EXPECTED_SNAPSHOT['dimension_statuses'][d.dimension_id], f'Dimension {d.dimension_id} status mismatch!'

print('ALL AUTOMATED RECONCILIATION ASSERTIONS PASSED (100% match with API AuditReport).')

ALL AUTOMATED RECONCILIATION ASSERTIONS PASSED (100% match with API AuditReport).


## Cell 14 — Complete UI-to-Notebook-to-Engine Coverage Map
Maps every metric, table, chart, and badge in the 6-Step Alpha Auditor UI and PDF report to the exact production Python function and notebook cell that verifies it.

| UI Step & Component | Displayed Outputs | Production Function | Notebook Verification Cell |
| :--- | :--- | :--- | :--- |
| **Step 1 & 2: Intake & Spec Builder** | `StrategySpec`, User vs. Software Defaults, Ambiguities | `engine.hypothesis_parser.interpret_hypothesis` | Cell 2 |
| **Step 3: Literature & Rigor Checklist** | Curated Papers, Causal & Split Safeguards | `engine.literature.build_literature_review` | Cell 3 & Cell 4 |
| **Step 4: KPI Strip & Split Table** | OOS / IS / Full Net Return, Excess pp, Sharpe, PSR, MDD, N | `engine.backtester._compute_period_metrics` | Cell 6 & Cell 7 |
| **Step 4: Equity, Drawdown & Signal Charts** | `equity_net`, `equity_benchmark`, `drawdown_net_pct`, `signal_values` | `engine.backtester.simulate_strategy_core` & `compute_causal_signals` | Cell 4, Cell 5, Cell 6, Cell 11 |
| **Step 4: Econometrics & 25-Cell DSR Grid** | Newey-West HAC $\alpha/\beta$, Stationary Bootstrap CI, Pairs ADF/OU, 5×5 DSR | `engine.econometrics.*` & `compute_parameter_sensitivity_surface` | Cell 8 & Cell 9 |
| **Step 5: Adversarial Stress Suite** | Outlier Removal, $t+2$ Latency, Regimes, Non-Overlapping MC Null, Drift | `engine.stress_tests.run_adversarial_stress_suite` | Cell 10 |
| **Step 6: Verdict, Multi-Question Breakdown & Blotter** | Co-Movement vs. Cointegration vs. Rule P&L, 5 Dimensions, Formula Ledger, Trades | `engine.stress_tests.build_hypothesis_scorecard_and_ledger` | Cell 5, Cell 12, Cell 13 |